In [2]:
# ============================================================
# MODULE N1 — CLEAN BRFSS-2015 EXPERIMENTAL FOUNDATION
#
# PROJECT:
#   Diverse Reliability-Guided Specialist MoE
#
# PURPOSE
# ------------------------------------------------------------
# Build a clean and reproducible development environment for:
#
#   - broad expert benchmarking
#   - specialist expert generation
#   - diversity experiments
#   - ensemble subset selection
#   - local competence modeling
#   - residual MoE
#   - sparse routing
#   - reliability-guided MoE
#
# IMPORTANT
# ------------------------------------------------------------
# BRFSS-2020 MUST NOT be loaded in this notebook.
#
# All architecture/model selection is performed using
# BRFSS-2015 DEV only.
#
# HOLDOUT is created and frozen but MUST NOT be used during
# architecture development.
# ============================================================


# ============================================================
# N1.1 IMPORTS
# ============================================================

import os
import json
import random
import warnings
import platform
import time

from pathlib import Path

import numpy as np
import pandas as pd

import sklearn

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ============================================================
# N1.2 GLOBAL REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ============================================================
# N1.3 PATHS
# ============================================================

DATA_PATH = Path(
    "/kaggle/input/datasets/"
    "alexteboul/"
    "heart-disease-health-indicators-dataset/"
    "heart_disease_health_indicators_BRFSS2015.csv"
)


WORK_DIR = Path(
    "/kaggle/working/"
    "diverse_specialist_moe"
)


PRED_DIR = WORK_DIR / "predictions"
MODEL_DIR = WORK_DIR / "models"
TAB_DIR = WORK_DIR / "tables"
LOG_DIR = WORK_DIR / "logs"
FIG_DIR = WORK_DIR / "figures"


for directory in [
    WORK_DIR,
    PRED_DIR,
    MODEL_DIR,
    TAB_DIR,
    LOG_DIR,
    FIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "Working directory:",
    WORK_DIR
)


# ============================================================
# N1.4 LOAD BRFSS-2015
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}"
    )


df = pd.read_csv(
    DATA_PATH
)


print(
    "\nRaw dataset shape:",
    df.shape
)


print(
    "\nColumns:"
)

print(
    list(df.columns)
)


# ============================================================
# N1.5 TARGET + FEATURE VALIDATION
# ============================================================

TARGET = "HeartDiseaseorAttack"


if TARGET not in df.columns:
    raise RuntimeError(
        f"Target '{TARGET}' not found."
    )


EXPECTED_FEATURES = [

    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


missing_features = [
    col
    for col in EXPECTED_FEATURES
    if col not in df.columns
]


if missing_features:
    raise RuntimeError(
        "Missing expected features: "
        + ", ".join(missing_features)
    )


FEATURES = EXPECTED_FEATURES.copy()


assert len(FEATURES) == 21


# ============================================================
# N1.6 ADD IMMUTABLE ROW ID
# ============================================================

df = df.copy()


df.insert(
    0,
    "ROW_ID",
    np.arange(
        len(df),
        dtype=np.int64
    )
)


# ============================================================
# N1.7 BASIC DATA AUDIT
# ============================================================

y_full = (
    df[
        TARGET
    ]
    .astype(int)
    .to_numpy()
)


X_full = (
    df[
        FEATURES
    ]
    .copy()
)


n_total = len(
    df
)


n_positive = int(
    y_full.sum()
)


n_negative = int(
    n_total
    -
    n_positive
)


prevalence = float(
    np.mean(
        y_full
    )
)


missing_total = int(
    X_full
    .isna()
    .sum()
    .sum()
)


duplicate_profiles = int(

    df[
        FEATURES
        +
        [
            TARGET
        ]
    ]
    .duplicated()
    .sum()
)


DATA_AUDIT_N1 = pd.DataFrame([

    {
        "N":
            n_total,

        "N_Features":
            len(
                FEATURES
            ),

        "Positive":
            n_positive,

        "Negative":
            n_negative,

        "Prevalence":
            prevalence,

        "Missing_Values":
            missing_total,

        "Duplicate_Profiles":
            duplicate_profiles,
    }
])


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE N1 — DATA AUDIT"
)

print(
    "=" * 120
)


display(
    DATA_AUDIT_N1
)


# ============================================================
# N1.8 FEATURE SCHEMA
# ============================================================

CATEGORICAL_FEATURES = [

    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "Diabetes",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income",
]


CONTINUOUS_FEATURES = [

    "BMI",
    "MentHlth",
    "PhysHlth",
]


assert set(
    CATEGORICAL_FEATURES
    +
    CONTINUOUS_FEATURES
) == set(
    FEATURES
)


FEATURE_GROUPS = {

    "demographic": [
        "Sex",
        "Age",
        "Education",
        "Income",
    ],

    "general_health": [
        "GenHlth",
        "MentHlth",
        "PhysHlth",
        "DiffWalk",
    ],

    "cardiometabolic": [
        "HighBP",
        "HighChol",
        "CholCheck",
        "BMI",
        "Diabetes",
    ],

    "cardiovascular": [
        "Stroke",
    ],

    "lifestyle": [
        "Smoker",
        "PhysActivity",
        "HvyAlcoholConsump",
    ],

    "nutrition": [
        "Fruits",
        "Veggies",
    ],

    "healthcare_access": [
        "AnyHealthcare",
        "NoDocbcCost",
    ],
}


schema_rows = []


for feature in FEATURES:

    feature_type = (

        "categorical"

        if feature
        in CATEGORICAL_FEATURES

        else
        "continuous"
    )


    groups = [

        group_name

        for group_name,
        columns
        in FEATURE_GROUPS.items()

        if feature
        in columns
    ]


    schema_rows.append({

        "Feature":
            feature,

        "Type":
            feature_type,

        "Group":
            (
                groups[0]
                if groups
                else "other"
            ),

        "Unique_Values":
            int(
                df[
                    feature
                ]
                .nunique()
            ),

        "Minimum":
            float(
                df[
                    feature
                ].min()
            ),

        "Maximum":
            float(
                df[
                    feature
                ].max()
            ),
    })


FEATURE_SCHEMA_N1 = pd.DataFrame(
    schema_rows
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE N1 — FEATURE SCHEMA"
)

print(
    "=" * 120
)


display(
    FEATURE_SCHEMA_N1
)


# ============================================================
# N1.9 FROZEN 80/20 DEV-HOLDOUT SPLIT
#
# IMPORTANT:
# Architecture development is allowed ONLY on DEV.
#
# HOLDOUT must remain untouched until the new architecture
# has been frozen.
# ============================================================

row_ids = df[
    "ROW_ID"
].to_numpy()


(
    DEV_IDS,
    HOLDOUT_IDS,

) = train_test_split(

    row_ids,

    test_size=0.20,

    random_state=SEED,

    stratify=y_full,
)


DEV_IDS = np.sort(
    DEV_IDS
)


HOLDOUT_IDS = np.sort(
    HOLDOUT_IDS
)


DEV_MASK = np.isin(
    row_ids,
    DEV_IDS
)


HOLDOUT_MASK = np.isin(
    row_ids,
    HOLDOUT_IDS
)


X_DEV = (

    df.loc[
        DEV_MASK,
        FEATURES
    ]
    .reset_index(
        drop=True
    )
)


y_DEV = (

    df.loc[
        DEV_MASK,
        TARGET
    ]
    .astype(int)
    .to_numpy()
)


ROW_DEV = (

    df.loc[
        DEV_MASK,
        "ROW_ID"
    ]
    .to_numpy()
)


X_HOLDOUT = (

    df.loc[
        HOLDOUT_MASK,
        FEATURES
    ]
    .reset_index(
        drop=True
    )
)


y_HOLDOUT = (

    df.loc[
        HOLDOUT_MASK,
        TARGET
    ]
    .astype(int)
    .to_numpy()
)


ROW_HOLDOUT = (

    df.loc[
        HOLDOUT_MASK,
        "ROW_ID"
    ]
    .to_numpy()
)


# ============================================================
# N1.10 SPLIT VERIFICATION
# ============================================================

SPLIT_SUMMARY_N1 = pd.DataFrame([

    {
        "Split":
            "FULL",

        "N":
            len(
                y_full
            ),

        "Positive":
            int(
                y_full.sum()
            ),

        "Negative":
            int(
                len(y_full)
                -
                y_full.sum()
            ),

        "Prevalence":
            float(
                y_full.mean()
            ),
    },

    {
        "Split":
            "DEV",

        "N":
            len(
                y_DEV
            ),

        "Positive":
            int(
                y_DEV.sum()
            ),

        "Negative":
            int(
                len(y_DEV)
                -
                y_DEV.sum()
            ),

        "Prevalence":
            float(
                y_DEV.mean()
            ),
    },

    {
        "Split":
            "HOLDOUT",

        "N":
            len(
                y_HOLDOUT
            ),

        "Positive":
            int(
                y_HOLDOUT.sum()
            ),

        "Negative":
            int(
                len(y_HOLDOUT)
                -
                y_HOLDOUT.sum()
            ),

        "Prevalence":
            float(
                y_HOLDOUT.mean()
            ),
    },
])


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE N1 — FROZEN SPLIT"
)

print(
    "=" * 120
)


display(
    SPLIT_SUMMARY_N1
)


# Expected approximately:
#
# FULL:
#   N = 253680
#   positive = 23893
#
# DEV:
#   N = 202944
#   positive ≈ 19114
#
# HOLDOUT:
#   N = 50736
#   positive ≈ 4779


# ============================================================
# N1.11 FIXED 5-FOLD DEV REGISTRY
# ============================================================

N_FOLDS = 5


SKF = StratifiedKFold(

    n_splits=N_FOLDS,

    shuffle=True,

    random_state=SEED,
)


DEV_FOLD = np.full(

    len(
        X_DEV
    ),

    -1,

    dtype=np.int16
)


fold_rows = []


for fold, (
    train_idx,
    valid_idx

) in enumerate(

    SKF.split(
        X_DEV,
        y_DEV
    )
):

    DEV_FOLD[
        valid_idx
    ] = fold


    fold_rows.append({

        "Fold":
            fold,

        "Train_N":
            len(
                train_idx
            ),

        "Valid_N":
            len(
                valid_idx
            ),

        "Train_Positive":
            int(
                y_DEV[
                    train_idx
                ].sum()
            ),

        "Valid_Positive":
            int(
                y_DEV[
                    valid_idx
                ].sum()
            ),

        "Train_Prevalence":
            float(
                y_DEV[
                    train_idx
                ].mean()
            ),

        "Valid_Prevalence":
            float(
                y_DEV[
                    valid_idx
                ].mean()
            ),
    })


assert np.all(
    DEV_FOLD >= 0
)


FOLD_SUMMARY_N1 = pd.DataFrame(
    fold_rows
)


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE N1 — FIXED DEV FOLDS"
)

print(
    "=" * 120
)


display(
    FOLD_SUMMARY_N1
)


# ============================================================
# N1.12 FOLD REGISTRY
# ============================================================

DEV_REGISTRY_N1 = pd.DataFrame({

    "ROW_ID":
        ROW_DEV,

    "TARGET":
        y_DEV,

    "FOLD":
        DEV_FOLD,
})


HOLDOUT_REGISTRY_N1 = pd.DataFrame({

    "ROW_ID":
        ROW_HOLDOUT,

    "TARGET":
        y_HOLDOUT,
})


# ============================================================
# N1.13 CORE METRIC FUNCTIONS
# ============================================================

def ece_score(
    y_true,
    probability,
    n_bins=15,
):

    y_true = np.asarray(
        y_true
    )


    probability = np.asarray(
        probability
    )


    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )


    bin_ids = np.digitize(

        probability,

        edges[
            1:-1
        ],
    )


    ece = 0.0


    for b in range(
        n_bins
    ):

        mask = (
            bin_ids
            ==
            b
        )


        if not mask.any():
            continue


        observed = y_true[
            mask
        ].mean()


        predicted = probability[
            mask
        ].mean()


        ece += (

            mask.mean()

            *

            abs(
                observed
                -
                predicted
            )
        )


    return float(
        ece
    )


def probability_metrics(
    y_true,
    probability,
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    probability = np.asarray(
        probability,
        dtype=float
    )


    probability = np.clip(

        probability,

        1e-7,

        1 - 1e-7
    )


    return {

        "PR_AUC":
            average_precision_score(
                y_true,
                probability
            ),

        "ROC_AUC":
            roc_auc_score(
                y_true,
                probability
            ),

        "Brier":
            brier_score_loss(
                y_true,
                probability
            ),

        "NLL":
            log_loss(
                y_true,
                probability
            ),

        "ECE":
            ece_score(
                y_true,
                probability
            ),
    }


# ============================================================
# N1.14 ERROR / OPERATING-POINT METRICS
# ============================================================

def threshold_metrics(
    y_true,
    probability,
    threshold,
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    pred = (

        np.asarray(
            probability
        )
        >=
        threshold

    ).astype(int)


    tn, fp, fn, tp = (

        confusion_matrix(
            y_true,
            pred,
            labels=[
                0,
                1
            ]
        )
        .ravel()
    )


    specificity = (

        tn
        /
        (
            tn + fp
        )

        if (
            tn + fp
        ) > 0

        else np.nan
    )


    return {

        "Threshold":
            threshold,

        "TP":
            int(tp),

        "FP":
            int(fp),

        "TN":
            int(tn),

        "FN":
            int(fn),

        "Precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "Specificity":
            specificity,

        "F1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "MCC":
            matthews_corrcoef(
                y_true,
                pred
            ),
    }


# ============================================================
# N1.15 EXPERIMENTAL MODEL REGISTRY
#
# Every model we train later gets one row here.
# ============================================================

EXPERIMENT_REGISTRY = pd.DataFrame(

    columns=[

        "Model_ID",
        "Family",
        "Role",
        "Specialization",

        "Feature_View",
        "Training_Objective",

        "Robustness_Augmentation",
        "Diversity_Method",

        "OOF_Available",

        "PR_AUC",
        "ROC_AUC",
        "Brier",
        "NLL",
        "ECE",

        "Status",
        "Notes",
    ]
)


# ============================================================
# N1.16 HELPER TO REGISTER EXPERIMENT
# ============================================================

def register_experiment(
    model_id,
    family,
    role,
    specialization,
    feature_view,
    training_objective,
    robustness_augmentation="None",
    diversity_method="None",
    oof_available=False,
    metrics=None,
    status="candidate",
    notes="",
):

    global EXPERIMENT_REGISTRY


    if metrics is None:

        metrics = {}


    row = {

        "Model_ID":
            model_id,

        "Family":
            family,

        "Role":
            role,

        "Specialization":
            specialization,

        "Feature_View":
            feature_view,

        "Training_Objective":
            training_objective,

        "Robustness_Augmentation":
            robustness_augmentation,

        "Diversity_Method":
            diversity_method,

        "OOF_Available":
            oof_available,

        "PR_AUC":
            metrics.get(
                "PR_AUC",
                np.nan
            ),

        "ROC_AUC":
            metrics.get(
                "ROC_AUC",
                np.nan
            ),

        "Brier":
            metrics.get(
                "Brier",
                np.nan
            ),

        "NLL":
            metrics.get(
                "NLL",
                np.nan
            ),

        "ECE":
            metrics.get(
                "ECE",
                np.nan
            ),

        "Status":
            status,

        "Notes":
            notes,
    }


    # Replace existing entry if model_id exists
    EXPERIMENT_REGISTRY = (

        EXPERIMENT_REGISTRY[
            EXPERIMENT_REGISTRY[
                "Model_ID"
            ]
            !=
            model_id
        ]
    )


    EXPERIMENT_REGISTRY = pd.concat(

        [
            EXPERIMENT_REGISTRY,

            pd.DataFrame(
                [
                    row
                ]
            ),
        ],

        ignore_index=True
    )


# ============================================================
# N1.17 OOF PREDICTION STORE
#
# Every candidate must place strict OOF predictions here.
# ============================================================

OOF_STORE = {}


def store_oof(
    model_id,
    predictions,
):

    predictions = np.asarray(
        predictions,
        dtype=float
    )


    if predictions.shape != (
        len(
            y_DEV
        ),
    ):

        raise ValueError(

            f"{model_id}: OOF shape "
            f"{predictions.shape} != "
            f"{(len(y_DEV),)}"
        )


    if not np.all(
        np.isfinite(
            predictions
        )
    ):

        raise ValueError(
            f"{model_id}: non-finite predictions."
        )


    OOF_STORE[
        model_id
    ] = predictions


# ============================================================
# N1.18 EXPERIMENT DESIGN CONFIGURATION
# ============================================================

EXPERIMENT_CONFIG = {

    "seed":
        SEED,

    "n_folds":
        N_FOLDS,

    "primary_metric":
        "PR_AUC",

    "secondary_metrics": [

        "ROC_AUC",
        "Brier",
        "NLL",
        "ECE",
    ],

    "selection_principle":
        (
            "Individual competence + "
            "error complementarity + "
            "ensemble contribution"
        ),

    "holdout_policy":
        (
            "Do not inspect until final architecture "
            "is frozen."
        ),

    "external_policy":
        (
            "BRFSS-2020 excluded entirely from "
            "development notebook."
        ),

    "candidate_experiment_families": [

        "global_classical",
        "global_neural",
        "class_sensitive",
        "hard_case_specialist",
        "robustness_specialist",
        "feature_view_specialist",
        "retrieval_expert",
        "diversity_regularized",
        "local_competence",
        "residual_moe",
        "sparse_moe",
        "reliability_guided_moe",
    ],
}


# ============================================================
# N1.19 BASELINE TARGETS FROM PREVIOUS NOTEBOOK
#
# These are comparison references only.
# They are NOT imported predictions.
# ============================================================

PREVIOUS_BENCHMARKS = pd.DataFrame([

    {
        "Model":
            "HGB",

        "PR_AUC":
            0.377012,

        "ROC_AUC":
            0.849628,
    },

    {
        "Model":
            "Repaired Core4",

        "PR_AUC":
            0.379284,

        "ROC_AUC":
            0.850402,
    },

    {
        "Model":
            "LSA-HE Core5",

        "PR_AUC":
            0.379751,

        "ROC_AUC":
            0.850608,
    },
])


print(
    "\n"
    + "=" * 120
)

print(
    "PREVIOUS BENCHMARKS TO RECONSTRUCT"
)

print(
    "=" * 120
)


display(
    PREVIOUS_BENCHMARKS
)


# ============================================================
# N1.20 SUCCESS CRITERIA FOR NEW MODEL
# ============================================================

SUCCESS_CRITERIA_N1 = pd.DataFrame([

    {
        "Dimension":
            "Predictive discrimination",

        "Desired_Result":
            (
                "PR-AUC > 0.379751 "
                "under strict DEV OOF"
            ),
    },

    {
        "Dimension":
            "Expert diversity",

        "Desired_Result":
            (
                "Reduced redundant error overlap "
                "without sacrificing competence"
            ),
    },

    {
        "Dimension":
            "Routing",

        "Desired_Result":
            (
                "Meaningful patient-specific specialization; "
                "avoid near-uniform gate collapse"
            ),
    },

    {
        "Dimension":
            "False-negative behavior",

        "Desired_Result":
            (
                "Improved FN-sensitive performance or "
                "review prioritization"
            ),
    },

    {
        "Dimension":
            "Reliability",

        "Desired_Result":
            (
                "Maintain/improve calibration and "
                "error-risk estimation"
            ),
    },

    {
        "Dimension":
            "Robustness",

        "Desired_Result":
            (
                "Improve degraded-input performance "
                "without harming clean performance materially"
            ),
    },

])


print(
    "\n"
    + "=" * 120
)

print(
    "NEW ARCHITECTURE SUCCESS CRITERIA"
)

print(
    "=" * 120
)


display(
    SUCCESS_CRITERIA_N1
)


# ============================================================
# N1.21 SAVE CORE REGISTRIES
# ============================================================

DATA_AUDIT_N1.to_csv(

    TAB_DIR
    /
    "n1_data_audit.csv",

    index=False
)


FEATURE_SCHEMA_N1.to_csv(

    TAB_DIR
    /
    "n1_feature_schema.csv",

    index=False
)


SPLIT_SUMMARY_N1.to_csv(

    TAB_DIR
    /
    "n1_split_summary.csv",

    index=False
)


FOLD_SUMMARY_N1.to_csv(

    TAB_DIR
    /
    "n1_fold_summary.csv",

    index=False
)


DEV_REGISTRY_N1.to_csv(

    WORK_DIR
    /
    "dev_fold_registry.csv",

    index=False
)


HOLDOUT_REGISTRY_N1.to_csv(

    WORK_DIR
    /
    "holdout_registry.csv",

    index=False
)


np.save(

    WORK_DIR
    /
    "dev_fold_assignment.npy",

    DEV_FOLD
)


PREVIOUS_BENCHMARKS.to_csv(

    TAB_DIR
    /
    "previous_benchmarks.csv",

    index=False
)


with open(

    LOG_DIR
    /
    "experiment_config.json",

    "w"

) as f:

    json.dump(

        EXPERIMENT_CONFIG,

        f,

        indent=2
    )


# ============================================================
# N1.22 ENVIRONMENT INFO
# ============================================================

ENVIRONMENT_N1 = {

    "python":
        platform.python_version(),

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "sklearn":
        sklearn.__version__,

    "seed":
        SEED,
}


with open(

    LOG_DIR
    /
    "environment.json",

    "w"

) as f:

    json.dump(

        ENVIRONMENT_N1,

        f,

        indent=2
    )


print(
    "\n"
    + "=" * 120
)

print(
    "MODULE N1 COMPLETE"
)

print(
    "=" * 120
)


print(
    "\nDevelopment rows:",
    len(
        X_DEV
    )
)


print(
    "Frozen holdout rows:",
    len(
        X_HOLDOUT
    )
)


print(
    "Number of fixed folds:",
    N_FOLDS
)


print(
    "\nBRFSS-2020 has NOT been loaded."
)


print(
    "\nNext:"
)

print(
    "MODULE N2 — Broad global expert benchmark pool"
)

Working directory: /kaggle/working/diverse_specialist_moe

Raw dataset shape: (253680, 22)

Columns:
['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

MODULE N1 — DATA AUDIT


,N,N_Features,Positive,Negative,Prevalence,Missing_Values,Duplicate_Profiles
0,253680,21,23893,229787,0.094186,0,23899



MODULE N1 — FEATURE SCHEMA


,Feature,Type,Group,Unique_Values,Minimum,Maximum
0,HighBP,categorical,cardiometabolic,2,0.0,1.0
1,HighChol,categorical,cardiometabolic,2,0.0,1.0
2,CholCheck,categorical,cardiometabolic,2,0.0,1.0
3,BMI,continuous,cardiometabolic,84,12.0,98.0
4,Smoker,categorical,lifestyle,2,0.0,1.0
5,Stroke,categorical,cardiovascular,2,0.0,1.0
6,Diabetes,categorical,cardiometabolic,3,0.0,2.0
7,PhysActivity,categorical,lifestyle,2,0.0,1.0
8,Fruits,categorical,nutrition,2,0.0,1.0
9,Veggies,categorical,nutrition,2,0.0,1.0



MODULE N1 — FROZEN SPLIT


...


PREVIOUS BENCHMARKS TO RECONSTRUCT


,Model,PR_AUC,ROC_AUC
0,HGB,0.377012,0.849628
1,Repaired Core4,0.379284,0.850402
2,LSA-HE Core5,0.379751,0.850608



NEW ARCHITECTURE SUCCESS CRITERIA


,Dimension,Desired_Result
0,Predictive discrimination,PR-AUC > 0.379751 under strict DEV OOF
1,Expert diversity,Reduced redundant error overlap without sacrif...
2,Routing,Meaningful patient-specific specialization; av...
3,False-negative behavior,Improved FN-sensitive performance or review pr...
4,Reliability,Maintain/improve calibration and error-risk es...
5,Robustness,Improve degraded-input performance without har...



MODULE N1 COMPLETE

Development rows: 202944
Frozen holdout rows: 50736
Number of fixed folds: 5

BRFSS-2020 has NOT been loaded.

Next:
MODULE N2 — Broad global expert benchmark pool


In [3]:
# ============================================================
# MODULE N2A — BROAD GLOBAL CLASSICAL EXPERT POOL
#
# PURPOSE
# ------------------------------------------------------------
# Establish a clean strict-OOF pool of global classical/tabular
# experts using EXACTLY the fixed DEV folds from Module N1.
#
# Models:
#   - Logistic Regression L2
#   - Logistic Regression Elastic-Net
#   - Gaussian Naive Bayes
#   - Random Forest
#   - Extra Trees
#   - HistGradientBoosting
#   - LightGBM
#   - XGBoost
#   - CatBoost
#
# IMPORTANT
# ------------------------------------------------------------
# - DEV only
# - HOLDOUT untouched
# - No BRFSS-2020
# - No model selection based on HOLDOUT
# - No specialist training yet
# ============================================================


# ============================================================
# N2A.1 IMPORTS
# ============================================================

import gc
import time
import copy
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)

warnings.filterwarnings("ignore")


# Optional external libraries
HAVE_LGBM = False
HAVE_XGB = False
HAVE_CAT = False


try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
    print("LightGBM available.")
except Exception as e:
    print("LightGBM unavailable:", e)


try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
    print("XGBoost available.")
except Exception as e:
    print("XGBoost unavailable:", e)


try:
    from catboost import CatBoostClassifier
    HAVE_CAT = True
    print("CatBoost available.")
except Exception as e:
    print("CatBoost unavailable:", e)


# ============================================================
# N2A.2 BASIC ARRAYS
# ============================================================

X_N2A = X_DEV.copy()
y_N2A = np.asarray(
    y_DEV,
    dtype=int
)

FOLD_N2A = np.asarray(
    DEV_FOLD,
    dtype=int
)


assert len(X_N2A) == len(y_N2A)
assert len(y_N2A) == len(FOLD_N2A)
assert set(np.unique(FOLD_N2A)) == set(range(N_FOLDS))


# ============================================================
# N2A.3 MODEL FACTORY
#
# IMPORTANT:
# These are GLOBAL experts.
# Do not intentionally specialize them yet.
# ============================================================

MODEL_FACTORIES_N2A = {}


# ------------------------------------------------------------
# Logistic Regression L2
# ------------------------------------------------------------

MODEL_FACTORIES_N2A[
    "LR_L2"
] = lambda: Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=3000,
            random_state=SEED,
        )
    ),
])


# ------------------------------------------------------------
# Logistic Regression Elastic-Net
# ------------------------------------------------------------

MODEL_FACTORIES_N2A[
    "LR_EN"
] = lambda: Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            penalty="elasticnet",
            C=1.0,
            l1_ratio=0.20,
            solver="saga",
            max_iter=4000,
            random_state=SEED,
            n_jobs=-1,
        )
    ),
])


# ------------------------------------------------------------
# Gaussian Naive Bayes
# ------------------------------------------------------------

MODEL_FACTORIES_N2A[
    "GNB"
] = lambda: GaussianNB()


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

MODEL_FACTORIES_N2A[
    "RF_GLOBAL"
] = lambda: RandomForestClassifier(

    n_estimators=500,

    max_depth=None,

    min_samples_leaf=5,

    max_features="sqrt",

    bootstrap=True,

    n_jobs=-1,

    random_state=SEED,
)


# ------------------------------------------------------------
# Extra Trees
# ------------------------------------------------------------

MODEL_FACTORIES_N2A[
    "ET_GLOBAL"
] = lambda: ExtraTreesClassifier(

    n_estimators=500,

    max_depth=None,

    min_samples_leaf=5,

    max_features="sqrt",

    bootstrap=False,

    n_jobs=-1,

    random_state=SEED,
)


# ------------------------------------------------------------
# HistGradientBoosting
#
# Close to our previous strong HGB family.
# ------------------------------------------------------------

MODEL_FACTORIES_N2A[
    "HGB_GLOBAL"
] = lambda: HistGradientBoostingClassifier(

    learning_rate=0.06,

    max_iter=350,

    max_leaf_nodes=31,

    min_samples_leaf=30,

    l2_regularization=1.0,

    early_stopping=False,

    random_state=SEED,
)


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

if HAVE_LGBM:

    MODEL_FACTORIES_N2A[
        "LGBM_GLOBAL"
    ] = lambda: LGBMClassifier(

        objective="binary",

        n_estimators=700,

        learning_rate=0.03,

        num_leaves=31,

        max_depth=-1,

        min_child_samples=40,

        subsample=0.90,

        colsample_bytree=0.85,

        reg_alpha=0.5,

        reg_lambda=2.0,

        random_state=SEED,

        n_jobs=-1,

        verbosity=-1,
    )


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

if HAVE_XGB:

    MODEL_FACTORIES_N2A[
        "XGB_GLOBAL"
    ] = lambda: XGBClassifier(

        objective="binary:logistic",

        eval_metric="aucpr",

        n_estimators=750,

        learning_rate=0.03,

        max_depth=4,

        min_child_weight=10,

        subsample=0.90,

        colsample_bytree=0.80,

        gamma=0.05,

        reg_alpha=1.0,

        reg_lambda=5.0,

        tree_method="hist",

        random_state=SEED,

        n_jobs=-1,
    )


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

if HAVE_CAT:

    MODEL_FACTORIES_N2A[
        "CAT_GLOBAL"
    ] = lambda: CatBoostClassifier(

        iterations=700,

        learning_rate=0.04,

        depth=6,

        loss_function="Logloss",

        eval_metric="PRAUC",

        l2_leaf_reg=5.0,

        random_seed=SEED,

        verbose=False,

        allow_writing_files=False,

        thread_count=-1,
    )


print(
    "\nCandidate global models:"
)

for model_id in MODEL_FACTORIES_N2A:
    print(" -", model_id)


# ============================================================
# N2A.4 MODEL METADATA
# ============================================================

MODEL_META_N2A = {

    "LR_L2": {
        "family": "LogisticRegression",
        "objective": "Log-loss / L2",
    },

    "LR_EN": {
        "family": "LogisticRegression",
        "objective": "Log-loss / Elastic-Net",
    },

    "GNB": {
        "family": "GaussianNB",
        "objective": "Generative likelihood",
    },

    "RF_GLOBAL": {
        "family": "RandomForest",
        "objective": "Global classification",
    },

    "ET_GLOBAL": {
        "family": "ExtraTrees",
        "objective": "Global classification",
    },

    "HGB_GLOBAL": {
        "family": "HistGradientBoosting",
        "objective": "Global log-loss",
    },

    "LGBM_GLOBAL": {
        "family": "LightGBM",
        "objective": "Global binary classification",
    },

    "XGB_GLOBAL": {
        "family": "XGBoost",
        "objective": "Global binary classification",
    },

    "CAT_GLOBAL": {
        "family": "CatBoost",
        "objective": "Global log-loss",
    },
}


# ============================================================
# N2A.5 GENERIC POSITIVE-PROBABILITY HELPER
# ============================================================

def positive_probability_n2a(
    model,
    X
):

    if hasattr(
        model,
        "predict_proba"
    ):

        p = model.predict_proba(
            X
        )

        if p.ndim == 2:
            return p[:, 1]

        return p


    if hasattr(
        model,
        "decision_function"
    ):

        z = model.decision_function(
            X
        )

        return (
            1.0
            /
            (
                1.0
                +
                np.exp(
                    -z
                )
            )
        )


    raise RuntimeError(
        f"No probability interface for {type(model)}"
    )


# ============================================================
# N2A.6 STRICT OOF TRAINING FUNCTION
# ============================================================

def run_oof_model_n2a(
    model_id,
    factory,
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        "TRAINING:",
        model_id
    )

    print(
        "=" * 110
    )


    oof = np.full(
        len(y_N2A),
        np.nan,
        dtype=float
    )


    fold_rows = []

    fitted_models = []

    total_start = time.perf_counter()


    for fold in range(
        N_FOLDS
    ):

        train_idx = np.where(
            FOLD_N2A != fold
        )[0]


        valid_idx = np.where(
            FOLD_N2A == fold
        )[0]


        X_train = X_N2A.iloc[
            train_idx
        ]


        y_train = y_N2A[
            train_idx
        ]


        X_valid = X_N2A.iloc[
            valid_idx
        ]


        y_valid = y_N2A[
            valid_idx
        ]


        model = factory()


        fold_start = time.perf_counter()


        model.fit(
            X_train,
            y_train
        )


        pred = positive_probability_n2a(
            model,
            X_valid
        )


        pred = np.clip(
            np.asarray(
                pred,
                dtype=float
            ),
            1e-7,
            1 - 1e-7
        )


        oof[
            valid_idx
        ] = pred


        fold_time = (
            time.perf_counter()
            -
            fold_start
        )


        m = probability_metrics(
            y_valid,
            pred
        )


        fold_rows.append({

            "Model_ID":
                model_id,

            "Fold":
                fold,

            "Train_N":
                len(
                    train_idx
                ),

            "Valid_N":
                len(
                    valid_idx
                ),

            "PR_AUC":
                m[
                    "PR_AUC"
                ],

            "ROC_AUC":
                m[
                    "ROC_AUC"
                ],

            "Brier":
                m[
                    "Brier"
                ],

            "NLL":
                m[
                    "NLL"
                ],

            "ECE":
                m[
                    "ECE"
                ],

            "Fit_Predict_Seconds":
                fold_time,
        })


        fitted_models.append(
            model
        )


        print(
            f"Fold {fold}: "
            f"PR={m['PR_AUC']:.6f} | "
            f"ROC={m['ROC_AUC']:.6f} | "
            f"Brier={m['Brier']:.6f} | "
            f"{fold_time:.1f}s"
        )


        gc.collect()


    if np.isnan(
        oof
    ).any():

        raise RuntimeError(
            f"{model_id}: incomplete OOF predictions."
        )


    total_time = (
        time.perf_counter()
        -
        total_start
    )


    pooled = probability_metrics(
        y_N2A,
        oof
    )


    print(
        "\nPOOLED OOF"
    )

    print(
        f"PR-AUC  : {pooled['PR_AUC']:.6f}"
    )

    print(
        f"ROC-AUC : {pooled['ROC_AUC']:.6f}"
    )

    print(
        f"Brier   : {pooled['Brier']:.6f}"
    )

    print(
        f"NLL     : {pooled['NLL']:.6f}"
    )

    print(
        f"ECE     : {pooled['ECE']:.6f}"
    )

    print(
        f"Time    : {total_time:.1f}s"
    )


    return {

        "model_id":
            model_id,

        "oof":
            oof,

        "fold_results":
            pd.DataFrame(
                fold_rows
            ),

        "pooled_metrics":
            pooled,

        "total_seconds":
            total_time,

        "fold_models":
            fitted_models,
    }


# ============================================================
# N2A.7 RUN ALL GLOBAL CANDIDATES
# ============================================================

RESULTS_N2A = {}

FOLD_RESULTS_N2A_LIST = []


for model_id, factory in (
    MODEL_FACTORIES_N2A.items()
):

    result = run_oof_model_n2a(
        model_id,
        factory
    )


    RESULTS_N2A[
        model_id
    ] = result


    OOF_STORE[
        model_id
    ] = result[
        "oof"
    ]


    FOLD_RESULTS_N2A_LIST.append(
        result[
            "fold_results"
        ]
    )


    meta = MODEL_META_N2A.get(
        model_id,
        {}
    )


    register_experiment(

        model_id=
            model_id,

        family=
            meta.get(
                "family",
                model_id
            ),

        role=
            "global",

        specialization=
            "none",

        feature_view=
            "all_21",

        training_objective=
            meta.get(
                "objective",
                "global classification"
            ),

        robustness_augmentation=
            "None",

        diversity_method=
            "Model-family diversity",

        oof_available=
            True,

        metrics=
            result[
                "pooled_metrics"
            ],

        status=
            "candidate",

        notes=
            (
                f"Strict 5-fold OOF; "
                f"{result['total_seconds']:.1f}s"
            ),
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        result[
            "oof"
        ]
    )


# ============================================================
# N2A.8 POOLED LEADERBOARD
# ============================================================

leaderboard_rows = []


for model_id, result in (
    RESULTS_N2A.items()
):

    m = result[
        "pooled_metrics"
    ]


    leaderboard_rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Total_Seconds":
            result[
                "total_seconds"
            ],
    })


LEADERBOARD_N2A = (

    pd.DataFrame(
        leaderboard_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE N2A — GLOBAL EXPERT LEADERBOARD"
)

print(
    "=" * 130
)


display(
    LEADERBOARD_N2A
)


# ============================================================
# N2A.9 FOLD-WISE RESULTS
# ============================================================

FOLD_RESULTS_N2A = pd.concat(

    FOLD_RESULTS_N2A_LIST,

    ignore_index=True
)


# ============================================================
# N2A.10 FOLD STABILITY SUMMARY
# ============================================================

FOLD_STABILITY_N2A = (

    FOLD_RESULTS_N2A
    .groupby(
        "Model_ID"
    )
    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_SD=(
            "ROC_AUC",
            "std"
        ),

    )
    .reset_index()
    .sort_values(
        "PR_Mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE N2A — FOLD STABILITY"
)

print(
    "=" * 130
)


display(
    FOLD_STABILITY_N2A
)


# ============================================================
# N2A.11 SIMPLE EQUAL-WEIGHT GLOBAL POOL TEST
#
# This is NOT expert selection.
#
# It simply tells us whether the entire broad model pool
# contains useful aggregate signal.
# ============================================================

GLOBAL_MODEL_IDS_N2A = list(
    RESULTS_N2A.keys()
)


P_GLOBAL_N2A = np.column_stack([

    OOF_STORE[
        model_id
    ]

    for model_id
    in GLOBAL_MODEL_IDS_N2A
])


GLOBAL_EQUAL_N2A = np.mean(
    P_GLOBAL_N2A,
    axis=1
)


GLOBAL_EQUAL_METRICS_N2A = (
    probability_metrics(
        y_N2A,
        GLOBAL_EQUAL_N2A
    )
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE N2A — ALL-GLOBAL EQUAL ENSEMBLE"
)

print(
    "=" * 130
)


print(
    GLOBAL_EQUAL_METRICS_N2A
)


# ============================================================
# N2A.12 BASIC PREDICTION CORRELATION
#
# This is only an early diagnostic.
# Full diversity analysis comes later in N5.
# ============================================================

CORR_N2A = pd.DataFrame(

    np.corrcoef(
        P_GLOBAL_N2A.T
    ),

    index=
        GLOBAL_MODEL_IDS_N2A,

    columns=
        GLOBAL_MODEL_IDS_N2A,
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE N2A — OOF PREDICTION CORRELATION"
)

print(
    "=" * 130
)


display(
    CORR_N2A.round(4)
)


# ============================================================
# N2A.13 BEST SINGLE MODEL
# ============================================================

BEST_GLOBAL_N2A = (

    LEADERBOARD_N2A
    .iloc[0]
    .copy()
)


print(
    "\nBest global classical expert:"
)

display(
    pd.DataFrame(
        [
            BEST_GLOBAL_N2A
        ]
    )
)


# ============================================================
# N2A.14 COMPARE WITH PREVIOUS BENCHMARK
# ============================================================

previous_core5_pr = float(

    PREVIOUS_BENCHMARKS.loc[

        PREVIOUS_BENCHMARKS[
            "Model"
        ]
        ==
        "LSA-HE Core5",

        "PR_AUC"

    ].iloc[0]
)


LEADERBOARD_N2A[
    "Delta_PR_vs_Old_Core5"
] = (

    LEADERBOARD_N2A[
        "PR_AUC"
    ]

    -
    previous_core5_pr
)


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE N2A — COMPARISON WITH OLD CORE5"
)

print(
    "=" * 130
)


display(
    LEADERBOARD_N2A[
        [
            "Model_ID",
            "PR_AUC",
            "ROC_AUC",
            "Delta_PR_vs_Old_Core5",
        ]
    ]
)


# ============================================================
# N2A.15 SAVE
# ============================================================

LEADERBOARD_N2A.to_csv(

    TAB_DIR
    /
    "n2a_global_expert_leaderboard.csv",

    index=False
)


FOLD_RESULTS_N2A.to_csv(

    TAB_DIR
    /
    "n2a_fold_results.csv",

    index=False
)


FOLD_STABILITY_N2A.to_csv(

    TAB_DIR
    /
    "n2a_fold_stability.csv",

    index=False
)


CORR_N2A.to_csv(

    TAB_DIR
    /
    "n2a_prediction_correlation.csv"
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "GLOBAL_EQUAL_N2A_OOF.npy",

    GLOBAL_EQUAL_N2A
)


with open(

    LOG_DIR
    /
    "n2a_manifest.json",

    "w"

) as f:

    json.dump(

        {
            "module":
                "N2A broad global classical expert pool",

            "models":
                GLOBAL_MODEL_IDS_N2A,

            "development_only":
                True,

            "holdout_used":
                False,

            "external_used":
                False,

            "primary_metric":
                "PR_AUC",
        },

        f,

        indent=2
    )


print(
    "\n"
    + "=" * 130
)

print(
    "MODULE N2A COMPLETE"
)

print(
    "=" * 130
)


print(
    "\nGlobal candidates trained:",
    len(
        GLOBAL_MODEL_IDS_N2A
    )
)


print(
    "OOF predictions stored:",
    len(
        OOF_STORE
    )
)


print(
    "\nHOLDOUT remains untouched."
)

print(
    "BRFSS-2020 remains absent."
)

print(
    "\nNext: MODULE N2B — controlled boosting/objective diversity."
)

LightGBM available.
XGBoost available.
CatBoost available.

Candidate global models:
 - LR_L2
 - LR_EN
 - GNB
 - RF_GLOBAL
 - ET_GLOBAL
 - HGB_GLOBAL
 - LGBM_GLOBAL
 - XGB_GLOBAL
 - CAT_GLOBAL

TRAINING: LR_L2
Fold 0: PR=0.371198 | ROC=0.848014 | Brier=0.070439 | 0.4s
Fold 1: PR=0.373917 | ROC=0.847658 | Brier=0.070306 | 0.4s
Fold 2: PR=0.371628 | ROC=0.845805 | Brier=0.070455 | 0.4s
Fold 3: PR=0.365533 | ROC=0.846182 | Brier=0.070737 | 0.3s
Fold 4: PR=0.380905 | ROC=0.848527 | Brier=0.069929 | 0.3s

POOLED OOF
PR-AUC  : 0.372212
ROC-AUC : 0.847227
Brier   : 0.070373
NLL     : 0.238269
ECE     : 0.008316
Time    : 3.0s

TRAINING: LR_EN
Fold 0: PR=0.371197 | ROC=0.848018 | Brier=0.070438 | 1.5s
Fold 1: PR=0.373904 | ROC=0.847656 | Brier=0.070306 | 1.6s
Fold 2: PR=0.371607 | ROC=0.845798 | Brier=0.070455 | 1.6s
Fold 3: PR=0.365536 | ROC=0.846184 | Brier=0.070736 | 1.7s
Fold 4: PR=0.380898 | ROC=0.848520 | Brier=0.069929 | 1.4s

POOLED OOF
PR-AUC  : 0.372200
ROC-AUC : 0.847224
Brier   : 0

,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE,Total_Seconds
0,XGB_GLOBAL,0.377869,0.850049,0.069840,0.235948,0.002161,22.613610
1,CAT_GLOBAL,0.376270,0.850128,0.069877,0.235976,0.002203,121.420385
2,LGBM_GLOBAL,0.375276,0.849220,0.069968,0.236437,0.002661,29.031527
3,HGB_GLOBAL,0.374130,0.848636,0.070050,0.236760,0.002997,27.405773
4,LR_L2,0.372212,0.847227,0.070373,0.238269,0.008316,2.970267
5,LR_EN,0.372200,0.847224,0.070373,0.238269,0.008280,8.733973
6,RF_GLOBAL,0.369146,0.844818,0.070410,0.238939,0.002134,122.475164
7,ET_GLOBAL,0.368413,0.845146,0.070489,0.238809,0.004857,114.570151
8,GNB,0.300301,0.805674,0.155772,0.972163,0.153109,1.127253



MODULE N2A — FOLD STABILITY


,Model_ID,PR_Mean,PR_SD,PR_Min,PR_Max,ROC_Mean,ROC_SD
8,XGB_GLOBAL,0.378449,0.006125,0.371245,0.387740,0.850074,0.001609
0,CAT_GLOBAL,0.376837,0.006111,0.371093,0.385545,0.850137,0.001560
4,LGBM_GLOBAL,0.375718,0.005253,0.370078,0.383698,0.849246,0.001648
3,HGB_GLOBAL,0.374521,0.005075,0.369178,0.382721,0.848666,0.001443
6,LR_L2,0.372636,0.005558,0.365533,0.380905,0.847237,0.001184
5,LR_EN,0.372628,0.005554,0.365536,0.380898,0.847235,0.001184
7,RF_GLOBAL,0.369585,0.005260,0.365026,0.377735,0.844843,0.001730
1,ET_GLOBAL,0.368920,0.005878,0.363534,0.378333,0.845175,0.001695
2,GNB,0.300425,0.005216,0.294927,0.308783,0.805674,0.001894



MODULE N2A — ALL-GLOBAL EQUAL ENSEMBLE
{'PR_AUC': np.float64(0.3729517813266626), 'ROC_AUC': np.float64(0.8469098310215153), 'Brier': np.float64(0.07081194683539643), 'NLL': 0.23875729214934652, 'ECE': 0.012049234226267292}

MODULE N2A — OOF PREDICTION CORRELATION


...


Best global classical expert:


,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE,Total_Seconds
0,XGB_GLOBAL,0.377869,0.850049,0.06984,0.235948,0.002161,22.61361



MODULE N2A — COMPARISON WITH OLD CORE5


,Model_ID,PR_AUC,ROC_AUC,Delta_PR_vs_Old_Core5
0,XGB_GLOBAL,0.377869,0.850049,-0.001882
1,CAT_GLOBAL,0.376270,0.850128,-0.003481
2,LGBM_GLOBAL,0.375276,0.849220,-0.004475
3,HGB_GLOBAL,0.374130,0.848636,-0.005621
4,LR_L2,0.372212,0.847227,-0.007539
5,LR_EN,0.372200,0.847224,-0.007551
6,RF_GLOBAL,0.369146,0.844818,-0.010605
7,ET_GLOBAL,0.368413,0.845146,-0.011338
8,GNB,0.300301,0.805674,-0.079450



MODULE N2A COMPLETE

Global candidates trained: 9
OOF predictions stored: 9

HOLDOUT remains untouched.
BRFSS-2020 remains absent.

Next: MODULE N2B — controlled boosting/objective diversity.


In [4]:
# ============================================================
# MODULE N2B — CONTROLLED BOOSTING / OBJECTIVE DIVERSITY
#
# PURPOSE
# ------------------------------------------------------------
# Deliberately create strong but behaviorally different
# boosted experts.
#
# Diversity mechanisms:
#   1. model capacity
#   2. positive-class emphasis
#   3. stronger regularization
#   4. row/feature subsampling
#   5. smoother vs interaction-rich trees
#
# IMPORTANT
# ------------------------------------------------------------
# This is NOT hyperparameter tuning against HOLDOUT.
#
# All candidates:
#   - use the same fixed DEV folds
#   - generate strict OOF predictions
#   - never inspect HOLDOUT
#   - never use BRFSS-2020
#
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
)


# ============================================================
# N2B.1 CONFIG
# ============================================================

MODEL_FACTORIES_N2B = {}

MODEL_META_N2B = {}


# ============================================================
# N2B.2 XGBOOST DIVERSITY FAMILY
# ============================================================

if HAVE_XGB:

    # --------------------------------------------------------
    # A. Smoother / shallower
    #
    # Intended behavior:
    # lower interaction complexity, stronger generalization
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "XGB_SMOOTH"
    ] = lambda: XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=900,
        learning_rate=0.025,

        max_depth=3,
        min_child_weight=15,

        subsample=0.90,
        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,
        reg_lambda=7.0,

        tree_method="hist",

        random_state=SEED + 101,

        n_jobs=-1,
    )


    MODEL_META_N2B[
        "XGB_SMOOTH"
    ] = {
        "family":
            "XGBoost",

        "specialization":
            "smooth",

        "objective":
            "global probability prediction",

        "diversity":
            "reduced depth + stronger regularization",
    }


    # --------------------------------------------------------
    # B. Interaction-rich
    #
    # Intended behavior:
    # capture more conditional interactions
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "XGB_INTERACTION"
    ] = lambda: XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=650,
        learning_rate=0.035,

        max_depth=6,
        min_child_weight=15,

        subsample=0.85,
        colsample_bytree=0.75,

        gamma=0.10,

        reg_alpha=1.5,
        reg_lambda=7.0,

        tree_method="hist",

        random_state=SEED + 102,

        n_jobs=-1,
    )


    MODEL_META_N2B[
        "XGB_INTERACTION"
    ] = {
        "family":
            "XGBoost",

        "specialization":
            "interaction-rich",

        "objective":
            "global probability prediction",

        "diversity":
            "deeper trees + feature subsampling",
    }


    # --------------------------------------------------------
    # C. Mild positive-sensitive
    #
    # Not extreme balancing.
    #
    # Designed to shift decision behavior toward positives
    # without destroying ranking/calibration too severely.
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "XGB_POS15"
    ] = lambda: XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=750,
        learning_rate=0.03,

        max_depth=4,
        min_child_weight=10,

        subsample=0.90,
        colsample_bytree=0.80,

        gamma=0.05,

        reg_alpha=1.0,
        reg_lambda=5.0,

        scale_pos_weight=1.5,

        tree_method="hist",

        random_state=SEED + 103,

        n_jobs=-1,
    )


    MODEL_META_N2B[
        "XGB_POS15"
    ] = {
        "family":
            "XGBoost",

        "specialization":
            "mild-positive-sensitive",

        "objective":
            "positive-weighted binary classification",

        "diversity":
            "class-cost diversity",
    }


    # --------------------------------------------------------
    # D. Stronger positive-sensitive
    #
    # More deliberately specialist.
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "XGB_POS25"
    ] = lambda: XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=750,
        learning_rate=0.03,

        max_depth=4,
        min_child_weight=10,

        subsample=0.90,
        colsample_bytree=0.80,

        gamma=0.05,

        reg_alpha=1.0,
        reg_lambda=5.0,

        scale_pos_weight=2.5,

        tree_method="hist",

        random_state=SEED + 104,

        n_jobs=-1,
    )


    MODEL_META_N2B[
        "XGB_POS25"
    ] = {
        "family":
            "XGBoost",

        "specialization":
            "strong-positive-sensitive",

        "objective":
            "positive-weighted binary classification",

        "diversity":
            "stronger class-cost diversity",
    }


    # --------------------------------------------------------
    # E. Stochastic / feature-diverse
    #
    # Stronger random subspace behavior.
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "XGB_SUBSPACE"
    ] = lambda: XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=900,
        learning_rate=0.025,

        max_depth=4,
        min_child_weight=10,

        subsample=0.70,
        colsample_bytree=0.55,

        gamma=0.05,

        reg_alpha=1.0,
        reg_lambda=5.0,

        tree_method="hist",

        random_state=SEED + 105,

        n_jobs=-1,
    )


    MODEL_META_N2B[
        "XGB_SUBSPACE"
    ] = {
        "family":
            "XGBoost",

        "specialization":
            "stochastic-subspace",

        "objective":
            "global probability prediction",

        "diversity":
            "aggressive row + feature subsampling",
    }


# ============================================================
# N2B.3 LIGHTGBM DIVERSITY FAMILY
# ============================================================

if HAVE_LGBM:

    # --------------------------------------------------------
    # A. Smooth
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "LGBM_SMOOTH"
    ] = lambda: LGBMClassifier(

        objective="binary",

        n_estimators=900,
        learning_rate=0.025,

        num_leaves=15,
        max_depth=5,

        min_child_samples=80,

        subsample=0.90,
        colsample_bytree=0.85,

        reg_alpha=1.0,
        reg_lambda=5.0,

        random_state=SEED + 201,

        n_jobs=-1,
        verbosity=-1,
    )


    MODEL_META_N2B[
        "LGBM_SMOOTH"
    ] = {
        "family":
            "LightGBM",

        "specialization":
            "smooth",

        "objective":
            "global binary classification",

        "diversity":
            "low-complexity leaves",
    }


    # --------------------------------------------------------
    # B. Interaction-rich
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "LGBM_INTERACTION"
    ] = lambda: LGBMClassifier(

        objective="binary",

        n_estimators=650,
        learning_rate=0.035,

        num_leaves=63,
        max_depth=8,

        min_child_samples=60,

        subsample=0.80,
        colsample_bytree=0.70,

        reg_alpha=1.0,
        reg_lambda=5.0,

        random_state=SEED + 202,

        n_jobs=-1,
        verbosity=-1,
    )


    MODEL_META_N2B[
        "LGBM_INTERACTION"
    ] = {
        "family":
            "LightGBM",

        "specialization":
            "interaction-rich",

        "objective":
            "global binary classification",

        "diversity":
            "larger leaf structure + subspace",
    }


    # --------------------------------------------------------
    # C. Positive-sensitive
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "LGBM_POS20"
    ] = lambda: LGBMClassifier(

        objective="binary",

        n_estimators=700,
        learning_rate=0.03,

        num_leaves=31,

        min_child_samples=40,

        subsample=0.90,
        colsample_bytree=0.85,

        reg_alpha=0.5,
        reg_lambda=2.0,

        scale_pos_weight=2.0,

        random_state=SEED + 203,

        n_jobs=-1,
        verbosity=-1,
    )


    MODEL_META_N2B[
        "LGBM_POS20"
    ] = {
        "family":
            "LightGBM",

        "specialization":
            "positive-sensitive",

        "objective":
            "positive-weighted classification",

        "diversity":
            "class-cost diversity",
    }


# ============================================================
# N2B.4 CATBOOST DIVERSITY FAMILY
# ============================================================

if HAVE_CAT:

    # --------------------------------------------------------
    # A. Smooth / shallow
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "CAT_SMOOTH"
    ] = lambda: CatBoostClassifier(

        iterations=900,

        learning_rate=0.03,

        depth=5,

        loss_function="Logloss",
        eval_metric="PRAUC",

        l2_leaf_reg=8.0,

        random_seed=SEED + 301,

        verbose=False,
        allow_writing_files=False,

        thread_count=-1,
    )


    MODEL_META_N2B[
        "CAT_SMOOTH"
    ] = {
        "family":
            "CatBoost",

        "specialization":
            "smooth",

        "objective":
            "global log-loss",

        "diversity":
            "shallower strongly regularized trees",
    }


    # --------------------------------------------------------
    # B. Interaction-rich
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "CAT_INTERACTION"
    ] = lambda: CatBoostClassifier(

        iterations=600,

        learning_rate=0.04,

        depth=8,

        loss_function="Logloss",
        eval_metric="PRAUC",

        l2_leaf_reg=8.0,

        random_seed=SEED + 302,

        verbose=False,
        allow_writing_files=False,

        thread_count=-1,
    )


    MODEL_META_N2B[
        "CAT_INTERACTION"
    ] = {
        "family":
            "CatBoost",

        "specialization":
            "interaction-rich",

        "objective":
            "global log-loss",

        "diversity":
            "deeper symmetric trees",
    }


    # --------------------------------------------------------
    # C. Positive-sensitive
    # --------------------------------------------------------

    MODEL_FACTORIES_N2B[
        "CAT_POS20"
    ] = lambda: CatBoostClassifier(

        iterations=700,

        learning_rate=0.04,

        depth=6,

        loss_function="Logloss",
        eval_metric="PRAUC",

        l2_leaf_reg=5.0,

        class_weights=[
            1.0,
            2.0
        ],

        random_seed=SEED + 303,

        verbose=False,
        allow_writing_files=False,

        thread_count=-1,
    )


    MODEL_META_N2B[
        "CAT_POS20"
    ] = {
        "family":
            "CatBoost",

        "specialization":
            "positive-sensitive",

        "objective":
            "positive-weighted log-loss",

        "diversity":
            "class-cost diversity",
    }


# ============================================================
# N2B.5 HGB DIVERSITY FAMILY
#
# HistGradientBoosting supports sample_weight through fit().
# For the weighted version we need a special wrapper.
# ============================================================

MODEL_FACTORIES_N2B[
    "HGB_SMOOTH"
] = lambda: HistGradientBoostingClassifier(

    learning_rate=0.045,

    max_iter=450,

    max_leaf_nodes=15,

    min_samples_leaf=60,

    l2_regularization=3.0,

    early_stopping=False,

    random_state=SEED + 401,
)


MODEL_META_N2B[
    "HGB_SMOOTH"
] = {
    "family":
        "HistGradientBoosting",

    "specialization":
        "smooth",

    "objective":
        "global log-loss",

    "diversity":
        "smaller trees + stronger smoothing",
}


MODEL_FACTORIES_N2B[
    "HGB_INTERACTION"
] = lambda: HistGradientBoostingClassifier(

    learning_rate=0.055,

    max_iter=350,

    max_leaf_nodes=63,

    min_samples_leaf=50,

    l2_regularization=3.0,

    early_stopping=False,

    random_state=SEED + 402,
)


MODEL_META_N2B[
    "HGB_INTERACTION"
] = {
    "family":
        "HistGradientBoosting",

    "specialization":
        "interaction-rich",

    "objective":
        "global log-loss",

    "diversity":
        "larger leaf capacity",
}


print(
    "=" * 130
)

print(
    "MODULE N2B — CONTROLLED DIVERSITY CANDIDATES"
)

print(
    "=" * 130
)


for model_id in MODEL_FACTORIES_N2B:

    meta = MODEL_META_N2B[
        model_id
    ]

    print(
        f"{model_id:22s} | "
        f"{meta['specialization']:24s} | "
        f"{meta['diversity']}"
    )


# ============================================================
# N2B.6 GENERIC OOF TRAINING
#
# Reuse the same N2A strict OOF structure.
# ============================================================

def run_oof_model_n2b(
    model_id,
    factory,
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        "TRAINING:",
        model_id
    )

    print(
        "=" * 110
    )


    oof = np.full(
        len(y_DEV),
        np.nan,
        dtype=float
    )


    fold_rows = []


    total_start = time.perf_counter()


    for fold in range(
        N_FOLDS
    ):

        train_idx = np.where(
            DEV_FOLD != fold
        )[0]


        valid_idx = np.where(
            DEV_FOLD == fold
        )[0]


        X_train = X_DEV.iloc[
            train_idx
        ]


        X_valid = X_DEV.iloc[
            valid_idx
        ]


        y_train = y_DEV[
            train_idx
        ]


        y_valid = y_DEV[
            valid_idx
        ]


        model = factory()


        t0 = time.perf_counter()


        model.fit(
            X_train,
            y_train
        )


        pred = positive_probability_n2a(
            model,
            X_valid
        )


        pred = np.clip(
            np.asarray(
                pred,
                dtype=float
            ),
            1e-7,
            1 - 1e-7
        )


        oof[
            valid_idx
        ] = pred


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        metrics = probability_metrics(
            y_valid,
            pred
        )


        fold_rows.append({

            "Model_ID":
                model_id,

            "Fold":
                fold,

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],

            "Seconds":
                elapsed,
        })


        print(
            f"Fold {fold}: "
            f"PR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f} | "
            f"Brier={metrics['Brier']:.6f} | "
            f"{elapsed:.1f}s"
        )


        del model
        gc.collect()


    if np.isnan(
        oof
    ).any():

        raise RuntimeError(
            f"Incomplete OOF for {model_id}"
        )


    pooled = probability_metrics(
        y_DEV,
        oof
    )


    total_time = (
        time.perf_counter()
        -
        total_start
    )


    print(
        "\nPOOLED:"
    )

    print(
        f"PR={pooled['PR_AUC']:.6f}"
    )

    print(
        f"ROC={pooled['ROC_AUC']:.6f}"
    )

    print(
        f"Brier={pooled['Brier']:.6f}"
    )

    print(
        f"NLL={pooled['NLL']:.6f}"
    )

    print(
        f"ECE={pooled['ECE']:.6f}"
    )


    return {

        "oof":
            oof,

        "pooled":
            pooled,

        "fold_results":
            pd.DataFrame(
                fold_rows
            ),

        "seconds":
            total_time,
    }


# ============================================================
# N2B.7 RUN CANDIDATES
# ============================================================

RESULTS_N2B = {}

fold_tables_n2b = []


for model_id, factory in (
    MODEL_FACTORIES_N2B.items()
):

    result = run_oof_model_n2b(

        model_id,
        factory
    )


    RESULTS_N2B[
        model_id
    ] = result


    store_oof(

        model_id,
        result[
            "oof"
        ]
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        result[
            "oof"
        ]
    )


    fold_tables_n2b.append(

        result[
            "fold_results"
        ]
    )


    meta = MODEL_META_N2B[
        model_id
    ]


    register_experiment(

        model_id=
            model_id,

        family=
            meta[
                "family"
            ],

        role=
            "global-diverse",

        specialization=
            meta[
                "specialization"
            ],

        feature_view=
            "all_21",

        training_objective=
            meta[
                "objective"
            ],

        robustness_augmentation=
            "None",

        diversity_method=
            meta[
                "diversity"
            ],

        oof_available=
            True,

        metrics=
            result[
                "pooled"
            ],

        status=
            "candidate",

        notes=
            (
                "Controlled behavioral-diversity "
                "candidate."
            ),
    )


FOLD_RESULTS_N2B = pd.concat(

    fold_tables_n2b,

    ignore_index=True
)


# ============================================================
# N2B.8 LEADERBOARD
# ============================================================

rows = []


for model_id, result in (
    RESULTS_N2B.items()
):

    m = result[
        "pooled"
    ]


    rows.append({

        "Model_ID":
            model_id,

        "Family":
            MODEL_META_N2B[
                model_id
            ][
                "family"
            ],

        "Specialization":
            MODEL_META_N2B[
                model_id
            ][
                "specialization"
            ],

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Seconds":
            result[
                "seconds"
            ],
    })


LEADERBOARD_N2B = (

    pd.DataFrame(
        rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N2B — CONTROLLED DIVERSITY LEADERBOARD"
)

print(
    "=" * 135
)


display(
    LEADERBOARD_N2B
)


# ============================================================
# N2B.9 COMBINED N2A + N2B LEADERBOARD
# ============================================================

base_n2a = (

    LEADERBOARD_N2A[
        [
            "Model_ID",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
        ]
    ]
    .copy()
)


base_n2a[
    "Source"
] = "N2A"


base_n2b = (

    LEADERBOARD_N2B[
        [
            "Model_ID",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
        ]
    ]
    .copy()
)


base_n2b[
    "Source"
] = "N2B"


ALL_GLOBAL_LEADERBOARD_N2 = (

    pd.concat(

        [
            base_n2a,
            base_n2b,
        ],

        ignore_index=True
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N2 — COMPLETE GLOBAL CANDIDATE LEADERBOARD"
)

print(
    "=" * 135
)


display(
    ALL_GLOBAL_LEADERBOARD_N2
)


# ============================================================
# N2B.10 STRONG-CANDIDATE CORRELATION
#
# We intentionally exclude extremely weak candidates here.
#
# Temporary threshold:
# PR >= 0.36
#
# This is diagnostic only.
# No final selection yet.
# ============================================================

STRONG_IDS_N2 = (

    ALL_GLOBAL_LEADERBOARD_N2.loc[

        ALL_GLOBAL_LEADERBOARD_N2[
            "PR_AUC"
        ]
        >=
        0.36,

        "Model_ID"

    ]
    .tolist()
)


P_STRONG_N2 = np.column_stack([

    OOF_STORE[
        model_id
    ]

    for model_id
    in STRONG_IDS_N2
])


CORR_STRONG_N2 = pd.DataFrame(

    np.corrcoef(
        P_STRONG_N2.T
    ),

    index=
        STRONG_IDS_N2,

    columns=
        STRONG_IDS_N2,
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N2 — STRONG CANDIDATE CORRELATION"
)

print(
    "=" * 135
)


display(
    CORR_STRONG_N2.round(4)
)


# ============================================================
# N2B.11 DISTANCE FROM XGB_GLOBAL
#
# Useful first measure of behavioral difference.
# ============================================================

anchor = np.asarray(
    OOF_STORE[
        "XGB_GLOBAL"
    ],
    dtype=float
)


distance_rows = []


for model_id in STRONG_IDS_N2:

    p = np.asarray(
        OOF_STORE[
            model_id
        ],
        dtype=float
    )


    distance_rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            float(

                ALL_GLOBAL_LEADERBOARD_N2.loc[

                    ALL_GLOBAL_LEADERBOARD_N2[
                        "Model_ID"
                    ]
                    ==
                    model_id,

                    "PR_AUC"

                ].iloc[0]
            ),

        "Corr_with_XGB_GLOBAL":
            float(
                np.corrcoef(
                    anchor,
                    p
                )[0, 1]
            ),

        "MeanAbsProbDifference":
            float(
                np.mean(
                    np.abs(
                        anchor
                        -
                        p
                    )
                )
            ),
    })


DIVERSITY_PREVIEW_N2 = (

    pd.DataFrame(
        distance_rows
    )
    .sort_values(
        [
            "Corr_with_XGB_GLOBAL",
            "PR_AUC",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N2 — EARLY DIVERSITY PREVIEW VS XGB"
)

print(
    "=" * 135
)


display(
    DIVERSITY_PREVIEW_N2
)


# ============================================================
# N2B.12 PAIRWISE TWO-MODEL ENSEMBLE SCREEN
#
# IMPORTANT:
# Pure discovery diagnostic.
#
# Equal-weight pair:
#
#       p_pair = (p_i + p_j) / 2
#
# This identifies whether any model adds useful complementary
# ranking to another.
# ============================================================

pair_rows = []


for i in range(
    len(
        STRONG_IDS_N2
    )
):

    for j in range(
        i + 1,
        len(
            STRONG_IDS_N2
        )
    ):

        id_a = STRONG_IDS_N2[
            i
        ]

        id_b = STRONG_IDS_N2[
            j
        ]


        pa = OOF_STORE[
            id_a
        ]


        pb = OOF_STORE[
            id_b
        ]


        p_pair = (
            0.5 * pa
            +
            0.5 * pb
        )


        metrics = probability_metrics(
            y_DEV,
            p_pair
        )


        corr = np.corrcoef(
            pa,
            pb
        )[0, 1]


        pair_rows.append({

            "Expert_A":
                id_a,

            "Expert_B":
                id_b,

            "Correlation":
                corr,

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],
        })


PAIR_SCREEN_N2 = (

    pd.DataFrame(
        pair_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N2 — TOP 20 EQUAL-WEIGHT PAIRS"
)

print(
    "=" * 135
)


display(
    PAIR_SCREEN_N2.head(
        20
    )
)


# ============================================================
# N2B.13 BEST PAIR VS BEST SINGLE
# ============================================================

BEST_SINGLE_PR_N2 = float(

    ALL_GLOBAL_LEADERBOARD_N2[
        "PR_AUC"
    ].max()
)


BEST_PAIR_PR_N2 = float(

    PAIR_SCREEN_N2[
        "PR_AUC"
    ].max()
)


PAIR_GAIN_N2 = (

    BEST_PAIR_PR_N2
    -
    BEST_SINGLE_PR_N2
)


print(
    "\nBest single PR:",
    BEST_SINGLE_PR_N2
)


print(
    "Best pair PR:",
    BEST_PAIR_PR_N2
)


print(
    "Pair gain:",
    PAIR_GAIN_N2
)


# ============================================================
# N2B.14 SAVE
# ============================================================

LEADERBOARD_N2B.to_csv(

    TAB_DIR
    /
    "n2b_controlled_diversity_leaderboard.csv",

    index=False
)


ALL_GLOBAL_LEADERBOARD_N2.to_csv(

    TAB_DIR
    /
    "n2_complete_global_leaderboard.csv",

    index=False
)


CORR_STRONG_N2.to_csv(

    TAB_DIR
    /
    "n2_strong_candidate_correlation.csv"
)


DIVERSITY_PREVIEW_N2.to_csv(

    TAB_DIR
    /
    "n2_diversity_preview.csv",

    index=False
)


PAIR_SCREEN_N2.to_csv(

    TAB_DIR
    /
    "n2_pairwise_ensemble_screen.csv",

    index=False
)


FOLD_RESULTS_N2B.to_csv(

    TAB_DIR
    /
    "n2b_fold_results.csv",

    index=False
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


with open(

    LOG_DIR
    /
    "n2b_manifest.json",

    "w"

) as f:

    json.dump(

        {
            "module":
                "Controlled boosting diversity",

            "n_new_candidates":
                len(
                    MODEL_FACTORIES_N2B
                ),

            "development_only":
                True,

            "holdout_used":
                False,

            "external_used":
                False,

            "expert_selection_finalized":
                False,
        },

        f,

        indent=2
    )


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N2B COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nNo expert has been permanently selected."
)

print(
    "HOLDOUT remains untouched."
)

print(
    "Next: N3 — neural/global representation-diverse experts."
)

MODULE N2B — CONTROLLED DIVERSITY CANDIDATES
XGB_SMOOTH             | smooth                   | reduced depth + stronger regularization
XGB_INTERACTION        | interaction-rich         | deeper trees + feature subsampling
XGB_POS15              | mild-positive-sensitive  | class-cost diversity
XGB_POS25              | strong-positive-sensitive | stronger class-cost diversity
XGB_SUBSPACE           | stochastic-subspace      | aggressive row + feature subsampling
LGBM_SMOOTH            | smooth                   | low-complexity leaves
LGBM_INTERACTION       | interaction-rich         | larger leaf structure + subspace
LGBM_POS20             | positive-sensitive       | class-cost diversity
CAT_SMOOTH             | smooth                   | shallower strongly regularized trees
CAT_INTERACTION        | interaction-rich         | deeper symmetric trees
CAT_POS20              | positive-sensitive       | class-cost diversity
HGB_SMOOTH             | smooth                   | smaller tr

,Model_ID,Family,Specialization,PR_AUC,ROC_AUC,Brier,NLL,ECE,Seconds
0,XGB_SMOOTH,XGBoost,smooth,0.378728,0.850209,0.069802,0.235856,0.002929,23.126378
1,XGB_SUBSPACE,XGBoost,stochastic-subspace,0.378562,0.850477,0.069792,0.235729,0.002664,28.315989
2,XGB_POS15,XGBoost,mild-positive-sensitive,0.378240,0.850049,0.071849,0.241930,0.030974,22.062660
3,CAT_SMOOTH,CatBoost,smooth,0.378151,0.850386,0.069787,0.235746,0.002194,148.066571
4,XGB_POS25,XGBoost,strong-positive-sensitive,0.377895,0.850137,0.081764,0.269415,0.079265,21.883885
5,LGBM_SMOOTH,LightGBM,smooth,0.377301,0.849855,0.069869,0.236057,0.002279,29.017135
6,CAT_POS20,CatBoost,positive-sensitive,0.377064,0.849966,0.076024,0.253951,0.056123,127.008038
7,HGB_SMOOTH,HistGradientBoosting,smooth,0.376940,0.849566,0.069901,0.236200,0.001944,32.121063
8,LGBM_POS20,LightGBM,positive-sensitive,0.375687,0.848968,0.076238,0.254288,0.055709,25.177116
9,XGB_INTERACTION,XGBoost,interaction-rich,0.374092,0.849059,0.070044,0.236604,0.003448,25.180513



MODULE N2 — COMPLETE GLOBAL CANDIDATE LEADERBOARD


,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE,Source
0,XGB_SMOOTH,0.378728,0.850209,0.069802,0.235856,0.002929,N2B
1,XGB_SUBSPACE,0.378562,0.850477,0.069792,0.235729,0.002664,N2B
2,XGB_POS15,0.378240,0.850049,0.071849,0.241930,0.030974,N2B
3,CAT_SMOOTH,0.378151,0.850386,0.069787,0.235746,0.002194,N2B
4,XGB_POS25,0.377895,0.850137,0.081764,0.269415,0.079265,N2B
5,XGB_GLOBAL,0.377869,0.850049,0.069840,0.235948,0.002161,N2A
6,LGBM_SMOOTH,0.377301,0.849855,0.069869,0.236057,0.002279,N2B
7,CAT_POS20,0.377064,0.849966,0.076024,0.253951,0.056123,N2B
8,HGB_SMOOTH,0.376940,0.849566,0.069901,0.236200,0.001944,N2B
9,CAT_GLOBAL,0.376270,0.850128,0.069877,0.235976,0.002203,N2A



MODULE N2 — STRONG CANDIDATE CORRELATION


...


MODULE N2 — EARLY DIVERSITY PREVIEW VS XGB


,Model_ID,PR_AUC,Corr_with_XGB_GLOBAL,MeanAbsProbDifference
0,ET_GLOBAL,0.368413,0.966510,0.019665
1,RF_GLOBAL,0.369146,0.970669,0.018963
2,LR_L2,0.372212,0.971989,0.016633
3,LR_EN,0.372200,0.971995,0.016633
4,XGB_POS25,0.377895,0.981225,0.079339
5,LGBM_POS20,0.375687,0.983781,0.055807
6,CAT_POS20,0.377064,0.984003,0.056293
7,HGB_INTERACTION,0.370194,0.984880,0.011632
8,CAT_INTERACTION,0.372436,0.986118,0.010219
9,LGBM_INTERACTION,0.371504,0.989095,0.009960



MODULE N2 — TOP 20 EQUAL-WEIGHT PAIRS


,Expert_A,Expert_B,Correlation,PR_AUC,ROC_AUC,Brier
0,XGB_SMOOTH,CAT_SMOOTH,0.993982,0.379339,0.850578,0.069747
1,XGB_SMOOTH,CAT_POS20,0.981529,0.379182,0.850482,0.071284
2,XGB_SUBSPACE,CAT_SMOOTH,0.995179,0.379106,0.850660,0.069751
3,XGB_SMOOTH,XGB_SUBSPACE,0.997151,0.379035,0.850483,0.069774
4,XGB_POS15,CAT_SMOOTH,0.991655,0.378907,0.850495,0.070261
5,XGB_SMOOTH,XGB_POS15,0.993824,0.378843,0.850255,0.070300
6,CAT_SMOOTH,HGB_SMOOTH,0.991118,0.378840,0.850426,0.069773
7,XGB_SMOOTH,CAT_GLOBAL,0.990265,0.378838,0.850570,0.069762
8,CAT_SMOOTH,XGB_GLOBAL,0.994416,0.378782,0.850512,0.069769
9,XGB_SUBSPACE,CAT_POS20,0.984381,0.378772,0.850472,0.071311



Best single PR: 0.3787275181167349
Best pair PR: 0.37933913560955274
Pair gain: 0.0006116174928178508

MODULE N2B COMPLETE

No expert has been permanently selected.
HOLDOUT remains untouched.
Next: N3 — neural/global representation-diverse experts.


In [5]:
# ============================================================
# MODULE N3A — NEURAL REPRESENTATION-DIVERSE EXPERT POOL
#
# Models:
#   1. MLP
#   2. Residual MLP
#   3. DeepFM
#   4. AutoInt
#
# PURPOSE:
#   Introduce representation diversity beyond tree ensembles.
#
# IMPORTANT:
#   - strict 5-fold OOF
#   - DEV only
#   - HOLDOUT untouched
#   - BRFSS-2020 untouched
# ============================================================

import os
import gc
import math
import time
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


# ============================================================
# N3A.1 DEVICE / REPRODUCIBILITY
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


def seed_everything_n3(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything_n3(SEED)


# ============================================================
# N3A.2 FEATURE SETUP
# ============================================================

CAT_COLS_N3 = CATEGORICAL_FEATURES.copy()

NUM_COLS_N3 = CONTINUOUS_FEATURES.copy()


print(
    "Categorical:",
    len(CAT_COLS_N3)
)

print(
    "Continuous:",
    len(NUM_COLS_N3)
)


# ============================================================
# N3A.3 CATEGORY CARDINALITIES
# ============================================================

CAT_CARDINALITIES_N3 = []


for col in CAT_COLS_N3:

    max_value = int(
        X_DEV[col].max()
    )

    CAT_CARDINALITIES_N3.append(
        max_value + 1
    )


print(
    "Cardinalities:",
    CAT_CARDINALITIES_N3
)


# ============================================================
# N3A.4 DATA PREPARATION
#
# Numerical features standardized inside each fold.
#
# Categorical variables remain integer encoded.
# ============================================================

def prepare_fold_n3(
    train_idx,
    valid_idx,
):

    X_train = X_DEV.iloc[
        train_idx
    ].copy()

    X_valid = X_DEV.iloc[
        valid_idx
    ].copy()


    scaler = StandardScaler()


    train_num = scaler.fit_transform(
        X_train[
            NUM_COLS_N3
        ]
    ).astype(
        np.float32
    )


    valid_num = scaler.transform(
        X_valid[
            NUM_COLS_N3
        ]
    ).astype(
        np.float32
    )


    train_cat = (

        X_train[
            CAT_COLS_N3
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    valid_cat = (

        X_valid[
            CAT_COLS_N3
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    train_y = y_DEV[
        train_idx
    ].astype(
        np.float32
    )


    valid_y = y_DEV[
        valid_idx
    ].astype(
        np.float32
    )


    return (
        train_cat,
        train_num,
        train_y,
        valid_cat,
        valid_num,
        valid_y,
        scaler,
    )


# ============================================================
# N3A.5 DATASET
# ============================================================

class TabularDatasetN3(
    Dataset
):

    def __init__(
        self,
        x_cat,
        x_num,
        y=None,
    ):

        self.x_cat = torch.tensor(
            x_cat,
            dtype=torch.long
        )

        self.x_num = torch.tensor(
            x_num,
            dtype=torch.float32
        )

        self.y = None

        if y is not None:

            self.y = torch.tensor(
                y,
                dtype=torch.float32
            )


    def __len__(
        self
    ):

        return len(
            self.x_cat
        )


    def __getitem__(
        self,
        idx
    ):

        if self.y is None:

            return (
                self.x_cat[idx],
                self.x_num[idx],
            )


        return (
            self.x_cat[idx],
            self.x_num[idx],
            self.y[idx],
        )


# ============================================================
# N3A.6 EMBEDDING BLOCK
# ============================================================

class EmbeddingBlockN3(
    nn.Module
):

    def __init__(
        self,
        cardinalities,
        emb_dim=8,
    ):

        super().__init__()


        self.embeddings = nn.ModuleList([

            nn.Embedding(
                cardinality,
                emb_dim
            )

            for cardinality
            in cardinalities
        ])


        self.emb_dim = emb_dim


    def forward(
        self,
        x_cat
    ):

        embs = [

            emb(
                x_cat[:, i]
            )

            for i, emb
            in enumerate(
                self.embeddings
            )
        ]


        return torch.stack(
            embs,
            dim=1
        )


# ============================================================
# N3A.7 MODEL 1 — STANDARD EMBEDDING MLP
# ============================================================

class TabMLP_N3(
    nn.Module
):

    def __init__(
        self,
        cardinalities,
        n_num,
        emb_dim=8,
        hidden=(256, 128, 64),
        dropout=0.20,
    ):

        super().__init__()


        self.embed = EmbeddingBlockN3(
            cardinalities,
            emb_dim
        )


        input_dim = (
            len(cardinalities)
            *
            emb_dim
            +
            n_num
        )


        layers = []


        previous = input_dim


        for h in hidden:

            layers.extend([

                nn.Linear(
                    previous,
                    h
                ),

                nn.BatchNorm1d(
                    h
                ),

                nn.ReLU(),

                nn.Dropout(
                    dropout
                ),
            ])


            previous = h


        self.mlp = nn.Sequential(
            *layers
        )


        self.head = nn.Linear(
            previous,
            1
        )


    def forward(
        self,
        x_cat,
        x_num
    ):

        emb = self.embed(
            x_cat
        )


        flat = emb.flatten(
            1
        )


        z = torch.cat(
            [
                flat,
                x_num
            ],
            dim=1
        )


        h = self.mlp(
            z
        )


        return self.head(
            h
        ).squeeze(1)


# ============================================================
# N3A.8 MODEL 2 — RESIDUAL MLP
# ============================================================

class ResidualBlockN3(
    nn.Module
):

    def __init__(
        self,
        dim,
        dropout=0.15,
    ):

        super().__init__()


        self.block = nn.Sequential(

            nn.Linear(
                dim,
                dim
            ),

            nn.BatchNorm1d(
                dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                dim,
                dim
            ),

            nn.BatchNorm1d(
                dim
            ),
        )


        self.activation = nn.GELU()


    def forward(
        self,
        x
    ):

        return self.activation(
            x
            +
            self.block(
                x
            )
        )


class ResMLP_N3(
    nn.Module
):

    def __init__(
        self,
        cardinalities,
        n_num,
        emb_dim=8,
        width=192,
        n_blocks=3,
        dropout=0.15,
    ):

        super().__init__()


        self.embed = EmbeddingBlockN3(
            cardinalities,
            emb_dim
        )


        input_dim = (
            len(cardinalities)
            *
            emb_dim
            +
            n_num
        )


        self.input_layer = nn.Sequential(

            nn.Linear(
                input_dim,
                width
            ),

            nn.BatchNorm1d(
                width
            ),

            nn.GELU(),
        )


        self.blocks = nn.Sequential(

            *[

                ResidualBlockN3(
                    width,
                    dropout
                )

                for _
                in range(
                    n_blocks
                )
            ]
        )


        self.head = nn.Linear(
            width,
            1
        )


    def forward(
        self,
        x_cat,
        x_num
    ):

        emb = self.embed(
            x_cat
        )


        flat = emb.flatten(
            1
        )


        x = torch.cat(
            [
                flat,
                x_num
            ],
            dim=1
        )


        h = self.input_layer(
            x
        )


        h = self.blocks(
            h
        )


        return self.head(
            h
        ).squeeze(1)


# ============================================================
# N3A.9 MODEL 3 — DEEPFM
# ============================================================

class DeepFM_N3(
    nn.Module
):

    def __init__(
        self,
        cardinalities,
        n_num,
        emb_dim=8,
        hidden=(192, 96),
        dropout=0.20,
    ):

        super().__init__()


        self.cardinalities = cardinalities


        self.embed = EmbeddingBlockN3(
            cardinalities,
            emb_dim
        )


        self.linear_embeddings = nn.ModuleList([

            nn.Embedding(
                c,
                1
            )

            for c
            in cardinalities
        ])


        deep_input = (
            len(cardinalities)
            *
            emb_dim
            +
            n_num
        )


        layers = []


        previous = deep_input


        for h in hidden:

            layers.extend([

                nn.Linear(
                    previous,
                    h
                ),

                nn.BatchNorm1d(
                    h
                ),

                nn.ReLU(),

                nn.Dropout(
                    dropout
                ),
            ])

            previous = h


        self.deep = nn.Sequential(
            *layers
        )


        self.deep_head = nn.Linear(
            previous,
            1
        )


        self.num_linear = nn.Linear(
            n_num,
            1
        )


        self.bias = nn.Parameter(
            torch.zeros(1)
        )


    def forward(
        self,
        x_cat,
        x_num
    ):

        emb = self.embed(
            x_cat
        )


        # ------------------------
        # Linear FM component
        # ------------------------

        linear_cat = torch.stack([

            emb_layer(
                x_cat[:, i]
            ).squeeze(-1)

            for i, emb_layer
            in enumerate(
                self.linear_embeddings
            )

        ], dim=1).sum(
            dim=1
        )


        linear_num = self.num_linear(
            x_num
        ).squeeze(1)


        # ------------------------
        # FM second-order term
        # ------------------------

        summed = emb.sum(
            dim=1
        )


        summed_square = (
            summed
            **
            2
        )


        square_summed = (
            emb
            **
            2
        ).sum(
            dim=1
        )


        fm = 0.5 * (

            summed_square
            -
            square_summed

        ).sum(
            dim=1
        )


        # ------------------------
        # Deep component
        # ------------------------

        deep_input = torch.cat(

            [
                emb.flatten(1),
                x_num
            ],

            dim=1
        )


        deep = self.deep_head(

            self.deep(
                deep_input
            )

        ).squeeze(1)


        return (

            self.bias
            +
            linear_cat
            +
            linear_num
            +
            fm
            +
            deep
        )


# ============================================================
# N3A.10 MODEL 4 — AUTOINTERACTION / AUTOINt
# ============================================================

class AutoInt_N3(
    nn.Module
):

    def __init__(
        self,
        cardinalities,
        n_num,
        emb_dim=16,
        n_heads=4,
        n_layers=2,
        dropout=0.15,
    ):

        super().__init__()


        self.embed = EmbeddingBlockN3(
            cardinalities,
            emb_dim
        )


        # numerical features get one learned token each

        self.num_weight = nn.Parameter(

            torch.randn(
                n_num,
                emb_dim
            )
            *
            0.02
        )


        self.num_bias = nn.Parameter(

            torch.zeros(
                n_num,
                emb_dim
            )
        )


        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=emb_dim,

                nhead=n_heads,

                dim_feedforward=
                    emb_dim * 4,

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True,
            )
        )


        self.encoder = nn.TransformerEncoder(

            encoder_layer,

            num_layers=n_layers
        )


        n_tokens = (
            len(cardinalities)
            +
            n_num
        )


        self.head = nn.Sequential(

            nn.LayerNorm(
                emb_dim
            ),

            nn.Linear(
                emb_dim,
                64
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                1
            ),
        )


    def forward(
        self,
        x_cat,
        x_num
    ):

        cat_tokens = self.embed(
            x_cat
        )


        num_tokens = (

            x_num.unsqueeze(
                -1
            )

            *
            self.num_weight.unsqueeze(
                0
            )

            +
            self.num_bias.unsqueeze(
                0
            )
        )


        tokens = torch.cat(

            [
                cat_tokens,
                num_tokens
            ],

            dim=1
        )


        encoded = self.encoder(
            tokens
        )


        pooled = encoded.mean(
            dim=1
        )


        return self.head(
            pooled
        ).squeeze(1)


# ============================================================
# N3A.11 MODEL FACTORY
# ============================================================

MODEL_FACTORIES_N3A = {

    "MLP_GLOBAL": lambda: TabMLP_N3(

        CAT_CARDINALITIES_N3,

        len(
            NUM_COLS_N3
        ),

        emb_dim=8,

        hidden=(
            256,
            128,
            64
        ),

        dropout=0.20,
    ),


    "RESMLP_GLOBAL": lambda: ResMLP_N3(

        CAT_CARDINALITIES_N3,

        len(
            NUM_COLS_N3
        ),

        emb_dim=8,

        width=192,

        n_blocks=3,

        dropout=0.15,
    ),


    "DEEPFM_GLOBAL": lambda: DeepFM_N3(

        CAT_CARDINALITIES_N3,

        len(
            NUM_COLS_N3
        ),

        emb_dim=8,

        hidden=(
            192,
            96
        ),

        dropout=0.20,
    ),


    "AUTOINT_GLOBAL": lambda: AutoInt_N3(

        CAT_CARDINALITIES_N3,

        len(
            NUM_COLS_N3
        ),

        emb_dim=16,

        n_heads=4,

        n_layers=2,

        dropout=0.15,
    ),
}


# ============================================================
# N3A.12 TRAINING CONFIG
# ============================================================

BATCH_SIZE_N3 = 2048

MAX_EPOCHS_N3 = 30

PATIENCE_N3 = 5

LR_N3 = 1e-3

WEIGHT_DECAY_N3 = 1e-4


# ============================================================
# N3A.13 PREDICTION HELPER
# ============================================================

@torch.no_grad()
def predict_neural_n3(
    model,
    loader,
):

    model.eval()


    predictions = []


    for batch in loader:

        x_cat = batch[0].to(
            DEVICE
        )

        x_num = batch[1].to(
            DEVICE
        )


        logits = model(
            x_cat,
            x_num
        )


        prob = torch.sigmoid(
            logits
        )


        predictions.append(

            prob
            .detach()
            .cpu()
            .numpy()
        )


    return np.concatenate(
        predictions
    )


# ============================================================
# N3A.14 TRAIN ONE FOLD
# ============================================================

def train_neural_fold_n3(
    model_factory,
    train_idx,
    valid_idx,
    seed,
):

    seed_everything_n3(
        seed
    )


    (
        train_cat,
        train_num,
        train_y,

        valid_cat,
        valid_num,
        valid_y,

        scaler,

    ) = prepare_fold_n3(

        train_idx,
        valid_idx
    )


    train_ds = TabularDatasetN3(

        train_cat,
        train_num,
        train_y
    )


    valid_ds = TabularDatasetN3(

        valid_cat,
        valid_num,
        valid_y
    )


    train_loader = DataLoader(

        train_ds,

        batch_size=
            BATCH_SIZE_N3,

        shuffle=True,

        num_workers=2,

        pin_memory=True,

        drop_last=False,
    )


    valid_loader = DataLoader(

        valid_ds,

        batch_size=
            BATCH_SIZE_N3 * 2,

        shuffle=False,

        num_workers=2,

        pin_memory=True,
    )


    model = model_factory().to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LR_N3,

        weight_decay=
            WEIGHT_DECAY_N3,
    )


    criterion = nn.BCEWithLogitsLoss()


    use_amp = (
        DEVICE.type
        ==
        "cuda"
    )


    scaler_amp = torch.amp.GradScaler(

        "cuda",

        enabled=use_amp
    )


    best_pr = -np.inf

    best_state = None

    best_epoch = None

    patience_counter = 0


    history = []


    for epoch in range(
        1,
        MAX_EPOCHS_N3 + 1
    ):

        model.train()


        running_loss = 0.0

        n_seen = 0


        for (
            x_cat,
            x_num,
            y_batch

        ) in train_loader:

            x_cat = x_cat.to(
                DEVICE,
                non_blocking=True
            )


            x_num = x_num.to(
                DEVICE,
                non_blocking=True
            )


            y_batch = y_batch.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.amp.autocast(

                device_type=
                    DEVICE.type,

                enabled=use_amp
            ):

                logits = model(
                    x_cat,
                    x_num
                )


                loss = criterion(
                    logits,
                    y_batch
                )


            scaler_amp.scale(
                loss
            ).backward()


            scaler_amp.step(
                optimizer
            )


            scaler_amp.update()


            running_loss += (

                loss.item()
                *
                len(
                    y_batch
                )
            )


            n_seen += len(
                y_batch
            )


        valid_pred = predict_neural_n3(

            model,
            valid_loader
        )


        valid_pr = (
            average_precision_score(

                valid_y,
                valid_pred
            )
        )


        valid_roc = (
            roc_auc_score(

                valid_y,
                valid_pred
            )
        )


        history.append({

            "epoch":
                epoch,

            "train_loss":
                running_loss
                /
                n_seen,

            "valid_pr":
                valid_pr,

            "valid_roc":
                valid_roc,
        })


        if valid_pr > (
            best_pr
            +
            1e-5
        ):

            best_pr = valid_pr

            best_epoch = epoch


            best_state = {

                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model
                .state_dict()
                .items()
            }


            patience_counter = 0


        else:

            patience_counter += 1


        if patience_counter >= (
            PATIENCE_N3
        ):

            break


    model.load_state_dict(
        best_state
    )


    valid_pred = predict_neural_n3(

        model,
        valid_loader
    )


    return {

        "model":
            model,

        "scaler":
            scaler,

        "pred":
            valid_pred,

        "best_epoch":
            best_epoch,

        "best_pr":
            best_pr,

        "history":
            pd.DataFrame(
                history
            ),
    }


# ============================================================
# N3A.15 STRICT OOF LOOP
# ============================================================

RESULTS_N3A = {}

FOLD_ROWS_N3A = []


for (
    model_id,
    factory

) in MODEL_FACTORIES_N3A.items():

    print(
        "\n"
        + "=" * 120
    )

    print(
        "TRAINING NEURAL EXPERT:",
        model_id
    )

    print(
        "=" * 120
    )


    oof = np.full(

        len(
            y_DEV
        ),

        np.nan,

        dtype=np.float32
    )


    start_total = time.perf_counter()


    for fold in range(
        N_FOLDS
    ):

        train_idx = np.where(

            DEV_FOLD != fold

        )[0]


        valid_idx = np.where(

            DEV_FOLD == fold

        )[0]


        t0 = time.perf_counter()


        result = train_neural_fold_n3(

            factory,

            train_idx,

            valid_idx,

            seed=
                SEED
                +
                fold
                +
                1000
        )


        oof[
            valid_idx
        ] = result[
            "pred"
        ]


        metrics = probability_metrics(

            y_DEV[
                valid_idx
            ],

            result[
                "pred"
            ]
        )


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        FOLD_ROWS_N3A.append({

            "Model_ID":
                model_id,

            "Fold":
                fold,

            "Best_Epoch":
                result[
                    "best_epoch"
                ],

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],

            "Seconds":
                elapsed,
        })


        print(
            f"Fold {fold}: "
            f"epoch={result['best_epoch']} | "
            f"PR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f} | "
            f"Brier={metrics['Brier']:.6f} | "
            f"{elapsed:.1f}s"
        )


        del result

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    if np.isnan(
        oof
    ).any():

        raise RuntimeError(
            f"Incomplete OOF: {model_id}"
        )


    pooled = probability_metrics(

        y_DEV,
        oof
    )


    total_seconds = (

        time.perf_counter()
        -
        start_total
    )


    RESULTS_N3A[
        model_id
    ] = {

        "oof":
            oof,

        "metrics":
            pooled,

        "seconds":
            total_seconds,
    }


    store_oof(

        model_id,
        oof
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        oof
    )


    register_experiment(

        model_id=
            model_id,

        family=
            model_id.replace(
                "_GLOBAL",
                ""
            ),

        role=
            "global-neural",

        specialization=
            "representation",

        feature_view=
            "all_21",

        training_objective=
            "BCE",

        robustness_augmentation=
            "None",

        diversity_method=
            "Neural representation diversity",

        oof_available=
            True,

        metrics=
            pooled,

        status=
            "candidate",

        notes=
            "Strict neural 5-fold OOF.",
    )


    print(
        "\nPOOLED"
    )

    print(
        f"PR    : {pooled['PR_AUC']:.6f}"
    )

    print(
        f"ROC   : {pooled['ROC_AUC']:.6f}"
    )

    print(
        f"Brier : {pooled['Brier']:.6f}"
    )

    print(
        f"NLL   : {pooled['NLL']:.6f}"
    )

    print(
        f"ECE   : {pooled['ECE']:.6f}"
    )


# ============================================================
# N3A.16 NEURAL LEADERBOARD
# ============================================================

rows = []


for (
    model_id,
    result

) in RESULTS_N3A.items():

    m = result[
        "metrics"
    ]


    rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Seconds":
            result[
                "seconds"
            ],
    })


LEADERBOARD_N3A = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3A — NEURAL EXPERT LEADERBOARD"
)

print(
    "=" * 135
)


display(
    LEADERBOARD_N3A
)


# ============================================================
# N3A.17 COMBINED LEADERBOARD
# ============================================================

neural_tmp = (

    LEADERBOARD_N3A[
        [
            "Model_ID",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
        ]
    ]
    .copy()
)


neural_tmp[
    "Source"
] = "N3A"


GLOBAL_PLUS_NEURAL_N3 = (

    pd.concat(

        [
            ALL_GLOBAL_LEADERBOARD_N2,
            neural_tmp,
        ],

        ignore_index=True
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3A — COMBINED GLOBAL + NEURAL LEADERBOARD"
)

print(
    "=" * 135
)


display(
    GLOBAL_PLUS_NEURAL_N3
)


# ============================================================
# N3A.18 NEURAL CORRELATION WITH STRONGEST TREE EXPERTS
# ============================================================

COMPARE_IDS_N3 = [

    "XGB_SMOOTH",
    "XGB_SUBSPACE",
    "CAT_SMOOTH",

] + list(
    MODEL_FACTORIES_N3A.keys()
)


P_COMPARE_N3 = np.column_stack([

    OOF_STORE[
        model_id
    ]

    for model_id
    in COMPARE_IDS_N3
])


CORR_N3 = pd.DataFrame(

    np.corrcoef(
        P_COMPARE_N3.T
    ),

    index=
        COMPARE_IDS_N3,

    columns=
        COMPARE_IDS_N3,
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3A — TREE / NEURAL CORRELATION"
)

print(
    "=" * 135
)


display(
    CORR_N3.round(4)
)


# ============================================================
# N3A.19 PAIR EACH NEURAL MODEL WITH XGB_SMOOTH + CAT_SMOOTH
# ============================================================

ANCHORS_N3 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
]


pair_rows = []


for neural_id in (
    MODEL_FACTORIES_N3A.keys()
):

    for anchor_id in ANCHORS_N3:

        pa = OOF_STORE[
            anchor_id
        ]

        pn = OOF_STORE[
            neural_id
        ]


        pair = (
            pa
            +
            pn
        ) / 2.0


        m = probability_metrics(
            y_DEV,
            pair
        )


        pair_rows.append({

            "Anchor":
                anchor_id,

            "Neural":
                neural_id,

            "Correlation":
                np.corrcoef(
                    pa,
                    pn
                )[0, 1],

            "PR_AUC":
                m[
                    "PR_AUC"
                ],

            "ROC_AUC":
                m[
                    "ROC_AUC"
                ],

            "Brier":
                m[
                    "Brier"
                ],
        })


PAIR_NEURAL_N3 = (

    pd.DataFrame(
        pair_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3A — TREE + NEURAL PAIR SCREEN"
)

print(
    "=" * 135
)


display(
    PAIR_NEURAL_N3
)


# ============================================================
# N3A.20 THREE-WAY SCREEN:
#
# XGB_SMOOTH + CAT_SMOOTH + each neural expert
# ============================================================

triple_rows = []


p_xgb = OOF_STORE[
    "XGB_SMOOTH"
]


p_cat = OOF_STORE[
    "CAT_SMOOTH"
]


for neural_id in (
    MODEL_FACTORIES_N3A.keys()
):

    p_neural = OOF_STORE[
        neural_id
    ]


    p_three = (

        p_xgb
        +
        p_cat
        +
        p_neural

    ) / 3.0


    m = probability_metrics(

        y_DEV,
        p_three
    )


    triple_rows.append({

        "Neural":
            neural_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],
    })


TRIPLE_SCREEN_N3 = (

    pd.DataFrame(
        triple_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3A — XGB + CAT + NEURAL TRIPLE SCREEN"
)

print(
    "=" * 135
)


display(
    TRIPLE_SCREEN_N3
)


# ============================================================
# N3A.21 SAVE
# ============================================================

FOLD_RESULTS_N3A = pd.DataFrame(
    FOLD_ROWS_N3A
)


LEADERBOARD_N3A.to_csv(

    TAB_DIR
    /
    "n3a_neural_leaderboard.csv",

    index=False
)


FOLD_RESULTS_N3A.to_csv(

    TAB_DIR
    /
    "n3a_neural_fold_results.csv",

    index=False
)


GLOBAL_PLUS_NEURAL_N3.to_csv(

    TAB_DIR
    /
    "n3a_combined_leaderboard.csv",

    index=False
)


CORR_N3.to_csv(

    TAB_DIR
    /
    "n3a_tree_neural_correlation.csv"
)


PAIR_NEURAL_N3.to_csv(

    TAB_DIR
    /
    "n3a_tree_neural_pair_screen.csv",

    index=False
)


TRIPLE_SCREEN_N3.to_csv(

    TAB_DIR
    /
    "n3a_tree_neural_triple_screen.csv",

    index=False
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3A COMPLETE"
)

print(
    "=" * 135
)


print(
    "\nNeural experts trained:",
    len(
        RESULTS_N3A
    )
)


print(
    "HOLDOUT untouched."
)

print(
    "BRFSS-2020 untouched."
)


print(
    "\nNext:"
)

print(
    "N3B — objective-diverse neural specialists"
)

Device: cuda
GPU: Tesla T4
Categorical: 18
Continuous: 3
Cardinalities: [2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 6, 2, 2, 14, 7, 9]

TRAINING NEURAL EXPERT: MLP_GLOBAL
Fold 0: epoch=7 | PR=0.372619 | ROC=0.850271 | Brier=0.070128 | 32.3s
Fold 1: epoch=5 | PR=0.379780 | ROC=0.848180 | Brier=0.070023 | 18.3s
Fold 2: epoch=7 | PR=0.375171 | ROC=0.847417 | Brier=0.070140 | 22.0s
Fold 3: epoch=5 | PR=0.367884 | ROC=0.846524 | Brier=0.070425 | 18.2s
Fold 4: epoch=5 | PR=0.385121 | ROC=0.849821 | Brier=0.069747 | 18.5s

POOLED
PR    : 0.374895
ROC   : 0.848307
Brier : 0.070093
NLL   : 0.237097
ECE   : 0.004825

TRAINING NEURAL EXPERT: RESMLP_GLOBAL
Fold 0: epoch=3 | PR=0.365016 | ROC=0.846747 | Brier=0.071128 | 16.1s
Fold 1: epoch=2 | PR=0.374475 | ROC=0.844253 | Brier=0.070367 | 14.1s
Fold 2: epoch=4 | PR=0.368943 | ROC=0.841858 | Brier=0.070584 | 18.0s
Fold 3: epoch=3 | PR=0.363838 | ROC=0.844576 | Brier=0.070626 | 15.9s
Fold 4: epoch=3 | PR=0.377218 | ROC=0.844645 | Brier=0.070534 | 16.0s

POO

,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE,Seconds
0,MLP_GLOBAL,0.374895,0.848307,0.070093,0.237097,0.004825,110.891899
1,AUTOINT_GLOBAL,0.373935,0.848527,0.070054,0.236817,0.002104,332.552559
2,RESMLP_GLOBAL,0.367362,0.843931,0.070648,0.239585,0.006719,81.769809
3,DEEPFM_GLOBAL,0.314390,0.807193,0.082755,0.391601,0.065529,213.819754



MODULE N3A — COMBINED GLOBAL + NEURAL LEADERBOARD


,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE,Source
0,XGB_SMOOTH,0.378728,0.850209,0.069802,0.235856,0.002929,N2B
1,XGB_SUBSPACE,0.378562,0.850477,0.069792,0.235729,0.002664,N2B
2,XGB_POS15,0.378240,0.850049,0.071849,0.241930,0.030974,N2B
3,CAT_SMOOTH,0.378151,0.850386,0.069787,0.235746,0.002194,N2B
4,XGB_POS25,0.377895,0.850137,0.081764,0.269415,0.079265,N2B
5,XGB_GLOBAL,0.377869,0.850049,0.069840,0.235948,0.002161,N2A
6,LGBM_SMOOTH,0.377301,0.849855,0.069869,0.236057,0.002279,N2B
7,CAT_POS20,0.377064,0.849966,0.076024,0.253951,0.056123,N2B
8,HGB_SMOOTH,0.376940,0.849566,0.069901,0.236200,0.001944,N2B
9,CAT_GLOBAL,0.376270,0.850128,0.069877,0.235976,0.002203,N2A



MODULE N3A — TREE / NEURAL CORRELATION


...


MODULE N3A — TREE + NEURAL PAIR SCREEN


,Anchor,Neural,Correlation,PR_AUC,ROC_AUC,Brier
0,XGB_SMOOTH,MLP_GLOBAL,0.980386,0.379311,0.850296,0.069777
1,CAT_SMOOTH,MLP_GLOBAL,0.979755,0.379267,0.850337,0.069763
2,CAT_SMOOTH,AUTOINT_GLOBAL,0.982784,0.378759,0.850293,0.069785
3,XGB_SMOOTH,AUTOINT_GLOBAL,0.983870,0.378503,0.850241,0.069800
4,XGB_SMOOTH,RESMLP_GLOBAL,0.964652,0.377186,0.849090,0.069906
5,CAT_SMOOTH,RESMLP_GLOBAL,0.964843,0.377065,0.849145,0.069899
6,XGB_SMOOTH,DEEPFM_GLOBAL,0.737208,0.359779,0.846258,0.072984
7,CAT_SMOOTH,DEEPFM_GLOBAL,0.735216,0.359437,0.846389,0.072959



MODULE N3A — XGB + CAT + NEURAL TRIPLE SCREEN


,Neural,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,MLP_GLOBAL,0.379960,0.850669,0.069718,0.235577,0.002478
1,AUTOINT_GLOBAL,0.379500,0.850586,0.069743,0.235604,0.002489
2,RESMLP_GLOBAL,0.378958,0.850100,0.069775,0.235859,0.002172
3,DEEPFM_GLOBAL,0.368666,0.848460,0.071157,0.240566,0.019857



MODULE N3A COMPLETE

Neural experts trained: 4
HOLDOUT untouched.
BRFSS-2020 untouched.

Next:
N3B — objective-diverse neural specialists


In [6]:
# ============================================================
# MODULE N3R — LEAKAGE-SAFE NEURAL OOF RECONSTRUCTION
#
# PURPOSE
# ------------------------------------------------------------
# Repair the early-stopping optimism in N3A.
#
# For every OUTER fold:
#
#   OUTER TRAIN
#       |
#       +---- INNER TRAIN
#       |
#       +---- INNER VALID
#
# Inner validation chooses the epoch.
#
# Then:
#
#   fresh model
#   + full OUTER TRAIN
#   + fixed chosen epoch
#   -> untouched OUTER VALID prediction
#
#
# Models repaired:
#   - MLP_GLOBAL
#   - AUTOINT_GLOBAL
#   - RESMLP_GLOBAL
#
# Current DeepFM is excluded because its standalone
# performance is far too weak. A repaired DeepFM-v2 can
# be introduced later as a new candidate.
#
# HOLDOUT remains untouched.
# ============================================================


import gc
import time
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ============================================================
# N3R.1 CONFIGURATION
# ============================================================

INNER_VALID_FRACTION_N3R = 0.15

MAX_EPOCHS_N3R = 35

PATIENCE_N3R = 5

BATCH_SIZE_N3R = 2048

LR_N3R = 1e-3

WEIGHT_DECAY_N3R = 1e-4


# Only repair models worth keeping alive
MODEL_FACTORIES_N3R = {

    "MLP_REPAIRED": (
        MODEL_FACTORIES_N3A[
            "MLP_GLOBAL"
        ]
    ),

    "AUTOINT_REPAIRED": (
        MODEL_FACTORIES_N3A[
            "AUTOINT_GLOBAL"
        ]
    ),

    "RESMLP_REPAIRED": (
        MODEL_FACTORIES_N3A[
            "RESMLP_GLOBAL"
        ]
    ),
}


# ============================================================
# N3R.2 PREPARE TRAIN / VALID WITH TRAIN-ONLY SCALER
# ============================================================

def prepare_arrays_n3r(
    fit_idx,
    eval_idx,
):

    X_fit = X_DEV.iloc[
        fit_idx
    ].copy()

    X_eval = X_DEV.iloc[
        eval_idx
    ].copy()


    scaler = StandardScaler()


    fit_num = scaler.fit_transform(

        X_fit[
            NUM_COLS_N3
        ]

    ).astype(
        np.float32
    )


    eval_num = scaler.transform(

        X_eval[
            NUM_COLS_N3
        ]

    ).astype(
        np.float32
    )


    fit_cat = (

        X_fit[
            CAT_COLS_N3
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    eval_cat = (

        X_eval[
            CAT_COLS_N3
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    fit_y = y_DEV[
        fit_idx
    ].astype(
        np.float32
    )


    eval_y = y_DEV[
        eval_idx
    ].astype(
        np.float32
    )


    return (
        fit_cat,
        fit_num,
        fit_y,
        eval_cat,
        eval_num,
        eval_y,
        scaler,
    )


# ============================================================
# N3R.3 LOADER HELPER
# ============================================================

def make_loader_n3r(
    x_cat,
    x_num,
    y=None,
    batch_size=2048,
    shuffle=False,
):

    ds = TabularDatasetN3(
        x_cat,
        x_num,
        y
    )


    return DataLoader(

        ds,

        batch_size=batch_size,

        shuffle=shuffle,

        num_workers=2,

        pin_memory=True,

        drop_last=False,
    )


# ============================================================
# N3R.4 TRAIN ONE EPOCH
# ============================================================

def train_one_epoch_n3r(
    model,
    loader,
    optimizer,
    criterion,
    scaler_amp,
):

    model.train()


    total_loss = 0.0

    n_seen = 0


    use_amp = (
        DEVICE.type
        ==
        "cuda"
    )


    for (
        x_cat,
        x_num,
        y_batch

    ) in loader:

        x_cat = x_cat.to(
            DEVICE,
            non_blocking=True
        )

        x_num = x_num.to(
            DEVICE,
            non_blocking=True
        )

        y_batch = y_batch.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(

            device_type=
                DEVICE.type,

            enabled=use_amp
        ):

            logits = model(
                x_cat,
                x_num
            )


            loss = criterion(
                logits,
                y_batch
            )


        scaler_amp.scale(
            loss
        ).backward()


        scaler_amp.step(
            optimizer
        )


        scaler_amp.update()


        total_loss += (
            loss.item()
            *
            len(y_batch)
        )


        n_seen += len(
            y_batch
        )


    return (
        total_loss
        /
        n_seen
    )


# ============================================================
# N3R.5 INNER EPOCH SELECTION
# ============================================================

def select_epoch_n3r(
    model_factory,
    outer_train_idx,
    seed,
):

    # --------------------------------------------------------
    # Split ONLY the outer-training observations
    # --------------------------------------------------------

    inner_train_idx, inner_valid_idx = (
        train_test_split(

            outer_train_idx,

            test_size=
                INNER_VALID_FRACTION_N3R,

            random_state=
                seed,

            stratify=
                y_DEV[
                    outer_train_idx
                ],
        )
    )


    (
        tr_cat,
        tr_num,
        tr_y,

        va_cat,
        va_num,
        va_y,

        scaler,

    ) = prepare_arrays_n3r(

        inner_train_idx,
        inner_valid_idx
    )


    train_loader = make_loader_n3r(

        tr_cat,
        tr_num,
        tr_y,

        batch_size=
            BATCH_SIZE_N3R,

        shuffle=True,
    )


    valid_loader = make_loader_n3r(

        va_cat,
        va_num,
        va_y,

        batch_size=
            BATCH_SIZE_N3R * 2,

        shuffle=False,
    )


    seed_everything_n3(
        seed
    )


    model = model_factory().to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LR_N3R,

        weight_decay=
            WEIGHT_DECAY_N3R,
    )


    criterion = (
        nn.BCEWithLogitsLoss()
    )


    scaler_amp = torch.amp.GradScaler(

        "cuda",

        enabled=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    best_pr = -np.inf

    best_epoch = 1

    patience = 0

    history = []


    for epoch in range(
        1,
        MAX_EPOCHS_N3R + 1
    ):

        train_loss = train_one_epoch_n3r(

            model,
            train_loader,
            optimizer,
            criterion,
            scaler_amp
        )


        valid_pred = predict_neural_n3(

            model,
            valid_loader
        )


        valid_pr = average_precision_score(

            va_y,
            valid_pred
        )


        valid_roc = roc_auc_score(

            va_y,
            valid_pred
        )


        history.append({

            "Epoch":
                epoch,

            "Train_Loss":
                train_loss,

            "Inner_PR":
                valid_pr,

            "Inner_ROC":
                valid_roc,
        })


        if valid_pr > (
            best_pr
            +
            1e-5
        ):

            best_pr = valid_pr

            best_epoch = epoch

            patience = 0


        else:

            patience += 1


        if patience >= (
            PATIENCE_N3R
        ):

            break


    del model

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return {

        "best_epoch":
            int(
                best_epoch
            ),

        "best_inner_pr":
            float(
                best_pr
            ),

        "history":
            pd.DataFrame(
                history
            ),
    }


# ============================================================
# N3R.6 RETRAIN FULL OUTER-TRAIN FOR FIXED EPOCH COUNT
# ============================================================

def retrain_outer_n3r(
    model_factory,
    outer_train_idx,
    outer_valid_idx,
    n_epochs,
    seed,
):

    (
        tr_cat,
        tr_num,
        tr_y,

        va_cat,
        va_num,
        va_y,

        scaler,

    ) = prepare_arrays_n3r(

        outer_train_idx,
        outer_valid_idx
    )


    train_loader = make_loader_n3r(

        tr_cat,
        tr_num,
        tr_y,

        batch_size=
            BATCH_SIZE_N3R,

        shuffle=True,
    )


    valid_loader = make_loader_n3r(

        va_cat,
        va_num,
        va_y,

        batch_size=
            BATCH_SIZE_N3R * 2,

        shuffle=False,
    )


    seed_everything_n3(
        seed
    )


    model = model_factory().to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LR_N3R,

        weight_decay=
            WEIGHT_DECAY_N3R,
    )


    criterion = (
        nn.BCEWithLogitsLoss()
    )


    scaler_amp = torch.amp.GradScaler(

        "cuda",

        enabled=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    for epoch in range(
        1,
        n_epochs + 1
    ):

        train_one_epoch_n3r(

            model,
            train_loader,
            optimizer,
            criterion,
            scaler_amp
        )


    outer_pred = predict_neural_n3(

        model,
        valid_loader
    )


    return (

        outer_pred,
        model,
        scaler
    )


# ============================================================
# N3R.7 STRICT OUTER OOF
# ============================================================

RESULTS_N3R = {}

FOLD_ROWS_N3R = []


for (
    model_id,
    factory

) in MODEL_FACTORIES_N3R.items():

    print(
        "\n"
        + "=" * 125
    )

    print(
        "REPAIRING:",
        model_id
    )

    print(
        "=" * 125
    )


    oof = np.full(

        len(
            y_DEV
        ),

        np.nan,

        dtype=np.float32
    )


    total_start = time.perf_counter()


    selected_epochs = []


    for fold in range(
        N_FOLDS
    ):

        outer_train_idx = np.where(

            DEV_FOLD != fold

        )[0]


        outer_valid_idx = np.where(

            DEV_FOLD == fold

        )[0]


        fold_seed = (
            SEED
            +
            5000
            +
            fold
        )


        # ----------------------------------------------------
        # STEP 1:
        # choose epoch using INNER validation only
        # ----------------------------------------------------

        selection = select_epoch_n3r(

            factory,

            outer_train_idx,

            seed=
                fold_seed
        )


        chosen_epoch = selection[
            "best_epoch"
        ]


        selected_epochs.append(
            chosen_epoch
        )


        # ----------------------------------------------------
        # STEP 2:
        # fresh model trained on FULL outer train
        # ----------------------------------------------------

        t0 = time.perf_counter()


        (
            pred,
            model,
            scaler

        ) = retrain_outer_n3r(

            factory,

            outer_train_idx,

            outer_valid_idx,

            n_epochs=
                chosen_epoch,

            seed=
                fold_seed + 100
        )


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        oof[
            outer_valid_idx
        ] = pred


        metrics = probability_metrics(

            y_DEV[
                outer_valid_idx
            ],

            pred
        )


        FOLD_ROWS_N3R.append({

            "Model_ID":
                model_id,

            "Fold":
                fold,

            "Selected_Epoch":
                chosen_epoch,

            "Inner_Best_PR":
                selection[
                    "best_inner_pr"
                ],

            "Outer_PR":
                metrics[
                    "PR_AUC"
                ],

            "Outer_ROC":
                metrics[
                    "ROC_AUC"
                ],

            "Outer_Brier":
                metrics[
                    "Brier"
                ],

            "Outer_NLL":
                metrics[
                    "NLL"
                ],

            "Outer_ECE":
                metrics[
                    "ECE"
                ],

            "Retrain_Seconds":
                elapsed,
        })


        print(

            f"Fold {fold} | "
            f"inner epoch={chosen_epoch:2d} | "
            f"inner PR={selection['best_inner_pr']:.6f} | "
            f"OUTER PR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f}"
        )


        del model
        del scaler

        gc.collect()


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    if np.isnan(
        oof
    ).any():

        raise RuntimeError(
            f"{model_id}: incomplete OOF"
        )


    pooled = probability_metrics(

        y_DEV,
        oof
    )


    total_seconds = (

        time.perf_counter()
        -
        total_start
    )


    RESULTS_N3R[
        model_id
    ] = {

        "oof":
            oof,

        "metrics":
            pooled,

        "epochs":
            selected_epochs,

        "seconds":
            total_seconds,
    }


    store_oof(
        model_id,
        oof
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        oof
    )


    register_experiment(

        model_id=
            model_id,

        family=
            model_id.replace(
                "_REPAIRED",
                ""
            ),

        role=
            "global-neural",

        specialization=
            "representation",

        feature_view=
            "all_21",

        training_objective=
            "BCE",

        robustness_augmentation=
            "None",

        diversity_method=
            "Neural representation diversity",

        oof_available=
            True,

        metrics=
            pooled,

        status=
            "candidate-clean",

        notes=
            (
                "Leakage-safe nested epoch selection; "
                "strict outer-fold OOF."
            ),
    )


    print(
        "\nPOOLED STRICT OOF"
    )

    print(
        f"PR-AUC  : {pooled['PR_AUC']:.6f}"
    )

    print(
        f"ROC-AUC : {pooled['ROC_AUC']:.6f}"
    )

    print(
        f"Brier   : {pooled['Brier']:.6f}"
    )

    print(
        f"NLL     : {pooled['NLL']:.6f}"
    )

    print(
        f"ECE     : {pooled['ECE']:.6f}"
    )

    print(
        "Epochs  :",
        selected_epochs
    )


# ============================================================
# N3R.8 LEADERBOARD
# ============================================================

rows = []


for model_id, result in (
    RESULTS_N3R.items()
):

    m = result[
        "metrics"
    ]


    rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Median_Epoch":
            float(
                np.median(
                    result[
                        "epochs"
                    ]
                )
            ),

        "Seconds":
            result[
                "seconds"
            ],
    })


LEADERBOARD_N3R = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3R — REPAIRED NEURAL LEADERBOARD"
)

print(
    "=" * 135
)


display(
    LEADERBOARD_N3R
)


# ============================================================
# N3R.9 ORIGINAL VS REPAIRED
# ============================================================

comparison_rows = []


mapping_n3r = {

    "MLP_GLOBAL":
        "MLP_REPAIRED",

    "AUTOINT_GLOBAL":
        "AUTOINT_REPAIRED",

    "RESMLP_GLOBAL":
        "RESMLP_REPAIRED",
}


for old_id, new_id in (
    mapping_n3r.items()
):

    old_pr = float(

        LEADERBOARD_N3A.loc[

            LEADERBOARD_N3A[
                "Model_ID"
            ]
            ==
            old_id,

            "PR_AUC"

        ].iloc[0]
    )


    new_pr = float(

        LEADERBOARD_N3R.loc[

            LEADERBOARD_N3R[
                "Model_ID"
            ]
            ==
            new_id,

            "PR_AUC"

        ].iloc[0]
    )


    comparison_rows.append({

        "Original":
            old_id,

        "Original_PR":
            old_pr,

        "Repaired":
            new_id,

        "Repaired_PR":
            new_pr,

        "Delta":
            new_pr
            -
            old_pr,
    })


REPAIR_COMPARISON_N3R = pd.DataFrame(
    comparison_rows
)


print(
    "\n"
    + "=" * 135
)

print(
    "ORIGINAL VS LEAKAGE-SAFE NEURAL OOF"
)

print(
    "=" * 135
)


display(
    REPAIR_COMPARISON_N3R
)


# ============================================================
# N3R.10 CORRELATION WITH TREE ANCHORS
# ============================================================

COMPARE_IDS_N3R = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "XGB_SUBSPACE",

    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "RESMLP_REPAIRED",
]


P_COMPARE_N3R = np.column_stack([

    OOF_STORE[
        mid
    ]

    for mid
    in COMPARE_IDS_N3R
])


CORR_N3R = pd.DataFrame(

    np.corrcoef(
        P_COMPARE_N3R.T
    ),

    index=
        COMPARE_IDS_N3R,

    columns=
        COMPARE_IDS_N3R,
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3R — CLEAN TREE / NEURAL CORRELATION"
)

print(
    "=" * 135
)


display(
    CORR_N3R.round(4)
)


# ============================================================
# N3R.11 CLEAN TREE + NEURAL ENSEMBLE SCREEN
# ============================================================

screen_rows = []


p_xgb = OOF_STORE[
    "XGB_SMOOTH"
]


p_cat = OOF_STORE[
    "CAT_SMOOTH"
]


for neural_id in [

    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "RESMLP_REPAIRED",

]:

    p_n = OOF_STORE[
        neural_id
    ]


    # two-model with XGB
    p_xgb_n = (
        p_xgb
        +
        p_n
    ) / 2


    mx = probability_metrics(

        y_DEV,
        p_xgb_n
    )


    # two-model with Cat
    p_cat_n = (
        p_cat
        +
        p_n
    ) / 2


    mc = probability_metrics(

        y_DEV,
        p_cat_n
    )


    # three-model
    p_three = (

        p_xgb
        +
        p_cat
        +
        p_n

    ) / 3


    mt = probability_metrics(

        y_DEV,
        p_three
    )


    screen_rows.extend([

        {
            "Combination":
                f"XGB_SMOOTH + {neural_id}",

            "PR_AUC":
                mx[
                    "PR_AUC"
                ],

            "ROC_AUC":
                mx[
                    "ROC_AUC"
                ],

            "Brier":
                mx[
                    "Brier"
                ],
        },

        {
            "Combination":
                f"CAT_SMOOTH + {neural_id}",

            "PR_AUC":
                mc[
                    "PR_AUC"
                ],

            "ROC_AUC":
                mc[
                    "ROC_AUC"
                ],

            "Brier":
                mc[
                    "Brier"
                ],
        },

        {
            "Combination":
                (
                    "XGB_SMOOTH + CAT_SMOOTH + "
                    f"{neural_id}"
                ),

            "PR_AUC":
                mt[
                    "PR_AUC"
                ],

            "ROC_AUC":
                mt[
                    "ROC_AUC"
                ],

            "Brier":
                mt[
                    "Brier"
                ],
        },
    ])


ENSEMBLE_SCREEN_N3R = (

    pd.DataFrame(
        screen_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3R — CLEAN ENSEMBLE SCREEN"
)

print(
    "=" * 135
)


display(
    ENSEMBLE_SCREEN_N3R
)


# ============================================================
# N3R.12 COMPARE TO OLD CORE5
# ============================================================

OLD_CORE5_PR = 0.379751


ENSEMBLE_SCREEN_N3R[
    "Delta_vs_Old_Core5"
] = (

    ENSEMBLE_SCREEN_N3R[
        "PR_AUC"
    ]

    -
    OLD_CORE5_PR
)


print(
    "\n"
    + "=" * 135
)

print(
    "CLEAN ENSEMBLES VS OLD CORE5"
)

print(
    "=" * 135
)


display(

    ENSEMBLE_SCREEN_N3R[
        [
            "Combination",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "Delta_vs_Old_Core5",
        ]
    ]
)


# ============================================================
# N3R.13 SAVE
# ============================================================

FOLD_RESULTS_N3R = pd.DataFrame(
    FOLD_ROWS_N3R
)


LEADERBOARD_N3R.to_csv(

    TAB_DIR
    /
    "n3r_repaired_neural_leaderboard.csv",

    index=False
)


FOLD_RESULTS_N3R.to_csv(

    TAB_DIR
    /
    "n3r_repaired_neural_fold_results.csv",

    index=False
)


REPAIR_COMPARISON_N3R.to_csv(

    TAB_DIR
    /
    "n3r_original_vs_repaired.csv",

    index=False
)


CORR_N3R.to_csv(

    TAB_DIR
    /
    "n3r_clean_tree_neural_correlation.csv"
)


ENSEMBLE_SCREEN_N3R.to_csv(

    TAB_DIR
    /
    "n3r_clean_ensemble_screen.csv",

    index=False
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


print(
    "\n"
    + "=" * 135
)

print(
    "MODULE N3R COMPLETE"
)

print(
    "=" * 135
)


print(
    "Neural OOF estimates are now leakage-safe."
)

print(
    "HOLDOUT remains untouched."
)

print(
    "BRFSS-2020 remains untouched."
)


REPAIRING: MLP_REPAIRED
Fold 0 | inner epoch=11 | inner PR=0.378553 | OUTER PR=0.371425 | ROC=0.848318
Fold 1 | inner epoch= 6 | inner PR=0.370266 | OUTER PR=0.376380 | ROC=0.847909
Fold 2 | inner epoch= 4 | inner PR=0.385124 | OUTER PR=0.371248 | ROC=0.846791
Fold 3 | inner epoch=10 | inner PR=0.398462 | OUTER PR=0.371273 | ROC=0.847807
Fold 4 | inner epoch= 8 | inner PR=0.380671 | OUTER PR=0.382233 | ROC=0.848725

POOLED STRICT OOF
PR-AUC  : 0.373965
ROC-AUC : 0.847718
Brier   : 0.070160
NLL     : 0.237351
ECE     : 0.004927
Epochs  : [11, 6, 4, 10, 8]

REPAIRING: AUTOINT_REPAIRED
Fold 0 | inner epoch=34 | inner PR=0.380107 | OUTER PR=0.374864 | ROC=0.850900
Fold 1 | inner epoch=31 | inner PR=0.370287 | OUTER PR=0.378408 | ROC=0.849152
Fold 2 | inner epoch=29 | inner PR=0.387613 | OUTER PR=0.375204 | ROC=0.847823
Fold 3 | inner epoch=16 | inner PR=0.394299 | OUTER PR=0.366695 | ROC=0.847764
Fold 4 | inner epoch=34 | inner PR=0.382909 | OUTER PR=0.385849 | ROC=0.851050

POOLED STRICT

,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE,Median_Epoch,Seconds
0,AUTOINT_REPAIRED,0.375838,0.849122,0.069971,0.236481,0.002132,31.0,673.483756
1,MLP_REPAIRED,0.373965,0.847718,0.070160,0.237351,0.004927,8.0,157.517470
2,RESMLP_REPAIRED,0.362738,0.842308,0.070819,0.240645,0.006515,4.0,120.956312



ORIGINAL VS LEAKAGE-SAFE NEURAL OOF


,Original,Original_PR,Repaired,Repaired_PR,Delta
0,MLP_GLOBAL,0.374895,MLP_REPAIRED,0.373965,-0.000930
1,AUTOINT_GLOBAL,0.373935,AUTOINT_REPAIRED,0.375838,0.001903
2,RESMLP_GLOBAL,0.367362,RESMLP_REPAIRED,0.362738,-0.004624



MODULE N3R — CLEAN TREE / NEURAL CORRELATION


...


MODULE N3R — CLEAN ENSEMBLE SCREEN


,Combination,PR_AUC,ROC_AUC,Brier
0,XGB_SMOOTH + CAT_SMOOTH + MLP_REPAIRED,0.379708,0.850546,0.069736
1,XGB_SMOOTH + CAT_SMOOTH + AUTOINT_REPAIRED,0.379658,0.850566,0.069742
2,CAT_SMOOTH + AUTOINT_REPAIRED,0.379120,0.850356,0.069772
3,XGB_SMOOTH + MLP_REPAIRED,0.378906,0.850089,0.069801
4,XGB_SMOOTH + AUTOINT_REPAIRED,0.378901,0.850275,0.069791
5,CAT_SMOOTH + MLP_REPAIRED,0.378825,0.850117,0.069795
6,XGB_SMOOTH + CAT_SMOOTH + RESMLP_REPAIRED,0.378346,0.849841,0.069815
7,XGB_SMOOTH + RESMLP_REPAIRED,0.375898,0.848618,0.069971
8,CAT_SMOOTH + RESMLP_REPAIRED,0.375682,0.848693,0.069968



CLEAN ENSEMBLES VS OLD CORE5


,Combination,PR_AUC,ROC_AUC,Brier,Delta_vs_Old_Core5
0,XGB_SMOOTH + CAT_SMOOTH + MLP_REPAIRED,0.379708,0.850546,0.069736,-0.000043
1,XGB_SMOOTH + CAT_SMOOTH + AUTOINT_REPAIRED,0.379658,0.850566,0.069742,-0.000093
2,CAT_SMOOTH + AUTOINT_REPAIRED,0.379120,0.850356,0.069772,-0.000631
3,XGB_SMOOTH + MLP_REPAIRED,0.378906,0.850089,0.069801,-0.000845
4,XGB_SMOOTH + AUTOINT_REPAIRED,0.378901,0.850275,0.069791,-0.000850
5,CAT_SMOOTH + MLP_REPAIRED,0.378825,0.850117,0.069795,-0.000926
6,XGB_SMOOTH + CAT_SMOOTH + RESMLP_REPAIRED,0.378346,0.849841,0.069815,-0.001405
7,XGB_SMOOTH + RESMLP_REPAIRED,0.375898,0.848618,0.069971,-0.003853
8,CAT_SMOOTH + RESMLP_REPAIRED,0.375682,0.848693,0.069968,-0.004069



MODULE N3R COMPLETE
Neural OOF estimates are now leakage-safe.
HOLDOUT remains untouched.
BRFSS-2020 remains untouched.


In [7]:
# ============================================================
# MODULE N3B — OBJECTIVE-DIVERSE NEURAL SPECIALISTS
#
# PURPOSE
# ------------------------------------------------------------
# Generate deliberately different neural experts through:
#
#   1. Focal loss
#   2. Asymmetric / FN-sensitive loss
#   3. Feature-token dropout robustness training
#
# Architectures:
#   - MLP
#   - AutoInt
#
# All evaluation uses leakage-safe nested epoch selection:
#
#   OUTER TRAIN
#       -> inner train / inner validation
#       -> choose epoch
#       -> fresh model on full outer train
#       -> untouched outer-valid prediction
#
# IMPORTANT
# ------------------------------------------------------------
# - DEV only
# - HOLDOUT untouched
# - BRFSS-2020 untouched
# - Hard-case weighting deferred to N4 because it requires
#   additional cross-fitting.
# ============================================================


import gc
import time
import copy
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader


# ============================================================
# N3B.1 CONFIGURATION
# ============================================================

MAX_EPOCHS_N3B = 20

PATIENCE_N3B = 5

BATCH_SIZE_N3B = 2048

LR_N3B = 1e-3

WEIGHT_DECAY_N3B = 1e-4

INNER_VALID_FRACTION_N3B = 0.15


# ============================================================
# N3B.2 LOSS FUNCTIONS
# ============================================================

class BinaryFocalLossN3B(nn.Module):
    """
    Standard binary focal loss.

    gamma > 0:
        emphasizes difficult examples.

    alpha:
        optional positive-class weighting.
    """

    def __init__(
        self,
        gamma=2.0,
        alpha=None,
    ):
        super().__init__()

        self.gamma = gamma
        self.alpha = alpha


    def forward(
        self,
        logits,
        targets,
    ):

        targets = targets.float()

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        prob = torch.sigmoid(
            logits
        )

        p_t = (
            targets * prob
            +
            (1.0 - targets)
            *
            (1.0 - prob)
        )

        focal_factor = (
            1.0 - p_t
        ) ** self.gamma


        loss = (
            focal_factor
            *
            bce
        )


        if self.alpha is not None:

            alpha_t = (
                targets
                *
                self.alpha

                +

                (1.0 - targets)
                *
                (1.0 - self.alpha)
            )

            loss = (
                alpha_t
                *
                loss
            )


        return loss.mean()


class AsymmetricBinaryLossN3B(nn.Module):
    """
    Positive-sensitive asymmetric BCE.

    positive_weight > 1 places greater emphasis on
    positive observations and therefore encourages
    FN-sensitive behavior.
    """

    def __init__(
        self,
        positive_weight=1.75,
    ):
        super().__init__()

        self.positive_weight = float(
            positive_weight
        )


    def forward(
        self,
        logits,
        targets,
    ):

        targets = targets.float()

        loss_positive = (
            -self.positive_weight
            *
            targets
            *
            nn.functional.logsigmoid(
                logits
            )
        )

        loss_negative = (
            -(1.0 - targets)
            *
            nn.functional.logsigmoid(
                -logits
            )
        )

        return (
            loss_positive
            +
            loss_negative
        ).mean()


# ============================================================
# N3B.3 ROBUST MLP
#
# Feature-token dropout:
#
# During training:
#
# categorical embedding vectors are dropped as WHOLE tokens,
# rather than replacing category values by 0.
#
# numerical features are independently zeroed after
# standardization.
#
# This avoids treating "0 category" as missing.
# ============================================================

class RobustMLP_N3B(nn.Module):

    def __init__(
        self,
        cardinalities,
        n_num,
        emb_dim=8,
        hidden=(256, 128, 64),
        dropout=0.20,
        feature_drop=0.10,
    ):

        super().__init__()


        self.embed = EmbeddingBlockN3(
            cardinalities,
            emb_dim
        )


        self.feature_drop = (
            feature_drop
        )


        input_dim = (
            len(cardinalities)
            *
            emb_dim
            +
            n_num
        )


        layers = []

        previous = input_dim


        for h in hidden:

            layers.extend([

                nn.Linear(
                    previous,
                    h
                ),

                nn.BatchNorm1d(
                    h
                ),

                nn.ReLU(),

                nn.Dropout(
                    dropout
                ),
            ])

            previous = h


        self.mlp = nn.Sequential(
            *layers
        )


        self.head = nn.Linear(
            previous,
            1
        )


    def forward(
        self,
        x_cat,
        x_num
    ):

        emb = self.embed(
            x_cat
        )


        if self.training:

            # ----------------------------------------------
            # Drop whole categorical feature embeddings
            # ----------------------------------------------

            cat_keep = (

                torch.rand(

                    emb.shape[0],
                    emb.shape[1],
                    1,

                    device=
                        emb.device
                )

                >
                self.feature_drop

            ).float()


            emb = (
                emb
                *
                cat_keep
            )


            # ----------------------------------------------
            # Drop numerical features
            # ----------------------------------------------

            num_keep = (

                torch.rand_like(
                    x_num
                )

                >
                self.feature_drop

            ).float()


            x_num = (
                x_num
                *
                num_keep
            )


        flat = emb.flatten(
            1
        )


        x = torch.cat(

            [
                flat,
                x_num
            ],

            dim=1
        )


        h = self.mlp(
            x
        )


        return self.head(
            h
        ).squeeze(1)


# ============================================================
# N3B.4 ROBUST AUTOINTERACTION
# ============================================================

class RobustAutoInt_N3B(nn.Module):

    def __init__(
        self,
        cardinalities,
        n_num,
        emb_dim=16,
        n_heads=4,
        n_layers=2,
        dropout=0.15,
        feature_drop=0.10,
    ):

        super().__init__()


        self.embed = EmbeddingBlockN3(
            cardinalities,
            emb_dim
        )


        self.feature_drop = (
            feature_drop
        )


        self.num_weight = nn.Parameter(

            torch.randn(
                n_num,
                emb_dim
            )
            *
            0.02
        )


        self.num_bias = nn.Parameter(

            torch.zeros(
                n_num,
                emb_dim
            )
        )


        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=
                    emb_dim,

                nhead=
                    n_heads,

                dim_feedforward=
                    emb_dim * 4,

                dropout=
                    dropout,

                activation=
                    "gelu",

                batch_first=
                    True,

                norm_first=
                    True,
            )
        )


        self.encoder = (
            nn.TransformerEncoder(

                encoder_layer,

                num_layers=
                    n_layers
            )
        )


        self.head = nn.Sequential(

            nn.LayerNorm(
                emb_dim
            ),

            nn.Linear(
                emb_dim,
                64
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                1
            ),
        )


    def forward(
        self,
        x_cat,
        x_num
    ):

        cat_tokens = self.embed(
            x_cat
        )


        num_tokens = (

            x_num.unsqueeze(
                -1
            )

            *
            self.num_weight.unsqueeze(
                0
            )

            +
            self.num_bias.unsqueeze(
                0
            )
        )


        tokens = torch.cat(

            [
                cat_tokens,
                num_tokens
            ],

            dim=1
        )


        if self.training:

            keep = (

                torch.rand(

                    tokens.shape[0],
                    tokens.shape[1],
                    1,

                    device=
                        tokens.device
                )

                >
                self.feature_drop

            ).float()


            tokens = (
                tokens
                *
                keep
            )


        encoded = self.encoder(
            tokens
        )


        pooled = encoded.mean(
            dim=1
        )


        return self.head(
            pooled
        ).squeeze(1)


# ============================================================
# N3B.5 SPECIALIST CONFIGURATION
# ============================================================

SPECIALISTS_N3B = {


    # --------------------------------------------------------
    # MLP — focal
    # --------------------------------------------------------

    "MLP_FOCAL": {

        "model_factory":
            lambda: TabMLP_N3(

                CAT_CARDINALITIES_N3,

                len(
                    NUM_COLS_N3
                ),

                emb_dim=8,

                hidden=(
                    256,
                    128,
                    64
                ),

                dropout=0.20,
            ),

        "loss_factory":
            lambda: BinaryFocalLossN3B(

                gamma=2.0,

                alpha=None
            ),

        "specialization":
            "hard-example/focal",

        "diversity":
            "objective diversity",
    },


    # --------------------------------------------------------
    # MLP — positive / FN-sensitive
    # --------------------------------------------------------

    "MLP_ASYM175": {

        "model_factory":
            lambda: TabMLP_N3(

                CAT_CARDINALITIES_N3,

                len(
                    NUM_COLS_N3
                ),

                emb_dim=8,

                hidden=(
                    256,
                    128,
                    64
                ),

                dropout=0.20,
            ),

        "loss_factory":
            lambda: AsymmetricBinaryLossN3B(

                positive_weight=1.75
            ),

        "specialization":
            "FN-sensitive",

        "diversity":
            "asymmetric class-cost objective",
    },


    # --------------------------------------------------------
    # MLP — robust feature dropout
    # --------------------------------------------------------

    "MLP_FEATDROP10": {

        "model_factory":
            lambda: RobustMLP_N3B(

                CAT_CARDINALITIES_N3,

                len(
                    NUM_COLS_N3
                ),

                emb_dim=8,

                hidden=(
                    256,
                    128,
                    64
                ),

                dropout=0.20,

                feature_drop=0.10,
            ),

        "loss_factory":
            lambda: nn.BCEWithLogitsLoss(),

        "specialization":
            "feature-robust",

        "diversity":
            "feature-token dropout",
    },


    # --------------------------------------------------------
    # AutoInt — focal
    # --------------------------------------------------------

    "AUTOINT_FOCAL": {

        "model_factory":
            lambda: AutoInt_N3(

                CAT_CARDINALITIES_N3,

                len(
                    NUM_COLS_N3
                ),

                emb_dim=16,

                n_heads=4,

                n_layers=2,

                dropout=0.15,
            ),

        "loss_factory":
            lambda: BinaryFocalLossN3B(

                gamma=2.0,

                alpha=None
            ),

        "specialization":
            "hard-example/focal",

        "diversity":
            "attention + focal objective",
    },


    # --------------------------------------------------------
    # AutoInt — FN-sensitive
    # --------------------------------------------------------

    "AUTOINT_ASYM175": {

        "model_factory":
            lambda: AutoInt_N3(

                CAT_CARDINALITIES_N3,

                len(
                    NUM_COLS_N3
                ),

                emb_dim=16,

                n_heads=4,

                n_layers=2,

                dropout=0.15,
            ),

        "loss_factory":
            lambda: AsymmetricBinaryLossN3B(

                positive_weight=1.75
            ),

        "specialization":
            "FN-sensitive",

        "diversity":
            "attention + asymmetric objective",
    },


    # --------------------------------------------------------
    # AutoInt — robust token dropout
    # --------------------------------------------------------

    "AUTOINT_FEATDROP10": {

        "model_factory":
            lambda: RobustAutoInt_N3B(

                CAT_CARDINALITIES_N3,

                len(
                    NUM_COLS_N3
                ),

                emb_dim=16,

                n_heads=4,

                n_layers=2,

                dropout=0.15,

                feature_drop=0.10,
            ),

        "loss_factory":
            lambda: nn.BCEWithLogitsLoss(),

        "specialization":
            "feature-robust",

        "diversity":
            "attention token dropout",
    },
}


print(
    "=" * 135
)

print(
    "MODULE N3B — SPECIALIST CANDIDATES"
)

print(
    "=" * 135
)


for model_id, config in (
    SPECIALISTS_N3B.items()
):

    print(

        f"{model_id:22s} | "
        f"{config['specialization']:22s} | "
        f"{config['diversity']}"
    )


# ============================================================
# N3B.6 GENERIC TRAIN EPOCH
# ============================================================

def train_epoch_n3b(
    model,
    loader,
    optimizer,
    criterion,
    scaler_amp,
):

    model.train()


    total_loss = 0.0

    n_seen = 0


    use_amp = (
        DEVICE.type
        ==
        "cuda"
    )


    for (
        x_cat,
        x_num,
        y_batch

    ) in loader:

        x_cat = x_cat.to(
            DEVICE,
            non_blocking=True
        )

        x_num = x_num.to(
            DEVICE,
            non_blocking=True
        )

        y_batch = y_batch.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(

            device_type=
                DEVICE.type,

            enabled=
                use_amp
        ):

            logits = model(
                x_cat,
                x_num
            )


            loss = criterion(
                logits,
                y_batch
            )


        scaler_amp.scale(
            loss
        ).backward()


        scaler_amp.step(
            optimizer
        )


        scaler_amp.update()


        total_loss += (

            loss.item()

            *
            len(
                y_batch
            )
        )


        n_seen += len(
            y_batch
        )


    return (
        total_loss
        /
        n_seen
    )


# ============================================================
# N3B.7 INNER EPOCH SELECTION
# ============================================================

def select_epoch_n3b(
    model_factory,
    loss_factory,
    outer_train_idx,
    seed,
):

    inner_train_idx, inner_valid_idx = (
        train_test_split(

            outer_train_idx,

            test_size=
                INNER_VALID_FRACTION_N3B,

            random_state=
                seed,

            stratify=
                y_DEV[
                    outer_train_idx
                ],
        )
    )


    (
        tr_cat,
        tr_num,
        tr_y,

        va_cat,
        va_num,
        va_y,

        _,

    ) = prepare_arrays_n3r(

        inner_train_idx,
        inner_valid_idx
    )


    train_loader = make_loader_n3r(

        tr_cat,
        tr_num,
        tr_y,

        batch_size=
            BATCH_SIZE_N3B,

        shuffle=True,
    )


    valid_loader = make_loader_n3r(

        va_cat,
        va_num,
        va_y,

        batch_size=
            BATCH_SIZE_N3B * 2,

        shuffle=False,
    )


    seed_everything_n3(
        seed
    )


    model = model_factory().to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LR_N3B,

        weight_decay=
            WEIGHT_DECAY_N3B,
    )


    criterion = loss_factory()


    scaler_amp = torch.amp.GradScaler(

        "cuda",

        enabled=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    best_pr = -np.inf

    best_epoch = 1

    patience = 0


    for epoch in range(
        1,
        MAX_EPOCHS_N3B + 1
    ):

        train_epoch_n3b(

            model,
            train_loader,
            optimizer,
            criterion,
            scaler_amp
        )


        pred = predict_neural_n3(

            model,
            valid_loader
        )


        pr = average_precision_score(

            va_y,
            pred
        )


        if pr > (
            best_pr
            +
            1e-5
        ):

            best_pr = pr

            best_epoch = epoch

            patience = 0


        else:

            patience += 1


        if patience >= (
            PATIENCE_N3B
        ):

            break


    del model

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return (
        int(
            best_epoch
        ),
        float(
            best_pr
        )
    )


# ============================================================
# N3B.8 FULL OUTER-TRAIN RETRAIN
# ============================================================

def retrain_outer_n3b(
    model_factory,
    loss_factory,
    outer_train_idx,
    outer_valid_idx,
    n_epochs,
    seed,
):

    (
        tr_cat,
        tr_num,
        tr_y,

        va_cat,
        va_num,
        va_y,

        _,

    ) = prepare_arrays_n3r(

        outer_train_idx,
        outer_valid_idx
    )


    train_loader = make_loader_n3r(

        tr_cat,
        tr_num,
        tr_y,

        batch_size=
            BATCH_SIZE_N3B,

        shuffle=True,
    )


    valid_loader = make_loader_n3r(

        va_cat,
        va_num,
        va_y,

        batch_size=
            BATCH_SIZE_N3B * 2,

        shuffle=False,
    )


    seed_everything_n3(
        seed
    )


    model = model_factory().to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LR_N3B,

        weight_decay=
            WEIGHT_DECAY_N3B,
    )


    criterion = loss_factory()


    scaler_amp = torch.amp.GradScaler(

        "cuda",

        enabled=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    for _ in range(
        n_epochs
    ):

        train_epoch_n3b(

            model,
            train_loader,
            optimizer,
            criterion,
            scaler_amp
        )


    pred = predict_neural_n3(

        model,
        valid_loader
    )


    return pred


# ============================================================
# N3B.9 STRICT SPECIALIST OOF LOOP
# ============================================================

RESULTS_N3B = {}

FOLD_ROWS_N3B = []


for (
    model_id,
    config

) in SPECIALISTS_N3B.items():

    print(
        "\n"
        + "=" * 130
    )

    print(
        "TRAINING SPECIALIST:",
        model_id
    )

    print(
        "=" * 130
    )


    oof = np.full(

        len(
            y_DEV
        ),

        np.nan,

        dtype=np.float32
    )


    selected_epochs = []


    total_start = time.perf_counter()


    for fold in range(
        N_FOLDS
    ):

        outer_train_idx = np.where(

            DEV_FOLD != fold

        )[0]


        outer_valid_idx = np.where(

            DEV_FOLD == fold

        )[0]


        seed_fold = (
            SEED
            +
            8000
            +
            fold
        )


        # ----------------------------------------------
        # Inner epoch selection
        # ----------------------------------------------

        (
            best_epoch,
            inner_best_pr

        ) = select_epoch_n3b(

            config[
                "model_factory"
            ],

            config[
                "loss_factory"
            ],

            outer_train_idx,

            seed=
                seed_fold
        )


        selected_epochs.append(
            best_epoch
        )


        # ----------------------------------------------
        # Fresh retraining on full outer train
        # ----------------------------------------------

        t0 = time.perf_counter()


        pred = retrain_outer_n3b(

            config[
                "model_factory"
            ],

            config[
                "loss_factory"
            ],

            outer_train_idx,
            outer_valid_idx,

            n_epochs=
                best_epoch,

            seed=
                seed_fold + 100
        )


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        oof[
            outer_valid_idx
        ] = pred


        metrics = probability_metrics(

            y_DEV[
                outer_valid_idx
            ],

            pred
        )


        FOLD_ROWS_N3B.append({

            "Model_ID":
                model_id,

            "Fold":
                fold,

            "Selected_Epoch":
                best_epoch,

            "Inner_PR":
                inner_best_pr,

            "Outer_PR":
                metrics[
                    "PR_AUC"
                ],

            "Outer_ROC":
                metrics[
                    "ROC_AUC"
                ],

            "Outer_Brier":
                metrics[
                    "Brier"
                ],

            "Outer_NLL":
                metrics[
                    "NLL"
                ],

            "Outer_ECE":
                metrics[
                    "ECE"
                ],

            "Seconds":
                elapsed,
        })


        print(

            f"Fold {fold}: "
            f"epoch={best_epoch:2d} | "
            f"innerPR={inner_best_pr:.6f} | "
            f"outerPR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f}"
        )


        gc.collect()


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    if np.isnan(
        oof
    ).any():

        raise RuntimeError(
            f"Incomplete OOF for {model_id}"
        )


    pooled = probability_metrics(

        y_DEV,
        oof
    )


    RESULTS_N3B[
        model_id
    ] = {

        "oof":
            oof,

        "metrics":
            pooled,

        "epochs":
            selected_epochs,

        "seconds":
            (
                time.perf_counter()
                -
                total_start
            ),
    }


    store_oof(
        model_id,
        oof
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        oof
    )


    register_experiment(

        model_id=
            model_id,

        family=
            (
                "AutoInt"
                if model_id.startswith(
                    "AUTOINT"
                )
                else
                "MLP"
            ),

        role=
            "neural-specialist",

        specialization=
            config[
                "specialization"
            ],

        feature_view=
            "all_21",

        training_objective=
            config[
                "specialization"
            ],

        robustness_augmentation=
            (
                "feature-token dropout"
                if "FEATDROP"
                in model_id
                else
                "None"
            ),

        diversity_method=
            config[
                "diversity"
            ],

        oof_available=
            True,

        metrics=
            pooled,

        status=
            "candidate-clean",

        notes=
            (
                "Nested epoch selection; "
                "strict outer-fold OOF."
            ),
    )


    print(
        "\nPOOLED"
    )

    print(
        f"PR    = {pooled['PR_AUC']:.6f}"
    )

    print(
        f"ROC   = {pooled['ROC_AUC']:.6f}"
    )

    print(
        f"Brier = {pooled['Brier']:.6f}"
    )

    print(
        f"NLL   = {pooled['NLL']:.6f}"
    )

    print(
        f"ECE   = {pooled['ECE']:.6f}"
    )

    print(
        "Epochs:",
        selected_epochs
    )


# ============================================================
# N3B.10 SPECIALIST LEADERBOARD
# ============================================================

rows = []


for (
    model_id,
    result

) in RESULTS_N3B.items():

    m = result[
        "metrics"
    ]


    rows.append({

        "Model_ID":
            model_id,

        "Specialization":
            SPECIALISTS_N3B[
                model_id
            ][
                "specialization"
            ],

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Median_Epoch":
            float(
                np.median(
                    result[
                        "epochs"
                    ]
                )
            ),
    })


LEADERBOARD_N3B = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N3B — OBJECTIVE-DIVERSE SPECIALIST LEADERBOARD"
)

print(
    "=" * 140
)


display(
    LEADERBOARD_N3B
)


# ============================================================
# N3B.11 CORRELATION AGAINST CURRENT ANCHORS
# ============================================================

ANCHOR_IDS_N3B = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",

    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",

]


SPECIALIST_IDS_N3B = list(
    SPECIALISTS_N3B.keys()
)


COMPARE_IDS_N3B = (

    ANCHOR_IDS_N3B
    +
    SPECIALIST_IDS_N3B
)


P_COMPARE_N3B = np.column_stack([

    OOF_STORE[
        model_id
    ]

    for model_id
    in COMPARE_IDS_N3B
])


CORR_N3B = pd.DataFrame(

    np.corrcoef(
        P_COMPARE_N3B.T
    ),

    index=
        COMPARE_IDS_N3B,

    columns=
        COMPARE_IDS_N3B,
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N3B — ANCHOR / SPECIALIST CORRELATION"
)

print(
    "=" * 140
)


display(
    CORR_N3B.round(4)
)


# ============================================================
# N3B.12 SPECIALIST ENSEMBLE CONTRIBUTION SCREEN
#
# Baseline:
#
#   XGB_SMOOTH + CAT_SMOOTH
#
# Add one specialist at a time.
# ============================================================

p_base2 = (

    OOF_STORE[
        "XGB_SMOOTH"
    ]

    +

    OOF_STORE[
        "CAT_SMOOTH"
    ]

) / 2.0


BASE2_METRICS_N3B = (
    probability_metrics(

        y_DEV,
        p_base2
    )
)


contribution_rows = []


for model_id in (
    SPECIALIST_IDS_N3B
):

    p = OOF_STORE[
        model_id
    ]


    p_three = (

        OOF_STORE[
            "XGB_SMOOTH"
        ]

        +

        OOF_STORE[
            "CAT_SMOOTH"
        ]

        +

        p

    ) / 3.0


    metrics = probability_metrics(

        y_DEV,
        p_three
    )


    contribution_rows.append({

        "Specialist":
            model_id,

        "Standalone_PR":
            RESULTS_N3B[
                model_id
            ][
                "metrics"
            ][
                "PR_AUC"
            ],

        "Corr_XGB":
            np.corrcoef(

                OOF_STORE[
                    "XGB_SMOOTH"
                ],

                p

            )[0, 1],

        "Corr_CAT":
            np.corrcoef(

                OOF_STORE[
                    "CAT_SMOOTH"
                ],

                p

            )[0, 1],

        "Ensemble_PR":
            metrics[
                "PR_AUC"
            ],

        "Ensemble_ROC":
            metrics[
                "ROC_AUC"
            ],

        "Ensemble_Brier":
            metrics[
                "Brier"
            ],

        "Delta_PR_vs_Base2":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                BASE2_METRICS_N3B[
                    "PR_AUC"
                ]
            ),

        "Delta_vs_Old_Core5":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


CONTRIBUTION_N3B = (

    pd.DataFrame(
        contribution_rows
    )

    .sort_values(
        "Ensemble_PR",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N3B — SPECIALIST CONTRIBUTION SCREEN"
)

print(
    "=" * 140
)


print(
    "Base XGB+CAT PR:",
    BASE2_METRICS_N3B[
        "PR_AUC"
    ]
)


display(
    CONTRIBUTION_N3B
)


# ============================================================
# N3B.13 ADD SPECIALISTS TO BEST CLEAN 3-MODEL BASE
#
# Current clean reference:
#
# XGB_SMOOTH
# + CAT_SMOOTH
# + MLP_REPAIRED
#
# ============================================================

REFERENCE3_IDS_N3B = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
]


p_reference3 = np.mean(

    np.column_stack([

        OOF_STORE[
            m
        ]

        for m
        in REFERENCE3_IDS_N3B
    ]),

    axis=1
)


REF3_METRICS_N3B = (
    probability_metrics(

        y_DEV,
        p_reference3
    )
)


four_rows = []


for model_id in (
    SPECIALIST_IDS_N3B
):

    p_four = np.mean(

        np.column_stack([

            OOF_STORE[
                m
            ]

            for m
            in (
                REFERENCE3_IDS_N3B
                +
                [
                    model_id
                ]
            )
        ]),

        axis=1
    )


    m = probability_metrics(

        y_DEV,
        p_four
    )


    four_rows.append({

        "Added_Specialist":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Delta_vs_Reference3":
            (
                m[
                    "PR_AUC"
                ]
                -
                REF3_METRICS_N3B[
                    "PR_AUC"
                ]
            ),

        "Delta_vs_Old_Core5":
            (
                m[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


FOUR_MODEL_SCREEN_N3B = (

    pd.DataFrame(
        four_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N3B — ADD-ONE TO CLEAN 3-MODEL REFERENCE"
)

print(
    "=" * 140
)


print(
    "Reference3 PR:",
    REF3_METRICS_N3B[
        "PR_AUC"
    ]
)


display(
    FOUR_MODEL_SCREEN_N3B
)


# ============================================================
# N3B.14 SPECIALIST PROBABILITY SHIFT
#
# Useful for seeing whether asymmetric/focal models merely
# rescale probabilities or actually change ranking behavior.
# ============================================================

shift_rows = []


anchor_prob = OOF_STORE[
    "MLP_REPAIRED"
]


for model_id in (
    SPECIALIST_IDS_N3B
):

    p = OOF_STORE[
        model_id
    ]


    shift_rows.append({

        "Model_ID":
            model_id,

        "Mean_Probability":
            float(
                np.mean(
                    p
                )
            ),

        "Observed_Prevalence":
            float(
                np.mean(
                    y_DEV
                )
            ),

        "MeanAbsDiff_vs_MLP":
            float(
                np.mean(
                    np.abs(
                        p
                        -
                        anchor_prob
                    )
                )
            ),

        "Correlation_vs_MLP":
            float(
                np.corrcoef(
                    p,
                    anchor_prob
                )[0, 1]
            ),
    })


PROB_SHIFT_N3B = (

    pd.DataFrame(
        shift_rows
    )

    .sort_values(
        "Correlation_vs_MLP"
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N3B — SPECIALIST PROBABILITY BEHAVIOR"
)

print(
    "=" * 140
)


display(
    PROB_SHIFT_N3B
)


# ============================================================
# N3B.15 SAVE
# ============================================================

FOLD_RESULTS_N3B = pd.DataFrame(
    FOLD_ROWS_N3B
)


LEADERBOARD_N3B.to_csv(

    TAB_DIR
    /
    "n3b_specialist_leaderboard.csv",

    index=False
)


FOLD_RESULTS_N3B.to_csv(

    TAB_DIR
    /
    "n3b_specialist_fold_results.csv",

    index=False
)


CORR_N3B.to_csv(

    TAB_DIR
    /
    "n3b_specialist_correlation.csv"
)


CONTRIBUTION_N3B.to_csv(

    TAB_DIR
    /
    "n3b_specialist_contribution.csv",

    index=False
)


FOUR_MODEL_SCREEN_N3B.to_csv(

    TAB_DIR
    /
    "n3b_four_model_screen.csv",

    index=False
)


PROB_SHIFT_N3B.to_csv(

    TAB_DIR
    /
    "n3b_probability_behavior.csv",

    index=False
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N3B COMPLETE"
)

print(
    "=" * 140
)


print(
    "Specialists trained:",
    len(
        SPECIALIST_IDS_N3B
    )
)


print(
    "HOLDOUT remains untouched."
)

print(
    "BRFSS-2020 remains untouched."
)


print(
    "\nNext:"
)

print(
    "N4 — Cross-fitted hard-case, FN/FP and feature-view specialists."
)

MODULE N3B — SPECIALIST CANDIDATES
MLP_FOCAL              | hard-example/focal     | objective diversity
MLP_ASYM175            | FN-sensitive           | asymmetric class-cost objective
MLP_FEATDROP10         | feature-robust         | feature-token dropout
AUTOINT_FOCAL          | hard-example/focal     | attention + focal objective
AUTOINT_ASYM175        | FN-sensitive           | attention + asymmetric objective
AUTOINT_FEATDROP10     | feature-robust         | attention token dropout

TRAINING SPECIALIST: MLP_FOCAL
Fold 0: epoch=11 | innerPR=0.379244 | outerPR=0.370452 | ROC=0.849169
Fold 1: epoch=11 | innerPR=0.361543 | outerPR=0.374173 | ROC=0.846304
Fold 2: epoch= 7 | innerPR=0.383386 | outerPR=0.372827 | ROC=0.846183
Fold 3: epoch=10 | innerPR=0.381360 | outerPR=0.368322 | ROC=0.846580
Fold 4: epoch=20 | innerPR=0.385513 | outerPR=0.383715 | ROC=0.848150

POOLED
PR    = 0.371329
ROC   = 0.846775
Brier = 0.095191
NLL   = 0.337286
ECE   = 0.148995
Epochs: [11, 11, 7, 10, 20]

TR

,Model_ID,Specialization,PR_AUC,ROC_AUC,Brier,NLL,ECE,Median_Epoch
0,AUTOINT_ASYM175,FN-sensitive,0.372998,0.848177,0.074681,0.250632,0.048586,19.0
1,MLP_FEATDROP10,feature-robust,0.372451,0.848026,0.070247,0.237403,0.005905,9.0
2,AUTOINT_FOCAL,hard-example/focal,0.372206,0.848033,0.096186,0.343394,0.154312,17.0
3,MLP_FOCAL,hard-example/focal,0.371329,0.846775,0.095191,0.337286,0.148995,11.0
4,MLP_ASYM175,FN-sensitive,0.371090,0.846907,0.075758,0.252634,0.050385,9.0
5,AUTOINT_FEATDROP10,feature-robust,0.370932,0.847051,0.070390,0.237955,0.007066,19.0



MODULE N3B — ANCHOR / SPECIALIST CORRELATION


,XGB_SMOOTH,CAT_SMOOTH,MLP_REPAIRED,AUTOINT_REPAIRED,MLP_FOCAL,MLP_ASYM175,MLP_FEATDROP10,AUTOINT_FOCAL,AUTOINT_ASYM175,AUTOINT_FEATDROP10
XGB_SMOOTH,1.0000,0.9940,0.9788,0.9880,0.8694,0.9657,0.9795,0.8882,0.9734,0.9751
CAT_SMOOTH,0.9940,1.0000,0.9790,0.9863,0.8738,0.9673,0.9793,0.8926,0.9744,0.9754
MLP_REPAIRED,0.9788,0.9790,1.0000,0.9788,0.8833,0.9733,0.9816,0.8950,0.9719,0.9706
AUTOINT_REPAIRED,0.9880,0.9863,0.9788,1.0000,0.8788,0.9691,0.9796,0.8990,0.9797,0.9809
MLP_FOCAL,0.8694,0.8738,0.8833,0.8788,1.0000,0.9193,0.8888,0.9771,0.9169,0.8902
MLP_ASYM175,0.9657,0.9673,0.9733,0.9691,0.9193,1.0000,0.9784,0.9226,0.9770,0.9719
MLP_FEATDROP10,0.9795,0.9793,0.9816,0.9796,0.8888,0.9784,1.0000,0.8983,0.9754,0.9745
AUTOINT_FOCAL,0.8882,0.8926,0.8950,0.8990,0.9771,0.9226,0.8983,1.0000,0.9438,0.9147
AUTOINT_ASYM175,0.9734,0.9744,0.9719,0.9797,0.9169,0.9770,0.9754,0.9438,1.0000,0.9885
AUTOINT_FEATDROP10,0.9751,0.9754,0.9706,0.9809,0.8902,0.9719,0.9745,0.9147,0.9885,1.0000



MODULE N3B — SPECIALIST CONTRIBUTION SCREEN
Base XGB+CAT PR: 0.37933913560955274


...


MODULE N3B — ADD-ONE TO CLEAN 3-MODEL REFERENCE
Reference3 PR: 0.3797075036189553


,Added_Specialist,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_vs_Reference3,Delta_vs_Old_Core5
0,MLP_FOCAL,0.379716,0.850252,0.071359,0.249244,0.038255,0.000008,-0.000035
1,AUTOINT_FOCAL,0.379612,0.850459,0.071428,0.250532,0.039604,-0.000096,-0.000139
2,AUTOINT_FEATDROP10,0.379377,0.850430,0.069768,0.235729,0.002762,-0.000330,-0.000374
3,AUTOINT_ASYM175,0.379346,0.850432,0.070079,0.236830,0.013177,-0.000361,-0.000405
4,MLP_FEATDROP10,0.379315,0.850476,0.069767,0.235688,0.002212,-0.000392,-0.000436
5,MLP_ASYM175,0.379276,0.850208,0.070145,0.236991,0.013609,-0.000431,-0.000475



MODULE N3B — SPECIALIST PROBABILITY BEHAVIOR


,Model_ID,Mean_Probability,Observed_Prevalence,MeanAbsDiff_vs_MLP,Correlation_vs_MLP
0,MLP_FOCAL,0.242110,0.094184,0.146508,0.883277
3,AUTOINT_FOCAL,0.247397,0.094184,0.152008,0.895033
5,AUTOINT_FEATDROP10,0.100335,0.094184,0.018584,0.970637
4,AUTOINT_ASYM175,0.142770,0.094184,0.045946,0.971915
1,MLP_ASYM175,0.144568,0.094184,0.047743,0.973342
2,MLP_FEATDROP10,0.098569,0.094184,0.014874,0.981611



MODULE N3B COMPLETE
Specialists trained: 6
HOLDOUT remains untouched.
BRFSS-2020 remains untouched.

Next:
N4 — Cross-fitted hard-case, FN/FP and feature-view specialists.


In [8]:
# ============================================================
# MODULE N4A — STRICT CROSS-FITTED FAILURE SPECIALISTS
#
# PURPOSE
# ------------------------------------------------------------
# Manufacture expert specialization from ACTUAL failure
# structure while preserving strict outer-fold isolation.
#
# Specialists:
#
#   1. XGB_HARD_CF
#      -> emphasizes generally difficult cases
#
#   2. XGB_FN_HARD_CF
#      -> emphasizes difficult positive cases / potential FNs
#
#   3. XGB_FP_HARD_CF
#      -> emphasizes difficult negative cases / potential FPs
#
#   4. XGB_HARD_DISAGREE_CF
#      -> emphasizes high-loss + high-disagreement cases
#
#
# STRICT DESIGN
# ------------------------------------------------------------
# For every outer fold:
#
# OUTER TRAIN
#     |
#     +--> 4-fold INNER cross-fitting
#             |
#             +--> XGB proxy
#             +--> HGB proxy
#
# Inner OOF probabilities generate difficulty weights.
#
# Then:
#
# full OUTER TRAIN + difficulty weights
#     -> train specialist
#     -> untouched OUTER VALID
#
#
# HOLDOUT IS NEVER USED.
# BRFSS-2020 IS NEVER USED.
# ============================================================


import gc
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
)

from xgboost import XGBClassifier


# ============================================================
# N4A.1 CONFIG
# ============================================================

INNER_FOLDS_N4A = 4

EPS_N4A = 1e-7


# Weight strengths
LAMBDA_HARD_N4A = 3.0

LAMBDA_FN_N4A = 5.0

LAMBDA_FP_N4A = 3.0

LAMBDA_DIS_N4A = 2.0


# ============================================================
# N4A.2 PROXY MODEL FACTORIES
#
# These models ONLY generate cross-fitted difficulty signals.
#
# They are deliberately:
#   - strong
#   - reasonably fast
#   - different enough to produce disagreement
# ============================================================

def make_proxy_xgb_n4a(
    seed
):

    return XGBClassifier(

        objective="binary:logistic",

        eval_metric="aucpr",

        n_estimators=800,

        learning_rate=0.025,

        max_depth=3,

        min_child_weight=15,

        subsample=0.90,

        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,

        reg_lambda=7.0,

        tree_method="hist",

        random_state=seed,

        n_jobs=-1,
    )


def make_proxy_hgb_n4a(
    seed
):

    return HistGradientBoostingClassifier(

        learning_rate=0.045,

        max_iter=450,

        max_leaf_nodes=15,

        min_samples_leaf=60,

        l2_regularization=3.0,

        early_stopping=False,

        random_state=seed,
    )


# ============================================================
# N4A.3 SPECIALIST MODEL FACTORY
#
# All specialists share the same strong smooth XGB
# architecture.
#
# Only their TRAINING DISTRIBUTION / WEIGHTS differ.
#
# This allows us to isolate specialization effects.
# ============================================================

def make_specialist_xgb_n4a(
    seed
):

    return XGBClassifier(

        objective="binary:logistic",

        eval_metric="aucpr",

        n_estimators=900,

        learning_rate=0.025,

        max_depth=3,

        min_child_weight=15,

        subsample=0.90,

        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,

        reg_lambda=7.0,

        tree_method="hist",

        random_state=seed,

        n_jobs=-1,
    )


# ============================================================
# N4A.4 SAFE NORMALIZATION
# ============================================================

def percentile_rank_n4a(
    values
):
    """
    Maps an array approximately onto [0, 1]
    using empirical rank.

    More robust than min-max when a few cases
    have very extreme loss values.
    """

    values = np.asarray(
        values,
        dtype=float
    )


    order = np.argsort(
        values,
        kind="mergesort"
    )


    ranks = np.empty(
        len(values),
        dtype=float
    )


    ranks[
        order
    ] = np.arange(
        len(values)
    )


    if len(values) <= 1:

        return np.zeros_like(
            values
        )


    return (
        ranks
        /
        (
            len(values)
            -
            1
        )
    )


# ============================================================
# N4A.5 INNER CROSS-FITTED DIFFICULTY GENERATOR
# ============================================================

def build_inner_difficulty_n4a(
    outer_train_idx,
    outer_fold,
):

    X_outer = X_DEV.iloc[
        outer_train_idx
    ].reset_index(
        drop=True
    )


    y_outer = np.asarray(

        y_DEV[
            outer_train_idx
        ],

        dtype=int
    )


    n_outer = len(
        outer_train_idx
    )


    p_xgb = np.full(

        n_outer,

        np.nan,

        dtype=float
    )


    p_hgb = np.full(

        n_outer,

        np.nan,

        dtype=float
    )


    skf_inner = StratifiedKFold(

        n_splits=
            INNER_FOLDS_N4A,

        shuffle=True,

        random_state=
            SEED
            +
            10000
            +
            outer_fold
    )


    print(
        f"\n  Building inner difficulty signals "
        f"for outer fold {outer_fold}..."
    )


    for (
        inner_fold,
        (
            tr_local,
            va_local
        )

    ) in enumerate(

        skf_inner.split(
            X_outer,
            y_outer
        )
    ):

        X_tr = X_outer.iloc[
            tr_local
        ]


        y_tr = y_outer[
            tr_local
        ]


        X_va = X_outer.iloc[
            va_local
        ]


        # ----------------------------------------------
        # XGB proxy
        # ----------------------------------------------

        xgb = make_proxy_xgb_n4a(

            SEED
            +
            11000
            +
            outer_fold * 100
            +
            inner_fold
        )


        xgb.fit(
            X_tr,
            y_tr
        )


        p_xgb[
            va_local
        ] = xgb.predict_proba(
            X_va
        )[:, 1]


        # ----------------------------------------------
        # HGB proxy
        # ----------------------------------------------

        hgb = make_proxy_hgb_n4a(

            SEED
            +
            12000
            +
            outer_fold * 100
            +
            inner_fold
        )


        hgb.fit(
            X_tr,
            y_tr
        )


        p_hgb[
            va_local
        ] = hgb.predict_proba(
            X_va
        )[:, 1]


        print(
            f"    inner fold {inner_fold} complete"
        )


        del xgb
        del hgb

        gc.collect()


    if (
        np.isnan(
            p_xgb
        ).any()
        or
        np.isnan(
            p_hgb
        ).any()
    ):

        raise RuntimeError(
            "Incomplete inner OOF predictions."
        )


    # ========================================================
    # Consensus probability
    # ========================================================

    p_base = (

        p_xgb
        +
        p_hgb

    ) / 2.0


    p_base = np.clip(

        p_base,

        EPS_N4A,

        1 - EPS_N4A
    )


    # ========================================================
    # Point-wise log-loss
    # ========================================================

    point_logloss = -(

        y_outer
        *
        np.log(
            p_base
        )

        +

        (
            1 - y_outer
        )
        *
        np.log(
            1 - p_base
        )
    )


    # ========================================================
    # Prediction disagreement
    # ========================================================

    disagreement = np.abs(

        p_xgb
        -
        p_hgb
    )


    # ========================================================
    # Global hardness percentile
    # ========================================================

    hard_rank = percentile_rank_n4a(
        point_logloss
    )


    dis_rank = percentile_rank_n4a(
        disagreement
    )


    # ========================================================
    # Class-specific hardness
    #
    # Rank positives only among positives.
    # Rank negatives only among negatives.
    # ========================================================

    pos_mask = (
        y_outer == 1
    )


    neg_mask = (
        y_outer == 0
    )


    fn_rank = np.zeros(
        n_outer,
        dtype=float
    )


    fp_rank = np.zeros(
        n_outer,
        dtype=float
    )


    # difficult positives:
    # low predicted probability -> high FN difficulty

    positive_difficulty = (

        1.0
        -
        p_base[
            pos_mask
        ]
    )


    fn_rank[
        pos_mask
    ] = percentile_rank_n4a(

        positive_difficulty
    )


    # difficult negatives:
    # high predicted probability -> high FP difficulty

    negative_difficulty = (

        p_base[
            neg_mask
        ]
    )


    fp_rank[
        neg_mask
    ] = percentile_rank_n4a(

        negative_difficulty
    )


    # ========================================================
    # TRAINING WEIGHTS
    # ========================================================


    # --------------------------------------------------------
    # General hard-case specialist
    # --------------------------------------------------------

    w_hard = (

        1.0
        +
        LAMBDA_HARD_N4A
        *
        hard_rank
    )


    # --------------------------------------------------------
    # FN specialist
    #
    # Only positives receive extra weight.
    # --------------------------------------------------------

    w_fn = np.ones(
        n_outer,
        dtype=float
    )


    w_fn[
        pos_mask
    ] += (

        LAMBDA_FN_N4A
        *
        fn_rank[
            pos_mask
        ]
    )


    # --------------------------------------------------------
    # FP specialist
    #
    # Only difficult negatives receive additional weight.
    # --------------------------------------------------------

    w_fp = np.ones(
        n_outer,
        dtype=float
    )


    w_fp[
        neg_mask
    ] += (

        LAMBDA_FP_N4A
        *
        fp_rank[
            neg_mask
        ]
    )


    # --------------------------------------------------------
    # Loss + disagreement specialist
    # --------------------------------------------------------

    w_hard_dis = (

        1.0

        +

        LAMBDA_HARD_N4A
        *
        hard_rank

        +

        LAMBDA_DIS_N4A
        *
        dis_rank
    )


    # ========================================================
    # NORMALIZE WEIGHTS
    #
    # Keep average weight = 1 so optimization scale remains
    # comparable across specialists.
    # ========================================================

    def normalize_weight(
        w
    ):

        return (
            w
            /
            np.mean(
                w
            )
        )


    weights = {

        "XGB_HARD_CF":
            normalize_weight(
                w_hard
            ),

        "XGB_FN_HARD_CF":
            normalize_weight(
                w_fn
            ),

        "XGB_FP_HARD_CF":
            normalize_weight(
                w_fp
            ),

        "XGB_HARD_DISAGREE_CF":
            normalize_weight(
                w_hard_dis
            ),
    }


    diagnostic = {

        "p_xgb":
            p_xgb,

        "p_hgb":
            p_hgb,

        "p_base":
            p_base,

        "point_logloss":
            point_logloss,

        "hard_rank":
            hard_rank,

        "disagreement":
            disagreement,

        "fn_rank":
            fn_rank,

        "fp_rank":
            fp_rank,
    }


    return (
        weights,
        diagnostic
    )


# ============================================================
# N4A.6 STRICT OUTER-FOLD SPECIALIST TRAINING
# ============================================================

SPECIALIST_IDS_N4A = [

    "XGB_HARD_CF",

    "XGB_FN_HARD_CF",

    "XGB_FP_HARD_CF",

    "XGB_HARD_DISAGREE_CF",
]


OOF_N4A = {

    model_id:
        np.full(
            len(
                y_DEV
            ),
            np.nan,
            dtype=float
        )

    for model_id
    in SPECIALIST_IDS_N4A
}


FOLD_ROWS_N4A = []

WEIGHT_DIAGNOSTICS_N4A = []


total_start_n4a = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 140
    )

    print(
        f"OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 140
    )


    outer_train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    X_train = X_DEV.iloc[
        outer_train_idx
    ]


    y_train = y_DEV[
        outer_train_idx
    ]


    X_valid = X_DEV.iloc[
        outer_valid_idx
    ]


    y_valid = y_DEV[
        outer_valid_idx
    ]


    # ========================================================
    # Strict cross-fitted difficulty scores
    # ========================================================

    (
        weights_dict,
        diagnostics

    ) = build_inner_difficulty_n4a(

        outer_train_idx,
        outer_fold
    )


    # ========================================================
    # Record weight diagnostics
    # ========================================================

    for model_id in (
        SPECIALIST_IDS_N4A
    ):

        w = weights_dict[
            model_id
        ]


        WEIGHT_DIAGNOSTICS_N4A.append({

            "Outer_Fold":
                outer_fold,

            "Model_ID":
                model_id,

            "Weight_Mean":
                float(
                    np.mean(
                        w
                    )
                ),

            "Weight_SD":
                float(
                    np.std(
                        w
                    )
                ),

            "Weight_Min":
                float(
                    np.min(
                        w
                    )
                ),

            "Weight_Max":
                float(
                    np.max(
                        w
                    )
                ),
        })


    # ========================================================
    # Train specialists
    # ========================================================

    for specialist_number, model_id in enumerate(

        SPECIALIST_IDS_N4A

    ):

        print(
            f"\n  Training {model_id}"
        )


        model = make_specialist_xgb_n4a(

            SEED
            +
            15000
            +
            outer_fold * 100
            +
            specialist_number
        )


        t0 = time.perf_counter()


        model.fit(

            X_train,

            y_train,

            sample_weight=
                weights_dict[
                    model_id
                ]
        )


        pred = model.predict_proba(

            X_valid

        )[:, 1]


        pred = np.clip(

            pred,

            EPS_N4A,

            1 - EPS_N4A
        )


        OOF_N4A[
            model_id
        ][
            outer_valid_idx
        ] = pred


        metrics = probability_metrics(

            y_valid,
            pred
        )


        elapsed = (

            time.perf_counter()
            -
            t0
        )


        FOLD_ROWS_N4A.append({

            "Model_ID":
                model_id,

            "Fold":
                outer_fold,

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],

            "Seconds":
                elapsed,
        })


        print(

            f"    PR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f} | "
            f"Brier={metrics['Brier']:.6f}"
        )


        del model
        gc.collect()


    del weights_dict
    del diagnostics

    gc.collect()


# ============================================================
# N4A.7 POOLED SPECIALIST RESULTS
# ============================================================

RESULTS_N4A = {}

rows = []


for model_id in (
    SPECIALIST_IDS_N4A
):

    pred = OOF_N4A[
        model_id
    ]


    if np.isnan(
        pred
    ).any():

        raise RuntimeError(
            f"Incomplete OOF for {model_id}"
        )


    metrics = probability_metrics(

        y_DEV,
        pred
    )


    RESULTS_N4A[
        model_id
    ] = {

        "oof":
            pred,

        "metrics":
            metrics,
    }


    rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            metrics[
                "ROC_AUC"
            ],

        "Brier":
            metrics[
                "Brier"
            ],

        "NLL":
            metrics[
                "NLL"
            ],

        "ECE":
            metrics[
                "ECE"
            ],
    })


    store_oof(
        model_id,
        pred
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        pred
    )


LEADERBOARD_N4A = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — FAILURE SPECIALIST LEADERBOARD"
)

print(
    "=" * 140
)


display(
    LEADERBOARD_N4A
)


# ============================================================
# N4A.8 REGISTER EXPERIMENTS
# ============================================================

SPECIALIZATION_META_N4A = {

    "XGB_HARD_CF":
        "cross-fitted hard-case",

    "XGB_FN_HARD_CF":
        "cross-fitted FN-hard",

    "XGB_FP_HARD_CF":
        "cross-fitted FP-hard",

    "XGB_HARD_DISAGREE_CF":
        "cross-fitted hard + disagreement",
}


for model_id in (
    SPECIALIST_IDS_N4A
):

    register_experiment(

        model_id=
            model_id,

        family=
            "XGBoost",

        role=
            "failure-specialist",

        specialization=
            SPECIALIZATION_META_N4A[
                model_id
            ],

        feature_view=
            "all_21",

        training_objective=
            "BCE with cross-fitted sample weighting",

        robustness_augmentation=
            "None",

        diversity_method=
            "cross-fitted failure-driven weighting",

        oof_available=
            True,

        metrics=
            RESULTS_N4A[
                model_id
            ][
                "metrics"
            ],

        status=
            "candidate-clean",

        notes=
            (
                "Difficulty generated strictly inside "
                "each outer training fold."
            ),
    )


# ============================================================
# N4A.9 CORRELATION WITH CURRENT ANCHORS
# ============================================================

ANCHORS_N4A = [

    "XGB_SMOOTH",

    "CAT_SMOOTH",

    "MLP_REPAIRED",

    "AUTOINT_REPAIRED",

    "MLP_FOCAL",
]


COMPARE_IDS_N4A = (

    ANCHORS_N4A
    +
    SPECIALIST_IDS_N4A
)


P_COMPARE_N4A = np.column_stack([

    OOF_STORE[
        model_id
    ]

    for model_id
    in COMPARE_IDS_N4A
])


CORR_N4A = pd.DataFrame(

    np.corrcoef(
        P_COMPARE_N4A.T
    ),

    index=
        COMPARE_IDS_N4A,

    columns=
        COMPARE_IDS_N4A,
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — FAILURE SPECIALIST CORRELATION"
)

print(
    "=" * 140
)


display(
    CORR_N4A.round(4)
)


# ============================================================
# N4A.10 CONTRIBUTION TO CLEAN TREE BASE
# ============================================================

BASE2_IDS_N4A = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
]


p_base2 = np.mean(

    np.column_stack([

        OOF_STORE[
            model_id
        ]

        for model_id
        in BASE2_IDS_N4A
    ]),

    axis=1
)


m_base2 = probability_metrics(

    y_DEV,
    p_base2
)


contribution_rows = []


for specialist in (
    SPECIALIST_IDS_N4A
):

    p_spec = OOF_STORE[
        specialist
    ]


    p_three = (

        p_base2 * 2.0

        +
        p_spec

    ) / 3.0


    metrics = probability_metrics(

        y_DEV,
        p_three
    )


    contribution_rows.append({

        "Specialist":
            specialist,

        "Standalone_PR":
            RESULTS_N4A[
                specialist
            ][
                "metrics"
            ][
                "PR_AUC"
            ],

        "Corr_XGB":
            float(
                np.corrcoef(

                    p_spec,

                    OOF_STORE[
                        "XGB_SMOOTH"
                    ]

                )[0, 1]
            ),

        "Corr_CAT":
            float(
                np.corrcoef(

                    p_spec,

                    OOF_STORE[
                        "CAT_SMOOTH"
                    ]

                )[0, 1]
            ),

        "Ensemble_PR":
            metrics[
                "PR_AUC"
            ],

        "Ensemble_ROC":
            metrics[
                "ROC_AUC"
            ],

        "Ensemble_Brier":
            metrics[
                "Brier"
            ],

        "Delta_vs_Base2":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                m_base2[
                    "PR_AUC"
                ]
            ),

        "Delta_vs_Old_Core5":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


CONTRIBUTION_N4A = (

    pd.DataFrame(
        contribution_rows
    )

    .sort_values(
        "Ensemble_PR",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — SPECIALIST CONTRIBUTION TO XGB+CAT"
)

print(
    "=" * 140
)


display(
    CONTRIBUTION_N4A
)


# ============================================================
# N4A.11 CONTRIBUTION TO CLEAN THREE-MODEL BASE
#
# Current reference:
#
# XGB_SMOOTH
# CAT_SMOOTH
# MLP_REPAIRED
# ============================================================

REFERENCE3_IDS_N4A = [

    "XGB_SMOOTH",

    "CAT_SMOOTH",

    "MLP_REPAIRED",
]


p_ref3 = np.mean(

    np.column_stack([

        OOF_STORE[
            model_id
        ]

        for model_id
        in REFERENCE3_IDS_N4A
    ]),

    axis=1
)


m_ref3 = probability_metrics(

    y_DEV,
    p_ref3
)


add_rows = []


for specialist in (
    SPECIALIST_IDS_N4A
):

    p_four = np.mean(

        np.column_stack([

            OOF_STORE[
                m
            ]

            for m
            in (
                REFERENCE3_IDS_N4A
                +
                [
                    specialist
                ]
            )
        ]),

        axis=1
    )


    metrics = probability_metrics(

        y_DEV,
        p_four
    )


    add_rows.append({

        "Added_Specialist":
            specialist,

        "PR_AUC":
            metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            metrics[
                "ROC_AUC"
            ],

        "Brier":
            metrics[
                "Brier"
            ],

        "NLL":
            metrics[
                "NLL"
            ],

        "ECE":
            metrics[
                "ECE"
            ],

        "Delta_vs_Reference3":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                m_ref3[
                    "PR_AUC"
                ]
            ),

        "Delta_vs_Old_Core5":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


ADD_TO_REF3_N4A = (

    pd.DataFrame(
        add_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — ADD FAILURE SPECIALIST TO CLEAN REFERENCE3"
)

print(
    "=" * 140
)


display(
    ADD_TO_REF3_N4A
)


# ============================================================
# N4A.12 CLASS-CONDITIONAL RANKING DIAGNOSTIC
#
# We do NOT yet claim FN improvement from ordinary PR-AUC.
#
# Instead examine whether specialists alter ranking in
# difficult positive and difficult negative regions.
#
# Compare probability assigned to positives/negatives.
# ============================================================

behavior_rows = []


for model_id in (

    [
        "XGB_SMOOTH",
        "CAT_SMOOTH",
        "MLP_REPAIRED",
    ]

    +
    SPECIALIST_IDS_N4A
):

    p = OOF_STORE[
        model_id
    ]


    behavior_rows.append({

        "Model_ID":
            model_id,

        "Mean_P_All":
            float(
                np.mean(
                    p
                )
            ),

        "Mean_P_Pos":
            float(
                np.mean(
                    p[
                        y_DEV == 1
                    ]
                )
            ),

        "Mean_P_Neg":
            float(
                np.mean(
                    p[
                        y_DEV == 0
                    ]
                )
            ),

        "PosMinusNeg":
            float(

                np.mean(
                    p[
                        y_DEV == 1
                    ]
                )

                -

                np.mean(
                    p[
                        y_DEV == 0
                    ]
                )
            ),
    })


BEHAVIOR_N4A = (

    pd.DataFrame(
        behavior_rows
    )

    .sort_values(
        "PosMinusNeg",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — CLASS-CONDITIONAL PREDICTION BEHAVIOR"
)

print(
    "=" * 140
)


display(
    BEHAVIOR_N4A
)


# ============================================================
# N4A.13 WEIGHT DIAGNOSTICS
# ============================================================

WEIGHT_DIAGNOSTICS_N4A = pd.DataFrame(

    WEIGHT_DIAGNOSTICS_N4A
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — SPECIALIST TRAINING-WEIGHT DIAGNOSTICS"
)

print(
    "=" * 140
)


display(
    WEIGHT_DIAGNOSTICS_N4A
)


# ============================================================
# N4A.14 FOLD RESULTS
# ============================================================

FOLD_RESULTS_N4A = pd.DataFrame(

    FOLD_ROWS_N4A
)


FOLD_STABILITY_N4A = (

    FOLD_RESULTS_N4A

    .groupby(
        "Model_ID"
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_SD=(
            "ROC_AUC",
            "std"
        ),
    )

    .reset_index()

    .sort_values(
        "PR_Mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A — FOLD STABILITY"
)

print(
    "=" * 140
)


display(
    FOLD_STABILITY_N4A
)


# ============================================================
# N4A.15 SAVE
# ============================================================

LEADERBOARD_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_leaderboard.csv",

    index=False
)


FOLD_RESULTS_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_folds.csv",

    index=False
)


FOLD_STABILITY_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_stability.csv",

    index=False
)


CORR_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_correlation.csv"
)


CONTRIBUTION_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_contribution_base2.csv",

    index=False
)


ADD_TO_REF3_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_contribution_ref3.csv",

    index=False
)


BEHAVIOR_N4A.to_csv(

    TAB_DIR
    /
    "n4a_failure_specialist_behavior.csv",

    index=False
)


WEIGHT_DIAGNOSTICS_N4A.to_csv(

    TAB_DIR
    /
    "n4a_weight_diagnostics.csv",

    index=False
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4A COMPLETE"
)

print(
    "=" * 140
)


print(
    f"\nTotal runtime: "
    f"{time.perf_counter() - total_start_n4a:.1f}s"
)


print(
    "\nAll failure labels/weights were generated "
    "inside outer-training data."
)


print(
    "HOLDOUT remains untouched."
)

print(
    "BRFSS-2020 remains untouched."
)


print(
    "\nNext:"
)

print(
    "N4B — feature-view and retrieval specialists."
)


OUTER FOLD 0

  Building inner difficulty signals for outer fold 0...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

  Training XGB_HARD_CF
    PR=0.377416 | ROC=0.852034 | Brier=0.070091

  Training XGB_FN_HARD_CF
    PR=0.365164 | ROC=0.851274 | Brier=0.099827

  Training XGB_FP_HARD_CF
    PR=0.377910 | ROC=0.851923 | Brier=0.079319

  Training XGB_HARD_DISAGREE_CF
    PR=0.378029 | ROC=0.852193 | Brier=0.069905

OUTER FOLD 1

  Building inner difficulty signals for outer fold 1...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

  Training XGB_HARD_CF
    PR=0.381439 | ROC=0.850276 | Brier=0.070093

  Training XGB_FN_HARD_CF
    PR=0.372913 | ROC=0.849217 | Brier=0.100137

  Training XGB_FP_HARD_CF
    PR=0.380822 | ROC=0.850259 | Brier=0.079264

  Training XGB_HARD_DISAGREE_CF
    PR=0.382086 | ROC=0.850321 | Brier=0.069909

OUTER FOLD 2

  Building inner difficulty s

,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,XGB_HARD_DISAGREE_CF,0.379261,0.850411,0.069908,0.237714,0.011970
1,XGB_FP_HARD_CF,0.379094,0.850293,0.079258,0.278480,0.059847
2,XGB_HARD_CF,0.378903,0.850321,0.070078,0.239010,0.016482
3,XGB_FN_HARD_CF,0.370405,0.849258,0.099601,0.323945,0.142192



MODULE N4A — FAILURE SPECIALIST CORRELATION


,XGB_SMOOTH,CAT_SMOOTH,MLP_REPAIRED,AUTOINT_REPAIRED,MLP_FOCAL,XGB_HARD_CF,XGB_FN_HARD_CF,XGB_FP_HARD_CF,XGB_HARD_DISAGREE_CF
XGB_SMOOTH,1.0000,0.9940,0.9788,0.9880,0.8694,0.9965,0.9039,0.9765,0.9977
CAT_SMOOTH,0.9940,1.0000,0.9790,0.9863,0.8738,0.9933,0.9053,0.9710,0.9944
MLP_REPAIRED,0.9788,0.9790,1.0000,0.9788,0.8833,0.9802,0.9070,0.9455,0.9804
AUTOINT_REPAIRED,0.9880,0.9863,0.9788,1.0000,0.8788,0.9877,0.9083,0.9582,0.9883
MLP_FOCAL,0.8694,0.8738,0.8833,0.8788,1.0000,0.8994,0.9706,0.8014,0.8910
XGB_HARD_CF,0.9965,0.9933,0.9802,0.9877,0.8994,1.0000,0.9302,0.9675,0.9991
XGB_FN_HARD_CF,0.9039,0.9053,0.9070,0.9083,0.9706,0.9302,1.0000,0.8266,0.9222
XGB_FP_HARD_CF,0.9765,0.9710,0.9455,0.9582,0.8014,0.9675,0.8266,1.0000,0.9713
XGB_HARD_DISAGREE_CF,0.9977,0.9944,0.9804,0.9883,0.8910,0.9991,0.9222,0.9713,1.0000



MODULE N4A — SPECIALIST CONTRIBUTION TO XGB+CAT


...


MODULE N4A — SPECIALIST TRAINING-WEIGHT DIAGNOSTICS


,Outer_Fold,Model_ID,Weight_Mean,Weight_SD,Weight_Min,Weight_Max
0,0,XGB_HARD_CF,1.0,0.346412,0.400000,1.600000
1,0,XGB_FN_HARD_CF,1.0,0.691301,0.809418,4.856505
2,0,XGB_FP_HARD_CF,1.0,0.395742,0.423958,1.695831
3,0,XGB_HARD_DISAGREE_CF,1.0,0.386759,0.286453,1.693166
4,1,XGB_HARD_CF,1.0,0.346412,0.400000,1.600000
5,1,XGB_FN_HARD_CF,1.0,0.691301,0.809418,4.856505
6,1,XGB_FP_HARD_CF,1.0,0.395742,0.423958,1.695831
7,1,XGB_HARD_DISAGREE_CF,1.0,0.386827,0.286747,1.696490
8,2,XGB_HARD_CF,1.0,0.346412,0.400000,1.600000
9,2,XGB_FN_HARD_CF,1.0,0.691301,0.809418,4.856505



MODULE N4A — FOLD STABILITY


,Model_ID,PR_Mean,PR_SD,PR_Min,PR_Max,ROC_Mean,ROC_SD
3,XGB_HARD_DISAGREE_CF,0.379726,0.006262,0.371977,0.388892,0.850420,0.001581
1,XGB_FP_HARD_CF,0.379620,0.006231,0.371999,0.389153,0.850311,0.001532
2,XGB_HARD_CF,0.379377,0.006040,0.371843,0.388263,0.850334,0.001592
0,XGB_FN_HARD_CF,0.370810,0.005234,0.365164,0.377327,0.849279,0.001527



MODULE N4A COMPLETE

Total runtime: 266.3s

All failure labels/weights were generated inside outer-training data.
HOLDOUT remains untouched.
BRFSS-2020 remains untouched.

Next:
N4B — feature-view and retrieval specialists.


In [9]:
# ============================================================
# MODULE N4B — FEATURE-VIEW + PROTOTYPE SPECIALISTS
#
# PURPOSE
# ------------------------------------------------------------
# Create experts that see fundamentally different information
# or use a fundamentally different prediction mechanism.
#
# FEATURE-VIEW EXPERTS:
#
#   1. CARDIOMETABOLIC
#   2. FUNCTIONAL / GENERAL HEALTH
#   3. LIFESTYLE
#   4. SOCIOECONOMIC / HEALTHCARE ACCESS
#
# NONPARAMETRIC-LIKE EXPERT:
#
#   5. PROTOTYPE RETRIEVAL
#
# All results are STRICT OUTER OOF.
#
# HOLDOUT untouched.
# BRFSS-2020 untouched.
# ============================================================


import gc
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans

from xgboost import XGBClassifier


# ============================================================
# N4B.1 FEATURE VIEWS
#
# A small demographic/general-health core is retained where
# useful because completely isolated feature families may be
# unrealistically weak.
# ============================================================

FEATURE_VIEWS_N4B = {


    # --------------------------------------------------------
    # Cardiometabolic / cardiovascular burden
    # --------------------------------------------------------

    "XGB_VIEW_CARDIO": [

        "Age",
        "Sex",

        "HighBP",
        "HighChol",
        "CholCheck",
        "BMI",
        "Stroke",
        "Diabetes",

        "GenHlth",
        "DiffWalk",
    ],


    # --------------------------------------------------------
    # Functional / self-reported health burden
    # --------------------------------------------------------

    "XGB_VIEW_FUNCTIONAL": [

        "Age",
        "Sex",

        "GenHlth",
        "MentHlth",
        "PhysHlth",
        "DiffWalk",

        "BMI",
        "Stroke",
    ],


    # --------------------------------------------------------
    # Lifestyle / behavior
    # --------------------------------------------------------

    "XGB_VIEW_LIFESTYLE": [

        "Age",
        "Sex",

        "Smoker",
        "PhysActivity",
        "Fruits",
        "Veggies",
        "HvyAlcoholConsump",

        "BMI",
        "GenHlth",
    ],


    # --------------------------------------------------------
    # Socioeconomic / healthcare access
    # --------------------------------------------------------

    "XGB_VIEW_SOCIO": [

        "Age",
        "Sex",

        "Education",
        "Income",

        "AnyHealthcare",
        "NoDocbcCost",

        "GenHlth",
        "DiffWalk",
    ],
}


# ============================================================
# N4B.2 VIEW MODEL FACTORY
# ============================================================

def make_view_xgb_n4b(
    seed
):

    return XGBClassifier(

        objective="binary:logistic",

        eval_metric="aucpr",

        n_estimators=900,

        learning_rate=0.025,

        max_depth=3,

        min_child_weight=15,

        subsample=0.90,

        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,

        reg_lambda=7.0,

        tree_method="hist",

        random_state=seed,

        n_jobs=-1,
    )


# ============================================================
# N4B.3 STRICT OOF FEATURE-VIEW TRAINING
# ============================================================

OOF_VIEW_N4B = {

    model_id:
        np.full(
            len(y_DEV),
            np.nan,
            dtype=float
        )

    for model_id
    in FEATURE_VIEWS_N4B
}


FOLD_ROWS_VIEW_N4B = []


for fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 130
    )

    print(
        f"FEATURE-VIEW OUTER FOLD {fold}"
    )

    print(
        "=" * 130
    )


    train_idx = np.where(
        DEV_FOLD != fold
    )[0]


    valid_idx = np.where(
        DEV_FOLD == fold
    )[0]


    for view_number, (
        model_id,
        features

    ) in enumerate(
        FEATURE_VIEWS_N4B.items()
    ):

        print(
            f"\n{model_id}"
        )

        print(
            "Features:",
            features
        )


        model = make_view_xgb_n4b(

            SEED
            +
            20000
            +
            fold * 100
            +
            view_number
        )


        t0 = time.perf_counter()


        model.fit(

            X_DEV.iloc[
                train_idx
            ][
                features
            ],

            y_DEV[
                train_idx
            ]
        )


        pred = model.predict_proba(

            X_DEV.iloc[
                valid_idx
            ][
                features
            ]

        )[:, 1]


        OOF_VIEW_N4B[
            model_id
        ][
            valid_idx
        ] = pred


        metrics = probability_metrics(

            y_DEV[
                valid_idx
            ],

            pred
        )


        elapsed = (
            time.perf_counter()
            -
            t0
        )


        FOLD_ROWS_VIEW_N4B.append({

            "Model_ID":
                model_id,

            "Fold":
                fold,

            "N_Features":
                len(
                    features
                ),

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],

            "Seconds":
                elapsed,
        })


        print(

            f"PR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f} | "
            f"Brier={metrics['Brier']:.6f}"
        )


        del model

        gc.collect()


# ============================================================
# N4B.4 PROTOTYPE RETRIEVAL EXPERT
#
# Concept:
#
#   1. Standardize features using outer TRAIN only.
#   2. Learn phenotype prototypes via MiniBatchKMeans.
#   3. Estimate prevalence associated with each prototype.
#   4. For a new patient, softly combine risks from nearest
#      prototypes.
#
# This is intentionally unlike tree/neural classifiers.
# ============================================================

PROTOTYPE_FEATURES_N4B = FEATURES.copy()

N_PROTOTYPES_N4B = 256

N_NEAREST_PROTO_N4B = 8

PROTO_SMOOTHING_N4B = 50.0


OOF_PROTO_N4B = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=float
)


PROTO_FOLD_ROWS_N4B = []


def prototype_predict_n4b(

    X_train,
    y_train,

    X_valid,

    seed,
):

    # --------------------------------------------------------
    # Standardize using training data only
    # --------------------------------------------------------

    scaler = StandardScaler()


    z_train = scaler.fit_transform(
        X_train
    ).astype(
        np.float32
    )


    z_valid = scaler.transform(
        X_valid
    ).astype(
        np.float32
    )


    # --------------------------------------------------------
    # Learn phenotype prototypes
    # --------------------------------------------------------

    km = MiniBatchKMeans(

        n_clusters=
            N_PROTOTYPES_N4B,

        batch_size=4096,

        n_init=3,

        max_iter=200,

        random_state=seed,

        reassignment_ratio=0.01,
    )


    train_cluster = km.fit_predict(
        z_train
    )


    # --------------------------------------------------------
    # Prototype risk with empirical-Bayes shrinkage
    # --------------------------------------------------------

    global_prev = float(
        np.mean(
            y_train
        )
    )


    proto_n = np.bincount(

        train_cluster,

        minlength=
            N_PROTOTYPES_N4B

    ).astype(
        float
    )


    proto_pos = np.bincount(

        train_cluster,

        weights=
            y_train,

        minlength=
            N_PROTOTYPES_N4B

    ).astype(
        float
    )


    proto_risk = (

        proto_pos

        +

        PROTO_SMOOTHING_N4B
        *
        global_prev

    ) / (

        proto_n

        +

        PROTO_SMOOTHING_N4B
    )


    # --------------------------------------------------------
    # Distance from validation samples to prototype centers
    # --------------------------------------------------------

    distances = km.transform(
        z_valid
    )


    # nearest K prototypes

    nearest_idx = np.argpartition(

        distances,

        kth=
            N_NEAREST_PROTO_N4B - 1,

        axis=1

    )[
        :,
        :N_NEAREST_PROTO_N4B
    ]


    nearest_dist = np.take_along_axis(

        distances,

        nearest_idx,

        axis=1
    )


    # --------------------------------------------------------
    # Adaptive temperature
    #
    # Median local distance gives scale.
    # --------------------------------------------------------

    local_scale = np.median(

        nearest_dist,

        axis=1,

        keepdims=True
    )


    local_scale = np.maximum(

        local_scale,

        1e-6
    )


    weights = np.exp(

        -nearest_dist

        /
        local_scale
    )


    weights = (

        weights

        /
        np.sum(
            weights,
            axis=1,
            keepdims=True
        )
    )


    nearest_risk = proto_risk[
        nearest_idx
    ]


    prediction = np.sum(

        weights
        *
        nearest_risk,

        axis=1
    )


    prediction = np.clip(

        prediction,

        1e-7,

        1 - 1e-7
    )


    # distance to nearest prototype can later become
    # a shift/reliability feature

    nearest_distance = np.min(

        nearest_dist,

        axis=1
    )


    return (
        prediction,
        nearest_distance
    )


PROTO_DISTANCE_OOF_N4B = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=float
)


for fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 130
    )

    print(
        f"PROTOTYPE OUTER FOLD {fold}"
    )

    print(
        "=" * 130
    )


    train_idx = np.where(
        DEV_FOLD != fold
    )[0]


    valid_idx = np.where(
        DEV_FOLD == fold
    )[0]


    t0 = time.perf_counter()


    (
        pred,
        nearest_distance

    ) = prototype_predict_n4b(

        X_DEV.iloc[
            train_idx
        ][
            PROTOTYPE_FEATURES_N4B
        ],

        y_DEV[
            train_idx
        ],

        X_DEV.iloc[
            valid_idx
        ][
            PROTOTYPE_FEATURES_N4B
        ],

        seed=
            SEED
            +
            25000
            +
            fold
    )


    OOF_PROTO_N4B[
        valid_idx
    ] = pred


    PROTO_DISTANCE_OOF_N4B[
        valid_idx
    ] = nearest_distance


    metrics = probability_metrics(

        y_DEV[
            valid_idx
        ],

        pred
    )


    elapsed = (
        time.perf_counter()
        -
        t0
    )


    PROTO_FOLD_ROWS_N4B.append({

        "Model_ID":
            "PROTO256",

        "Fold":
            fold,

        "PR_AUC":
            metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            metrics[
                "ROC_AUC"
            ],

        "Brier":
            metrics[
                "Brier"
            ],

        "NLL":
            metrics[
                "NLL"
            ],

        "ECE":
            metrics[
                "ECE"
            ],

        "MeanNearestDistance":
            float(
                np.mean(
                    nearest_distance
                )
            ),

        "Seconds":
            elapsed,
    })


    print(

        f"PR={metrics['PR_AUC']:.6f} | "
        f"ROC={metrics['ROC_AUC']:.6f} | "
        f"Brier={metrics['Brier']:.6f} | "
        f"{elapsed:.1f}s"
    )


# ============================================================
# N4B.5 STORE ALL NEW OOF EXPERTS
# ============================================================

NEW_EXPERT_IDS_N4B = []


for model_id in (
    FEATURE_VIEWS_N4B
):

    pred = OOF_VIEW_N4B[
        model_id
    ]


    if np.isnan(
        pred
    ).any():

        raise RuntimeError(
            f"Incomplete OOF: {model_id}"
        )


    store_oof(
        model_id,
        pred
    )


    np.save(

        PRED_DIR
        /
        f"{model_id}_OOF.npy",

        pred
    )


    NEW_EXPERT_IDS_N4B.append(
        model_id
    )


# Prototype

if np.isnan(
    OOF_PROTO_N4B
).any():

    raise RuntimeError(
        "Incomplete prototype OOF."
    )


store_oof(

    "PROTO256",

    OOF_PROTO_N4B
)


np.save(

    PRED_DIR
    /
    "PROTO256_OOF.npy",

    OOF_PROTO_N4B
)


np.save(

    PRED_DIR
    /
    "PROTO256_DISTANCE_OOF.npy",

    PROTO_DISTANCE_OOF_N4B
)


NEW_EXPERT_IDS_N4B.append(
    "PROTO256"
)


# ============================================================
# N4B.6 LEADERBOARD
# ============================================================

leader_rows = []


for model_id in (
    NEW_EXPERT_IDS_N4B
):

    p = OOF_STORE[
        model_id
    ]


    metrics = probability_metrics(

        y_DEV,
        p
    )


    leader_rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            metrics[
                "ROC_AUC"
            ],

        "Brier":
            metrics[
                "Brier"
            ],

        "NLL":
            metrics[
                "NLL"
            ],

        "ECE":
            metrics[
                "ECE"
            ],
    })


LEADERBOARD_N4B = (

    pd.DataFrame(
        leader_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B — FEATURE-VIEW / PROTOTYPE LEADERBOARD"
)

print(
    "=" * 140
)


display(
    LEADERBOARD_N4B
)


# ============================================================
# N4B.7 REGISTER EXPERIMENTS
# ============================================================

for model_id, features in (
    FEATURE_VIEWS_N4B.items()
):

    metrics = probability_metrics(

        y_DEV,

        OOF_STORE[
            model_id
        ]
    )


    register_experiment(

        model_id=
            model_id,

        family=
            "XGBoost",

        role=
            "feature-view-specialist",

        specialization=
            model_id.replace(
                "XGB_VIEW_",
                ""
            ).lower(),

        feature_view=
            ",".join(
                features
            ),

        training_objective=
            "global BCE",

        robustness_augmentation=
            "None",

        diversity_method=
            "restricted feature view",

        oof_available=
            True,

        metrics=
            metrics,

        status=
            "candidate-clean",

        notes=
            "Strict outer-fold OOF.",
    )


proto_metrics = probability_metrics(

    y_DEV,
    OOF_PROTO_N4B
)


register_experiment(

    model_id=
        "PROTO256",

    family=
        "PrototypeRisk",

    role=
        "retrieval-specialist",

    specialization=
        "phenotype prototypes",

    feature_view=
        "all_21",

    training_objective=
        "cluster prevalence estimation",

    robustness_augmentation=
        "None",

    diversity_method=
        "prototype retrieval",

    oof_available=
        True,

    metrics=
        proto_metrics,

    status=
        "candidate-clean",

    notes=
        (
            "256 phenotype prototypes; "
            "soft nearest-prototype risk."
        ),
)


# ============================================================
# N4B.8 CORRELATION WITH CURRENT IMPORTANT EXPERTS
# ============================================================

CURRENT_ANCHORS_N4B = [

    "XGB_SMOOTH",

    "CAT_SMOOTH",

    "MLP_REPAIRED",

    "AUTOINT_REPAIRED",

    "MLP_FOCAL",

    "XGB_HARD_DISAGREE_CF",

    "XGB_FP_HARD_CF",
]


COMPARE_IDS_N4B = (

    CURRENT_ANCHORS_N4B

    +
    NEW_EXPERT_IDS_N4B
)


P_COMPARE_N4B = np.column_stack([

    OOF_STORE[
        m
    ]

    for m
    in COMPARE_IDS_N4B
])


CORR_N4B = pd.DataFrame(

    np.corrcoef(
        P_COMPARE_N4B.T
    ),

    index=
        COMPARE_IDS_N4B,

    columns=
        COMPARE_IDS_N4B,
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B — FEATURE-VIEW / PROTOTYPE CORRELATION"
)

print(
    "=" * 140
)


display(
    CORR_N4B.round(4)
)


# ============================================================
# N4B.9 CONTRIBUTION TO PREVIOUS CLEAN REFERENCE3
# ============================================================

REF3_IDS_N4B = [

    "XGB_SMOOTH",

    "CAT_SMOOTH",

    "MLP_REPAIRED",
]


p_ref3 = np.mean(

    np.column_stack([

        OOF_STORE[
            m
        ]

        for m
        in REF3_IDS_N4B
    ]),

    axis=1
)


REF3_METRICS_N4B = probability_metrics(

    y_DEV,
    p_ref3
)


ref3_rows = []


for model_id in (
    NEW_EXPERT_IDS_N4B
):

    p_new = np.mean(

        np.column_stack([

            OOF_STORE[
                m
            ]

            for m
            in (
                REF3_IDS_N4B
                +
                [
                    model_id
                ]
            )
        ]),

        axis=1
    )


    metrics = probability_metrics(

        y_DEV,
        p_new
    )


    ref3_rows.append({

        "Added_Expert":
            model_id,

        "Standalone_PR":
            probability_metrics(

                y_DEV,

                OOF_STORE[
                    model_id
                ]

            )[
                "PR_AUC"
            ],

        "PR_AUC":
            metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            metrics[
                "ROC_AUC"
            ],

        "Brier":
            metrics[
                "Brier"
            ],

        "Delta_vs_Ref3":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                REF3_METRICS_N4B[
                    "PR_AUC"
                ]
            ),

        "Delta_vs_Old_Core5":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


ADD_REF3_N4B = (

    pd.DataFrame(
        ref3_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B — ADD-ONE TO CLEAN REFERENCE3"
)

print(
    "=" * 140
)


display(
    ADD_REF3_N4B
)


# ============================================================
# N4B.10 CONTRIBUTION TO CURRENT BEST 4-MODEL SYSTEM
#
# Current best clean result from N4A:
#
# XGB_SMOOTH
# + CAT_SMOOTH
# + MLP_REPAIRED
# + XGB_HARD_DISAGREE_CF
#
# PR ~ 0.379902
# ============================================================

BEST4_IDS_N4B = [

    "XGB_SMOOTH",

    "CAT_SMOOTH",

    "MLP_REPAIRED",

    "XGB_HARD_DISAGREE_CF",
]


p_best4 = np.mean(

    np.column_stack([

        OOF_STORE[
            m
        ]

        for m
        in BEST4_IDS_N4B
    ]),

    axis=1
)


BEST4_METRICS_N4B = probability_metrics(

    y_DEV,
    p_best4
)


best4_rows = []


for model_id in (
    NEW_EXPERT_IDS_N4B
):

    ids = (

        BEST4_IDS_N4B

        +
        [
            model_id
        ]
    )


    p_new = np.mean(

        np.column_stack([

            OOF_STORE[
                m
            ]

            for m
            in ids
        ]),

        axis=1
    )


    metrics = probability_metrics(

        y_DEV,
        p_new
    )


    best4_rows.append({

        "Added_Expert":
            model_id,

        "PR_AUC":
            metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            metrics[
                "ROC_AUC"
            ],

        "Brier":
            metrics[
                "Brier"
            ],

        "NLL":
            metrics[
                "NLL"
            ],

        "ECE":
            metrics[
                "ECE"
            ],

        "Delta_vs_Best4":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                BEST4_METRICS_N4B[
                    "PR_AUC"
                ]
            ),

        "Delta_vs_Old_Core5":
            (
                metrics[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


ADD_BEST4_N4B = (

    pd.DataFrame(
        best4_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B — ADD-ONE TO CURRENT BEST4"
)

print(
    "=" * 140
)


print(
    "Current Best4:",
    BEST4_METRICS_N4B
)


display(
    ADD_BEST4_N4B
)


# ============================================================
# N4B.11 DIVERSITY / COMPETENCE SUMMARY
# ============================================================

anchor_xgb = OOF_STORE[
    "XGB_SMOOTH"
]


div_rows = []


for model_id in (
    NEW_EXPERT_IDS_N4B
):

    p = OOF_STORE[
        model_id
    ]


    m = probability_metrics(

        y_DEV,
        p
    )


    div_rows.append({

        "Model_ID":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "Corr_XGB":
            float(
                np.corrcoef(

                    anchor_xgb,

                    p

                )[0, 1]
            ),

        "Corr_CAT":
            float(
                np.corrcoef(

                    OOF_STORE[
                        "CAT_SMOOTH"
                    ],

                    p

                )[0, 1]
            ),

        "MeanAbsDiff_XGB":
            float(
                np.mean(
                    np.abs(
                        anchor_xgb
                        -
                        p
                    )
                )
            ),
    })


DIVERSITY_N4B = (

    pd.DataFrame(
        div_rows
    )

    .sort_values(
        [
            "Corr_XGB",
            "PR_AUC",
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B — COMPETENCE / DIVERSITY SUMMARY"
)

print(
    "=" * 140
)


display(
    DIVERSITY_N4B
)


# ============================================================
# N4B.12 PROTOTYPE-DISTANCE DIAGNOSTIC
#
# This distance may later become an explicit shift /
# reliability signal for the MoE gate.
# ============================================================

distance_quantiles = np.quantile(

    PROTO_DISTANCE_OOF_N4B,

    [
        0.0,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.0
    ]
)


PROTO_DISTANCE_SUMMARY_N4B = pd.DataFrame({

    "Quantile": [

        "0%",
        "25%",
        "50%",
        "75%",
        "90%",
        "95%",
        "99%",
        "100%",
    ],

    "NearestPrototypeDistance":
        distance_quantiles
})


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B — PROTOTYPE DISTANCE SUMMARY"
)

print(
    "=" * 140
)


display(
    PROTO_DISTANCE_SUMMARY_N4B
)


# ============================================================
# N4B.13 SAVE
# ============================================================

FOLD_VIEW_N4B = pd.DataFrame(
    FOLD_ROWS_VIEW_N4B
)


FOLD_PROTO_N4B = pd.DataFrame(
    PROTO_FOLD_ROWS_N4B
)


LEADERBOARD_N4B.to_csv(

    TAB_DIR
    /
    "n4b_feature_view_prototype_leaderboard.csv",

    index=False
)


FOLD_VIEW_N4B.to_csv(

    TAB_DIR
    /
    "n4b_feature_view_folds.csv",

    index=False
)


FOLD_PROTO_N4B.to_csv(

    TAB_DIR
    /
    "n4b_prototype_folds.csv",

    index=False
)


CORR_N4B.to_csv(

    TAB_DIR
    /
    "n4b_feature_view_prototype_correlation.csv"
)


ADD_REF3_N4B.to_csv(

    TAB_DIR
    /
    "n4b_add_to_reference3.csv",

    index=False
)


ADD_BEST4_N4B.to_csv(

    TAB_DIR
    /
    "n4b_add_to_best4.csv",

    index=False
)


DIVERSITY_N4B.to_csv(

    TAB_DIR
    /
    "n4b_competence_diversity.csv",

    index=False
)


PROTO_DISTANCE_SUMMARY_N4B.to_csv(

    TAB_DIR
    /
    "n4b_prototype_distance_summary.csv",

    index=False
)


EXPERIMENT_REGISTRY.to_csv(

    TAB_DIR
    /
    "experiment_registry.csv",

    index=False
)


print(
    "\n"
    + "=" * 140
)

print(
    "MODULE N4B COMPLETE"
)

print(
    "=" * 140
)


print(
    "\nNew experts:",
    NEW_EXPERT_IDS_N4B
)


print(
    "\nCurrent Best4 PR:",
    BEST4_METRICS_N4B[
        "PR_AUC"
    ]
)


print(
    "\nHOLDOUT remains untouched."
)

print(
    "BRFSS-2020 remains untouched."
)


print(
    "\nNext:"
)

print(
    "N5 — full error complementarity + oracle routing headroom."
)


FEATURE-VIEW OUTER FOLD 0

XGB_VIEW_CARDIO
Features: ['Age', 'Sex', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'Diabetes', 'GenHlth', 'DiffWalk']
PR=0.372073 | ROC=0.848769 | Brier=0.070131

XGB_VIEW_FUNCTIONAL
Features: ['Age', 'Sex', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'BMI', 'Stroke']
PR=0.346369 | ROC=0.832883 | Brier=0.071843

XGB_VIEW_LIFESTYLE
Features: ['Age', 'Sex', 'Smoker', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'BMI', 'GenHlth']
PR=0.316853 | ROC=0.822170 | Brier=0.073354

XGB_VIEW_SOCIO
Features: ['Age', 'Sex', 'Education', 'Income', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'DiffWalk']
PR=0.318437 | ROC=0.825236 | Brier=0.073207

FEATURE-VIEW OUTER FOLD 1

XGB_VIEW_CARDIO
Features: ['Age', 'Sex', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'Diabetes', 'GenHlth', 'DiffWalk']
PR=0.372213 | ROC=0.846691 | Brier=0.070311

XGB_VIEW_FUNCTIONAL
Features: ['Age', 'Sex', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'BMI', 'Stroke'

,Model_ID,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,XGB_VIEW_CARDIO,0.372252,0.847208,0.070190,0.237488,0.002416
1,XGB_VIEW_FUNCTIONAL,0.348840,0.833085,0.071763,0.244514,0.001654
2,XGB_VIEW_SOCIO,0.322894,0.826736,0.072952,0.248094,0.001093
3,XGB_VIEW_LIFESTYLE,0.321083,0.824460,0.073097,0.249152,0.001363
4,PROTO256,0.311338,0.815309,0.074910,0.257767,0.021061



MODULE N4B — FEATURE-VIEW / PROTOTYPE CORRELATION


,XGB_SMOOTH,CAT_SMOOTH,MLP_REPAIRED,AUTOINT_REPAIRED,MLP_FOCAL,XGB_HARD_DISAGREE_CF,XGB_FP_HARD_CF,XGB_VIEW_CARDIO,XGB_VIEW_FUNCTIONAL,XGB_VIEW_LIFESTYLE,XGB_VIEW_SOCIO,PROTO256
XGB_SMOOTH,1.0000,0.9940,0.9788,0.9880,0.8694,0.9977,0.9765,0.9825,0.9292,0.8806,0.8824,0.8333
CAT_SMOOTH,0.9940,1.0000,0.9790,0.9863,0.8738,0.9944,0.9710,0.9785,0.9278,0.8784,0.8823,0.8390
MLP_REPAIRED,0.9788,0.9790,1.0000,0.9788,0.8833,0.9804,0.9455,0.9683,0.9109,0.8625,0.8682,0.8454
AUTOINT_REPAIRED,0.9880,0.9863,0.9788,1.0000,0.8788,0.9883,0.9582,0.9765,0.9259,0.8826,0.8844,0.8410
MLP_FOCAL,0.8694,0.8738,0.8833,0.8788,1.0000,0.8910,0.8014,0.8657,0.8335,0.8265,0.8348,0.8321
XGB_HARD_DISAGREE_CF,0.9977,0.9944,0.9804,0.9883,0.8910,1.0000,0.9713,0.9821,0.9315,0.8862,0.8881,0.8441
XGB_FP_HARD_CF,0.9765,0.9710,0.9455,0.9582,0.8014,0.9713,1.0000,0.9582,0.9083,0.8378,0.8387,0.8050
XGB_VIEW_CARDIO,0.9825,0.9785,0.9683,0.9765,0.8657,0.9821,0.9582,1.0000,0.9387,0.8715,0.8847,0.8309
XGB_VIEW_FUNCTIONAL,0.9292,0.9278,0.9109,0.9259,0.8335,0.9315,0.9083,0.9387,1.0000,0.9203,0.9336,0.7930
XGB_VIEW_LIFESTYLE,0.8806,0.8784,0.8625,0.8826,0.8265,0.8862,0.8378,0.8715,0.9203,1.0000,0.9560,0.6632



MODULE N4B — ADD-ONE TO CLEAN REFERENCE3


...


MODULE N4B — COMPETENCE / DIVERSITY SUMMARY


,Model_ID,PR_AUC,Corr_XGB,Corr_CAT,MeanAbsDiff_XGB
4,PROTO256,0.311338,0.833261,0.839041,0.047998
2,XGB_VIEW_LIFESTYLE,0.321083,0.880584,0.878384,0.034600
3,XGB_VIEW_SOCIO,0.322894,0.882371,0.882279,0.034594
1,XGB_VIEW_FUNCTIONAL,0.348840,0.929153,0.927815,0.029415
0,XGB_VIEW_CARDIO,0.372252,0.982487,0.978462,0.014044



MODULE N4B — PROTOTYPE DISTANCE SUMMARY


,Quantile,NearestPrototypeDistance
0,0%,0.254310
1,25%,1.506533
2,50%,2.235047
3,75%,2.952883
4,90%,3.574491
5,95%,3.965527
6,99%,4.838120
7,100%,9.236485



MODULE N4B COMPLETE

New experts: ['XGB_VIEW_CARDIO', 'XGB_VIEW_FUNCTIONAL', 'XGB_VIEW_LIFESTYLE', 'XGB_VIEW_SOCIO', 'PROTO256']

Current Best4 PR: 0.3799018405419231

HOLDOUT remains untouched.
BRFSS-2020 remains untouched.

Next:
N5 — full error complementarity + oracle routing headroom.


In [10]:
# ============================================================
# MODULE N5 — ERROR COMPLEMENTARITY + ROUTING HEADROOM
#
# PURPOSE
# ------------------------------------------------------------
# Determine whether the candidate pool contains REAL,
# exploitable expert specialization.
#
# This module does NOT train new base predictors.
#
# Analyses:
#
#   A. Competent expert-pool construction
#   B. Strict cross-fitted operating thresholds
#   C. Error overlap / Jaccard / double-fault
#   D. Positive rescue / negative rescue
#   E. Unique-correct / unique-error behavior
#   F. Equal-weight subset search
#   G. Hindsight oracle upper bound
#   H. Cross-fitted risk-bin competence routing
#   I. Go / no-go diagnostics for MoE
#
# IMPORTANT
# ------------------------------------------------------------
# HOLDOUT untouched.
# BRFSS-2020 untouched.
#
# ============================================================


import itertools
import math
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    f1_score,
    confusion_matrix,
)


# ============================================================
# N5.1 DEFINE A REPRESENTATIVE COMPETENT POOL
#
# We deliberately do NOT use every near-duplicate model.
#
# Goal:
#   strong + behaviorally distinct + interpretable roles
# ============================================================

CANDIDATE_POOL_N5 = [

    # strongest smooth/global trees
    "XGB_SMOOTH",
    "CAT_SMOOTH",

    # stochastic tree variant
    "XGB_SUBSPACE",

    # neural representation diversity
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",

    # objective diversity
    "MLP_FOCAL",

    # failure-driven specialists
    "XGB_HARD_DISAGREE_CF",
    "XGB_FP_HARD_CF",
    "XGB_FN_HARD_CF",

    # feature-view specialist
    "XGB_VIEW_CARDIO",
]


# verify all exist
missing_n5 = [

    m
    for m in CANDIDATE_POOL_N5
    if m not in OOF_STORE
]


if missing_n5:

    raise RuntimeError(
        f"Missing OOF predictions: {missing_n5}"
    )


print(
    "=" * 145
)

print(
    "MODULE N5 — REPRESENTATIVE COMPETENT EXPERT POOL"
)

print(
    "=" * 145
)


for m in CANDIDATE_POOL_N5:

    metric = probability_metrics(
        y_DEV,
        OOF_STORE[m]
    )

    print(
        f"{m:25s} "
        f"PR={metric['PR_AUC']:.6f} | "
        f"ROC={metric['ROC_AUC']:.6f} | "
        f"Brier={metric['Brier']:.6f}"
    )


# ============================================================
# N5.2 REFERENCE ENSEMBLES
# ============================================================

REF3_IDS_N5 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
]


BEST4_IDS_N5 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


P_REF3_N5 = np.mean(

    np.column_stack([

        OOF_STORE[m]
        for m in REF3_IDS_N5

    ]),

    axis=1
)


P_BEST4_N5 = np.mean(

    np.column_stack([

        OOF_STORE[m]
        for m in BEST4_IDS_N5

    ]),

    axis=1
)


METRIC_REF3_N5 = probability_metrics(
    y_DEV,
    P_REF3_N5
)


METRIC_BEST4_N5 = probability_metrics(
    y_DEV,
    P_BEST4_N5
)


print(
    "\nREF3:",
    METRIC_REF3_N5
)

print(
    "BEST4:",
    METRIC_BEST4_N5
)


# ============================================================
# N5.3 CROSS-FITTED THRESHOLD SELECTION
#
# For every model and every outer fold:
#
# threshold is optimized using OTHER FOUR folds only,
# then applied to current held-out fold.
#
# This prevents threshold leakage.
# ============================================================

def best_f1_threshold_n5(
    y_true,
    prob,
    grid=None,
):

    if grid is None:

        grid = np.linspace(
            0.03,
            0.50,
            300
        )


    best_t = None
    best_f1 = -np.inf


    for t in grid:

        pred = (
            prob >= t
        ).astype(int)


        score = f1_score(

            y_true,
            pred,
            zero_division=0
        )


        if score > best_f1:

            best_f1 = score
            best_t = t


    return (
        float(best_t),
        float(best_f1)
    )


def crossfitted_binary_predictions_n5(
    probability
):

    probability = np.asarray(
        probability,
        dtype=float
    )


    pred_cf = np.zeros(
        len(y_DEV),
        dtype=int
    )


    thresholds = []


    for fold in range(
        N_FOLDS
    ):

        train_mask = (
            DEV_FOLD != fold
        )


        valid_mask = (
            DEV_FOLD == fold
        )


        threshold, train_f1 = (
            best_f1_threshold_n5(

                y_DEV[
                    train_mask
                ],

                probability[
                    train_mask
                ]
            )
        )


        pred_cf[
            valid_mask
        ] = (

            probability[
                valid_mask
            ]

            >=
            threshold

        ).astype(int)


        thresholds.append({

            "Fold":
                fold,

            "Threshold":
                threshold,

            "Train_F1":
                train_f1,
        })


    return (

        pred_cf,

        pd.DataFrame(
            thresholds
        )
    )


# ============================================================
# N5.4 BUILD STRICT ERROR MATRICES
# ============================================================

BINARY_PRED_N5 = {}

ERROR_N5 = {}

THRESHOLD_TABLES_N5 = {}


for model_id in (
    CANDIDATE_POOL_N5
):

    pred_cf, threshold_table = (
        crossfitted_binary_predictions_n5(

            OOF_STORE[
                model_id
            ]
        )
    )


    BINARY_PRED_N5[
        model_id
    ] = pred_cf


    ERROR_N5[
        model_id
    ] = (

        pred_cf
        !=
        y_DEV

    ).astype(int)


    THRESHOLD_TABLES_N5[
        model_id
    ] = threshold_table


# ============================================================
# N5.5 CROSS-FITTED OPERATING-POINT SUMMARY
# ============================================================

operating_rows = []


for model_id in (
    CANDIDATE_POOL_N5
):

    pred = BINARY_PRED_N5[
        model_id
    ]


    tn, fp, fn, tp = (

        confusion_matrix(

            y_DEV,
            pred,
            labels=[0, 1]

        )
        .ravel()
    )


    operating_rows.append({

        "Model_ID":
            model_id,

        "TP":
            int(tp),

        "FN":
            int(fn),

        "FP":
            int(fp),

        "TN":
            int(tn),

        "Recall":
            tp / (tp + fn),

        "Precision":
            tp / (tp + fp),

        "Specificity":
            tn / (tn + fp),

        "F1":
            f1_score(
                y_DEV,
                pred
            ),

        "Total_Error":
            int(
                fp + fn
            ),
    })


OPERATING_N5 = (

    pd.DataFrame(
        operating_rows
    )

    .sort_values(
        "F1",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — CROSS-FITTED OPERATING-POINT SUMMARY"
)

print(
    "=" * 145
)


display(
    OPERATING_N5
)


# ============================================================
# N5.6 ERROR-OVERLAP METRICS
# ============================================================

def jaccard_binary_n5(
    a,
    b
):

    a = np.asarray(
        a,
        dtype=bool
    )

    b = np.asarray(
        b,
        dtype=bool
    )


    union = np.sum(
        a | b
    )


    if union == 0:
        return 0.0


    return float(

        np.sum(
            a & b
        )

        /
        union
    )


pair_rows = []


for i in range(
    len(
        CANDIDATE_POOL_N5
    )
):

    for j in range(
        i + 1,
        len(
            CANDIDATE_POOL_N5
        )
    ):

        a_id = CANDIDATE_POOL_N5[i]
        b_id = CANDIDATE_POOL_N5[j]


        pa = OOF_STORE[a_id]
        pb = OOF_STORE[b_id]


        pred_a = BINARY_PRED_N5[
            a_id
        ]


        pred_b = BINARY_PRED_N5[
            b_id
        ]


        err_a = ERROR_N5[
            a_id
        ]


        err_b = ERROR_N5[
            b_id
        ]


        error_jaccard = (
            jaccard_binary_n5(
                err_a,
                err_b
            )
        )


        double_fault = float(

            np.mean(
                (err_a == 1)
                &
                (err_b == 1)
            )
        )


        classification_disagreement = float(

            np.mean(
                pred_a
                !=
                pred_b
            )
        )


        # unique success of A where B fails

        a_rescues_b = int(

            np.sum(

                (err_a == 0)

                &

                (err_b == 1)
            )
        )


        b_rescues_a = int(

            np.sum(

                (err_b == 0)

                &

                (err_a == 1)
            )
        )


        # positive-only rescue
        pos = (
            y_DEV == 1
        )


        a_pos_rescue = int(

            np.sum(

                pos

                &

                (pred_a == 1)

                &

                (pred_b == 0)
            )
        )


        b_pos_rescue = int(

            np.sum(

                pos

                &

                (pred_b == 1)

                &

                (pred_a == 0)
            )
        )


        pair_rows.append({

            "Expert_A":
                a_id,

            "Expert_B":
                b_id,

            "Prob_Correlation":
                float(
                    np.corrcoef(
                        pa,
                        pb
                    )[0, 1]
                ),

            "Error_Jaccard":
                error_jaccard,

            "Double_Fault":
                double_fault,

            "Classification_Disagreement":
                classification_disagreement,

            "A_Rescues_B":
                a_rescues_b,

            "B_Rescues_A":
                b_rescues_a,

            "A_Positive_Rescue":
                a_pos_rescue,

            "B_Positive_Rescue":
                b_pos_rescue,
        })


PAIR_COMPLEMENTARITY_N5 = (

    pd.DataFrame(
        pair_rows
    )

    .sort_values(

        [
            "Error_Jaccard",
            "Double_Fault"
        ],

        ascending=[
            True,
            True
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — MOST COMPLEMENTARY ERROR PAIRS"
)

print(
    "=" * 145
)


display(
    PAIR_COMPLEMENTARITY_N5.head(
        25
    )
)


# ============================================================
# N5.7 UNIQUE CORRECT / UNIQUE ERROR ANALYSIS
# ============================================================

error_matrix = np.column_stack([

    ERROR_N5[m]

    for m in CANDIDATE_POOL_N5
])


pred_matrix_binary = np.column_stack([

    BINARY_PRED_N5[m]

    for m in CANDIDATE_POOL_N5
])


n_errors_per_case = np.sum(
    error_matrix,
    axis=1
)


n_correct_per_case = (

    len(
        CANDIDATE_POOL_N5
    )

    -
    n_errors_per_case
)


difficulty_summary_n5 = pd.DataFrame({

    "N_Experts_Wrong":
        n_errors_per_case
})


DIFFICULTY_COUNTS_N5 = (

    difficulty_summary_n5[
        "N_Experts_Wrong"
    ]

    .value_counts()

    .sort_index()

    .rename_axis(
        "N_Experts_Wrong"
    )

    .reset_index(
        name="N_Cases"
    )
)


DIFFICULTY_COUNTS_N5[
    "Fraction"
] = (

    DIFFICULTY_COUNTS_N5[
        "N_Cases"
    ]

    /
    len(
        y_DEV
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — CASE DIFFICULTY DISTRIBUTION"
)

print(
    "=" * 145
)


display(
    DIFFICULTY_COUNTS_N5
)


# ============================================================
# N5.8 UNIQUE POSITIVE RECOVERY
#
# For each model:
# positives caught by that model but missed by ALL others.
# ============================================================

unique_rows = []


positive_mask = (
    y_DEV == 1
)


for j, model_id in enumerate(
    CANDIDATE_POOL_N5
):

    other_idx = [

        k

        for k in range(
            len(
                CANDIDATE_POOL_N5
            )
        )

        if k != j
    ]


    other_positive_predictions = (

        pred_matrix_binary[
            :,
            other_idx
        ]

        .sum(
            axis=1
        )
    )


    unique_positive = (

        positive_mask

        &

        (
            pred_matrix_binary[
                :,
                j
            ]
            ==
            1
        )

        &

        (
            other_positive_predictions
            ==
            0
        )
    )


    unique_correct = (

        (
            error_matrix[
                :,
                j
            ]
            ==
            0
        )

        &

        (
            error_matrix[
                :,
                other_idx
            ]
            .sum(
                axis=1
            )

            ==
            len(
                other_idx
            )
        )
    )


    unique_rows.append({

        "Model_ID":
            model_id,

        "Unique_Positive_Rescues":
            int(
                np.sum(
                    unique_positive
                )
            ),

        "Unique_Correct_Cases":
            int(
                np.sum(
                    unique_correct
                )
            ),
    })


UNIQUE_VALUE_N5 = (

    pd.DataFrame(
        unique_rows
    )

    .sort_values(
        "Unique_Positive_Rescues",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — UNIQUE EXPERT VALUE"
)

print(
    "=" * 145
)


display(
    UNIQUE_VALUE_N5
)


# ============================================================
# N5.9 EQUAL-WEIGHT SUBSET SEARCH
#
# Discovery analysis only.
#
# Search representative expert subsets of size 2..6.
#
# This asks:
# Can static expert selection improve on current Best4?
# ============================================================

subset_rows = []


MAX_SUBSET_SIZE_N5 = 6


for subset_size in range(
    2,
    MAX_SUBSET_SIZE_N5 + 1
):

    for subset in itertools.combinations(

        CANDIDATE_POOL_N5,
        subset_size
    ):

        p = np.mean(

            np.column_stack([

                OOF_STORE[m]

                for m in subset
            ]),

            axis=1
        )


        metrics = probability_metrics(

            y_DEV,
            p
        )


        subset_rows.append({

            "N_Experts":
                subset_size,

            "Experts":
                " | ".join(
                    subset
                ),

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],
        })


SUBSET_SEARCH_N5 = (

    pd.DataFrame(
        subset_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — TOP STATIC EXPERT SUBSETS"
)

print(
    "=" * 145
)


display(
    SUBSET_SEARCH_N5.head(
        30
    )
)


BEST_BY_SIZE_N5 = (

    SUBSET_SEARCH_N5

    .sort_values(
        [
            "N_Experts",
            "PR_AUC"
        ],

        ascending=[
            True,
            False
        ]
    )

    .groupby(
        "N_Experts"
    )

    .head(1)

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — BEST STATIC ENSEMBLE BY SIZE"
)

print(
    "=" * 145
)


display(
    BEST_BY_SIZE_N5
)


# ============================================================
# N5.10 HINDSIGHT PER-SAMPLE ORACLE
#
# IMPORTANT:
# This is NOT a usable model.
#
# It uses the true label to select the expert producing the
# lowest sample-wise log loss.
#
# This is only a theoretical ceiling demonstrating whether
# experts possess complementary correct predictions.
# ============================================================

P_POOL_N5 = np.column_stack([

    np.clip(
        OOF_STORE[m],
        1e-7,
        1 - 1e-7
    )

    for m in CANDIDATE_POOL_N5
])


sample_losses_n5 = -(

    y_DEV[:, None]
    *
    np.log(
        P_POOL_N5
    )

    +

    (
        1 - y_DEV[:, None]
    )
    *
    np.log(
        1 - P_POOL_N5
    )
)


oracle_idx_n5 = np.argmin(

    sample_losses_n5,

    axis=1
)


oracle_prob_n5 = P_POOL_N5[

    np.arange(
        len(
            y_DEV
        )
    ),

    oracle_idx_n5
]


ORACLE_METRICS_N5 = probability_metrics(

    y_DEV,
    oracle_prob_n5
)


oracle_usage = pd.Series(

    [
        CANDIDATE_POOL_N5[i]
        for i in oracle_idx_n5
    ]

).value_counts()


ORACLE_USAGE_N5 = (

    oracle_usage

    .rename_axis(
        "Model_ID"
    )

    .reset_index(
        name="Selected_N"
    )
)


ORACLE_USAGE_N5[
    "Selected_Fraction"
] = (

    ORACLE_USAGE_N5[
        "Selected_N"
    ]

    /
    len(
        y_DEV
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — HINDSIGHT ORACLE UPPER BOUND"
)

print(
    "=" * 145
)


print(
    ORACLE_METRICS_N5
)


display(
    ORACLE_USAGE_N5
)


# ============================================================
# N5.11 CROSS-FITTED RISK-BIN COMPETENCE ROUTER
#
# This is much more meaningful than hindsight oracle.
#
# For each outer fold:
#
# 1. Use OTHER FOUR folds as router-training data.
# 2. Create quantile bins from a stable baseline probability.
# 3. Within each risk bin, determine which expert has lowest
#    Brier loss.
# 4. Route held-out fold samples according to their risk bin.
#
# Thus expert selection for each held-out patient uses NO
# label information from that patient.
# ============================================================

ROUTER_BASE_IDS_N5 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
]


P_ROUTER_BASE_N5 = np.mean(

    np.column_stack([

        OOF_STORE[m]
        for m in ROUTER_BASE_IDS_N5

    ]),

    axis=1
)


N_RISK_BINS_N5 = 10


P_ROUTED_RISKBIN_N5 = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=float
)


ROUTER_DECISIONS_N5 = []


for fold in range(
    N_FOLDS
):

    train_mask = (
        DEV_FOLD != fold
    )


    valid_mask = (
        DEV_FOLD == fold
    )


    train_base = P_ROUTER_BASE_N5[
        train_mask
    ]


    valid_base = P_ROUTER_BASE_N5[
        valid_mask
    ]


    # --------------------------------------------------------
    # Learn bin boundaries from outer-training folds only
    # --------------------------------------------------------

    quantiles = np.linspace(

        0,
        1,
        N_RISK_BINS_N5 + 1
    )


    edges = np.quantile(

        train_base,
        quantiles
    )


    # avoid duplicate boundaries

    edges = np.unique(
        edges
    )


    if len(edges) < 3:

        raise RuntimeError(
            "Too few unique risk-bin boundaries."
        )


    train_bin = np.digitize(

        train_base,

        edges[
            1:-1
        ],

        right=True
    )


    valid_bin = np.digitize(

        valid_base,

        edges[
            1:-1
        ],

        right=True
    )


    y_train_router = y_DEV[
        train_mask
    ]


    train_indices_global = np.where(
        train_mask
    )[0]


    valid_indices_global = np.where(
        valid_mask
    )[0]


    # --------------------------------------------------------
    # Choose best expert independently within each bin
    # --------------------------------------------------------

    for bin_id in np.unique(
        train_bin
    ):

        train_bin_mask = (
            train_bin == bin_id
        )


        valid_bin_mask = (
            valid_bin == bin_id
        )


        if np.sum(
            valid_bin_mask
        ) == 0:

            continue


        candidate_scores = []


        for model_id in (
            CANDIDATE_POOL_N5
        ):

            p_train_model = OOF_STORE[
                model_id
            ][
                train_indices_global[
                    train_bin_mask
                ]
            ]


            y_train_bin = y_train_router[
                train_bin_mask
            ]


            brier = brier_score_loss(

                y_train_bin,
                p_train_model
            )


            candidate_scores.append(

                (
                    brier,
                    model_id
                )
            )


        candidate_scores.sort(
            key=lambda x: x[0]
        )


        best_brier, best_model = (
            candidate_scores[0]
        )


        target_indices = (

            valid_indices_global[
                valid_bin_mask
            ]
        )


        P_ROUTED_RISKBIN_N5[
            target_indices
        ] = OOF_STORE[
            best_model
        ][
            target_indices
        ]


        ROUTER_DECISIONS_N5.append({

            "Fold":
                fold,

            "Risk_Bin":
                int(
                    bin_id
                ),

            "Best_Expert":
                best_model,

            "Train_Brier":
                float(
                    best_brier
                ),

            "N_Train_Bin":
                int(
                    np.sum(
                        train_bin_mask
                    )
                ),

            "N_Valid_Bin":
                int(
                    np.sum(
                        valid_bin_mask
                    )
                ),
        })


if np.isnan(
    P_ROUTED_RISKBIN_N5
).any():

    raise RuntimeError(
        "Risk-bin router left predictions missing."
    )


RISK_BIN_ROUTER_METRICS_N5 = probability_metrics(

    y_DEV,
    P_ROUTED_RISKBIN_N5
)


ROUTER_DECISIONS_N5 = pd.DataFrame(

    ROUTER_DECISIONS_N5
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — CROSS-FITTED RISK-BIN COMPETENCE ROUTER"
)

print(
    "=" * 145
)


print(
    "Reference3:",
    METRIC_REF3_N5
)


print(
    "Best4:",
    METRIC_BEST4_N5
)


print(
    "Risk-bin router:",
    RISK_BIN_ROUTER_METRICS_N5
)


print(
    "\nRouter decisions:"
)


display(
    ROUTER_DECISIONS_N5
)


# ============================================================
# N5.12 ROUTER EXPERT USAGE
# ============================================================

ROUTER_USAGE_N5 = (

    ROUTER_DECISIONS_N5

    .groupby(
        "Best_Expert"
    )[
        "N_Valid_Bin"
    ]

    .sum()

    .sort_values(
        ascending=False
    )

    .rename(
        "Routed_N"
    )

    .reset_index()
)


ROUTER_USAGE_N5[
    "Routed_Fraction"
] = (

    ROUTER_USAGE_N5[
        "Routed_N"
    ]

    /
    len(
        y_DEV
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — RISK-BIN ROUTER EXPERT USAGE"
)

print(
    "=" * 145
)


display(
    ROUTER_USAGE_N5
)


# ============================================================
# N5.13 CLASS-CONDITIONAL EXPERT COMPETENCE
#
# Compare Brier loss separately for positives and negatives.
#
# This tells us whether some specialists really are better
# for different error modes.
# ============================================================

class_rows = []


for model_id in (
    CANDIDATE_POOL_N5
):

    p = OOF_STORE[
        model_id
    ]


    pos_mask = (
        y_DEV == 1
    )


    neg_mask = (
        y_DEV == 0
    )


    pos_brier = np.mean(

        (
            1
            -
            p[
                pos_mask
            ]
        ) ** 2
    )


    neg_brier = np.mean(

        (
            p[
                neg_mask
            ]
        ) ** 2
    )


    class_rows.append({

        "Model_ID":
            model_id,

        "Positive_Brier":
            pos_brier,

        "Negative_Brier":
            neg_brier,

        "Positive_MeanP":
            float(
                np.mean(
                    p[
                        pos_mask
                    ]
                )
            ),

        "Negative_MeanP":
            float(
                np.mean(
                    p[
                        neg_mask
                    ]
                )
            ),
    })


CLASS_COMPETENCE_N5 = pd.DataFrame(

    class_rows
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — CLASS-CONDITIONAL EXPERT COMPETENCE"
)

print(
    "=" * 145
)


display(

    CLASS_COMPETENCE_N5.sort_values(
        "Positive_Brier"
    )
)


# ============================================================
# N5.14 SUMMARY / GO-NO-GO
# ============================================================

best_static_n5 = (

    SUBSET_SEARCH_N5
    .iloc[0]
)


best_static_pr_n5 = float(

    best_static_n5[
        "PR_AUC"
    ]
)


risk_router_pr_n5 = float(

    RISK_BIN_ROUTER_METRICS_N5[
        "PR_AUC"
    ]
)


best4_pr_n5 = float(

    METRIC_BEST4_N5[
        "PR_AUC"
    ]
)


old_core5_pr_n5 = 0.379751


SUMMARY_N5 = pd.DataFrame([

    {
        "System":
            "Old Core5",

        "PR_AUC":
            old_core5_pr_n5,

        "Delta_vs_Old_Core5":
            0.0,
    },

    {
        "System":
            "Current Best4",

        "PR_AUC":
            best4_pr_n5,

        "Delta_vs_Old_Core5":
            best4_pr_n5
            -
            old_core5_pr_n5,
    },

    {
        "System":
            "Best static subset",

        "PR_AUC":
            best_static_pr_n5,

        "Delta_vs_Old_Core5":
            best_static_pr_n5
            -
            old_core5_pr_n5,
    },

    {
        "System":
            "Cross-fitted risk-bin router",

        "PR_AUC":
            risk_router_pr_n5,

        "Delta_vs_Old_Core5":
            risk_router_pr_n5
            -
            old_core5_pr_n5,
    },
])


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — HEADROOM SUMMARY"
)

print(
    "=" * 145
)


display(
    SUMMARY_N5
)


print(
    "\nHindsight oracle metrics:"
)

print(
    ORACLE_METRICS_N5
)


# ============================================================
# N5.15 INTERPRETIVE FLAGS
# ============================================================

STATIC_GAIN_N5 = (

    best_static_pr_n5
    -
    best4_pr_n5
)


ROUTING_GAIN_N5 = (

    risk_router_pr_n5
    -
    best_static_pr_n5
)


print(
    "\n"
    + "=" * 145
)

print(
    "N5 — DECISION FLAGS"
)

print(
    "=" * 145
)


print(
    f"Best4 PR              : "
    f"{best4_pr_n5:.6f}"
)


print(
    f"Best static subset PR : "
    f"{best_static_pr_n5:.6f}"
)


print(
    f"Risk-bin router PR    : "
    f"{risk_router_pr_n5:.6f}"
)


print(
    f"Static gain vs Best4  : "
    f"{STATIC_GAIN_N5:+.6f}"
)


print(
    f"Routing gain vs static: "
    f"{ROUTING_GAIN_N5:+.6f}"
)


if ROUTING_GAIN_N5 > 0.0005:

    print(
        "\n>>> STRONG SIGNAL:"
    )

    print(
        "Conditional expert competence appears exploitable."
    )

    print(
        "Proceed aggressively toward local-competence MoE."
    )


elif ROUTING_GAIN_N5 > 0:

    print(
        "\n>>> MODERATE SIGNAL:"
    )

    print(
        "Some conditional routing headroom exists."
    )

    print(
        "Proceed to local competence, but require strict "
        "improvement over static ensemble."
    )


else:

    print(
        "\n>>> WEAK RISK-BIN SIGNAL:"
    )

    print(
        "Simple risk-region routing did not beat static selection."
    )

    print(
        "Do NOT abandon MoE yet: local neighborhood competence "
        "may still reveal specialization."
    )


# ============================================================
# N5.16 SAVE
# ============================================================

OPERATING_N5.to_csv(

    TAB_DIR
    /
    "n5_operating_points.csv",

    index=False
)


PAIR_COMPLEMENTARITY_N5.to_csv(

    TAB_DIR
    /
    "n5_pair_complementarity.csv",

    index=False
)


DIFFICULTY_COUNTS_N5.to_csv(

    TAB_DIR
    /
    "n5_case_difficulty.csv",

    index=False
)


UNIQUE_VALUE_N5.to_csv(

    TAB_DIR
    /
    "n5_unique_expert_value.csv",

    index=False
)


SUBSET_SEARCH_N5.to_csv(

    TAB_DIR
    /
    "n5_static_subset_search.csv",

    index=False
)


BEST_BY_SIZE_N5.to_csv(

    TAB_DIR
    /
    "n5_best_subset_by_size.csv",

    index=False
)


ORACLE_USAGE_N5.to_csv(

    TAB_DIR
    /
    "n5_hindsight_oracle_usage.csv",

    index=False
)


ROUTER_DECISIONS_N5.to_csv(

    TAB_DIR
    /
    "n5_risk_bin_router_decisions.csv",

    index=False
)


ROUTER_USAGE_N5.to_csv(

    TAB_DIR
    /
    "n5_risk_bin_router_usage.csv",

    index=False
)


CLASS_COMPETENCE_N5.to_csv(

    TAB_DIR
    /
    "n5_class_conditional_competence.csv",

    index=False
)


SUMMARY_N5.to_csv(

    TAB_DIR
    /
    "n5_headroom_summary.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "N5_RISK_BIN_ROUTER_OOF.npy",

    P_ROUTED_RISKBIN_N5
)


print(
    "\n"
    + "=" * 145
)

print(
    "MODULE N5 COMPLETE"
)

print(
    "=" * 145
)


print(
    "\nNo HOLDOUT used."
)

print(
    "No BRFSS-2020 used."
)

print(
    "\nNext decision:"
)

print(
    "N6 — local-neighborhood competence routing / MoE"
)

MODULE N5 — REPRESENTATIVE COMPETENT EXPERT POOL
XGB_SMOOTH                PR=0.378728 | ROC=0.850209 | Brier=0.069802
CAT_SMOOTH                PR=0.378151 | ROC=0.850386 | Brier=0.069787
XGB_SUBSPACE              PR=0.378562 | ROC=0.850477 | Brier=0.069792
MLP_REPAIRED              PR=0.373965 | ROC=0.847718 | Brier=0.070160
AUTOINT_REPAIRED          PR=0.375838 | ROC=0.849122 | Brier=0.069971
MLP_FOCAL                 PR=0.371329 | ROC=0.846775 | Brier=0.095191
XGB_HARD_DISAGREE_CF      PR=0.379261 | ROC=0.850411 | Brier=0.069908
XGB_FP_HARD_CF            PR=0.379094 | ROC=0.850293 | Brier=0.079258
XGB_FN_HARD_CF            PR=0.370405 | ROC=0.849258 | Brier=0.099601
XGB_VIEW_CARDIO           PR=0.372252 | ROC=0.847208 | Brier=0.070190

REF3: {'PR_AUC': np.float64(0.3797075036189553), 'ROC_AUC': np.float64(0.8505462993304812), 'Brier': np.float64(0.06973576340813259), 'NLL': 0.23561295678947142, 'ECE': 0.0018557527478071428}
BEST4: {'PR_AUC': np.float64(0.3799018405419231), 'ROC_AUC

,Model_ID,TP,FN,FP,TN,Recall,Precision,Specificity,F1,Total_Error
0,XGB_HARD_DISAGREE_CF,10383,8731,19672,164158,0.543214,0.345467,0.892988,0.422339,28403
1,XGB_FP_HARD_CF,10377,8737,19854,163976,0.542900,0.343257,0.891998,0.420590,28591
2,XGB_SMOOTH,10729,8385,21210,162620,0.561316,0.335922,0.884622,0.420308,29595
3,CAT_SMOOTH,10287,8827,19577,164253,0.538192,0.344462,0.893505,0.420066,28404
4,XGB_SUBSPACE,10238,8876,19468,164362,0.535628,0.344644,0.894098,0.419418,28344
5,AUTOINT_REPAIRED,10996,8118,22430,161400,0.575285,0.328965,0.877985,0.418576,30548
6,MLP_REPAIRED,10757,8357,21613,162217,0.562781,0.332314,0.882429,0.417877,29970
7,XGB_FN_HARD_CF,10618,8496,21100,162730,0.555509,0.334763,0.885220,0.417768,29596
8,XGB_VIEW_CARDIO,10550,8564,21033,162797,0.551951,0.334040,0.885585,0.416198,29597
9,MLP_FOCAL,10381,8733,20569,163261,0.543110,0.335412,0.888109,0.414709,29302



N5 — MOST COMPLEMENTARY ERROR PAIRS


,Expert_A,Expert_B,Prob_Correlation,Error_Jaccard,Double_Fault,Classification_Disagreement,A_Rescues_B,B_Rescues_A,A_Positive_Rescue,B_Positive_Rescue
0,MLP_FOCAL,XGB_VIEW_CARDIO,0.865701,0.733802,0.122832,0.044559,4669,4374,852,1021
1,MLP_REPAIRED,XGB_VIEW_CARDIO,0.968313,0.746577,0.125463,0.042588,4135,4508,996,789
2,MLP_FOCAL,XGB_FN_HARD_CF,0.970571,0.754692,0.124823,0.040573,4264,3970,737,974
3,AUTOINT_REPAIRED,MLP_FOCAL,0.878762,0.765591,0.127878,0.039154,3350,4596,1112,497
4,CAT_SMOOTH,MLP_FOCAL,0.873763,0.770503,0.123743,0.036857,4189,3291,724,818
5,XGB_SMOOTH,MLP_FOCAL,0.869390,0.772991,0.126528,0.037158,3624,3917,944,596
6,MLP_FOCAL,XGB_FP_HARD_CF,0.801412,0.774281,0.124488,0.036291,3327,4038,743,739
7,MLP_FOCAL,XGB_HARD_DISAGREE_CF,0.890959,0.774883,0.124138,0.036064,3210,4109,732,734
8,AUTOINT_REPAIRED,XGB_VIEW_CARDIO,0.976525,0.777335,0.129617,0.037128,3292,4243,1001,555
9,XGB_FN_HARD_CF,XGB_VIEW_CARDIO,0.895919,0.778048,0.127631,0.036409,3695,3694,789,721



N5 — CASE DIFFICULTY DISTRIBUTION


...


N5 — CLASS-CONDITIONAL EXPERT COMPETENCE


,Model_ID,Positive_Brier,Negative_Brier,Positive_MeanP,Negative_MeanP
8,XGB_FN_HARD_CF,0.299530,0.078813,0.476013,0.211457
5,MLP_FOCAL,0.384718,0.065087,0.387014,0.227043
6,XGB_HARD_DISAGREE_CF,0.558549,0.019100,0.271350,0.088825
3,MLP_REPAIRED,0.565249,0.018682,0.267407,0.080227
2,XGB_SUBSPACE,0.575967,0.017161,0.261168,0.076727
0,XGB_SMOOTH,0.576482,0.017119,0.261082,0.076761
1,CAT_SMOOTH,0.576845,0.017065,0.260247,0.076873
4,AUTOINT_REPAIRED,0.579760,0.016965,0.256844,0.076254
9,XGB_VIEW_CARDIO,0.581740,0.017001,0.256475,0.077255
7,XGB_FP_HARD_CF,0.819364,0.002304,0.098715,0.027643



N5 — HEADROOM SUMMARY


,System,PR_AUC,Delta_vs_Old_Core5
0,Old Core5,0.379751,0.000000
1,Current Best4,0.379902,0.000151
2,Best static subset,0.380049,0.000298
3,Cross-fitted risk-bin router,0.378034,-0.001717



Hindsight oracle metrics:
{'PR_AUC': np.float64(0.9866636988277055), 'ROC_AUC': np.float64(0.998371852133448), 'Brier': np.float64(0.029225423984974994), 'NLL': 0.10025958302874192, 'ECE': 0.06257751203092139}

N5 — DECISION FLAGS
Best4 PR              : 0.379902
Best static subset PR : 0.380049
Risk-bin router PR    : 0.378034
Static gain vs Best4  : +0.000148
Routing gain vs static: -0.002015

>>> WEAK RISK-BIN SIGNAL:
Simple risk-region routing did not beat static selection.
Do NOT abandon MoE yet: local neighborhood competence may still reveal specialization.

MODULE N5 COMPLETE

No HOLDOUT used.
No BRFSS-2020 used.

Next decision:
N6 — local-neighborhood competence routing / MoE


In [11]:
# ============================================================
# MODULE N6A — STRICT LOCAL-COMPETENCE MoE
#
# PURPOSE
# ------------------------------------------------------------
# Test whether expert competence genuinely varies across
# local patient neighborhoods.
#
# STRICT NESTED DESIGN
# ------------------------------------------------------------
#
# For each OUTER fold:
#
#   OUTER TRAIN
#       |
#       +--- 4-fold INNER OOF prediction for every expert
#       |       |
#       |       +--- produces per-patient expert loss
#       |
#       +--- fit neighborhood representation
#       |
#       +--- train each expert on FULL outer train
#               |
#               +--- predict untouched outer valid
#
# For every OUTER-VALID patient:
#
#   1. Find similar patients in OUTER TRAIN.
#   2. Examine each expert's INNER-OOF error among neighbors.
#   3. Estimate local competence.
#   4. Route using:
#
#      - hard best expert
#      - soft local competence
#      - sparse top-2 local competence
#      - reliability-shrunk residual routing
#
#
# NO outer-valid label is used for routing.
# HOLDOUT untouched.
# BRFSS-2020 untouched.
# ============================================================


import gc
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# ============================================================
# N6A.1 CONFIG
# ============================================================

INNER_FOLDS_N6A = 4

K_LOCAL_N6A = 50

EPS_N6A = 1e-7


# soft routing temperature
TEMPERATURE_N6A = 1.0


# residual/local routing strength
MAX_LOCAL_ALPHA_N6A = 0.75


# ============================================================
# N6A.2 EXPERT POOL
#
# Compact pool deliberately chosen to contain:
#
#   - strong experts
#   - model-family variation
#   - stochastic variation
#   - feature-view variation
#
# ============================================================

EXPERT_IDS_N6A = [

    "XGB_SMOOTH_N6",
    "CAT_SMOOTH_N6",
    "XGB_SUBSPACE_N6",
    "HGB_SMOOTH_N6",
    "XGB_CARDIO_N6",
]


CARDIO_FEATURES_N6A = [

    "Age",
    "Sex",

    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Stroke",
    "Diabetes",

    "GenHlth",
    "DiffWalk",
]


# ============================================================
# N6A.3 MODEL FACTORIES
# ============================================================

def make_xgb_smooth_n6(
    seed
):

    return XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=900,
        learning_rate=0.025,

        max_depth=3,
        min_child_weight=15,

        subsample=0.90,
        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,
        reg_lambda=7.0,

        tree_method="hist",

        random_state=seed,
        n_jobs=-1,
    )


def make_xgb_subspace_n6(
    seed
):

    return XGBClassifier(

        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=900,
        learning_rate=0.025,

        max_depth=4,
        min_child_weight=10,

        subsample=0.70,
        colsample_bytree=0.55,

        gamma=0.05,

        reg_alpha=1.0,
        reg_lambda=5.0,

        tree_method="hist",

        random_state=seed,
        n_jobs=-1,
    )


def make_cat_smooth_n6(
    seed
):

    return CatBoostClassifier(

        iterations=900,
        learning_rate=0.03,

        depth=5,

        loss_function="Logloss",
        eval_metric="PRAUC",

        l2_leaf_reg=8.0,

        random_seed=seed,

        verbose=False,
        allow_writing_files=False,

        thread_count=-1,
    )


def make_hgb_smooth_n6(
    seed
):

    return HistGradientBoostingClassifier(

        learning_rate=0.045,

        max_iter=450,

        max_leaf_nodes=15,

        min_samples_leaf=60,

        l2_regularization=3.0,

        early_stopping=False,

        random_state=seed,
    )


def build_model_n6(
    model_id,
    seed
):

    if model_id == "XGB_SMOOTH_N6":

        return make_xgb_smooth_n6(
            seed
        )


    if model_id == "CAT_SMOOTH_N6":

        return make_cat_smooth_n6(
            seed
        )


    if model_id == "XGB_SUBSPACE_N6":

        return make_xgb_subspace_n6(
            seed
        )


    if model_id == "HGB_SMOOTH_N6":

        return make_hgb_smooth_n6(
            seed
        )


    if model_id == "XGB_CARDIO_N6":

        return make_xgb_smooth_n6(
            seed
        )


    raise ValueError(
        model_id
    )


def model_features_n6(
    model_id
):

    if model_id == "XGB_CARDIO_N6":

        return CARDIO_FEATURES_N6A


    return FEATURES


# ============================================================
# N6A.4 INNER OOF EXPERT PREDICTIONS
#
# These predictions are produced WITHOUT the outer validation
# fold participating anywhere.
#
# Therefore neighborhood competence estimates remain strict.
# ============================================================

def build_inner_expert_oof_n6(

    outer_train_idx,
    outer_fold,
):

    X_outer = X_DEV.iloc[
        outer_train_idx
    ].reset_index(
        drop=True
    )


    y_outer = y_DEV[
        outer_train_idx
    ]


    n_outer = len(
        outer_train_idx
    )


    inner_predictions = {

        model_id:
            np.full(
                n_outer,
                np.nan,
                dtype=np.float32
            )

        for model_id
        in EXPERT_IDS_N6A
    }


    inner_cv = StratifiedKFold(

        n_splits=
            INNER_FOLDS_N6A,

        shuffle=True,

        random_state=
            SEED
            +
            30000
            +
            outer_fold
    )


    for (
        inner_fold,
        (
            train_local,
            valid_local
        )

    ) in enumerate(

        inner_cv.split(
            X_outer,
            y_outer
        )
    ):

        print(
            f"      inner fold "
            f"{inner_fold}/{INNER_FOLDS_N6A - 1}"
        )


        for model_number, model_id in enumerate(

            EXPERT_IDS_N6A
        ):

            features = model_features_n6(
                model_id
            )


            model = build_model_n6(

                model_id,

                seed=
                    SEED
                    +
                    31000
                    +
                    outer_fold * 100
                    +
                    inner_fold * 10
                    +
                    model_number
            )


            model.fit(

                X_outer.iloc[
                    train_local
                ][
                    features
                ],

                y_outer[
                    train_local
                ]
            )


            pred = model.predict_proba(

                X_outer.iloc[
                    valid_local
                ][
                    features
                ]

            )[:, 1]


            inner_predictions[
                model_id
            ][
                valid_local
            ] = pred


            del model

            gc.collect()


    for model_id in EXPERT_IDS_N6A:

        if np.isnan(
            inner_predictions[
                model_id
            ]
        ).any():

            raise RuntimeError(
                f"Incomplete inner OOF: {model_id}"
            )


    return (
        inner_predictions,
        y_outer
    )


# ============================================================
# N6A.5 TRAIN FULL OUTER EXPERTS
# ============================================================

def train_outer_experts_n6(

    outer_train_idx,
    outer_valid_idx,
    outer_fold,
):

    predictions = {}


    X_train = X_DEV.iloc[
        outer_train_idx
    ]


    X_valid = X_DEV.iloc[
        outer_valid_idx
    ]


    y_train = y_DEV[
        outer_train_idx
    ]


    for model_number, model_id in enumerate(

        EXPERT_IDS_N6A
    ):

        features = model_features_n6(
            model_id
        )


        model = build_model_n6(

            model_id,

            seed=
                SEED
                +
                35000
                +
                outer_fold * 100
                +
                model_number
        )


        model.fit(

            X_train[
                features
            ],

            y_train
        )


        predictions[
            model_id
        ] = model.predict_proba(

            X_valid[
                features
            ]

        )[:, 1]


        del model

        gc.collect()


    return predictions


# ============================================================
# N6A.6 LOCAL FEATURE REPRESENTATION
#
# All 21 BRFSS predictors are standardized using OUTER TRAIN
# only.
#
# Most "categorical" variables here are binary or ordinal,
# so standardized mixed-space distance is defensible as an
# initial local-competence representation.
#
# Later we can test learned embeddings/Gower distance.
# ============================================================

def build_neighbor_space_n6(

    outer_train_idx,
    outer_valid_idx,
):

    scaler = StandardScaler()


    Z_train = scaler.fit_transform(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ]
    ).astype(
        np.float32
    )


    Z_valid = scaler.transform(

        X_DEV.iloc[
            outer_valid_idx
        ][
            FEATURES
        ]
    ).astype(
        np.float32
    )


    return (
        Z_train,
        Z_valid
    )


# ============================================================
# N6A.7 LOCAL COMPETENCE
#
# Competence = neighborhood mean Brier loss.
#
# Query patient label is NEVER used.
# ============================================================

def calculate_local_competence_n6(

    Z_train,
    Z_valid,

    y_outer_train,

    inner_predictions,
):

    print(
        "    fitting neighborhood index..."
    )


    knn = NearestNeighbors(

        n_neighbors=
            K_LOCAL_N6A,

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1,
    )


    knn.fit(
        Z_train
    )


    distances, indices = (
        knn.kneighbors(

            Z_valid,

            return_distance=True
        )
    )


    # --------------------------------------------------------
    # Per-training-profile loss for every expert
    # --------------------------------------------------------

    losses = {}


    global_losses = {}


    for model_id in (
        EXPERT_IDS_N6A
    ):

        p = np.clip(

            inner_predictions[
                model_id
            ],

            EPS_N6A,

            1 - EPS_N6A
        )


        point_brier = (

            p
            -
            y_outer_train

        ) ** 2


        losses[
            model_id
        ] = point_brier


        global_losses[
            model_id
        ] = float(
            np.mean(
                point_brier
            )
        )


    # --------------------------------------------------------
    # Distance weighting
    #
    # Close neighbors contribute more.
    # Scale is query-specific.
    # --------------------------------------------------------

    scale = np.median(

        distances,

        axis=1,

        keepdims=True
    )


    scale = np.maximum(
        scale,
        1e-6
    )


    neighbor_weight = np.exp(

        -distances
        /
        scale
    )


    neighbor_weight /= np.sum(

        neighbor_weight,

        axis=1,

        keepdims=True
    )


    # --------------------------------------------------------
    # Local Brier for each expert
    # --------------------------------------------------------

    local_loss_matrix = np.zeros(

        (
            len(
                Z_valid
            ),

            len(
                EXPERT_IDS_N6A
            )
        ),

        dtype=np.float32
    )


    for j, model_id in enumerate(

        EXPERT_IDS_N6A
    ):

        neighbor_losses = losses[
            model_id
        ][
            indices
        ]


        local_loss_matrix[
            :,
            j
        ] = np.sum(

            neighbor_weight
            *
            neighbor_losses,

            axis=1
        )


    global_loss_vector = np.array(

        [
            global_losses[
                model_id
            ]

            for model_id
            in EXPERT_IDS_N6A
        ],

        dtype=np.float32
    )


    return {

        "local_loss":
            local_loss_matrix,

        "global_loss":
            global_loss_vector,

        "distances":
            distances,

        "neighbor_indices":
            indices,
    }


# ============================================================
# N6A.8 SOFTMAX UTILITY
# ============================================================

def stable_softmax_n6(
    x,
    axis=1
):

    x = np.asarray(
        x,
        dtype=float
    )


    x = (
        x
        -
        np.max(
            x,
            axis=axis,
            keepdims=True
        )
    )


    ex = np.exp(
        x
    )


    return (

        ex

        /
        np.sum(
            ex,
            axis=axis,
            keepdims=True
        )
    )


# ============================================================
# N6A.9 ROUTING FUNCTIONS
# ============================================================

def routing_weights_n6(
    local_loss,
    global_loss,
):

    # --------------------------------------------------------
    # LOCAL ADVANTAGE
    #
    # positive:
    # expert performs locally BETTER than its global average.
    #
    # negative:
    # locally worse.
    # --------------------------------------------------------

    relative_advantage = (

        global_loss[
            None,
            :
        ]

        -
        local_loss

    ) / (

        global_loss[
            None,
            :
        ]

        +
        EPS_N6A
    )


    # --------------------------------------------------------
    # A. Soft competence routing
    # --------------------------------------------------------

    w_soft = stable_softmax_n6(

        relative_advantage

        /
        TEMPERATURE_N6A,

        axis=1
    )


    # --------------------------------------------------------
    # B. Top-2 sparse routing
    # --------------------------------------------------------

    w_top2 = np.zeros_like(
        w_soft
    )


    top2_idx = np.argpartition(

        w_soft,

        kth=
            w_soft.shape[1] - 2,

        axis=1

    )[
        :,
        -2:
    ]


    rows = np.arange(
        len(
            w_soft
        )
    )[
        :,
        None
    ]


    selected_weights = w_soft[
        rows,
        top2_idx
    ]


    selected_weights /= np.sum(

        selected_weights,

        axis=1,

        keepdims=True
    )


    w_top2[
        rows,
        top2_idx
    ] = selected_weights


    # --------------------------------------------------------
    # C. Hard routing
    # --------------------------------------------------------

    hard_idx = np.argmin(

        local_loss,

        axis=1
    )


    w_hard = np.zeros_like(
        w_soft
    )


    w_hard[

        np.arange(
            len(
                w_soft
            )
        ),

        hard_idx

    ] = 1.0


    return {

        "soft":
            w_soft,

        "top2":
            w_top2,

        "hard":
            w_hard,

        "relative_advantage":
            relative_advantage,
    }


# ============================================================
# N6A.10 ADAPTIVE SHRINKAGE STRENGTH
#
# If experts have almost identical local competence,
# do NOT trust routing strongly.
#
# If competence separation is clear, routing receives more
# influence.
# ============================================================

def local_alpha_n6(
    local_loss
):

    best = np.min(
        local_loss,
        axis=1
    )


    worst = np.max(
        local_loss,
        axis=1
    )


    mean_loss = np.mean(
        local_loss,
        axis=1
    )


    separation = (

        worst
        -
        best

    ) / (

        mean_loss
        +
        EPS_N6A
    )


    # Smooth saturating function
    alpha = (

        separation

        /
        (
            separation
            +
            0.15
        )
    )


    alpha *= (
        MAX_LOCAL_ALPHA_N6A
    )


    return np.clip(

        alpha,

        0.0,

        MAX_LOCAL_ALPHA_N6A
    )


# ============================================================
# N6A.11 GLOBAL OOF ARRAYS
# ============================================================

OOF_EXPERT_N6A = {

    model_id:
        np.full(
            len(
                y_DEV
            ),
            np.nan,
            dtype=np.float32
        )

    for model_id
    in EXPERT_IDS_N6A
}


OOF_HARD_N6A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


OOF_SOFT_N6A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


OOF_TOP2_N6A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


OOF_RESIDUAL_N6A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


OOF_STATIC5_N6A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


ALPHA_OOF_N6A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


LOCAL_BEST_EXPERT_N6A = np.full(

    len(
        y_DEV
    ),

    -1,

    dtype=np.int16
)


LOCAL_LOSS_OOF_N6A = np.full(

    (
        len(
            y_DEV
        ),

        len(
            EXPERT_IDS_N6A
        )
    ),

    np.nan,

    dtype=np.float32
)


# ============================================================
# N6A.12 STRICT OUTER LOOP
# ============================================================

FOLD_RESULTS_N6A = []


total_start_n6 = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 145
    )

    print(
        f"N6A OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 145
    )


    outer_train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    # ========================================================
    # STEP 1
    # Strict inner OOF expert predictions
    # ========================================================

    print(
        "\n[1/5] Building INNER OOF expert predictions..."
    )


    (
        inner_pred,
        y_outer_train

    ) = build_inner_expert_oof_n6(

        outer_train_idx,
        outer_fold
    )


    # ========================================================
    # STEP 2
    # Train full outer experts and predict outer-valid
    # ========================================================

    print(
        "\n[2/5] Training full OUTER experts..."
    )


    outer_pred = train_outer_experts_n6(

        outer_train_idx,
        outer_valid_idx,
        outer_fold
    )


    P_outer = np.column_stack([

        outer_pred[
            model_id
        ]

        for model_id
        in EXPERT_IDS_N6A
    ])


    # store individual strict OOF
    for j, model_id in enumerate(

        EXPERT_IDS_N6A
    ):

        OOF_EXPERT_N6A[
            model_id
        ][
            outer_valid_idx
        ] = P_outer[
            :,
            j
        ]


    # ========================================================
    # STEP 3
    # Build neighborhood space
    # ========================================================

    print(
        "\n[3/5] Building local patient neighborhood..."
    )


    (
        Z_train,
        Z_valid

    ) = build_neighbor_space_n6(

        outer_train_idx,
        outer_valid_idx
    )


    local_info = calculate_local_competence_n6(

        Z_train,
        Z_valid,

        y_outer_train,

        inner_pred
    )


    local_loss = local_info[
        "local_loss"
    ]


    global_loss = local_info[
        "global_loss"
    ]


    LOCAL_LOSS_OOF_N6A[
        outer_valid_idx
    ] = local_loss


    # ========================================================
    # STEP 4
    # Construct routing weights
    # ========================================================

    print(
        "\n[4/5] Computing competence routing..."
    )


    weights = routing_weights_n6(

        local_loss,
        global_loss
    )


    w_soft = weights[
        "soft"
    ]


    w_top2 = weights[
        "top2"
    ]


    w_hard = weights[
        "hard"
    ]


    # equal five-expert static baseline

    p_static = np.mean(

        P_outer,

        axis=1
    )


    # hard local expert

    p_hard = np.sum(

        w_hard
        *
        P_outer,

        axis=1
    )


    # soft local weighting

    p_soft = np.sum(

        w_soft
        *
        P_outer,

        axis=1
    )


    # sparse top-2 weighting

    p_top2 = np.sum(

        w_top2
        *
        P_outer,

        axis=1
    )


    # ========================================================
    # Reliability-shrunk residual routing
    #
    # p_final =
    #       static
    #       +
    #       alpha(x) * [localMoE - static]
    #
    # ========================================================

    alpha = local_alpha_n6(
        local_loss
    )


    p_residual = (

        p_static

        +

        alpha
        *
        (
            p_top2
            -
            p_static
        )
    )


    OOF_STATIC5_N6A[
        outer_valid_idx
    ] = p_static


    OOF_HARD_N6A[
        outer_valid_idx
    ] = p_hard


    OOF_SOFT_N6A[
        outer_valid_idx
    ] = p_soft


    OOF_TOP2_N6A[
        outer_valid_idx
    ] = p_top2


    OOF_RESIDUAL_N6A[
        outer_valid_idx
    ] = p_residual


    ALPHA_OOF_N6A[
        outer_valid_idx
    ] = alpha


    LOCAL_BEST_EXPERT_N6A[
        outer_valid_idx
    ] = np.argmin(

        local_loss,

        axis=1
    )


    # ========================================================
    # STEP 5
    # Fold metrics
    # ========================================================

    print(
        "\n[5/5] Fold evaluation"
    )


    y_valid = y_DEV[
        outer_valid_idx
    ]


    systems_fold = {

        "STATIC5":
            p_static,

        "LOCAL_HARD":
            p_hard,

        "LOCAL_SOFT":
            p_soft,

        "LOCAL_TOP2":
            p_top2,

        "LOCAL_RESIDUAL":
            p_residual,
    }


    for system_name, pred in (

        systems_fold.items()
    ):

        metrics = probability_metrics(

            y_valid,
            pred
        )


        FOLD_RESULTS_N6A.append({

            "Fold":
                outer_fold,

            "System":
                system_name,

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],
        })


        print(

            f"{system_name:16s} | "
            f"PR={metrics['PR_AUC']:.6f} | "
            f"ROC={metrics['ROC_AUC']:.6f} | "
            f"Brier={metrics['Brier']:.6f}"
        )


    del inner_pred
    del outer_pred
    del local_info
    del Z_train
    del Z_valid

    gc.collect()


# ============================================================
# N6A.13 VERIFY COMPLETE OOF
# ============================================================

arrays_to_check_n6 = {

    "STATIC5":
        OOF_STATIC5_N6A,

    "LOCAL_HARD":
        OOF_HARD_N6A,

    "LOCAL_SOFT":
        OOF_SOFT_N6A,

    "LOCAL_TOP2":
        OOF_TOP2_N6A,

    "LOCAL_RESIDUAL":
        OOF_RESIDUAL_N6A,
}


for name, arr in (

    arrays_to_check_n6.items()
):

    if np.isnan(
        arr
    ).any():

        raise RuntimeError(
            f"Incomplete OOF: {name}"
        )


# ============================================================
# N6A.14 POOLED RESULTS
# ============================================================

rows = []


for system_name, pred in (

    arrays_to_check_n6.items()
):

    m = probability_metrics(

        y_DEV,
        pred
    )


    rows.append({

        "System":
            system_name,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Delta_PR_vs_STATIC5":
            (
                m[
                    "PR_AUC"
                ]

                -
                probability_metrics(

                    y_DEV,
                    OOF_STATIC5_N6A

                )[
                    "PR_AUC"
                ]
            ),

        "Delta_PR_vs_Old_Core5":
            (
                m[
                    "PR_AUC"
                ]
                -
                0.379751
            ),
    })


RESULTS_N6A = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "MODULE N6A — STRICT LOCAL-COMPETENCE RESULTS"
)

print(
    "=" * 145
)


display(
    RESULTS_N6A
)


# ============================================================
# N6A.15 INDIVIDUAL EXPERT STRICT OOF
#
# Important because these are freshly generated under the
# same nested structure as routing.
# ============================================================

expert_rows = []


for model_id in (

    EXPERT_IDS_N6A
):

    m = probability_metrics(

        y_DEV,

        OOF_EXPERT_N6A[
            model_id
        ]
    )


    expert_rows.append({

        "Expert":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],
    })


EXPERT_RESULTS_N6A = (

    pd.DataFrame(
        expert_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N6A — STRICT EXPERT PERFORMANCE"
)

print(
    "=" * 145
)


display(
    EXPERT_RESULTS_N6A
)


# ============================================================
# N6A.16 LOCAL EXPERT USAGE
# ============================================================

usage = pd.Series(

    [

        EXPERT_IDS_N6A[
            idx
        ]

        for idx
        in LOCAL_BEST_EXPERT_N6A
    ]

).value_counts()


USAGE_N6A = (

    usage

    .rename_axis(
        "Expert"
    )

    .reset_index(
        name="Locally_Best_N"
    )
)


USAGE_N6A[
    "Fraction"
] = (

    USAGE_N6A[
        "Locally_Best_N"
    ]

    /
    len(
        y_DEV
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N6A — LOCALLY BEST EXPERT USAGE"
)

print(
    "=" * 145
)


display(
    USAGE_N6A
)


# ============================================================
# N6A.17 ROUTING STRENGTH
# ============================================================

ALPHA_SUMMARY_N6A = pd.DataFrame([

    {
        "Mean":
            float(
                np.mean(
                    ALPHA_OOF_N6A
                )
            ),

        "SD":
            float(
                np.std(
                    ALPHA_OOF_N6A
                )
            ),

        "P25":
            float(
                np.quantile(
                    ALPHA_OOF_N6A,
                    0.25
                )
            ),

        "Median":
            float(
                np.median(
                    ALPHA_OOF_N6A
                )
            ),

        "P75":
            float(
                np.quantile(
                    ALPHA_OOF_N6A,
                    0.75
                )
            ),

        "P90":
            float(
                np.quantile(
                    ALPHA_OOF_N6A,
                    0.90
                )
            ),

        "Max":
            float(
                np.max(
                    ALPHA_OOF_N6A
                )
            ),
    }
])


print(
    "\n"
    + "=" * 145
)

print(
    "N6A — ADAPTIVE ROUTING STRENGTH"
)

print(
    "=" * 145
)


display(
    ALPHA_SUMMARY_N6A
)


# ============================================================
# N6A.18 LOCAL-COMPETENCE SEPARATION
#
# How different are experts locally?
# ============================================================

local_best = np.min(

    LOCAL_LOSS_OOF_N6A,

    axis=1
)


local_worst = np.max(

    LOCAL_LOSS_OOF_N6A,

    axis=1
)


local_mean = np.mean(

    LOCAL_LOSS_OOF_N6A,

    axis=1
)


local_gap = (

    local_worst
    -
    local_best
)


relative_gap = (

    local_gap

    /
    (
        local_mean
        +
        EPS_N6A
    )
)


COMPETENCE_GAP_N6A = pd.DataFrame([

    {
        "Mean_Local_Loss_Gap":
            float(
                np.mean(
                    local_gap
                )
            ),

        "Median_Local_Loss_Gap":
            float(
                np.median(
                    local_gap
                )
            ),

        "Mean_Relative_Gap":
            float(
                np.mean(
                    relative_gap
                )
            ),

        "Median_Relative_Gap":
            float(
                np.median(
                    relative_gap
                )
            ),

        "P90_Relative_Gap":
            float(
                np.quantile(
                    relative_gap,
                    0.90
                )
            ),
    }
])


print(
    "\n"
    + "=" * 145
)

print(
    "N6A — LOCAL COMPETENCE SEPARATION"
)

print(
    "=" * 145
)


display(
    COMPETENCE_GAP_N6A
)


# ============================================================
# N6A.19 FOLD STABILITY
# ============================================================

FOLD_RESULTS_N6A = pd.DataFrame(

    FOLD_RESULTS_N6A
)


FOLD_STABILITY_N6A = (

    FOLD_RESULTS_N6A

    .groupby(
        "System"
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_SD=(
            "ROC_AUC",
            "std"
        ),

        Brier_Mean=(
            "Brier",
            "mean"
        ),
    )

    .reset_index()

    .sort_values(
        "PR_Mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 145
)

print(
    "N6A — FOLD STABILITY"
)

print(
    "=" * 145
)


display(
    FOLD_STABILITY_N6A
)


# ============================================================
# N6A.20 SAVE
# ============================================================

RESULTS_N6A.to_csv(

    TAB_DIR
    /
    "n6a_local_competence_results.csv",

    index=False
)


EXPERT_RESULTS_N6A.to_csv(

    TAB_DIR
    /
    "n6a_strict_expert_results.csv",

    index=False
)


USAGE_N6A.to_csv(

    TAB_DIR
    /
    "n6a_local_expert_usage.csv",

    index=False
)


ALPHA_SUMMARY_N6A.to_csv(

    TAB_DIR
    /
    "n6a_alpha_summary.csv",

    index=False
)


COMPETENCE_GAP_N6A.to_csv(

    TAB_DIR
    /
    "n6a_competence_gap.csv",

    index=False
)


FOLD_RESULTS_N6A.to_csv(

    TAB_DIR
    /
    "n6a_fold_results.csv",

    index=False
)


FOLD_STABILITY_N6A.to_csv(

    TAB_DIR
    /
    "n6a_fold_stability.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "N6A_STATIC5_OOF.npy",

    OOF_STATIC5_N6A
)


np.save(

    PRED_DIR
    /
    "N6A_LOCAL_SOFT_OOF.npy",

    OOF_SOFT_N6A
)


np.save(

    PRED_DIR
    /
    "N6A_LOCAL_TOP2_OOF.npy",

    OOF_TOP2_N6A
)


np.save(

    PRED_DIR
    /
    "N6A_LOCAL_RESIDUAL_OOF.npy",

    OOF_RESIDUAL_N6A
)


np.save(

    PRED_DIR
    /
    "N6A_LOCAL_LOSS.npy",

    LOCAL_LOSS_OOF_N6A
)


np.save(

    PRED_DIR
    /
    "N6A_ALPHA.npy",

    ALPHA_OOF_N6A
)


print(
    "\n"
    + "=" * 145
)

print(
    "MODULE N6A COMPLETE"
)

print(
    "=" * 145
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - total_start_n6:.1f}s"
)


print(
    "\nHOLDOUT remains untouched."
)

print(
    "BRFSS-2020 remains untouched."
)


N6A OUTER FOLD 0

[1/5] Building INNER OOF expert predictions...
      inner fold 0/3
      inner fold 1/3
      inner fold 2/3
      inner fold 3/3

[2/5] Training full OUTER experts...

[3/5] Building local patient neighborhood...
    fitting neighborhood index...

[4/5] Computing competence routing...

[5/5] Fold evaluation
STATIC5          | PR=0.377443 | ROC=0.852160 | Brier=0.069748
LOCAL_HARD       | PR=0.373932 | ROC=0.850905 | Brier=0.069930
LOCAL_SOFT       | PR=0.377430 | ROC=0.852165 | Brier=0.069748
LOCAL_TOP2       | PR=0.375943 | ROC=0.851808 | Brier=0.069828
LOCAL_RESIDUAL   | PR=0.377326 | ROC=0.852181 | Brier=0.069755

N6A OUTER FOLD 1

[1/5] Building INNER OOF expert predictions...
      inner fold 0/3
      inner fold 1/3
      inner fold 2/3
      inner fold 3/3

[2/5] Training full OUTER experts...

[3/5] Building local patient neighborhood...
    fitting neighborhood index...

[4/5] Computing competence routing...

[5/5] Fold evaluation
STATIC5          | PR=0.3

,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_STATIC5,Delta_PR_vs_Old_Core5
0,STATIC5,0.379218,0.850530,0.069754,0.235657,0.002570,0.000000,-0.000533
1,LOCAL_SOFT,0.379205,0.850528,0.069754,0.235658,0.002605,-0.000013,-0.000546
2,LOCAL_RESIDUAL,0.379166,0.850518,0.069756,0.235650,0.002421,-0.000052,-0.000585
3,LOCAL_TOP2,0.378058,0.849942,0.069827,0.235956,0.002166,-0.001160,-0.001693
4,LOCAL_HARD,0.376341,0.849096,0.069938,0.236426,0.001943,-0.002877,-0.003410



N6A — STRICT EXPERT PERFORMANCE


,Expert,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,XGB_SUBSPACE_N6,0.378822,0.850490,0.069781,0.235709,0.002405
1,XGB_SMOOTH_N6,0.378598,0.850216,0.069809,0.235862,0.002456
2,CAT_SMOOTH_N6,0.377645,0.850431,0.069792,0.235746,0.002291
3,HGB_SMOOTH_N6,0.376940,0.849566,0.069901,0.236200,0.001944
4,XGB_CARDIO_N6,0.372023,0.847075,0.070208,0.237560,0.002183



N6A — LOCALLY BEST EXPERT USAGE


...


N6A — LOCAL COMPETENCE SEPARATION


,Mean_Local_Loss_Gap,Median_Local_Loss_Gap,Mean_Relative_Gap,Median_Relative_Gap,P90_Relative_Gap
0,0.002315,0.000876,0.246955,0.036404,0.88309



N6A — FOLD STABILITY


,System,PR_Mean,PR_SD,PR_Min,PR_Max,ROC_Mean,ROC_SD,Brier_Mean
4,STATIC5,0.379691,0.006215,0.372492,0.389324,0.850547,0.001560,0.069754
2,LOCAL_SOFT,0.379686,0.006218,0.372521,0.389364,0.850545,0.001561,0.069754
1,LOCAL_RESIDUAL,0.379680,0.006126,0.372689,0.389270,0.850536,0.001556,0.069756
3,LOCAL_TOP2,0.378562,0.005754,0.372522,0.387893,0.849966,0.001589,0.069827
0,LOCAL_HARD,0.376786,0.005603,0.371749,0.386239,0.849121,0.001692,0.069938



MODULE N6A COMPLETE

Runtime: 1117.8s

HOLDOUT remains untouched.
BRFSS-2020 remains untouched.


In [12]:
# ============================================================
# MODULE N7A-0 — SHOW FULL TOP STATIC SUBSETS
# ============================================================

pd.set_option(
    "display.max_colwidth",
    None
)

pd.set_option(
    "display.width",
    250
)

print(
    "=" * 160
)

print(
    "TOP 20 STATIC SUBSETS — FULL EXPERT NAMES"
)

print(
    "=" * 160
)

display(
    SUBSET_SEARCH_N5[
        [
            "N_Experts",
            "Experts",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
        ]
    ].head(20)
)


print(
    "\n"
    + "=" * 160
)

print(
    "BEST STATIC SUBSET BY SIZE — FULL EXPERT NAMES"
)

print(
    "=" * 160
)

display(
    BEST_BY_SIZE_N5[
        [
            "N_Experts",
            "Experts",
            "PR_AUC",
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE",
        ]
    ]
)

TOP 20 STATIC SUBSETS — FULL EXPERT NAMES


,N_Experts,Experts,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,6,XGB_SMOOTH | CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | MLP_FOCAL | XGB_HARD_DISAGREE_CF,0.380049,0.850525,0.070550,0.243705,0.027460
1,5,XGB_SMOOTH | CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.380049,0.850681,0.069728,0.235674,0.003105
2,6,CAT_SMOOTH | XGB_SUBSPACE | MLP_REPAIRED | AUTOINT_REPAIRED | MLP_FOCAL | XGB_HARD_DISAGREE_CF,0.380028,0.850545,0.070551,0.243685,0.027475
3,6,XGB_SMOOTH | CAT_SMOOTH | XGB_SUBSPACE | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.380012,0.850732,0.069722,0.235615,0.002803
4,6,CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | MLP_FOCAL | XGB_HARD_DISAGREE_CF | XGB_FP_HARD_CF,0.379987,0.850461,0.070321,0.242435,0.022627
5,6,CAT_SMOOTH | XGB_SUBSPACE | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF | XGB_FP_HARD_CF,0.379987,0.850728,0.069922,0.236106,0.008080
6,6,XGB_SMOOTH | CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF | XGB_FP_HARD_CF,0.379985,0.850690,0.069924,0.236129,0.008023
7,5,CAT_SMOOTH | XGB_SUBSPACE | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.379983,0.850718,0.069726,0.235652,0.003120
8,6,XGB_SMOOTH | CAT_SMOOTH | XGB_SUBSPACE | MLP_REPAIRED | AUTOINT_REPAIRED | MLP_FOCAL,0.379966,0.850514,0.070432,0.242817,0.025466
9,4,CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.379961,0.850617,0.069741,0.235762,0.003887



BEST STATIC SUBSET BY SIZE — FULL EXPERT NAMES


,N_Experts,Experts,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,2,CAT_SMOOTH | XGB_HARD_DISAGREE_CF,0.379458,0.850627,0.069768,0.236212,0.006059
1,3,CAT_SMOOTH | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.379791,0.850604,0.069747,0.235854,0.004247
2,4,CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.379961,0.850617,0.069741,0.235762,0.003887
3,5,XGB_SMOOTH | CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | XGB_HARD_DISAGREE_CF,0.380049,0.850681,0.069728,0.235674,0.003105
4,6,XGB_SMOOTH | CAT_SMOOTH | MLP_REPAIRED | AUTOINT_REPAIRED | MLP_FOCAL | XGB_HARD_DISAGREE_CF,0.380049,0.850525,0.070550,0.243705,0.027460


In [13]:
# ============================================================
# MODULE N7A — FREEZE FINAL PREDICTIVE BACKBONE
# ============================================================

FINAL_BACKBONE_ID = "LSA_HE_5"

FINAL_EXPERTS = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


FINAL_SELECTION_REASON = {

    "primary_metric":
        "PR_AUC",

    "secondary_metrics":
        [
            "ROC_AUC",
            "Brier",
            "NLL",
            "ECE"
        ],

    "selection_summary":
        (
            "Five-expert heterogeneous ensemble selected "
            "after discovery-stage DEV analysis. "
            "It provided the best balance between PR-AUC, "
            "ROC-AUC, calibration, and architectural complexity."
        ),

    "routing_decision":
        (
            "Dynamic routing rejected because risk-bin, "
            "local-hard, local-soft, top-2, and residual "
            "local-competence routing did not outperform "
            "static heterogeneous aggregation."
        ),

    "focal_decision":
        (
            "MLP_FOCAL excluded from predictive backbone "
            "because its negligible PR-AUC contribution was "
            "accompanied by substantial calibration degradation."
        ),

    "development_external_data_used":
        False,

    "development_holdout_used":
        False,

    "important_provenance_note":
        (
            "The 2015 holdout was inspected in earlier project "
            "development and is therefore not an untouched "
            "validation set. It remains excluded from current "
            "architecture development and will only be used "
            "as a secondary post-freeze evaluation set."
        ),
}


P_FINAL_DISCOVERY = np.mean(

    np.column_stack([

        OOF_STORE[
            model_id
        ]

        for model_id
        in FINAL_EXPERTS

    ]),

    axis=1
)


FINAL_DISCOVERY_METRICS = probability_metrics(

    y_DEV,
    P_FINAL_DISCOVERY
)


print(
    "=" * 145
)

print(
    "MODULE N7A — FROZEN PREDICTIVE BACKBONE"
)

print(
    "=" * 145
)


print(
    "\nBackbone:",
    FINAL_BACKBONE_ID
)


print(
    "\nFrozen experts:"
)


for i, expert in enumerate(
    FINAL_EXPERTS,
    start=1
):

    print(
        f"  {i}. {expert}"
    )


print(
    "\nDiscovery-stage metrics:"
)


for key, value in (
    FINAL_DISCOVERY_METRICS.items()
):

    print(
        f"{key:10s}: {value:.6f}"
    )


print(
    "\nIMPORTANT:"
)

print(
    "These are discovery-stage OOF metrics."
)

print(
    "They are NOT yet the final confirmatory estimate."
)


# save frozen definition

with open(

    LOG_DIR
    /
    "n7a_frozen_backbone.json",

    "w"

) as f:

    json.dump(

        {
            "backbone_id":
                FINAL_BACKBONE_ID,

            "experts":
                FINAL_EXPERTS,

            "discovery_metrics":
                {
                    key:
                        float(value)

                    for key, value
                    in FINAL_DISCOVERY_METRICS.items()
                },

            "selection_reason":
                FINAL_SELECTION_REASON,
        },

        f,

        indent=2
    )


np.save(

    PRED_DIR
    /
    "N7A_FINAL_DISCOVERY_OOF.npy",

    P_FINAL_DISCOVERY
)


print(
    "\nArchitecture frozen."
)

print(
    "No further expert addition/removal should be based "
    "on DEV performance after this point."
)

MODULE N7A — FROZEN PREDICTIVE BACKBONE

Backbone: LSA_HE_5

Frozen experts:
  1. XGB_SMOOTH
  2. CAT_SMOOTH
  3. MLP_REPAIRED
  4. AUTOINT_REPAIRED
  5. XGB_HARD_DISAGREE_CF

Discovery-stage metrics:
PR_AUC    : 0.380049
ROC_AUC   : 0.850681
Brier     : 0.069728
NLL       : 0.235674
ECE       : 0.003105

IMPORTANT:
These are discovery-stage OOF metrics.
They are NOT yet the final confirmatory estimate.

Architecture frozen.
No further expert addition/removal should be based on DEV performance after this point.


In [14]:
# ============================================================
# MODULE N7B — STRICT NESTED RECONSTRUCTION OF FROZEN LSA-HE-5
#
# FROZEN ARCHITECTURE
# ------------------------------------------------------------
# 1. XGB_SMOOTH
# 2. CAT_SMOOTH
# 3. MLP_REPAIRED
# 4. AUTOINT_REPAIRED
# 5. XGB_HARD_DISAGREE_CF
#
# IMPORTANT
# ------------------------------------------------------------
# Expert composition is FROZEN.
#
# This module DOES NOT select experts.
#
# It reconstructs every expert strictly inside each outer fold:
#
#   - tree experts trained on outer train only
#   - neural epoch chosen using inner validation only
#   - hard/disagreement weights generated by inner OOF only
#
# The resulting OOF estimate is a strict reconstruction of
# the frozen model-training protocol.
#
# It is NOT a fully independent external confirmation because
# the architecture was discovered using the DEV population.
# ============================================================


import gc
import time
import json

import numpy as np
import pandas as pd


# ============================================================
# N7B.1 FIXED ARCHITECTURE
# ============================================================

FINAL_EXPERTS_N7B = [

    "XGB_SMOOTH",

    "CAT_SMOOTH",

    "MLP_REPAIRED",

    "AUTOINT_REPAIRED",

    "XGB_HARD_DISAGREE_CF",
]


assert FINAL_EXPERTS_N7B == FINAL_EXPERTS


# ============================================================
# N7B.2 OOF STORAGE
# ============================================================

OOF_N7B = {

    model_id:
        np.full(
            len(y_DEV),
            np.nan,
            dtype=np.float32
        )

    for model_id
    in FINAL_EXPERTS_N7B
}


OOF_LSAHE5_N7B = np.full(

    len(y_DEV),

    np.nan,

    dtype=np.float32
)


FOLD_ROWS_N7B = []

NEURAL_EPOCH_ROWS_N7B = []

HARD_WEIGHT_ROWS_N7B = []


total_start_n7b = time.perf_counter()


# ============================================================
# N7B.3 OUTER LOOP
# ============================================================

for fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 150
    )

    print(
        f"N7B — OUTER FOLD {fold}"
    )

    print(
        "=" * 150
    )


    outer_train_idx = np.where(

        DEV_FOLD != fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == fold

    )[0]


    X_train = X_DEV.iloc[
        outer_train_idx
    ]


    X_valid = X_DEV.iloc[
        outer_valid_idx
    ]


    y_train = y_DEV[
        outer_train_idx
    ]


    y_valid = y_DEV[
        outer_valid_idx
    ]


    fold_predictions = {}


    # ========================================================
    # EXPERT 1 — XGB SMOOTH
    # ========================================================

    print(
        "\n[1/5] XGB_SMOOTH"
    )


    xgb = XGBClassifier(

        objective="binary:logistic",

        eval_metric="aucpr",

        n_estimators=900,

        learning_rate=0.025,

        max_depth=3,

        min_child_weight=15,

        subsample=0.90,

        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,

        reg_lambda=7.0,

        tree_method="hist",

        random_state=
            SEED
            +
            41000
            +
            fold,

        n_jobs=-1,
    )


    xgb.fit(
        X_train,
        y_train
    )


    p_xgb = xgb.predict_proba(
        X_valid
    )[:, 1]


    fold_predictions[
        "XGB_SMOOTH"
    ] = p_xgb


    del xgb

    gc.collect()


    # ========================================================
    # EXPERT 2 — CATBOOST SMOOTH
    # ========================================================

    print(
        "[2/5] CAT_SMOOTH"
    )


    cat = CatBoostClassifier(

        iterations=900,

        learning_rate=0.03,

        depth=5,

        loss_function="Logloss",

        eval_metric="PRAUC",

        l2_leaf_reg=8.0,

        random_seed=
            SEED
            +
            42000
            +
            fold,

        verbose=False,

        allow_writing_files=False,

        thread_count=-1,
    )


    cat.fit(
        X_train,
        y_train
    )


    p_cat = cat.predict_proba(
        X_valid
    )[:, 1]


    fold_predictions[
        "CAT_SMOOTH"
    ] = p_cat


    del cat

    gc.collect()


    # ========================================================
    # EXPERT 3 — MLP REPAIRED
    #
    # Inner split selects epoch.
    # Fresh model retrained on full outer train.
    # ========================================================

    print(
        "[3/5] MLP_REPAIRED"
    )


    mlp_factory = MODEL_FACTORIES_N3A[
        "MLP_GLOBAL"
    ]


    mlp_seed = (
        SEED
        +
        43000
        +
        fold
    )


    mlp_selection = select_epoch_n3r(

        mlp_factory,

        outer_train_idx,

        seed=
            mlp_seed
    )


    mlp_epoch = mlp_selection[
        "best_epoch"
    ]


    (
        p_mlp,
        model_mlp,
        scaler_mlp

    ) = retrain_outer_n3r(

        mlp_factory,

        outer_train_idx,

        outer_valid_idx,

        n_epochs=
            mlp_epoch,

        seed=
            mlp_seed + 100
    )


    fold_predictions[
        "MLP_REPAIRED"
    ] = p_mlp


    NEURAL_EPOCH_ROWS_N7B.append({

        "Fold":
            fold,

        "Expert":
            "MLP_REPAIRED",

        "Selected_Epoch":
            mlp_epoch,

        "Inner_Best_PR":
            mlp_selection[
                "best_inner_pr"
            ],
    })


    del model_mlp
    del scaler_mlp

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    # ========================================================
    # EXPERT 4 — AUTOINTERACTION REPAIRED
    # ========================================================

    print(
        "[4/5] AUTOINT_REPAIRED"
    )


    autoint_factory = MODEL_FACTORIES_N3A[
        "AUTOINT_GLOBAL"
    ]


    autoint_seed = (
        SEED
        +
        44000
        +
        fold
    )


    autoint_selection = select_epoch_n3r(

        autoint_factory,

        outer_train_idx,

        seed=
            autoint_seed
    )


    autoint_epoch = autoint_selection[
        "best_epoch"
    ]


    (
        p_autoint,
        model_autoint,
        scaler_autoint

    ) = retrain_outer_n3r(

        autoint_factory,

        outer_train_idx,

        outer_valid_idx,

        n_epochs=
            autoint_epoch,

        seed=
            autoint_seed + 100
    )


    fold_predictions[
        "AUTOINT_REPAIRED"
    ] = p_autoint


    NEURAL_EPOCH_ROWS_N7B.append({

        "Fold":
            fold,

        "Expert":
            "AUTOINT_REPAIRED",

        "Selected_Epoch":
            autoint_epoch,

        "Inner_Best_PR":
            autoint_selection[
                "best_inner_pr"
            ],
    })


    del model_autoint
    del scaler_autoint

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    # ========================================================
    # EXPERT 5 — CROSS-FITTED HARD + DISAGREEMENT SPECIALIST
    #
    # Difficulty weights are recreated INSIDE current
    # outer-training partition.
    # ========================================================

    print(
        "[5/5] XGB_HARD_DISAGREE_CF"
    )


    (
        weights_dict,
        diagnostics

    ) = build_inner_difficulty_n4a(

        outer_train_idx,
        fold
    )


    hard_weights = weights_dict[
        "XGB_HARD_DISAGREE_CF"
    ]


    HARD_WEIGHT_ROWS_N7B.append({

        "Fold":
            fold,

        "Weight_Mean":
            float(
                np.mean(
                    hard_weights
                )
            ),

        "Weight_SD":
            float(
                np.std(
                    hard_weights
                )
            ),

        "Weight_Min":
            float(
                np.min(
                    hard_weights
                )
            ),

        "Weight_Max":
            float(
                np.max(
                    hard_weights
                )
            ),
    })


    hard_model = make_specialist_xgb_n4a(

        SEED
        +
        45000
        +
        fold
    )


    hard_model.fit(

        X_train,

        y_train,

        sample_weight=
            hard_weights
    )


    p_hard = hard_model.predict_proba(

        X_valid

    )[:, 1]


    fold_predictions[
        "XGB_HARD_DISAGREE_CF"
    ] = p_hard


    del hard_model
    del weights_dict
    del diagnostics

    gc.collect()


    # ========================================================
    # STORE EXPERT OOF
    # ========================================================

    for model_id in (
        FINAL_EXPERTS_N7B
    ):

        pred = np.asarray(

            fold_predictions[
                model_id
            ],

            dtype=np.float32
        )


        OOF_N7B[
            model_id
        ][
            outer_valid_idx
        ] = pred


    # ========================================================
    # FINAL EQUAL-WEIGHT FROZEN ENSEMBLE
    # ========================================================

    P_fold = np.column_stack([

        fold_predictions[
            model_id
        ]

        for model_id
        in FINAL_EXPERTS_N7B
    ])


    p_ensemble = np.mean(

        P_fold,

        axis=1
    )


    OOF_LSAHE5_N7B[
        outer_valid_idx
    ] = p_ensemble


    # ========================================================
    # FOLD METRICS — INDIVIDUAL EXPERTS
    # ========================================================

    for model_id in (
        FINAL_EXPERTS_N7B
    ):

        metrics = probability_metrics(

            y_valid,

            fold_predictions[
                model_id
            ]
        )


        FOLD_ROWS_N7B.append({

            "Fold":
                fold,

            "System":
                model_id,

            "PR_AUC":
                metrics[
                    "PR_AUC"
                ],

            "ROC_AUC":
                metrics[
                    "ROC_AUC"
                ],

            "Brier":
                metrics[
                    "Brier"
                ],

            "NLL":
                metrics[
                    "NLL"
                ],

            "ECE":
                metrics[
                    "ECE"
                ],
        })


    # ========================================================
    # FOLD METRICS — FINAL BACKBONE
    # ========================================================

    ensemble_metrics = probability_metrics(

        y_valid,
        p_ensemble
    )


    FOLD_ROWS_N7B.append({

        "Fold":
            fold,

        "System":
            "LSA_HE_5",

        "PR_AUC":
            ensemble_metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            ensemble_metrics[
                "ROC_AUC"
            ],

        "Brier":
            ensemble_metrics[
                "Brier"
            ],

        "NLL":
            ensemble_metrics[
                "NLL"
            ],

        "ECE":
            ensemble_metrics[
                "ECE"
            ],
    })


    print(
        "\nFOLD RESULT"
    )


    print(

        f"LSA-HE-5 | "
        f"PR={ensemble_metrics['PR_AUC']:.6f} | "
        f"ROC={ensemble_metrics['ROC_AUC']:.6f} | "
        f"Brier={ensemble_metrics['Brier']:.6f} | "
        f"NLL={ensemble_metrics['NLL']:.6f} | "
        f"ECE={ensemble_metrics['ECE']:.6f}"
    )


    del fold_predictions
    del P_fold

    gc.collect()


# ============================================================
# N7B.4 COMPLETENESS CHECK
# ============================================================

if np.isnan(
    OOF_LSAHE5_N7B
).any():

    raise RuntimeError(
        "Incomplete LSA-HE-5 OOF."
    )


for model_id in (
    FINAL_EXPERTS_N7B
):

    if np.isnan(
        OOF_N7B[
            model_id
        ]
    ).any():

        raise RuntimeError(

            f"Incomplete OOF for {model_id}"
        )


# ============================================================
# N7B.5 POOLED STRICT RESULTS
# ============================================================

pooled_rows = []


for model_id in (
    FINAL_EXPERTS_N7B
):

    m = probability_metrics(

        y_DEV,

        OOF_N7B[
            model_id
        ]
    )


    pooled_rows.append({

        "System":
            model_id,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],
    })


m_final_n7b = probability_metrics(

    y_DEV,

    OOF_LSAHE5_N7B
)


pooled_rows.append({

    "System":
        "LSA_HE_5",

    "PR_AUC":
        m_final_n7b[
            "PR_AUC"
        ],

    "ROC_AUC":
        m_final_n7b[
            "ROC_AUC"
        ],

    "Brier":
        m_final_n7b[
            "Brier"
        ],

    "NLL":
        m_final_n7b[
            "NLL"
        ],

    "ECE":
        m_final_n7b[
            "ECE"
        ],
})


POOLED_RESULTS_N7B = (

    pd.DataFrame(
        pooled_rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE N7B — STRICT NESTED RECONSTRUCTION"
)

print(
    "=" * 150
)


display(
    POOLED_RESULTS_N7B
)


# ============================================================
# N7B.6 DISCOVERY VS STRICT RECONSTRUCTION
# ============================================================

DISCOVERY_VS_N7B = pd.DataFrame([

    {
        "Estimate":
            "Discovery OOF",

        "PR_AUC":
            FINAL_DISCOVERY_METRICS[
                "PR_AUC"
            ],

        "ROC_AUC":
            FINAL_DISCOVERY_METRICS[
                "ROC_AUC"
            ],

        "Brier":
            FINAL_DISCOVERY_METRICS[
                "Brier"
            ],

        "NLL":
            FINAL_DISCOVERY_METRICS[
                "NLL"
            ],

        "ECE":
            FINAL_DISCOVERY_METRICS[
                "ECE"
            ],
    },

    {
        "Estimate":
            "Strict nested reconstruction",

        "PR_AUC":
            m_final_n7b[
                "PR_AUC"
            ],

        "ROC_AUC":
            m_final_n7b[
                "ROC_AUC"
            ],

        "Brier":
            m_final_n7b[
                "Brier"
            ],

        "NLL":
            m_final_n7b[
                "NLL"
            ],

        "ECE":
            m_final_n7b[
                "ECE"
            ],
    },
])


DISCOVERY_VS_N7B[
    "Delta_PR_vs_Discovery"
] = (

    DISCOVERY_VS_N7B[
        "PR_AUC"
    ]

    -
    FINAL_DISCOVERY_METRICS[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 150
)

print(
    "N7B — DISCOVERY VS STRICT RECONSTRUCTION"
)

print(
    "=" * 150
)


display(
    DISCOVERY_VS_N7B
)


# ============================================================
# N7B.7 FOLD STABILITY
# ============================================================

FOLD_RESULTS_N7B = pd.DataFrame(

    FOLD_ROWS_N7B
)


FOLD_STABILITY_N7B = (

    FOLD_RESULTS_N7B

    .groupby(
        "System"
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_SD=(
            "ROC_AUC",
            "std"
        ),

        Brier_Mean=(
            "Brier",
            "mean"
        ),

        NLL_Mean=(
            "NLL",
            "mean"
        ),

        ECE_Mean=(
            "ECE",
            "mean"
        ),
    )

    .reset_index()

    .sort_values(
        "PR_Mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 150
)

print(
    "N7B — FOLD STABILITY"
)

print(
    "=" * 150
)


display(
    FOLD_STABILITY_N7B
)


# ============================================================
# N7B.8 NEURAL EPOCHS
# ============================================================

NEURAL_EPOCHS_N7B = pd.DataFrame(

    NEURAL_EPOCH_ROWS_N7B
)


print(
    "\n"
    + "=" * 150
)

print(
    "N7B — NESTED NEURAL EPOCH SELECTION"
)

print(
    "=" * 150
)


display(
    NEURAL_EPOCHS_N7B
)


# ============================================================
# N7B.9 HARD SPECIALIST WEIGHTS
# ============================================================

HARD_WEIGHTS_N7B = pd.DataFrame(

    HARD_WEIGHT_ROWS_N7B
)


print(
    "\n"
    + "=" * 150
)

print(
    "N7B — HARD-DISAGREEMENT WEIGHT DIAGNOSTICS"
)

print(
    "=" * 150
)


display(
    HARD_WEIGHTS_N7B
)


# ============================================================
# N7B.10 EXPERT CORRELATION
# ============================================================

P_EXPERTS_N7B = np.column_stack([

    OOF_N7B[
        model_id
    ]

    for model_id
    in FINAL_EXPERTS_N7B
])


CORR_N7B = pd.DataFrame(

    np.corrcoef(
        P_EXPERTS_N7B.T
    ),

    index=
        FINAL_EXPERTS_N7B,

    columns=
        FINAL_EXPERTS_N7B,
)


print(
    "\n"
    + "=" * 150
)

print(
    "N7B — FROZEN EXPERT CORRELATION"
)

print(
    "=" * 150
)


display(
    CORR_N7B.round(4)
)


# ============================================================
# N7B.11 COMPARE WITH HISTORICAL BENCHMARKS
# ============================================================

COMPARISON_N7B = pd.DataFrame([

    {
        "System":
            "Old HGB",

        "PR_AUC":
            0.377012,

        "ROC_AUC":
            0.849628,
    },

    {
        "System":
            "Old Core5",

        "PR_AUC":
            0.379751,

        "ROC_AUC":
            0.850608,
    },

    {
        "System":
            "Frozen LSA-HE-5 strict reconstruction",

        "PR_AUC":
            m_final_n7b[
                "PR_AUC"
            ],

        "ROC_AUC":
            m_final_n7b[
                "ROC_AUC"
            ],
    },
])


COMPARISON_N7B[
    "Delta_PR_vs_Old_Core5"
] = (

    COMPARISON_N7B[
        "PR_AUC"
    ]

    -
    0.379751
)


print(
    "\n"
    + "=" * 150
)

print(
    "N7B — HISTORICAL BENCHMARK COMPARISON"
)

print(
    "=" * 150
)


display(
    COMPARISON_N7B
)


# ============================================================
# N7B.12 SAVE
# ============================================================

POOLED_RESULTS_N7B.to_csv(

    TAB_DIR
    /
    "n7b_strict_nested_results.csv",

    index=False
)


DISCOVERY_VS_N7B.to_csv(

    TAB_DIR
    /
    "n7b_discovery_vs_nested.csv",

    index=False
)


FOLD_RESULTS_N7B.to_csv(

    TAB_DIR
    /
    "n7b_fold_results.csv",

    index=False
)


FOLD_STABILITY_N7B.to_csv(

    TAB_DIR
    /
    "n7b_fold_stability.csv",

    index=False
)


NEURAL_EPOCHS_N7B.to_csv(

    TAB_DIR
    /
    "n7b_neural_epochs.csv",

    index=False
)


HARD_WEIGHTS_N7B.to_csv(

    TAB_DIR
    /
    "n7b_hard_weight_diagnostics.csv",

    index=False
)


CORR_N7B.to_csv(

    TAB_DIR
    /
    "n7b_expert_correlation.csv"
)


COMPARISON_N7B.to_csv(

    TAB_DIR
    /
    "n7b_historical_comparison.csv",

    index=False
)


for model_id in (
    FINAL_EXPERTS_N7B
):

    np.save(

        PRED_DIR
        /
        f"N7B_{model_id}_OOF.npy",

        OOF_N7B[
            model_id
        ]
    )


np.save(

    PRED_DIR
    /
    "N7B_LSA_HE_5_OOF.npy",

    OOF_LSAHE5_N7B
)


with open(

    LOG_DIR
    /
    "n7b_strict_reconstruction.json",

    "w"

) as f:

    json.dump(

        {
            "architecture":
                "LSA_HE_5",

            "experts":
                FINAL_EXPERTS_N7B,

            "protocol":
                (
                    "Strict nested reconstruction of frozen "
                    "architecture."
                ),

            "architecture_selection_locked":
                True,

            "holdout_used_current_development":
                False,

            "external_2020_used_current_development":
                False,

            "historical_provenance":
                (
                    "The 2015 holdout and BRFSS-2020 had been "
                    "examined in prior project stages and are "
                    "not claimed as never-seen datasets."
                ),

            "metrics":
                {
                    k:
                        float(v)

                    for k, v
                    in m_final_n7b.items()
                },
        },

        f,

        indent=2
    )


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE N7B COMPLETE"
)

print(
    "=" * 150
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - total_start_n7b:.1f}s"
)


print(
    "\nArchitecture remains frozen."
)

print(
    "No expert selection performed in N7B."
)


N7B — OUTER FOLD 0

[1/5] XGB_SMOOTH
[2/5] CAT_SMOOTH
[3/5] MLP_REPAIRED
[4/5] AUTOINT_REPAIRED
[5/5] XGB_HARD_DISAGREE_CF

  Building inner difficulty signals for outer fold 0...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

FOLD RESULT
LSA-HE-5 | PR=0.378993 | ROC=0.852518 | Brier=0.069673 | NLL=0.234947 | ECE=0.002848

N7B — OUTER FOLD 1

[1/5] XGB_SMOOTH
[2/5] CAT_SMOOTH
[3/5] MLP_REPAIRED
[4/5] AUTOINT_REPAIRED
[5/5] XGB_HARD_DISAGREE_CF

  Building inner difficulty signals for outer fold 1...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

FOLD RESULT
LSA-HE-5 | PR=0.381858 | ROC=0.850508 | Brier=0.069755 | NLL=0.235752 | ECE=0.004541

N7B — OUTER FOLD 2

[1/5] XGB_SMOOTH
[2/5] CAT_SMOOTH
[3/5] MLP_REPAIRED
[4/5] AUTOINT_REPAIRED
[5/5] XGB_HARD_DISAGREE_CF

  Building inner difficulty signals for outer fold 2...
    inner fold 0 complete
    inner fold 1 complete
 

,System,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,LSA_HE_5,0.379645,0.850617,0.069730,0.235667,0.002774
1,XGB_HARD_DISAGREE_CF,0.379299,0.850415,0.069907,0.237708,0.011919
2,XGB_SMOOTH,0.378606,0.850151,0.069811,0.235886,0.002514
3,CAT_SMOOTH,0.377730,0.850254,0.069808,0.235825,0.001903
4,AUTOINT_REPAIRED,0.375803,0.849055,0.069970,0.236755,0.005183
5,MLP_REPAIRED,0.372623,0.847616,0.070172,0.237435,0.004625



N7B — DISCOVERY VS STRICT RECONSTRUCTION


,Estimate,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_Discovery
0,Discovery OOF,0.380049,0.850681,0.069728,0.235674,0.003105,0.000000
1,Strict nested reconstruction,0.379645,0.850617,0.069730,0.235667,0.002774,-0.000404



N7B — FOLD STABILITY


...


N7B — FROZEN EXPERT CORRELATION


,XGB_SMOOTH,CAT_SMOOTH,MLP_REPAIRED,AUTOINT_REPAIRED,XGB_HARD_DISAGREE_CF
XGB_SMOOTH,1.0000,0.9940,0.9787,0.9876,0.9978
CAT_SMOOTH,0.9940,1.0000,0.9783,0.9861,0.9943
MLP_REPAIRED,0.9787,0.9783,1.0000,0.9783,0.9799
AUTOINT_REPAIRED,0.9876,0.9861,0.9783,1.0000,0.9871
XGB_HARD_DISAGREE_CF,0.9978,0.9943,0.9799,0.9871,1.0000



N7B — HISTORICAL BENCHMARK COMPARISON


,System,PR_AUC,ROC_AUC,Delta_PR_vs_Old_Core5
0,Old HGB,0.377012,0.849628,-0.002739
1,Old Core5,0.379751,0.850608,0.000000
2,Frozen LSA-HE-5 strict reconstruction,0.379645,0.850617,-0.000106



MODULE N7B COMPLETE

Runtime: 1273.0s

Architecture remains frozen.
No expert selection performed in N7B.


In [15]:
# ============================================================
# MODULE N8 — STRICT CROSS-FITTED CALIBRATION
#
# Frozen predictor:
#   N7B LSA-HE-5
#
# Compare:
#   1. RAW
#   2. PLATT
#   3. ISOTONIC
#
# Calibration fitting is cross-fitted:
#
# other 4 outer folds -> fit calibrator
# current outer fold  -> evaluate calibrator
#
# No architecture tuning.
# No expert tuning.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# N8.1 INPUT
# ============================================================

P_RAW_N8 = np.asarray(
    OOF_LSAHE5_N7B,
    dtype=float
)


assert not np.isnan(
    P_RAW_N8
).any()


EPS_N8 = 1e-7


# ============================================================
# N8.2 LOGIT HELPER
# ============================================================

def logit_n8(
    p
):

    p = np.clip(
        p,
        EPS_N8,
        1 - EPS_N8
    )


    return np.log(
        p / (1 - p)
    )


# ============================================================
# N8.3 STORAGE
# ============================================================

P_PLATT_N8 = np.full(

    len(y_DEV),

    np.nan,

    dtype=float
)


P_ISO_N8 = np.full(

    len(y_DEV),

    np.nan,

    dtype=float
)


CALIBRATION_FOLD_ROWS_N8 = []


# ============================================================
# N8.4 CROSS-FITTED CALIBRATION
# ============================================================

for fold in range(
    N_FOLDS
):

    train_mask = (
        DEV_FOLD != fold
    )


    valid_mask = (
        DEV_FOLD == fold
    )


    y_cal_train = y_DEV[
        train_mask
    ]


    p_cal_train = P_RAW_N8[
        train_mask
    ]


    p_cal_valid = P_RAW_N8[
        valid_mask
    ]


    y_cal_valid = y_DEV[
        valid_mask
    ]


    # ========================================================
    # PLATT
    # ========================================================

    X_train_logit = logit_n8(

        p_cal_train

    ).reshape(
        -1,
        1
    )


    X_valid_logit = logit_n8(

        p_cal_valid

    ).reshape(
        -1,
        1
    )


    platt = LogisticRegression(

        penalty=None,

        solver="lbfgs",

        max_iter=2000,
    )


    platt.fit(

        X_train_logit,

        y_cal_train
    )


    pred_platt = platt.predict_proba(

        X_valid_logit

    )[:, 1]


    P_PLATT_N8[
        valid_mask
    ] = pred_platt


    # ========================================================
    # ISOTONIC
    # ========================================================

    iso = IsotonicRegression(

        y_min=EPS_N8,

        y_max=1 - EPS_N8,

        out_of_bounds="clip"
    )


    iso.fit(

        p_cal_train,

        y_cal_train
    )


    pred_iso = iso.predict(

        p_cal_valid
    )


    P_ISO_N8[
        valid_mask
    ] = pred_iso


    # ========================================================
    # FOLD RESULTS
    # ========================================================

    systems_fold = {

        "RAW":
            p_cal_valid,

        "PLATT":
            pred_platt,

        "ISOTONIC":
            pred_iso,
    }


    for system_name, pred in (
        systems_fold.items()
    ):

        m = probability_metrics(

            y_cal_valid,
            pred
        )


        CALIBRATION_FOLD_ROWS_N8.append({

            "Fold":
                fold,

            "System":
                system_name,

            "PR_AUC":
                m[
                    "PR_AUC"
                ],

            "ROC_AUC":
                m[
                    "ROC_AUC"
                ],

            "Brier":
                m[
                    "Brier"
                ],

            "NLL":
                m[
                    "NLL"
                ],

            "ECE":
                m[
                    "ECE"
                ],
        })


# ============================================================
# N8.5 COMPLETENESS
# ============================================================

for name, arr in {

    "PLATT":
        P_PLATT_N8,

    "ISOTONIC":
        P_ISO_N8,

}.items():

    if np.isnan(
        arr
    ).any():

        raise RuntimeError(
            f"Incomplete predictions: {name}"
        )


# ============================================================
# N8.6 CALIBRATION INTERCEPT + SLOPE
#
# Logistic calibration model:
#
# logit(y) = intercept + slope * logit(p)
#
# Ideal:
#   intercept = 0
#   slope = 1
# ============================================================

def calibration_intercept_slope_n8(
    y,
    p
):

    X = logit_n8(
        p
    ).reshape(
        -1,
        1
    )


    model = LogisticRegression(

        penalty=None,

        solver="lbfgs",

        max_iter=2000,
    )


    model.fit(
        X,
        y
    )


    return (

        float(
            model.intercept_[0]
        ),

        float(
            model.coef_[0, 0]
        )
    )


# ============================================================
# N8.7 POOLED RESULTS
# ============================================================

SYSTEMS_N8 = {

    "RAW":
        P_RAW_N8,

    "PLATT":
        P_PLATT_N8,

    "ISOTONIC":
        P_ISO_N8,
}


rows = []


for name, pred in (
    SYSTEMS_N8.items()
):

    m = probability_metrics(

        y_DEV,
        pred
    )


    intercept, slope = (
        calibration_intercept_slope_n8(

            y_DEV,
            pred
        )
    )


    rows.append({

        "System":
            name,

        "PR_AUC":
            m[
                "PR_AUC"
            ],

        "ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Brier":
            m[
                "Brier"
            ],

        "NLL":
            m[
                "NLL"
            ],

        "ECE":
            m[
                "ECE"
            ],

        "Mean_P":
            float(
                np.mean(
                    pred
                )
            ),

        "Observed_Prevalence":
            float(
                np.mean(
                    y_DEV
                )
            ),

        "Calibration_Intercept":
            intercept,

        "Calibration_Slope":
            slope,
    })


CALIBRATION_RESULTS_N8 = pd.DataFrame(
    rows
)


print(
    "=" * 150
)

print(
    "MODULE N8 — CROSS-FITTED CALIBRATION RESULTS"
)

print(
    "=" * 150
)


display(
    CALIBRATION_RESULTS_N8
)


# ============================================================
# N8.8 DELTA VS RAW
# ============================================================

raw_row = (

    CALIBRATION_RESULTS_N8

    .loc[
        CALIBRATION_RESULTS_N8[
            "System"
        ]
        ==
        "RAW"
    ]

    .iloc[0]
)


CALIBRATION_DELTA_N8 = (

    CALIBRATION_RESULTS_N8
    .copy()
)


for metric in [

    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
    "ECE",

]:

    CALIBRATION_DELTA_N8[

        f"Delta_{metric}_vs_RAW"

    ] = (

        CALIBRATION_DELTA_N8[
            metric
        ]

        -
        raw_row[
            metric
        ]
    )


print(
    "\n"
    + "=" * 150
)

print(
    "N8 — CHANGE RELATIVE TO RAW"
)

print(
    "=" * 150
)


display(
    CALIBRATION_DELTA_N8
)


# ============================================================
# N8.9 FOLD STABILITY
# ============================================================

CALIBRATION_FOLDS_N8 = pd.DataFrame(

    CALIBRATION_FOLD_ROWS_N8
)


CALIBRATION_STABILITY_N8 = (

    CALIBRATION_FOLDS_N8

    .groupby(
        "System"
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        Brier_Mean=(
            "Brier",
            "mean"
        ),

        NLL_Mean=(
            "NLL",
            "mean"
        ),

        ECE_Mean=(
            "ECE",
            "mean"
        ),
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 150
)

print(
    "N8 — CALIBRATION FOLD STABILITY"
)

print(
    "=" * 150
)


display(
    CALIBRATION_STABILITY_N8
)


# ============================================================
# N8.10 SIMPLE SELECTION RULE
#
# Calibration method selected primarily by:
#
#   Brier + NLL + ECE
#
# PR / ROC should not materially deteriorate.
# ============================================================

eligible_n8 = (

    CALIBRATION_RESULTS_N8

    .copy()
)


eligible_n8[
    "CalibrationScore"
] = (

    eligible_n8[
        "Brier"
    ]

    +

    0.25
    *
    eligible_n8[
        "NLL"
    ]

    +

    0.50
    *
    eligible_n8[
        "ECE"
    ]
)


CALIBRATION_RANK_N8 = (

    eligible_n8

    .sort_values(
        "CalibrationScore"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 150
)

print(
    "N8 — CALIBRATION QUALITY RANKING"
)

print(
    "=" * 150
)


display(
    CALIBRATION_RANK_N8
)


# ============================================================
# N8.11 SAVE
# ============================================================

CALIBRATION_RESULTS_N8.to_csv(

    TAB_DIR
    /
    "n8_calibration_results.csv",

    index=False
)


CALIBRATION_DELTA_N8.to_csv(

    TAB_DIR
    /
    "n8_calibration_delta.csv",

    index=False
)


CALIBRATION_FOLDS_N8.to_csv(

    TAB_DIR
    /
    "n8_calibration_folds.csv",

    index=False
)


CALIBRATION_STABILITY_N8.to_csv(

    TAB_DIR
    /
    "n8_calibration_stability.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "N8_RAW_OOF.npy",

    P_RAW_N8
)


np.save(

    PRED_DIR
    /
    "N8_PLATT_OOF.npy",

    P_PLATT_N8
)


np.save(

    PRED_DIR
    /
    "N8_ISOTONIC_OOF.npy",

    P_ISO_N8
)


print(
    "\n"
    + "=" * 150
)

print(
    "MODULE N8 COMPLETE"
)

print(
    "=" * 150
)


print(
    "\nPredictive architecture remains frozen."
)

print(
    "Next: N9 — LPNS / disagreement / confidence reliability."
)

MODULE N8 — CROSS-FITTED CALIBRATION RESULTS


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Mean_P,Observed_Prevalence,Calibration_Intercept,Calibration_Slope
0,RAW,0.379645,0.850617,0.069730,0.235667,0.002774,0.096437,0.094184,0.004235,1.023259
1,PLATT,0.379490,0.850587,0.069734,0.235625,0.002761,0.094203,0.094184,-0.000488,0.999927
2,ISOTONIC,0.375987,0.850021,0.069751,0.235833,0.001184,0.094189,0.094184,-0.013707,0.991282



N8 — CHANGE RELATIVE TO RAW


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Mean_P,Observed_Prevalence,Calibration_Intercept,Calibration_Slope,Delta_PR_AUC_vs_RAW,Delta_ROC_AUC_vs_RAW,Delta_Brier_vs_RAW,Delta_NLL_vs_RAW,Delta_ECE_vs_RAW
0,RAW,0.379645,0.850617,0.069730,0.235667,0.002774,0.096437,0.094184,0.004235,1.023259,0.000000,0.000000,0.000000,0.000000,0.000000
1,PLATT,0.379490,0.850587,0.069734,0.235625,0.002761,0.094203,0.094184,-0.000488,0.999927,-0.000155,-0.000029,0.000004,-0.000041,-0.000013
2,ISOTONIC,0.375987,0.850021,0.069751,0.235833,0.001184,0.094189,0.094184,-0.013707,0.991282,-0.003658,-0.000595,0.000021,0.000166,-0.001590



N8 — CALIBRATION FOLD STABILITY


,System,PR_Mean,PR_SD,ROC_Mean,Brier_Mean,NLL_Mean,ECE_Mean
0,ISOTONIC,0.372377,0.004825,0.850506,0.069751,0.235833,0.003733
1,PLATT,0.380295,0.005796,0.850667,0.069734,0.235625,0.004216
2,RAW,0.380295,0.005796,0.850667,0.069730,0.235667,0.004119



N8 — CALIBRATION QUALITY RANKING


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Mean_P,Observed_Prevalence,Calibration_Intercept,Calibration_Slope,CalibrationScore
0,ISOTONIC,0.375987,0.850021,0.069751,0.235833,0.001184,0.094189,0.094184,-0.013707,0.991282,0.129301
1,PLATT,0.379490,0.850587,0.069734,0.235625,0.002761,0.094203,0.094184,-0.000488,0.999927,0.130021
2,RAW,0.379645,0.850617,0.069730,0.235667,0.002774,0.096437,0.094184,0.004235,1.023259,0.130034



MODULE N8 COMPLETE

Predictive architecture remains frozen.
Next: N9 — LPNS / disagreement / confidence reliability.


In [16]:
# ============================================================
# MODULE N9 — STRICT LPNS RELIABILITY FRAMEWORK
#
# FROZEN PREDICTOR:
#   LSA-HE-5 from N7B
#
# PRIMARY probability:
#   RAW LSA-HE-5
#
# PURPOSE
# ------------------------------------------------------------
# Predict whether the frozen classifier will make an error.
#
# Feature families:
#
#   C = confidence features
#   D = heterogeneous expert disagreement
#   L = Local Prediction Neighborhood Stability (LPNS)
#
# Compare:
#
#   C
#   D
#   L
#   C+D
#   C+L
#   C+D+L
#
#
# STRICT OUTER-FOLD DESIGN
# ------------------------------------------------------------
#
# For each outer fold:
#
#   Outer TRAIN:
#       - choose operating threshold
#       - construct neighborhood reference bank
#       - construct reliability-training features
#       - train reliability models
#
#   Outer VALID:
#       - use frozen expert OOF probabilities
#       - compute LPNS only against outer TRAIN reference bank
#       - predict error probability
#
# Outer-valid labels never participate in:
#   - threshold selection
#   - neighborhood construction
#   - LPNS feature construction
#   - reliability-model fitting
#
#
# IMPORTANT
# ------------------------------------------------------------
# 2015 holdout remains excluded from current development.
# It is historically inspected and will only be used later
# as a secondary post-freeze evaluation.
#
# BRFSS-2020 is not used here.
# ============================================================


import gc
import time
import itertools

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# N9.1 CONFIGURATION
# ============================================================

K_LPNS_N9 = 20

REFERENCE_SIZE_N9 = 80000

EPS_N9 = 1e-7


FINAL_EXPERTS_N9 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


# ============================================================
# N9.2 STRICT N7B EXPERT MATRIX
# ============================================================

P_EXPERT_N9 = np.column_stack([

    np.asarray(
        OOF_N7B[
            expert
        ],
        dtype=np.float32
    )

    for expert
    in FINAL_EXPERTS_N9
])


P_BASE_N9 = np.asarray(

    OOF_LSAHE5_N7B,

    dtype=np.float32
)


assert P_EXPERT_N9.shape == (
    len(y_DEV),
    5
)


assert not np.isnan(
    P_EXPERT_N9
).any()


assert not np.isnan(
    P_BASE_N9
).any()


# ============================================================
# N9.3 BINARY ENTROPY
# ============================================================

def binary_entropy_n9(
    p
):

    p = np.clip(

        np.asarray(
            p,
            dtype=float
        ),

        EPS_N9,
        1 - EPS_N9
    )


    return -(

        p * np.log(p)

        +

        (1 - p)
        *
        np.log(
            1 - p
        )
    )


# ============================================================
# N9.4 OPERATING THRESHOLD
#
# Selected from outer TRAIN only.
#
# Objective:
#   maximize F1
#
# The threshold is part of the frozen classification
# operating point used to define the reliability target.
# ============================================================

def select_threshold_n9(
    y,
    p
):

    thresholds = np.linspace(

        0.03,
        0.50,
        300
    )


    best_threshold = None

    best_f1 = -np.inf


    for threshold in thresholds:

        pred = (
            p >= threshold
        ).astype(int)


        tp = np.sum(

            (pred == 1)
            &
            (y == 1)
        )


        fp = np.sum(

            (pred == 1)
            &
            (y == 0)
        )


        fn = np.sum(

            (pred == 0)
            &
            (y == 1)
        )


        denominator = (

            2 * tp
            +
            fp
            +
            fn
        )


        f1 = (

            0.0

            if denominator == 0

            else

            (
                2 * tp
                /
                denominator
            )
        )


        if f1 > best_f1:

            best_f1 = f1

            best_threshold = threshold


    return (

        float(
            best_threshold
        ),

        float(
            best_f1
        )
    )


# ============================================================
# N9.5 CONFIDENCE FEATURES
#
# 5 features:
#
#   1. base probability
#   2. uncertainty around p=0.5
#   3. distance to operating threshold
#   4. entropy
#   5. predicted class
# ============================================================

CONFIDENCE_FEATURE_NAMES_N9 = [

    "base_probability",

    "uncertainty_05",

    "threshold_margin",

    "binary_entropy",

    "predicted_class",
]


def build_confidence_features_n9(
    p,
    threshold
):

    p = np.asarray(
        p,
        dtype=float
    )


    uncertainty = (

        1.0
        -
        np.abs(
            2 * p
            -
            1
        )
    )


    margin = np.abs(

        p
        -
        threshold
    )


    entropy = binary_entropy_n9(
        p
    )


    predicted_class = (

        p >= threshold

    ).astype(float)


    return np.column_stack([

        p,

        uncertainty,

        margin,

        entropy,

        predicted_class,
    ]).astype(
        np.float32
    )


# ============================================================
# N9.6 DISAGREEMENT FEATURES
#
# 10 features:
#
# 5 expert deviations from consensus
# + expert SD
# + expert range
# + mean absolute deviation
# + max absolute deviation
# + mean pairwise absolute disagreement
# ============================================================

DISAGREEMENT_FEATURE_NAMES_N9 = (

    [

        f"absdev_{expert}"

        for expert
        in FINAL_EXPERTS_N9
    ]

    +

    [
        "expert_std",
        "expert_range",
        "expert_mean_absdev",
        "expert_max_absdev",
        "expert_pairwise_mean_absdiff",
    ]
)


def build_disagreement_features_n9(
    expert_prob
):

    expert_prob = np.asarray(

        expert_prob,

        dtype=float
    )


    consensus = np.mean(

        expert_prob,

        axis=1,

        keepdims=True
    )


    absdev = np.abs(

        expert_prob
        -
        consensus
    )


    expert_std = np.std(

        expert_prob,

        axis=1
    )


    expert_range = (

        np.max(
            expert_prob,
            axis=1
        )

        -

        np.min(
            expert_prob,
            axis=1
        )
    )


    mean_absdev = np.mean(

        absdev,

        axis=1
    )


    max_absdev = np.max(

        absdev,

        axis=1
    )


    pairwise_values = []


    for i, j in itertools.combinations(

        range(
            expert_prob.shape[1]
        ),

        2
    ):

        pairwise_values.append(

            np.abs(

                expert_prob[:, i]

                -

                expert_prob[:, j]
            )
        )


    pairwise_mean = np.mean(

        np.column_stack(
            pairwise_values
        ),

        axis=1
    )


    return np.column_stack([

        absdev,

        expert_std,

        expert_range,

        mean_absdev,

        max_absdev,

        pairwise_mean,

    ]).astype(
        np.float32
    )


# ============================================================
# N9.7 LPNS FEATURE NAMES
#
# Per expert:
#
#   - weighted prediction deviation from query
#   - weighted neighborhood SD
#   - absolute query-vs-local-mean bias
#
# 5 experts * 3 = 15
#
# Aggregates:
#
#   - mean deviation
#   - max deviation
#   - mean local SD
#   - max local bias
#
# Total = 19
# ============================================================

LPNS_FEATURE_NAMES_N9 = []


for expert in FINAL_EXPERTS_N9:

    LPNS_FEATURE_NAMES_N9.extend([

        f"lpns_dev_{expert}",

        f"lpns_std_{expert}",

        f"lpns_bias_{expert}",
    ])


LPNS_FEATURE_NAMES_N9.extend([

    "lpns_mean_dev",

    "lpns_max_dev",

    "lpns_mean_std",

    "lpns_max_bias",
])


assert len(
    LPNS_FEATURE_NAMES_N9
) == 19


# ============================================================
# N9.8 REFERENCE BANK
#
# Deterministic random subset from OUTER TRAIN only.
#
# No outcome stratification is used.
# ============================================================

def choose_reference_bank_n9(
    outer_train_idx,
    fold
):

    rng = np.random.default_rng(

        SEED
        +
        50000
        +
        fold
    )


    n_reference = min(

        REFERENCE_SIZE_N9,

        len(
            outer_train_idx
        )
    )


    reference_idx = rng.choice(

        outer_train_idx,

        size=
            n_reference,

        replace=False
    )


    return np.asarray(

        reference_idx,

        dtype=int
    )


# ============================================================
# N9.9 LPNS COMPUTATION
#
# Mixed BRFSS space:
# standardized using outer TRAIN only.
#
# For a query q and expert j:
#
# dev_j =
#   sum_i w_i |p_j(q)-p_j(neighbor_i)|
#
# std_j =
#   weighted std of neighbor predictions
#
# bias_j =
#   |p_j(q)-weighted neighborhood mean|
#
# Distance weights:
#
#   exp(-d / local_median_distance)
# ============================================================

def compute_lpns_features_n9(

    query_global_idx,

    outer_train_idx,

    reference_global_idx,

    scaler,

    neighbor_model,

):

    query_global_idx = np.asarray(

        query_global_idx,

        dtype=int
    )


    # --------------------------------------------------------
    # Query representation
    # --------------------------------------------------------

    Z_query = scaler.transform(

        X_DEV.iloc[
            query_global_idx
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    # --------------------------------------------------------
    # Ask for k+1 because TRAIN queries can themselves belong
    # to the reference bank.
    # --------------------------------------------------------

    n_neighbors_query = min(

        K_LPNS_N9 + 1,

        len(
            reference_global_idx
        )
    )


    distances, ref_local_idx = (

        neighbor_model.kneighbors(

            Z_query,

            n_neighbors=
                n_neighbors_query,

            return_distance=True
        )
    )


    ref_global_matrix = (

        reference_global_idx[
            ref_local_idx
        ]
    )


    n_query = len(
        query_global_idx
    )


    selected_global = np.empty(

        (
            n_query,
            K_LPNS_N9
        ),

        dtype=int
    )


    selected_distance = np.empty(

        (
            n_query,
            K_LPNS_N9
        ),

        dtype=np.float32
    )


    # --------------------------------------------------------
    # Explicit self-removal where applicable
    # --------------------------------------------------------

    for i in range(
        n_query
    ):

        candidate_global = (
            ref_global_matrix[i]
        )


        candidate_distance = (
            distances[i]
        )


        keep = (

            candidate_global

            !=

            query_global_idx[i]
        )


        candidate_global = (
            candidate_global[
                keep
            ]
        )


        candidate_distance = (
            candidate_distance[
                keep
            ]
        )


        if len(
            candidate_global
        ) < K_LPNS_N9:

            raise RuntimeError(

                "Insufficient non-self neighbors."
            )


        selected_global[
            i
        ] = candidate_global[
            :K_LPNS_N9
        ]


        selected_distance[
            i
        ] = candidate_distance[
            :K_LPNS_N9
        ]


    # --------------------------------------------------------
    # Distance weights
    # --------------------------------------------------------

    local_scale = np.median(

        selected_distance,

        axis=1,

        keepdims=True
    )


    local_scale = np.maximum(

        local_scale,

        1e-6
    )


    weights = np.exp(

        -selected_distance

        /

        local_scale
    )


    weights /= np.sum(

        weights,

        axis=1,

        keepdims=True
    )


    # --------------------------------------------------------
    # Expert predictions
    # --------------------------------------------------------

    query_pred = P_EXPERT_N9[
        query_global_idx
    ]


    neighbor_pred = P_EXPERT_N9[
        selected_global
    ]


    # shapes:
    #
    # query_pred    -> [Q, 5]
    # neighbor_pred -> [Q, K, 5]


    weighted_mean = np.sum(

        weights[
            :,
            :,
            None
        ]

        *

        neighbor_pred,

        axis=1
    )


    deviation = np.sum(

        weights[
            :,
            :,
            None
        ]

        *

        np.abs(

            neighbor_pred

            -

            query_pred[
                :,
                None,
                :
            ]
        ),

        axis=1
    )


    variance = np.sum(

        weights[
            :,
            :,
            None
        ]

        *

        (

            neighbor_pred

            -

            weighted_mean[
                :,
                None,
                :
            ]

        ) ** 2,

        axis=1
    )


    local_std = np.sqrt(

        np.maximum(
            variance,
            0.0
        )
    )


    local_bias = np.abs(

        query_pred

        -

        weighted_mean
    )


    # --------------------------------------------------------
    # Interleave:
    #
    # expert1 dev/std/bias
    # expert2 dev/std/bias
    # ...
    # --------------------------------------------------------

    per_expert = []


    for j in range(
        len(
            FINAL_EXPERTS_N9
        )
    ):

        per_expert.extend([

            deviation[:, j],

            local_std[:, j],

            local_bias[:, j],
        ])


    aggregate = [

        np.mean(
            deviation,
            axis=1
        ),

        np.max(
            deviation,
            axis=1
        ),

        np.mean(
            local_std,
            axis=1
        ),

        np.max(
            local_bias,
            axis=1
        ),
    ]


    features = np.column_stack(

        per_expert

        +

        aggregate
    ).astype(
        np.float32
    )


    assert features.shape[1] == 19


    return features


# ============================================================
# N9.10 RELIABILITY MODEL
#
# Fixed model across ALL feature blocks.
#
# No hyperparameter search.
# ============================================================

def make_reliability_head_n9(
    seed
):

    return HistGradientBoostingClassifier(

        learning_rate=0.05,

        max_iter=250,

        max_leaf_nodes=15,

        min_samples_leaf=40,

        l2_regularization=2.0,

        early_stopping=False,

        random_state=seed,
    )


# ============================================================
# N9.11 FEATURE BLOCK DEFINITIONS
# ============================================================

BLOCKS_N9 = {

    "CONF":
        ["C"],

    "DIS":
        ["D"],

    "LPNS":
        ["L"],

    "CONF_DIS":
        ["C", "D"],

    "CONF_LPNS":
        ["C", "L"],

    "CONF_DIS_LPNS":
        ["C", "D", "L"],
}


# ============================================================
# N9.12 OOF RELIABILITY SCORE STORAGE
# ============================================================

RELIABILITY_OOF_N9 = {

    block:
        np.full(
            len(
                y_DEV
            ),

            np.nan,

            dtype=np.float32
        )

    for block
    in BLOCKS_N9
}


ERROR_TARGET_OOF_N9 = np.full(

    len(
        y_DEV
    ),

    -1,

    dtype=np.int8
)


THRESHOLD_OOF_N9 = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


FOLD_ROWS_N9 = []

FEATURE_DIM_ROWS_N9 = []

REFERENCE_ROWS_N9 = []


# ============================================================
# N9.13 HELPER — COMBINE BLOCKS
# ============================================================

def concatenate_blocks_n9(

    block_codes,

    C,
    D,
    L,
):

    matrices = []


    for code in block_codes:

        if code == "C":

            matrices.append(
                C
            )


        elif code == "D":

            matrices.append(
                D
            )


        elif code == "L":

            matrices.append(
                L
            )


        else:

            raise ValueError(
                code
            )


    return np.concatenate(

        matrices,

        axis=1
    )


# ============================================================
# N9.14 STRICT OUTER RELIABILITY LOOP
# ============================================================

total_start_n9 = time.perf_counter()


for fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 155
    )

    print(
        f"N9 — RELIABILITY OUTER FOLD {fold}"
    )

    print(
        "=" * 155
    )


    outer_train_idx = np.where(

        DEV_FOLD != fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == fold

    )[0]


    # ========================================================
    # STEP 1 — threshold from outer train only
    # ========================================================

    threshold, threshold_train_f1 = (

        select_threshold_n9(

            y_DEV[
                outer_train_idx
            ],

            P_BASE_N9[
                outer_train_idx
            ]
        )
    )


    print(
        f"\nOperating threshold = "
        f"{threshold:.5f}"
    )


    THRESHOLD_OOF_N9[
        outer_valid_idx
    ] = threshold


    # ========================================================
    # Error labels
    # ========================================================

    pred_train_class = (

        P_BASE_N9[
            outer_train_idx
        ]

        >=
        threshold

    ).astype(int)


    pred_valid_class = (

        P_BASE_N9[
            outer_valid_idx
        ]

        >=
        threshold

    ).astype(int)


    error_train = (

        pred_train_class

        !=

        y_DEV[
            outer_train_idx
        ]

    ).astype(int)


    error_valid = (

        pred_valid_class

        !=

        y_DEV[
            outer_valid_idx
        ]

    ).astype(int)


    ERROR_TARGET_OOF_N9[
        outer_valid_idx
    ] = error_valid


    print(
        f"Train error prevalence: "
        f"{np.mean(error_train):.6f}"
    )


    print(
        f"Valid error prevalence: "
        f"{np.mean(error_valid):.6f}"
    )


    # ========================================================
    # STEP 2 — Confidence features
    # ========================================================

    C_train = build_confidence_features_n9(

        P_BASE_N9[
            outer_train_idx
        ],

        threshold
    )


    C_valid = build_confidence_features_n9(

        P_BASE_N9[
            outer_valid_idx
        ],

        threshold
    )


    # ========================================================
    # STEP 3 — Disagreement features
    # ========================================================

    D_train = build_disagreement_features_n9(

        P_EXPERT_N9[
            outer_train_idx
        ]
    )


    D_valid = build_disagreement_features_n9(

        P_EXPERT_N9[
            outer_valid_idx
        ]
    )


    # ========================================================
    # STEP 4 — Reference neighborhood bank
    # ========================================================

    reference_idx = choose_reference_bank_n9(

        outer_train_idx,

        fold
    )


    REFERENCE_ROWS_N9.append({

        "Fold":
            fold,

        "Outer_Train_N":
            len(
                outer_train_idx
            ),

        "Reference_N":
            len(
                reference_idx
            ),

        "K":
            K_LPNS_N9,
    })


    scaler = StandardScaler()


    scaler.fit(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ]
    )


    Z_reference = scaler.transform(

        X_DEV.iloc[
            reference_idx
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    neighbor_model = NearestNeighbors(

        n_neighbors=
            K_LPNS_N9 + 1,

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1,
    )


    neighbor_model.fit(
        Z_reference
    )


    # ========================================================
    # STEP 5 — LPNS for outer TRAIN
    # ========================================================

    print(
        "\nComputing TRAIN LPNS..."
    )


    L_train = compute_lpns_features_n9(

        outer_train_idx,

        outer_train_idx,

        reference_idx,

        scaler,

        neighbor_model,
    )


    # ========================================================
    # STEP 6 — LPNS for outer VALID
    # ========================================================

    print(
        "Computing VALID LPNS..."
    )


    L_valid = compute_lpns_features_n9(

        outer_valid_idx,

        outer_train_idx,

        reference_idx,

        scaler,

        neighbor_model,
    )


    # ========================================================
    # STEP 7 — Reliability heads
    # ========================================================

    for block_number, (
        block_name,
        block_codes

    ) in enumerate(
        BLOCKS_N9.items()
    ):

        X_rel_train = concatenate_blocks_n9(

            block_codes,

            C_train,
            D_train,
            L_train
        )


        X_rel_valid = concatenate_blocks_n9(

            block_codes,

            C_valid,
            D_valid,
            L_valid
        )


        model = make_reliability_head_n9(

            SEED
            +
            60000
            +
            fold * 100
            +
            block_number
        )


        model.fit(

            X_rel_train,

            error_train
        )


        error_score = model.predict_proba(

            X_rel_valid

        )[:, 1]


        RELIABILITY_OOF_N9[
            block_name
        ][
            outer_valid_idx
        ] = error_score


        m = probability_metrics(

            error_valid,

            error_score
        )


        FOLD_ROWS_N9.append({

            "Fold":
                fold,

            "Block":
                block_name,

            "N_Features":
                X_rel_train.shape[1],

            "Error_PR_AUC":
                m[
                    "PR_AUC"
                ],

            "Error_ROC_AUC":
                m[
                    "ROC_AUC"
                ],

            "Error_Brier":
                m[
                    "Brier"
                ],

            "Error_NLL":
                m[
                    "NLL"
                ],

            "Error_ECE":
                m[
                    "ECE"
                ],
        })


        print(

            f"{block_name:15s} | "
            f"n={X_rel_train.shape[1]:2d} | "
            f"ErrorPR={m['PR_AUC']:.6f} | "
            f"ROC={m['ROC_AUC']:.6f}"
        )


        if fold == 0:

            FEATURE_DIM_ROWS_N9.append({

                "Block":
                    block_name,

                "N_Features":
                    X_rel_train.shape[1],
            })


        del model
        del X_rel_train
        del X_rel_valid

        gc.collect()


    del C_train
    del C_valid

    del D_train
    del D_valid

    del L_train
    del L_valid

    del scaler
    del Z_reference
    del neighbor_model

    gc.collect()


# ============================================================
# N9.15 COMPLETENESS
# ============================================================

if np.any(
    ERROR_TARGET_OOF_N9 < 0
):

    raise RuntimeError(
        "Incomplete reliability targets."
    )


for block, score in (

    RELIABILITY_OOF_N9.items()
):

    if np.isnan(
        score
    ).any():

        raise RuntimeError(
            f"Incomplete reliability score: {block}"
        )


# ============================================================
# N9.16 POOLED RELIABILITY RESULTS
# ============================================================

rows = []


for block_name, score in (

    RELIABILITY_OOF_N9.items()
):

    m = probability_metrics(

        ERROR_TARGET_OOF_N9,

        score
    )


    rows.append({

        "Block":
            block_name,

        "N_Features":
            dict(

                (
                    row[
                        "Block"
                    ],

                    row[
                        "N_Features"
                    ]
                )

                for row
                in FEATURE_DIM_ROWS_N9
            )[
                block_name
            ],

        "Error_PR_AUC":
            m[
                "PR_AUC"
            ],

        "Error_ROC_AUC":
            m[
                "ROC_AUC"
            ],

        "Error_Brier":
            m[
                "Brier"
            ],

        "Error_NLL":
            m[
                "NLL"
            ],

        "Error_ECE":
            m[
                "ECE"
            ],
    })


RELIABILITY_RESULTS_N9 = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "Error_PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 155
)

print(
    "MODULE N9 — STRICT RELIABILITY RESULTS"
)

print(
    "=" * 155
)


print(
    f"\nOverall error prevalence = "
    f"{np.mean(ERROR_TARGET_OOF_N9):.6f}"
)


display(
    RELIABILITY_RESULTS_N9
)


# ============================================================
# N9.17 INCREMENTAL LPNS VALUE
# ============================================================

def get_metric_n9(
    block,
    metric
):

    return float(

        RELIABILITY_RESULTS_N9.loc[

            RELIABILITY_RESULTS_N9[
                "Block"
            ]
            ==
            block,

            metric

        ].iloc[0]
    )


INCREMENTAL_N9 = pd.DataFrame([

    {
        "Comparison":
            "CONF_DIS vs CONF",

        "Delta_Error_PR":
            (
                get_metric_n9(
                    "CONF_DIS",
                    "Error_PR_AUC"
                )

                -

                get_metric_n9(
                    "CONF",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_metric_n9(
                    "CONF_DIS",
                    "Error_ROC_AUC"
                )

                -

                get_metric_n9(
                    "CONF",
                    "Error_ROC_AUC"
                )
            ),
    },


    {
        "Comparison":
            "CONF_LPNS vs CONF",

        "Delta_Error_PR":
            (
                get_metric_n9(
                    "CONF_LPNS",
                    "Error_PR_AUC"
                )

                -

                get_metric_n9(
                    "CONF",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_metric_n9(
                    "CONF_LPNS",
                    "Error_ROC_AUC"
                )

                -

                get_metric_n9(
                    "CONF",
                    "Error_ROC_AUC"
                )
            ),
    },


    {
        "Comparison":
            "FULL vs CONF_DIS",

        "Delta_Error_PR":
            (
                get_metric_n9(
                    "CONF_DIS_LPNS",
                    "Error_PR_AUC"
                )

                -

                get_metric_n9(
                    "CONF_DIS",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_metric_n9(
                    "CONF_DIS_LPNS",
                    "Error_ROC_AUC"
                )

                -

                get_metric_n9(
                    "CONF_DIS",
                    "Error_ROC_AUC"
                )
            ),
    },


    {
        "Comparison":
            "FULL vs CONF_LPNS",

        "Delta_Error_PR":
            (
                get_metric_n9(
                    "CONF_DIS_LPNS",
                    "Error_PR_AUC"
                )

                -

                get_metric_n9(
                    "CONF_LPNS",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_metric_n9(
                    "CONF_DIS_LPNS",
                    "Error_ROC_AUC"
                )

                -

                get_metric_n9(
                    "CONF_LPNS",
                    "Error_ROC_AUC"
                )
            ),
    },
])


print(
    "\n"
    + "=" * 155
)

print(
    "N9 — INCREMENTAL INFORMATION"
)

print(
    "=" * 155
)


display(
    INCREMENTAL_N9
)


# ============================================================
# N9.18 FOLD STABILITY
# ============================================================

FOLD_RESULTS_N9 = pd.DataFrame(

    FOLD_ROWS_N9
)


FOLD_STABILITY_N9 = (

    FOLD_RESULTS_N9

    .groupby(
        "Block"
    )

    .agg(

        ErrorPR_Mean=(
            "Error_PR_AUC",
            "mean"
        ),

        ErrorPR_SD=(
            "Error_PR_AUC",
            "std"
        ),

        ErrorPR_Min=(
            "Error_PR_AUC",
            "min"
        ),

        ErrorPR_Max=(
            "Error_PR_AUC",
            "max"
        ),

        ErrorROC_Mean=(
            "Error_ROC_AUC",
            "mean"
        ),

        ErrorROC_SD=(
            "Error_ROC_AUC",
            "std"
        ),
    )

    .reset_index()

    .sort_values(
        "ErrorPR_Mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 155
)

print(
    "N9 — RELIABILITY FOLD STABILITY"
)

print(
    "=" * 155
)


display(
    FOLD_STABILITY_N9
)


# ============================================================
# N9.19 OPERATING THRESHOLDS
# ============================================================

THRESHOLD_SUMMARY_N9 = pd.DataFrame({

    "Fold":
        np.arange(
            N_FOLDS
        ),

    "Threshold":
        [

            float(
                THRESHOLD_OOF_N9[
                    DEV_FOLD == fold
                ][0]
            )

            for fold
            in range(
                N_FOLDS
            )
        ]
})


print(
    "\n"
    + "=" * 155
)

print(
    "N9 — CROSS-FITTED OPERATING THRESHOLDS"
)

print(
    "=" * 155
)


display(
    THRESHOLD_SUMMARY_N9
)


# ============================================================
# N9.20 REFERENCE-BANK AUDIT
# ============================================================

REFERENCE_AUDIT_N9 = pd.DataFrame(

    REFERENCE_ROWS_N9
)


print(
    "\n"
    + "=" * 155
)

print(
    "N9 — LPNS REFERENCE-BANK AUDIT"
)

print(
    "=" * 155
)


display(
    REFERENCE_AUDIT_N9
)


# ============================================================
# N9.21 SAVE
# ============================================================

RELIABILITY_RESULTS_N9.to_csv(

    TAB_DIR
    /
    "n9_strict_reliability_results.csv",

    index=False
)


INCREMENTAL_N9.to_csv(

    TAB_DIR
    /
    "n9_incremental_lpns_value.csv",

    index=False
)


FOLD_RESULTS_N9.to_csv(

    TAB_DIR
    /
    "n9_reliability_fold_results.csv",

    index=False
)


FOLD_STABILITY_N9.to_csv(

    TAB_DIR
    /
    "n9_reliability_fold_stability.csv",

    index=False
)


THRESHOLD_SUMMARY_N9.to_csv(

    TAB_DIR
    /
    "n9_operating_thresholds.csv",

    index=False
)


REFERENCE_AUDIT_N9.to_csv(

    TAB_DIR
    /
    "n9_reference_bank_audit.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "N9_ERROR_TARGET_OOF.npy",

    ERROR_TARGET_OOF_N9
)


for block_name, score in (

    RELIABILITY_OOF_N9.items()
):

    np.save(

        PRED_DIR
        /
        f"N9_{block_name}_ERROR_SCORE_OOF.npy",

        score
    )


print(
    "\n"
    + "=" * 155
)

print(
    "MODULE N9 COMPLETE"
)

print(
    "=" * 155
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - total_start_n9:.1f}s"
)


print(
    "\nFrozen predictive architecture unchanged."
)

print(
    "Primary prediction probability remains RAW LSA-HE-5."
)

print(
    "Next: N10 — selective review and statistical testing."
)


N9 — RELIABILITY OUTER FOLD 0

Operating threshold = 0.22177
Train error prevalence: 0.138290
Valid error prevalence: 0.135603

Computing TRAIN LPNS...
Computing VALID LPNS...
CONF            | n= 5 | ErrorPR=0.627625 | ROC=0.908134
DIS             | n=10 | ErrorPR=0.437890 | ROC=0.858035
LPNS            | n=19 | ErrorPR=0.498337 | ROC=0.882168
CONF_DIS        | n=15 | ErrorPR=0.629397 | ROC=0.908056
CONF_LPNS       | n=24 | ErrorPR=0.630995 | ROC=0.908304
CONF_DIS_LPNS   | n=34 | ErrorPR=0.628528 | ROC=0.908530

N9 — RELIABILITY OUTER FOLD 1

Operating threshold = 0.22177
Train error prevalence: 0.137520
Valid error prevalence: 0.138683

Computing TRAIN LPNS...
Computing VALID LPNS...
CONF            | n= 5 | ErrorPR=0.626828 | ROC=0.908782
DIS             | n=10 | ErrorPR=0.448422 | ROC=0.859465
LPNS            | n=19 | ErrorPR=0.501495 | ROC=0.881179
CONF_DIS        | n=15 | ErrorPR=0.631044 | ROC=0.909058
CONF_LPNS       | n=24 | ErrorPR=0.631008 | ROC=0.909031
CONF_DIS_LPNS   | n

,Block,N_Features,Error_PR_AUC,Error_ROC_AUC,Error_Brier,Error_NLL,Error_ECE
0,CONF_LPNS,24,0.634225,0.908365,0.069843,0.235941,0.001092
1,CONF_DIS_LPNS,34,0.632909,0.908436,0.069860,0.235919,0.001167
2,CONF,5,0.632345,0.908241,0.069839,0.235964,0.001569
3,CONF_DIS,15,0.632035,0.908148,0.069894,0.236106,0.001094
4,LPNS,19,0.501199,0.882346,0.084757,0.271094,0.006561
5,DIS,10,0.450728,0.859822,0.090735,0.289985,0.003695



N9 — INCREMENTAL INFORMATION


,Comparison,Delta_Error_PR,Delta_Error_ROC
0,CONF_DIS vs CONF,-0.000310,-0.000092
1,CONF_LPNS vs CONF,0.001880,0.000124
2,FULL vs CONF_DIS,0.000874,0.000287
3,FULL vs CONF_LPNS,-0.001316,0.000071



N9 — RELIABILITY FOLD STABILITY


...


N9 — CROSS-FITTED OPERATING THRESHOLDS


,Fold,Threshold
0,0,0.221773
1,1,0.221773
2,2,0.221773
3,3,0.221773
4,4,0.221773



N9 — LPNS REFERENCE-BANK AUDIT


,Fold,Outer_Train_N,Reference_N,K
0,0,162355,80000,20
1,1,162355,80000,20
2,2,162355,80000,20
3,3,162355,80000,20
4,4,162356,80000,20



MODULE N9 COMPLETE

Runtime: 277.7s

Frozen predictive architecture unchanged.
Primary prediction probability remains RAW LSA-HE-5.
Next: N10 — selective review and statistical testing.


In [17]:
# ============================================================
# MODULE N10A — RELIABILITY-GUIDED RESIDUAL MoE SCREEN
#
# PURPOSE
# ------------------------------------------------------------
# Test whether LPNS-based reliability can safely control
# adaptive expert weighting.
#
# No new base experts.
# No architecture change.
#
# Uses:
#   - frozen N7B expert OOF predictions
#   - frozen LSA-HE-5 consensus
#   - strict N9 reliability scores
#
# IMPORTANT:
# This is a SCREENING / mechanism experiment.
#
# We compare several deterministic gates and several
# reliability-controlled residual strengths.
# ============================================================


import numpy as np
import pandas as pd


# ============================================================
# N10A.1 INPUT
# ============================================================

P_EXPERT_N10 = np.column_stack([

    OOF_N7B[
        expert
    ]

    for expert
    in FINAL_EXPERTS_N7B
]).astype(float)


P_ANCHOR_N10 = np.mean(

    P_EXPERT_N10,

    axis=1
)


R_CONF_N10 = np.asarray(

    RELIABILITY_OOF_N9[
        "CONF"
    ],

    dtype=float
)


R_CL_N10 = np.asarray(

    RELIABILITY_OOF_N9[
        "CONF_LPNS"
    ],

    dtype=float
)


R_CDL_N10 = np.asarray(

    RELIABILITY_OOF_N9[
        "CONF_DIS_LPNS"
    ],

    dtype=float
)


EPS_N10 = 1e-8


# ============================================================
# N10A.2 STABLE SOFTMAX
# ============================================================

def softmax_n10(
    x,
    axis=1
):

    x = np.asarray(
        x,
        dtype=float
    )


    x = (

        x

        -

        np.max(
            x,
            axis=axis,
            keepdims=True
        )
    )


    z = np.exp(
        x
    )


    return (

        z

        /

        np.sum(
            z,
            axis=axis,
            keepdims=True
        )
    )


# ============================================================
# N10A.3 DETERMINISTIC ADAPTIVE GATES
# ============================================================

def gate_sharp_n10(
    P,
    tau=4.0
):

    utility = np.abs(

        P
        -
        0.5
    )


    return softmax_n10(

        tau
        *
        utility
    )


def gate_consensus_n10(
    P,
    tau=12.0
):

    anchor = np.mean(

        P,

        axis=1,

        keepdims=True
    )


    utility = -(

        np.abs(

            P
            -
            anchor
        )
    )


    return softmax_n10(

        tau
        *
        utility
    )


def gate_specialist_n10(
    P,
    tau=8.0
):

    anchor = np.mean(

        P,

        axis=1,

        keepdims=True
    )


    utility = np.abs(

        P
        -
        anchor
    )


    return softmax_n10(

        tau
        *
        utility
    )


# ============================================================
# N10A.4 GATE PREDICTIONS
# ============================================================

GATES_N10 = {}


for name, weight in {

    "SHARP":
        gate_sharp_n10(
            P_EXPERT_N10
        ),

    "CONSENSUS":
        gate_consensus_n10(
            P_EXPERT_N10
        ),

    "SPECIALIST":
        gate_specialist_n10(
            P_EXPERT_N10
        ),

}.items():

    GATES_N10[
        name
    ] = {

        "weights":
            weight,

        "prediction":
            np.sum(

                weight
                *
                P_EXPERT_N10,

                axis=1
            )
    }


# ============================================================
# N10A.5 TRUST FUNCTIONS
#
# r = predicted probability of classifier error.
#
# Higher r => less adaptive routing.
# ============================================================

def trust_linear_n10(
    r,
    alpha_max=0.50
):

    return (

        alpha_max

        *
        (
            1.0
            -
            np.clip(
                r,
                0,
                1
            )
        )
    )


def trust_sigmoid_n10(

    r,

    threshold=0.15,

    scale=0.05,

    alpha_max=0.50,
):

    z = (

        threshold
        -
        r

    ) / scale


    return (

        alpha_max

        /

        (
            1.0
            +
            np.exp(
                -z
            )
        )
    )


# ============================================================
# N10A.6 SCREEN TRUST SETTINGS
#
# These are sensitivity experiments.
#
# We are NOT freezing the best one yet.
# ============================================================

TRUST_VARIANTS_N10 = {

    "NONE": np.ones(
        len(
            y_DEV
        )
    ),

    "CONF_LINEAR_025":
        trust_linear_n10(
            R_CONF_N10,
            alpha_max=0.25
        ),

    "CONF_LPNS_LINEAR_025":
        trust_linear_n10(
            R_CL_N10,
            alpha_max=0.25
        ),

    "CONF_LPNS_LINEAR_050":
        trust_linear_n10(
            R_CL_N10,
            alpha_max=0.50
        ),

    "FULL_LINEAR_025":
        trust_linear_n10(
            R_CDL_N10,
            alpha_max=0.25
        ),

    "CONF_LPNS_SIGMOID":
        trust_sigmoid_n10(

            R_CL_N10,

            threshold=0.15,

            scale=0.05,

            alpha_max=0.50
        ),
}


# ============================================================
# N10A.7 RESIDUAL MIXING
# ============================================================

rows = []


anchor_metrics_n10 = probability_metrics(

    y_DEV,
    P_ANCHOR_N10
)


for gate_name, gate_obj in (
    GATES_N10.items()
):

    p_gate = gate_obj[
        "prediction"
    ]


    gate_metrics = probability_metrics(

        y_DEV,
        p_gate
    )


    rows.append({

        "Gate":
            gate_name,

        "Trust":
            "FULL_GATE",

        "PR_AUC":
            gate_metrics[
                "PR_AUC"
            ],

        "ROC_AUC":
            gate_metrics[
                "ROC_AUC"
            ],

        "Brier":
            gate_metrics[
                "Brier"
            ],

        "NLL":
            gate_metrics[
                "NLL"
            ],

        "ECE":
            gate_metrics[
                "ECE"
            ],

        "Delta_PR_vs_Anchor":
            (
                gate_metrics[
                    "PR_AUC"
                ]
                -
                anchor_metrics_n10[
                    "PR_AUC"
                ]
            ),
    })


    for trust_name, alpha in (
        TRUST_VARIANTS_N10.items()
    ):

        if trust_name == "NONE":

            continue


        p_residual = (

            P_ANCHOR_N10

            +

            alpha

            *
            (
                p_gate
                -
                P_ANCHOR_N10
            )
        )


        p_residual = np.clip(

            p_residual,

            1e-7,

            1 - 1e-7
        )


        m = probability_metrics(

            y_DEV,
            p_residual
        )


        rows.append({

            "Gate":
                gate_name,

            "Trust":
                trust_name,

            "PR_AUC":
                m[
                    "PR_AUC"
                ],

            "ROC_AUC":
                m[
                    "ROC_AUC"
                ],

            "Brier":
                m[
                    "Brier"
                ],

            "NLL":
                m[
                    "NLL"
                ],

            "ECE":
                m[
                    "ECE"
                ],

            "Delta_PR_vs_Anchor":
                (
                    m[
                        "PR_AUC"
                    ]
                    -
                    anchor_metrics_n10[
                        "PR_AUC"
                    ]
                ),
        })


RESULTS_N10A = (

    pd.DataFrame(
        rows
    )

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "=" * 155
)

print(
    "MODULE N10A — RELIABILITY-GUIDED RESIDUAL MoE SCREEN"
)

print(
    "=" * 155
)


print(
    "\nAnchor LSA-HE-5:"
)

print(
    anchor_metrics_n10
)


display(
    RESULTS_N10A
)


# ============================================================
# N10A.8 DOES RELIABILITY IDENTIFY HARMFUL ROUTING?
#
# Compare:
#
# absolute loss of gate
# minus
# absolute loss of anchor.
#
# Positive = adaptive gate was worse.
# ============================================================

harm_rows = []


anchor_loss = (

    P_ANCHOR_N10
    -
    y_DEV

) ** 2


for gate_name, gate_obj in (
    GATES_N10.items()
):

    p_gate = gate_obj[
        "prediction"
    ]


    gate_loss = (

        p_gate
        -
        y_DEV

    ) ** 2


    routing_harm = (

        gate_loss
        -
        anchor_loss
    )


    harmful = (

        routing_harm
        >
        0
    ).astype(int)


    for reliability_name, r in {

        "CONF":
            R_CONF_N10,

        "CONF_LPNS":
            R_CL_N10,

        "FULL":
            R_CDL_N10,

    }.items():

        pr = average_precision_score(

            harmful,
            r
        )


        roc = roc_auc_score(

            harmful,
            r
        )


        harm_rows.append({

            "Gate":
                gate_name,

            "Reliability":
                reliability_name,

            "Harm_Prevalence":
                float(
                    np.mean(
                        harmful
                    )
                ),

            "Harm_PR_AUC":
                pr,

            "Harm_ROC_AUC":
                roc,

            "Mean_Harm":
                float(
                    np.mean(
                        routing_harm
                    )
                ),

            "Mean_Harm_HarmfulOnly":
                float(
                    np.mean(

                        routing_harm[
                            harmful == 1
                        ]
                    )
                ),
        })


HARM_RESULTS_N10A = (

    pd.DataFrame(
        harm_rows
    )

    .sort_values(
        "Harm_PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 155
)

print(
    "N10A — CAN RELIABILITY IDENTIFY HARMFUL ROUTING?"
)

print(
    "=" * 155
)


display(
    HARM_RESULTS_N10A
)


# ============================================================
# N10A.9 ROUTING MAGNITUDE
# ============================================================

magnitude_rows = []


for gate_name, gate_obj in (
    GATES_N10.items()
):

    p_gate = gate_obj[
        "prediction"
    ]


    correction = np.abs(

        p_gate
        -
        P_ANCHOR_N10
    )


    magnitude_rows.append({

        "Gate":
            gate_name,

        "MeanAbsCorrection":
            float(
                np.mean(
                    correction
                )
            ),

        "MedianAbsCorrection":
            float(
                np.median(
                    correction
                )
            ),

        "P90AbsCorrection":
            float(
                np.quantile(
                    correction,
                    0.90
                )
            ),

        "P99AbsCorrection":
            float(
                np.quantile(
                    correction,
                    0.99
                )
            ),

        "MaxAbsCorrection":
            float(
                np.max(
                    correction
                )
            ),
    })


ROUTING_MAGNITUDE_N10A = pd.DataFrame(

    magnitude_rows
)


print(
    "\n"
    + "=" * 155
)

print(
    "N10A — ADAPTIVE CORRECTION MAGNITUDE"
)

print(
    "=" * 155
)


display(
    ROUTING_MAGNITUDE_N10A
)


# ============================================================
# N10A.10 TRUST DISTRIBUTION
# ============================================================

trust_rows = []


for trust_name, alpha in (
    TRUST_VARIANTS_N10.items()
):

    if trust_name == "NONE":
        continue


    trust_rows.append({

        "Trust":
            trust_name,

        "MeanAlpha":
            float(
                np.mean(
                    alpha
                )
            ),

        "P10":
            float(
                np.quantile(
                    alpha,
                    0.10
                )
            ),

        "Median":
            float(
                np.median(
                    alpha
                )
            ),

        "P90":
            float(
                np.quantile(
                    alpha,
                    0.90
                )
            ),

        "Max":
            float(
                np.max(
                    alpha
                )
            ),
    })


TRUST_SUMMARY_N10A = pd.DataFrame(

    trust_rows
)


print(
    "\n"
    + "=" * 155
)

print(
    "N10A — RELIABILITY-CONTROLLED ROUTING STRENGTH"
)

print(
    "=" * 155
)


display(
    TRUST_SUMMARY_N10A
)


# ============================================================
# N10A.11 SAVE
# ============================================================

RESULTS_N10A.to_csv(

    TAB_DIR
    /
    "n10a_residual_moe_screen.csv",

    index=False
)


HARM_RESULTS_N10A.to_csv(

    TAB_DIR
    /
    "n10a_routing_harm_prediction.csv",

    index=False
)


ROUTING_MAGNITUDE_N10A.to_csv(

    TAB_DIR
    /
    "n10a_routing_magnitude.csv",

    index=False
)


TRUST_SUMMARY_N10A.to_csv(

    TAB_DIR
    /
    "n10a_trust_strength.csv",

    index=False
)


print(
    "\nMODULE N10A COMPLETE"
)

MODULE N10A — RELIABILITY-GUIDED RESIDUAL MoE SCREEN

Anchor LSA-HE-5:
{'PR_AUC': np.float64(0.3796450707882225), 'ROC_AUC': np.float64(0.8506166182615538), 'Brier': np.float64(0.0697298004096943), 'NLL': 0.23566657615581602, 'ECE': 0.0027742093872892525}


,Gate,Trust,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_Anchor
0,CONSENSUS,FULL_GATE,0.379709,0.850620,0.069727,0.235653,0.002660,0.000064
1,CONSENSUS,CONF_LPNS_LINEAR_050,0.379668,0.850619,0.069729,0.235660,0.002701,0.000023
2,CONSENSUS,CONF_LPNS_LINEAR_025,0.379661,0.850618,0.069729,0.235663,0.002743,0.000016
3,CONSENSUS,FULL_LINEAR_025,0.379661,0.850618,0.069729,0.235663,0.002747,0.000016
4,CONSENSUS,CONF_LINEAR_025,0.379660,0.850618,0.069729,0.235663,0.002765,0.000015
5,CONSENSUS,CONF_LPNS_SIGMOID,0.379642,0.850618,0.069730,0.235663,0.002775,-0.000003
6,SPECIALIST,CONF_LPNS_SIGMOID,0.379639,0.850615,0.069730,0.235670,0.002784,-0.000006
7,SHARP,CONF_LPNS_LINEAR_025,0.379639,0.850615,0.069730,0.235664,0.002765,-0.000006
8,SHARP,CONF_LINEAR_025,0.379638,0.850615,0.069730,0.235664,0.002772,-0.000007
9,SHARP,FULL_LINEAR_025,0.379638,0.850615,0.069730,0.235664,0.002757,-0.000007



N10A — CAN RELIABILITY IDENTIFY HARMFUL ROUTING?


,Gate,Reliability,Harm_Prevalence,Harm_PR_AUC,Harm_ROC_AUC,Mean_Harm,Mean_Harm_HarmfulOnly
0,SPECIALIST,CONF,0.772839,0.635470,0.192367,0.000008,0.000199
1,SPECIALIST,CONF_LPNS,0.772839,0.635013,0.191059,0.000008,0.000199
2,SPECIALIST,FULL,0.772839,0.635008,0.191841,0.000008,0.000199
3,CONSENSUS,FULL,0.227526,0.460052,0.807353,-0.000003,0.000704
4,CONSENSUS,CONF_LPNS,0.227526,0.459768,0.808118,-0.000003,0.000704
5,CONSENSUS,CONF,0.227526,0.459153,0.806822,-0.000003,0.000704
6,SHARP,CONF,0.091631,0.260927,0.832895,0.000001,0.002640
7,SHARP,FULL,0.091631,0.260916,0.833104,0.000001,0.002640
8,SHARP,CONF_LPNS,0.091631,0.260254,0.832904,0.000001,0.002640



N10A — ADAPTIVE CORRECTION MAGNITUDE


,Gate,MeanAbsCorrection,MedianAbsCorrection,P90AbsCorrection,P99AbsCorrection,MaxAbsCorrection
0,SHARP,0.000775,0.000266,0.001986,0.007312,0.060861
1,CONSENSUS,0.000506,0.000132,0.001202,0.006182,0.066995
2,SPECIALIST,0.000450,0.000095,0.000970,0.006113,0.145510



N10A — RELIABILITY-CONTROLLED ROUTING STRENGTH


,Trust,MeanAlpha,P10,Median,P90,Max
0,CONF_LINEAR_025,0.215567,0.093685,0.240244,0.249298,0.249938
1,CONF_LPNS_LINEAR_025,0.215523,0.096009,0.240220,0.249278,0.249928
2,CONF_LPNS_LINEAR_050,0.431045,0.192019,0.480439,0.498557,0.499855
3,FULL_LINEAR_025,0.215544,0.096418,0.240264,0.249276,0.249934
4,CONF_LPNS_SIGMOID,0.354220,0.000045,0.450908,0.474949,0.476156



MODULE N10A COMPLETE


In [18]:
# ============================================================
# MODULE N10B — STRICT CONSENSUS-REGULARIZED LEARNED MoE
#
# FINAL routing experiment.
#
# Frozen five experts:
#   1. XGB_SMOOTH
#   2. CAT_SMOOTH
#   3. MLP_REPAIRED
#   4. AUTOINT_REPAIRED
#   5. XGB_HARD_DISAGREE_CF
#
# STRICT OUTER-FOLD DESIGN
# ------------------------------------------------------------
#
# For each outer fold:
#
#   OUTER TRAIN
#       |
#       +-- 3-fold inner cross-fitting of all five experts
#       |      -> gate-training expert predictions
#       |
#       +-- gate feature preprocessing
#       +-- gate epoch selection using gate-training split
#       +-- fresh gate retraining
#
#   OUTER VALID
#       |
#       +-- use already strict N7B expert predictions
#       +-- frozen learned gate
#       +-- final bounded residual prediction
#
#
# Important:
# Existing DEV-wide OOF predictions are NOT reused as
# gate-training predictions.
#
# LPNS is NOT used as the routing controller.
# LPNS remains the separate reliability component.
#
# Current development does not use:
#   - historically inspected 2015 holdout
#   - previously inspected BRFSS-2020
# ============================================================


import gc
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    average_precision_score,
)


# ============================================================
# N10B.1 FIXED CONFIGURATION
#
# Fixed before outer-fold results are inspected.
# No hyperparameter search.
# ============================================================

GATE_INNER_FOLDS_N10B = 3

GATE_MAX_EPOCHS_N10B = 50

GATE_PATIENCE_N10B = 7

GATE_BATCH_N10B = 4096

GATE_LR_N10B = 1e-3

GATE_WEIGHT_DECAY_N10B = 1e-4


# bounded prediction correction
DELTA_MAX_N10B = 0.03


# regularization
LAMBDA_UNIFORM_N10B = 0.01

LAMBDA_RESIDUAL_N10B = 0.02


# fixed neural training epochs derived BEFORE N10B
#
# N7B:
# MLP epochs     = [6, 6, 12, 8, 9]
# AutoInt epochs = [35, 35, 35, 35, 25]
#
# Freeze medians rather than reselecting using N10B folds.

MLP_FIXED_EPOCH_N10B = 8

AUTOINT_FIXED_EPOCH_N10B = 35


FINAL_EXPERTS_N10B = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


N_EXPERTS_N10B = len(
    FINAL_EXPERTS_N10B
)


# ============================================================
# N10B.2 TREE MODEL FACTORIES
# ============================================================

def make_xgb_smooth_n10b(
    seed
):

    return XGBClassifier(

        objective="binary:logistic",

        eval_metric="aucpr",

        n_estimators=900,

        learning_rate=0.025,

        max_depth=3,

        min_child_weight=15,

        subsample=0.90,

        colsample_bytree=0.85,

        gamma=0.10,

        reg_alpha=1.0,

        reg_lambda=7.0,

        tree_method="hist",

        random_state=seed,

        n_jobs=-1,
    )


def make_cat_smooth_n10b(
    seed
):

    return CatBoostClassifier(

        iterations=900,

        learning_rate=0.03,

        depth=5,

        loss_function="Logloss",

        eval_metric="PRAUC",

        l2_leaf_reg=8.0,

        random_seed=seed,

        verbose=False,

        allow_writing_files=False,

        thread_count=-1,
    )


# ============================================================
# N10B.3 GENERATE ONE INNER-FOLD EXPERT SET
#
# Train on inner_train_global
# Predict inner_valid_global
#
# Nothing from outer-valid enters this function.
# ============================================================

def fit_predict_inner_experts_n10b(

    inner_train_global,
    inner_valid_global,

    outer_fold,
    inner_fold,
):

    predictions = {}


    X_tr = X_DEV.iloc[
        inner_train_global
    ]


    X_va = X_DEV.iloc[
        inner_valid_global
    ]


    y_tr = y_DEV[
        inner_train_global
    ]


    # ========================================================
    # 1. XGB Smooth
    # ========================================================

    xgb = make_xgb_smooth_n10b(

        SEED
        +
        70000
        +
        outer_fold * 1000
        +
        inner_fold * 100
        +
        1
    )


    xgb.fit(
        X_tr,
        y_tr
    )


    predictions[
        "XGB_SMOOTH"
    ] = xgb.predict_proba(
        X_va
    )[:, 1]


    del xgb
    gc.collect()


    # ========================================================
    # 2. CatBoost Smooth
    # ========================================================

    cat = make_cat_smooth_n10b(

        SEED
        +
        70000
        +
        outer_fold * 1000
        +
        inner_fold * 100
        +
        2
    )


    cat.fit(
        X_tr,
        y_tr
    )


    predictions[
        "CAT_SMOOTH"
    ] = cat.predict_proba(
        X_va
    )[:, 1]


    del cat
    gc.collect()


    # ========================================================
    # 3. MLP
    #
    # Epoch count fixed before N10B.
    # Scaling fitted on inner train only by retrain_outer_n3r.
    # ========================================================

    mlp_factory = MODEL_FACTORIES_N3A[
        "MLP_GLOBAL"
    ]


    (
        p_mlp,
        mlp_model,
        mlp_scaler

    ) = retrain_outer_n3r(

        mlp_factory,

        inner_train_global,
        inner_valid_global,

        n_epochs=
            MLP_FIXED_EPOCH_N10B,

        seed=
            (
                SEED
                +
                71000
                +
                outer_fold * 1000
                +
                inner_fold * 100
            )
    )


    predictions[
        "MLP_REPAIRED"
    ] = p_mlp


    del mlp_model
    del mlp_scaler

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # ========================================================
    # 4. AutoInt
    #
    # Epoch count fixed before N10B.
    # ========================================================

    autoint_factory = MODEL_FACTORIES_N3A[
        "AUTOINT_GLOBAL"
    ]


    (
        p_autoint,
        autoint_model,
        autoint_scaler

    ) = retrain_outer_n3r(

        autoint_factory,

        inner_train_global,
        inner_valid_global,

        n_epochs=
            AUTOINT_FIXED_EPOCH_N10B,

        seed=
            (
                SEED
                +
                72000
                +
                outer_fold * 1000
                +
                inner_fold * 100
            )
    )


    predictions[
        "AUTOINT_REPAIRED"
    ] = p_autoint


    del autoint_model
    del autoint_scaler

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # ========================================================
    # 5. Hard + Disagreement specialist
    #
    # Difficulty weights are themselves generated ONLY from
    # inner_train_global through additional cross-fitting.
    # ========================================================

    synthetic_fold_id = (

        100

        +
        outer_fold * 10

        +
        inner_fold
    )


    (
        weight_dict,
        diagnostics

    ) = build_inner_difficulty_n4a(

        inner_train_global,

        synthetic_fold_id
    )


    specialist = make_specialist_xgb_n4a(

        SEED
        +
        73000
        +
        outer_fold * 1000
        +
        inner_fold * 100
    )


    specialist.fit(

        X_tr,

        y_tr,

        sample_weight=
            weight_dict[
                "XGB_HARD_DISAGREE_CF"
            ]
    )


    predictions[
        "XGB_HARD_DISAGREE_CF"
    ] = specialist.predict_proba(

        X_va

    )[:, 1]


    del specialist
    del weight_dict
    del diagnostics

    gc.collect()


    return predictions


# ============================================================
# N10B.4 CREATE CROSS-FITTED GATE-TRAINING EXPERT PREDICTIONS
# ============================================================

def build_gate_training_oof_n10b(

    outer_train_idx,
    outer_fold,
):

    n_outer = len(
        outer_train_idx
    )


    P_inner = np.full(

        (
            n_outer,
            N_EXPERTS_N10B
        ),

        np.nan,

        dtype=np.float32
    )


    y_outer = y_DEV[
        outer_train_idx
    ]


    inner_cv = StratifiedKFold(

        n_splits=
            GATE_INNER_FOLDS_N10B,

        shuffle=True,

        random_state=
            SEED
            +
            74000
            +
            outer_fold
    )


    for inner_fold, (
        tr_local,
        va_local

    ) in enumerate(

        inner_cv.split(

            outer_train_idx,

            y_outer
        )
    ):

        print(
            f"\n      Gate inner fold "
            f"{inner_fold + 1}/"
            f"{GATE_INNER_FOLDS_N10B}"
        )


        inner_train_global = (

            outer_train_idx[
                tr_local
            ]
        )


        inner_valid_global = (

            outer_train_idx[
                va_local
            ]
        )


        predictions = (
            fit_predict_inner_experts_n10b(

                inner_train_global,

                inner_valid_global,

                outer_fold,

                inner_fold,
            )
        )


        for j, expert in enumerate(

            FINAL_EXPERTS_N10B
        ):

            P_inner[
                va_local,
                j
            ] = predictions[
                expert
            ]


        del predictions

        gc.collect()


    if np.isnan(
        P_inner
    ).any():

        raise RuntimeError(

            "Incomplete gate-training expert OOF."
        )


    return P_inner


# ============================================================
# N10B.5 GATE FEATURE BUILDER
#
# Inputs:
#
# 21 standardized patient variables
# 5 expert probabilities
# 1 anchor probability
# 5 expert absolute deviations
# 4 aggregate disagreement statistics
#
# Total = 36 features
# ============================================================

def build_gate_features_n10b(

    raw_feature_matrix,

    expert_probability,

    raw_scaler,
):

    Z = raw_scaler.transform(

        raw_feature_matrix

    ).astype(
        np.float32
    )


    P = np.asarray(

        expert_probability,

        dtype=np.float32
    )


    anchor = np.mean(

        P,

        axis=1,

        keepdims=True
    )


    absdev = np.abs(

        P
        -
        anchor
    )


    expert_std = np.std(

        P,

        axis=1,

        keepdims=True
    )


    expert_range = (

        np.max(
            P,
            axis=1,
            keepdims=True
        )

        -

        np.min(
            P,
            axis=1,
            keepdims=True
        )
    )


    mean_absdev = np.mean(

        absdev,

        axis=1,

        keepdims=True
    )


    max_absdev = np.max(

        absdev,

        axis=1,

        keepdims=True
    )


    X_gate = np.concatenate([

        Z,

        P,

        anchor,

        absdev,

        expert_std,

        expert_range,

        mean_absdev,

        max_absdev,

    ], axis=1)


    return X_gate.astype(
        np.float32
    )


# ============================================================
# N10B.6 SMALL CONSENSUS-REGULARIZED GATE
# ============================================================

class ConsensusRegularizedGateN10B(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        n_experts,
    ):

        super().__init__()


        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.GELU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                32
            ),

            nn.GELU(),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                32,
                n_experts
            ),
        )


    def forward(
        self,
        x
    ):

        logits = self.net(
            x
        )


        weights = torch.softmax(

            logits,

            dim=1
        )


        return weights


# ============================================================
# N10B.7 BOUNDED RESIDUAL MoE
# ============================================================

def bounded_moe_prediction_n10b(

    weights,
    expert_prob,
):

    anchor = torch.mean(

        expert_prob,

        dim=1
    )


    adaptive = torch.sum(

        weights
        *
        expert_prob,

        dim=1
    )


    raw_delta = (

        adaptive
        -
        anchor
    )


    bounded_delta = (

        DELTA_MAX_N10B

        *
        torch.tanh(

            raw_delta

            /
            DELTA_MAX_N10B
        )
    )


    final_prob = (

        anchor
        +
        bounded_delta
    )


    final_prob = torch.clamp(

        final_prob,

        1e-6,

        1 - 1e-6
    )


    return (

        final_prob,
        anchor,
        adaptive,
        bounded_delta
    )


# ============================================================
# N10B.8 TRAINING LOSS
# ============================================================

def gate_loss_n10b(

    weights,
    expert_prob,
    y,
):

    (
        final_prob,
        anchor,
        adaptive,
        bounded_delta

    ) = bounded_moe_prediction_n10b(

        weights,
        expert_prob
    )


    # predictive loss
    pred_loss = F.binary_cross_entropy(

        final_prob,

        y
    )


    # --------------------------------------------------------
    # Uniform-weight KL regularization
    #
    # KL(w || uniform)
    # --------------------------------------------------------

    uniform_log = np.log(

        1.0
        /
        N_EXPERTS_N10B
    )


    kl_uniform = torch.mean(

        torch.sum(

            weights

            *
            (

                torch.log(
                    weights
                    +
                    1e-8
                )

                -
                uniform_log
            ),

            dim=1
        )
    )


    # --------------------------------------------------------
    # Residual magnitude penalty
    # --------------------------------------------------------

    residual_penalty = torch.mean(

        (
            bounded_delta

            /
            DELTA_MAX_N10B
        ) ** 2
    )


    total_loss = (

        pred_loss

        +

        LAMBDA_UNIFORM_N10B
        *
        kl_uniform

        +

        LAMBDA_RESIDUAL_N10B
        *
        residual_penalty
    )


    return (

        total_loss,

        pred_loss,

        kl_uniform,

        residual_penalty,
    )


# ============================================================
# N10B.9 NUMPY -> TORCH
# ============================================================

def tensors_n10b(

    X_gate,
    P_expert,
    y,
):

    return (

        torch.tensor(

            X_gate,

            dtype=torch.float32,

            device=DEVICE
        ),

        torch.tensor(

            P_expert,

            dtype=torch.float32,

            device=DEVICE
        ),

        torch.tensor(

            y,

            dtype=torch.float32,

            device=DEVICE
        ),
    )


# ============================================================
# N10B.10 MINI-BATCH TRAIN ONE EPOCH
# ============================================================

def train_gate_epoch_n10b(

    model,
    optimizer,

    X,
    P,
    y,

    seed,
):

    model.train()


    rng = np.random.default_rng(
        seed
    )


    order = rng.permutation(
        len(y)
    )


    losses = []


    for start in range(

        0,

        len(order),

        GATE_BATCH_N10B
    ):

        idx = order[

            start:
            start
            +
            GATE_BATCH_N10B
        ]


        xb = X[
            idx
        ]


        pb = P[
            idx
        ]


        yb = y[
            idx
        ]


        optimizer.zero_grad(
            set_to_none=True
        )


        weights = model(
            xb
        )


        (
            loss,
            _,
            _,
            _

        ) = gate_loss_n10b(

            weights,
            pb,
            yb
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=5.0
        )


        optimizer.step()


        losses.append(
            float(
                loss.detach().cpu()
            )
        )


    return float(
        np.mean(
            losses
        )
    )


# ============================================================
# N10B.11 PREDICT GATE
# ============================================================

@torch.no_grad()
def predict_gate_n10b(

    model,
    X_gate,
    P_expert,
):

    model.eval()


    X_t = torch.tensor(

        X_gate,

        dtype=torch.float32,

        device=DEVICE
    )


    P_t = torch.tensor(

        P_expert,

        dtype=torch.float32,

        device=DEVICE
    )


    all_prob = []

    all_weight = []

    all_delta = []


    batch = 8192


    for start in range(

        0,

        len(
            X_gate
        ),

        batch
    ):

        xb = X_t[
            start:
            start + batch
        ]


        pb = P_t[
            start:
            start + batch
        ]


        weights = model(
            xb
        )


        (
            final_prob,
            _,
            _,
            bounded_delta

        ) = bounded_moe_prediction_n10b(

            weights,
            pb
        )


        all_prob.append(

            final_prob.cpu().numpy()
        )


        all_weight.append(

            weights.cpu().numpy()
        )


        all_delta.append(

            bounded_delta.cpu().numpy()
        )


    return (

        np.concatenate(
            all_prob
        ),

        np.vstack(
            all_weight
        ),

        np.concatenate(
            all_delta
        ),
    )


# ============================================================
# N10B.12 GATE EPOCH SELECTION
#
# Gate-training expert predictions are already cross-fitted.
#
# We split those rows into:
#   gate-train
#   gate-validation
#
# Outer-valid remains completely untouched.
# ============================================================

def select_gate_epoch_n10b(

    X_gate,
    P_gate,
    y_gate,

    seed,
):

    indices = np.arange(
        len(
            y_gate
        )
    )


    tr_idx, va_idx = train_test_split(

        indices,

        test_size=0.15,

        stratify=y_gate,

        random_state=seed,
    )


    X_t, P_t, y_t = tensors_n10b(

        X_gate,
        P_gate,
        y_gate
    )


    seed_everything_n3(
        seed
    )


    model = ConsensusRegularizedGateN10B(

        input_dim=
            X_gate.shape[1],

        n_experts=
            N_EXPERTS_N10B
    ).to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            GATE_LR_N10B,

        weight_decay=
            GATE_WEIGHT_DECAY_N10B
    )


    best_epoch = 1

    best_pr = -np.inf

    patience = 0


    for epoch in range(

        1,

        GATE_MAX_EPOCHS_N10B + 1
    ):

        train_gate_epoch_n10b(

            model,
            optimizer,

            X_t[
                tr_idx
            ],

            P_t[
                tr_idx
            ],

            y_t[
                tr_idx
            ],

            seed=
                seed
                +
                epoch
        )


        pred_valid, _, _ = (

            predict_gate_n10b(

                model,

                X_gate[
                    va_idx
                ],

                P_gate[
                    va_idx
                ]
            )
        )


        pr = average_precision_score(

            y_gate[
                va_idx
            ],

            pred_valid
        )


        if pr > (
            best_pr
            +
            1e-6
        ):

            best_pr = pr

            best_epoch = epoch

            patience = 0


        else:

            patience += 1


        if patience >= (
            GATE_PATIENCE_N10B
        ):

            break


    del model
    del optimizer
    del X_t
    del P_t
    del y_t

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return (

        int(
            best_epoch
        ),

        float(
            best_pr
        )
    )


# ============================================================
# N10B.13 RETRAIN GATE ON ALL OUTER-TRAIN GATE DATA
# ============================================================

def retrain_gate_n10b(

    X_gate,
    P_gate,
    y_gate,

    n_epochs,
    seed,
):

    X_t, P_t, y_t = tensors_n10b(

        X_gate,
        P_gate,
        y_gate
    )


    seed_everything_n3(
        seed
    )


    model = ConsensusRegularizedGateN10B(

        input_dim=
            X_gate.shape[1],

        n_experts=
            N_EXPERTS_N10B
    ).to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            GATE_LR_N10B,

        weight_decay=
            GATE_WEIGHT_DECAY_N10B
    )


    for epoch in range(

        1,
        n_epochs + 1
    ):

        train_gate_epoch_n10b(

            model,
            optimizer,

            X_t,
            P_t,
            y_t,

            seed=
                seed
                +
                epoch
        )


    del optimizer
    del X_t
    del P_t
    del y_t

    gc.collect()


    return model


# ============================================================
# N10B.14 OOF STORAGE
# ============================================================

P_MOE_N10B = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


WEIGHT_OOF_N10B = np.full(

    (
        len(
            y_DEV
        ),

        N_EXPERTS_N10B
    ),

    np.nan,

    dtype=np.float32
)


DELTA_OOF_N10B = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


FOLD_ROWS_N10B = []

EPOCH_ROWS_N10B = []


# ============================================================
# N10B.15 STRICT OUTER LOOP
# ============================================================

total_start_n10b = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 160
    )

    print(
        f"N10B — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 160
    )


    outer_train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    # ========================================================
    # STEP 1 — cross-fitted expert probabilities for gate train
    # ========================================================

    print(
        "\n[1/5] Building strict inner expert predictions..."
    )


    P_gate_train = (

        build_gate_training_oof_n10b(

            outer_train_idx,

            outer_fold
        )
    )


    # ========================================================
    # STEP 2 — outer-valid strict expert probabilities
    #
    # Already generated in N7B from models trained strictly
    # without current outer-valid fold.
    # ========================================================

    print(
        "\n[2/5] Loading strict outer-valid expert predictions..."
    )


    P_gate_valid = np.column_stack([

        OOF_N7B[
            expert
        ][
            outer_valid_idx
        ]

        for expert
        in FINAL_EXPERTS_N10B
    ]).astype(
        np.float32
    )


    # ========================================================
    # STEP 3 — gate feature preprocessing
    # ========================================================

    print(
        "\n[3/5] Building gate features..."
    )


    raw_scaler = StandardScaler()


    raw_scaler.fit(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ]
    )


    X_gate_train = (

        build_gate_features_n10b(

            X_DEV.iloc[
                outer_train_idx
            ][
                FEATURES
            ],

            P_gate_train,

            raw_scaler
        )
    )


    X_gate_valid = (

        build_gate_features_n10b(

            X_DEV.iloc[
                outer_valid_idx
            ][
                FEATURES
            ],

            P_gate_valid,

            raw_scaler
        )
    )


    y_gate_train = y_DEV[
        outer_train_idx
    ]


    # ========================================================
    # STEP 4 — gate epoch selection
    # ========================================================

    print(
        "\n[4/5] Selecting gate epoch..."
    )


    best_epoch, inner_best_pr = (

        select_gate_epoch_n10b(

            X_gate_train,

            P_gate_train,

            y_gate_train,

            seed=
                SEED
                +
                76000
                +
                outer_fold
        )
    )


    print(
        f"Selected gate epoch: "
        f"{best_epoch}"
    )


    print(
        f"Gate internal validation PR: "
        f"{inner_best_pr:.6f}"
    )


    EPOCH_ROWS_N10B.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            best_epoch,

        "Inner_Gate_PR":
            inner_best_pr,
    })


    # ========================================================
    # STEP 5 — retrain full gate and evaluate outer valid
    # ========================================================

    print(
        "\n[5/5] Retraining full gate..."
    )


    gate = retrain_gate_n10b(

        X_gate_train,

        P_gate_train,

        y_gate_train,

        n_epochs=
            best_epoch,

        seed=
            SEED
            +
            77000
            +
            outer_fold
    )


    (
        p_valid,
        weights_valid,
        delta_valid

    ) = predict_gate_n10b(

        gate,

        X_gate_valid,

        P_gate_valid
    )


    P_MOE_N10B[
        outer_valid_idx
    ] = p_valid


    WEIGHT_OOF_N10B[
        outer_valid_idx
    ] = weights_valid


    DELTA_OOF_N10B[
        outer_valid_idx
    ] = delta_valid


    # --------------------------------------------------------
    # anchor on exactly same expert predictions
    # --------------------------------------------------------

    p_anchor_fold = np.mean(

        P_gate_valid,

        axis=1
    )


    m_anchor = probability_metrics(

        y_DEV[
            outer_valid_idx
        ],

        p_anchor_fold
    )


    m_moe = probability_metrics(

        y_DEV[
            outer_valid_idx
        ],

        p_valid
    )


    FOLD_ROWS_N10B.append({

        "Fold":
            outer_fold,

        "Anchor_PR":
            m_anchor[
                "PR_AUC"
            ],

        "MoE_PR":
            m_moe[
                "PR_AUC"
            ],

        "Delta_PR":
            (
                m_moe[
                    "PR_AUC"
                ]
                -
                m_anchor[
                    "PR_AUC"
                ]
            ),

        "Anchor_ROC":
            m_anchor[
                "ROC_AUC"
            ],

        "MoE_ROC":
            m_moe[
                "ROC_AUC"
            ],

        "Anchor_Brier":
            m_anchor[
                "Brier"
            ],

        "MoE_Brier":
            m_moe[
                "Brier"
            ],

        "Anchor_NLL":
            m_anchor[
                "NLL"
            ],

        "MoE_NLL":
            m_moe[
                "NLL"
            ],

        "Anchor_ECE":
            m_anchor[
                "ECE"
            ],

        "MoE_ECE":
            m_moe[
                "ECE"
            ],
    })


    print(
        "\nFOLD RESULT"
    )


    print(

        f"Anchor PR = "
        f"{m_anchor['PR_AUC']:.6f}"
    )


    print(

        f"MoE PR    = "
        f"{m_moe['PR_AUC']:.6f}"
    )


    print(

        f"Delta     = "
        f"{m_moe['PR_AUC'] - m_anchor['PR_AUC']:+.6f}"
    )


    del gate

    del P_gate_train
    del P_gate_valid

    del X_gate_train
    del X_gate_valid

    del raw_scaler

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# N10B.16 COMPLETENESS CHECK
# ============================================================

if np.isnan(
    P_MOE_N10B
).any():

    raise RuntimeError(
        "Incomplete MoE OOF predictions."
    )


if np.isnan(
    WEIGHT_OOF_N10B
).any():

    raise RuntimeError(
        "Incomplete gate weights."
    )


# ============================================================
# N10B.17 POOLED RESULTS
# ============================================================

P_ANCHOR_N10B = np.mean(

    np.column_stack([

        OOF_N7B[
            expert
        ]

        for expert
        in FINAL_EXPERTS_N10B
    ]),

    axis=1
)


ANCHOR_METRICS_N10B = probability_metrics(

    y_DEV,
    P_ANCHOR_N10B
)


MOE_METRICS_N10B = probability_metrics(

    y_DEV,
    P_MOE_N10B
)


RESULTS_N10B = pd.DataFrame([

    {
        "System":
            "Frozen_LSA_HE_5_Anchor",

        **ANCHOR_METRICS_N10B,
    },

    {
        "System":
            "Consensus_Regularized_MoE",

        **MOE_METRICS_N10B,
    },
])


RESULTS_N10B[
    "Delta_PR_vs_Anchor"
] = (

    RESULTS_N10B[
        "PR_AUC"
    ]

    -
    ANCHOR_METRICS_N10B[
        "PR_AUC"
    ]
)


print(
    "\n"
    + "=" * 160
)

print(
    "MODULE N10B — STRICT CONSENSUS-REGULARIZED MoE RESULTS"
)

print(
    "=" * 160
)


display(
    RESULTS_N10B
)


# ============================================================
# N10B.18 FOLD STABILITY
# ============================================================

FOLD_RESULTS_N10B = pd.DataFrame(

    FOLD_ROWS_N10B
)


print(
    "\n"
    + "=" * 160
)

print(
    "N10B — FOLD RESULTS"
)

print(
    "=" * 160
)


display(
    FOLD_RESULTS_N10B
)


print(
    "\nMean fold delta PR:",
    FOLD_RESULTS_N10B[
        "Delta_PR"
    ].mean()
)


print(
    "Positive-delta folds:",
    int(
        np.sum(
            FOLD_RESULTS_N10B[
                "Delta_PR"
            ]
            >
            0
        )
    ),
    "/",
    N_FOLDS
)


# ============================================================
# N10B.19 ROUTING WEIGHT SUMMARY
# ============================================================

weight_rows = []


for j, expert in enumerate(

    FINAL_EXPERTS_N10B
):

    w = WEIGHT_OOF_N10B[
        :,
        j
    ]


    weight_rows.append({

        "Expert":
            expert,

        "MeanWeight":
            float(
                np.mean(
                    w
                )
            ),

        "SD":
            float(
                np.std(
                    w
                )
            ),

        "P10":
            float(
                np.quantile(
                    w,
                    0.10
                )
            ),

        "Median":
            float(
                np.median(
                    w
                )
            ),

        "P90":
            float(
                np.quantile(
                    w,
                    0.90
                )
            ),

        "Min":
            float(
                np.min(
                    w
                )
            ),

        "Max":
            float(
                np.max(
                    w
                )
            ),
    })


WEIGHT_SUMMARY_N10B = pd.DataFrame(

    weight_rows
)


print(
    "\n"
    + "=" * 160
)

print(
    "N10B — EXPERT WEIGHT DISTRIBUTION"
)

print(
    "=" * 160
)


display(
    WEIGHT_SUMMARY_N10B
)


# ============================================================
# N10B.20 ROUTING ENTROPY / EFFECTIVE EXPERTS
# ============================================================

gate_entropy = -(

    WEIGHT_OOF_N10B

    *
    np.log(

        WEIGHT_OOF_N10B
        +
        1e-8
    )

).sum(
    axis=1
)


normalized_entropy = (

    gate_entropy

    /
    np.log(
        N_EXPERTS_N10B
    )
)


effective_experts = np.exp(

    gate_entropy
)


ROUTING_SUMMARY_N10B = pd.DataFrame([

    {
        "NormalizedEntropy_Mean":
            float(
                np.mean(
                    normalized_entropy
                )
            ),

        "NormalizedEntropy_SD":
            float(
                np.std(
                    normalized_entropy
                )
            ),

        "NormalizedEntropy_P10":
            float(
                np.quantile(
                    normalized_entropy,
                    0.10
                )
            ),

        "NormalizedEntropy_Median":
            float(
                np.median(
                    normalized_entropy
                )
            ),

        "NormalizedEntropy_P90":
            float(
                np.quantile(
                    normalized_entropy,
                    0.90
                )
            ),

        "EffectiveExperts_Mean":
            float(
                np.mean(
                    effective_experts
                )
            ),

        "EffectiveExperts_Median":
            float(
                np.median(
                    effective_experts
                )
            ),
    }
])


print(
    "\n"
    + "=" * 160
)

print(
    "N10B — ROUTING ENTROPY"
)

print(
    "=" * 160
)


display(
    ROUTING_SUMMARY_N10B
)


# ============================================================
# N10B.21 CORRECTION MAGNITUDE
# ============================================================

CORRECTION_SUMMARY_N10B = pd.DataFrame([

    {
        "MeanSignedCorrection":
            float(
                np.mean(
                    DELTA_OOF_N10B
                )
            ),

        "MeanAbsCorrection":
            float(
                np.mean(
                    np.abs(
                        DELTA_OOF_N10B
                    )
                )
            ),

        "MedianAbsCorrection":
            float(
                np.median(
                    np.abs(
                        DELTA_OOF_N10B
                    )
                )
            ),

        "P90AbsCorrection":
            float(
                np.quantile(
                    np.abs(
                        DELTA_OOF_N10B
                    ),
                    0.90
                )
            ),

        "P99AbsCorrection":
            float(
                np.quantile(
                    np.abs(
                        DELTA_OOF_N10B
                    ),
                    0.99
                )
            ),

        "MaxAbsCorrection":
            float(
                np.max(
                    np.abs(
                        DELTA_OOF_N10B
                    )
                )
            ),
    }
])


print(
    "\n"
    + "=" * 160
)

print(
    "N10B — BOUNDED CORRECTION MAGNITUDE"
)

print(
    "=" * 160
)


display(
    CORRECTION_SUMMARY_N10B
)


# ============================================================
# N10B.22 DOMINANT-EXPERT USAGE
# ============================================================

dominant_idx = np.argmax(

    WEIGHT_OOF_N10B,

    axis=1
)


dominant_name = [

    FINAL_EXPERTS_N10B[
        i
    ]

    for i
    in dominant_idx
]


DOMINANT_USAGE_N10B = (

    pd.Series(
        dominant_name
    )

    .value_counts()

    .rename_axis(
        "Expert"
    )

    .reset_index(
        name="Dominant_N"
    )
)


DOMINANT_USAGE_N10B[
    "Dominant_Fraction"
] = (

    DOMINANT_USAGE_N10B[
        "Dominant_N"
    ]

    /
    len(
        y_DEV
    )
)


print(
    "\n"
    + "=" * 160
)

print(
    "N10B — DOMINANT EXPERT USAGE"
)

print(
    "=" * 160
)


display(
    DOMINANT_USAGE_N10B
)


# ============================================================
# N10B.23 GATE EPOCHS
# ============================================================

GATE_EPOCHS_N10B = pd.DataFrame(

    EPOCH_ROWS_N10B
)


print(
    "\n"
    + "=" * 160
)

print(
    "N10B — GATE EPOCH SELECTION"
)

print(
    "=" * 160
)


display(
    GATE_EPOCHS_N10B
)


# ============================================================
# N10B.24 SAVE
# ============================================================

RESULTS_N10B.to_csv(

    TAB_DIR
    /
    "n10b_consensus_regularized_moe.csv",

    index=False
)


FOLD_RESULTS_N10B.to_csv(

    TAB_DIR
    /
    "n10b_fold_results.csv",

    index=False
)


WEIGHT_SUMMARY_N10B.to_csv(

    TAB_DIR
    /
    "n10b_expert_weight_summary.csv",

    index=False
)


ROUTING_SUMMARY_N10B.to_csv(

    TAB_DIR
    /
    "n10b_routing_entropy.csv",

    index=False
)


CORRECTION_SUMMARY_N10B.to_csv(

    TAB_DIR
    /
    "n10b_correction_summary.csv",

    index=False
)


DOMINANT_USAGE_N10B.to_csv(

    TAB_DIR
    /
    "n10b_dominant_expert_usage.csv",

    index=False
)


GATE_EPOCHS_N10B.to_csv(

    TAB_DIR
    /
    "n10b_gate_epochs.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "N10B_MOE_OOF.npy",

    P_MOE_N10B
)


np.save(

    PRED_DIR
    /
    "N10B_GATE_WEIGHTS.npy",

    WEIGHT_OOF_N10B
)


np.save(

    PRED_DIR
    /
    "N10B_CORRECTION.npy",

    DELTA_OOF_N10B
)


print(
    "\n"
    + "=" * 160
)

print(
    "MODULE N10B COMPLETE"
)

print(
    "=" * 160
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - total_start_n10b:.1f}s"
)


print(
    "\nNo 2015 secondary holdout used."
)

print(
    "No BRFSS-2020 used."
)

print(
    "\nThis completes predictive routing development."
)


N10B — OUTER FOLD 0

[1/5] Building strict inner expert predictions...

      Gate inner fold 1/3

  Building inner difficulty signals for outer fold 100...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

      Gate inner fold 2/3

  Building inner difficulty signals for outer fold 101...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

      Gate inner fold 3/3

  Building inner difficulty signals for outer fold 102...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

[2/5] Loading strict outer-valid expert predictions...

[3/5] Building gate features...

[4/5] Selecting gate epoch...
Selected gate epoch: 2
Gate internal validation PR: 0.378523

[5/5] Retraining full gate...

FOLD RESULT
Anchor PR = 0.378993
MoE PR    = 0.378941
Delta     = -0.000051

N10B — OUTER FOLD 1

[1/5] Building strict inner expert predictions.

,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_Anchor
0,Frozen_LSA_HE_5_Anchor,0.379645,0.850617,0.069730,0.235667,0.002774,0.000000
1,Consensus_Regularized_MoE,0.379643,0.850618,0.069729,0.235647,0.002496,-0.000002



N10B — FOLD RESULTS


,Fold,Anchor_PR,MoE_PR,Delta_PR,Anchor_ROC,MoE_ROC,Anchor_Brier,MoE_Brier,Anchor_NLL,MoE_NLL,Anchor_ECE,MoE_ECE
0,0,0.378993,0.378941,-0.000051,0.852518,0.852534,0.069673,0.069673,0.234947,0.234919,0.002848,0.002723
1,1,0.381858,0.381914,0.000056,0.850508,0.850506,0.069755,0.069752,0.235752,0.235727,0.004541,0.004295
2,2,0.379144,0.379100,-0.000044,0.849177,0.849169,0.069828,0.069827,0.236326,0.236312,0.004185,0.004177
3,3,0.372725,0.372715,-0.000010,0.849145,0.849161,0.070082,0.070081,0.236670,0.236650,0.003733,0.003633
4,4,0.388756,0.388711,-0.000045,0.851989,0.851969,0.069311,0.069313,0.234637,0.234628,0.005289,0.004865



Mean fold delta PR: -1.9029349466404e-05
Positive-delta folds: 1 / 5

N10B — EXPERT WEIGHT DISTRIBUTION


...


N10B — DOMINANT EXPERT USAGE


,Expert,Dominant_N,Dominant_Fraction
0,CAT_SMOOTH,75413,0.371595
1,AUTOINT_REPAIRED,41607,0.205017
2,XGB_SMOOTH,41278,0.203396
3,MLP_REPAIRED,31822,0.156802
4,XGB_HARD_DISAGREE_CF,12824,0.063190



N10B — GATE EPOCH SELECTION


,Fold,Selected_Epoch,Inner_Gate_PR
0,0,2,0.378523
1,1,1,0.385804
2,2,27,0.398833
3,3,14,0.390170
4,4,10,0.368026



MODULE N10B COMPLETE

Runtime: 1732.3s

No 2015 secondary holdout used.
No BRFSS-2020 used.

This completes predictive routing development.


In [19]:
# ============================================================
# MODULE N11-FAST — OPTIMIZED PAIRED SIGNIFICANCE TESTING
#
# FINAL:
#   Consensus-Regularized MoE (N10B)
#
# BASELINES:
#   Static 5-expert anchor
#   XGB_HARD_DISAGREE_CF
#   XGB_SMOOTH
#   CAT_SMOOTH
#   AUTOINT_REPAIRED
#   MLP_REPAIRED
#
# DESIGN
# ------------------------------------------------------------
# PR-AUC / ROC-AUC:
#   500 paired stratified bootstrap replicates
#
# Brier / NLL:
#   fast paired bootstrap on row-level loss differences
#
# IMPORTANT:
#   Same resampled subjects are used for final and baseline.
#
# Use 500 for screening/all comparisons.
# Only rerun important borderline comparisons with 2000+
# later if needed.
# ============================================================


import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# N11F.1 CONFIGURATION
# ============================================================

N_BOOT_RANK_N11F = 500

N_BOOT_LOSS_N11F = 2000

EPS_N11F = 1e-7

SEED_N11F = SEED + 91000


# ============================================================
# N11F.2 FINAL SYSTEM + BASELINES
# ============================================================

Y_N11F = np.asarray(
    y_DEV,
    dtype=np.int8
)


P_FINAL_N11F = np.asarray(
    P_MOE_N10B,
    dtype=np.float64
)


BASELINES_N11F = {

    "LSA_HE_5_Anchor":
        np.asarray(
            P_ANCHOR_N10B,
            dtype=np.float64
        ),

    "XGB_HARD_DISAGREE_CF":
        np.asarray(
            OOF_N7B[
                "XGB_HARD_DISAGREE_CF"
            ],
            dtype=np.float64
        ),

    "XGB_SMOOTH":
        np.asarray(
            OOF_N7B[
                "XGB_SMOOTH"
            ],
            dtype=np.float64
        ),

    "CAT_SMOOTH":
        np.asarray(
            OOF_N7B[
                "CAT_SMOOTH"
            ],
            dtype=np.float64
        ),

    "AUTOINT_REPAIRED":
        np.asarray(
            OOF_N7B[
                "AUTOINT_REPAIRED"
            ],
            dtype=np.float64
        ),

    "MLP_REPAIRED":
        np.asarray(
            OOF_N7B[
                "MLP_REPAIRED"
            ],
            dtype=np.float64
        ),
}


# ============================================================
# N11F.3 SANITY CHECK
# ============================================================

assert len(
    P_FINAL_N11F
) == len(
    Y_N11F
)


assert not np.isnan(
    P_FINAL_N11F
).any()


for name, p in (
    BASELINES_N11F.items()
):

    if len(p) != len(Y_N11F):

        raise RuntimeError(
            f"Length mismatch: {name}"
        )


    if np.isnan(p).any():

        raise RuntimeError(
            f"NaN predictions: {name}"
        )


# ============================================================
# N11F.4 ORIGINAL POINT METRICS
# ============================================================

def point_metrics_n11f(
    y,
    p
):

    p_clip = np.clip(
        p,
        EPS_N11F,
        1 - EPS_N11F
    )


    return {

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),

        "Brier":
            np.mean(
                (
                    p - y
                ) ** 2
            ),

        "NLL":
            np.mean(
                -(
                    y
                    *
                    np.log(
                        p_clip
                    )

                    +

                    (1 - y)
                    *
                    np.log(
                        1 - p_clip
                    )
                )
            ),
    }


FINAL_POINT_N11F = point_metrics_n11f(
    Y_N11F,
    P_FINAL_N11F
)


# ============================================================
# N11F.5 PRECOMPUTE STRATA
# ============================================================

POS_IDX_N11F = np.where(
    Y_N11F == 1
)[0]


NEG_IDX_N11F = np.where(
    Y_N11F == 0
)[0]


N_POS_N11F = len(
    POS_IDX_N11F
)


N_NEG_N11F = len(
    NEG_IDX_N11F
)


print(
    "Positive rows:",
    N_POS_N11F
)


print(
    "Negative rows:",
    N_NEG_N11F
)


# ============================================================
# N11F.6 FAST PR / ROC PAIRED BOOTSTRAP
#
# Generate one bootstrap sample and evaluate BOTH metrics
# for final and baseline in the same iteration.
#
# This avoids separate PR and ROC bootstrap loops.
# ============================================================

def bootstrap_rank_metrics_n11f(

    p_baseline,

    n_boot,

    seed,
):

    rng = np.random.default_rng(
        seed
    )


    delta_pr = np.empty(
        n_boot,
        dtype=np.float64
    )


    delta_roc = np.empty(
        n_boot,
        dtype=np.float64
    )


    for b in range(
        n_boot
    ):

        # ---------------------------------------------
        # stratified paired resampling
        # ---------------------------------------------

        sampled_pos = rng.choice(

            POS_IDX_N11F,

            size=N_POS_N11F,

            replace=True
        )


        sampled_neg = rng.choice(

            NEG_IDX_N11F,

            size=N_NEG_N11F,

            replace=True
        )


        idx = np.concatenate([

            sampled_pos,
            sampled_neg
        ])


        y_b = Y_N11F[
            idx
        ]


        pf = P_FINAL_N11F[
            idx
        ]


        pb = p_baseline[
            idx
        ]


        # ---------------------------------------------
        # PR-AUC
        # ---------------------------------------------

        pr_final = average_precision_score(
            y_b,
            pf
        )


        pr_base = average_precision_score(
            y_b,
            pb
        )


        delta_pr[b] = (
            pr_final
            -
            pr_base
        )


        # ---------------------------------------------
        # ROC-AUC
        # ---------------------------------------------

        roc_final = roc_auc_score(
            y_b,
            pf
        )


        roc_base = roc_auc_score(
            y_b,
            pb
        )


        delta_roc[b] = (
            roc_final
            -
            roc_base
        )


    return (
        delta_pr,
        delta_roc
    )


# ============================================================
# N11F.7 EMPIRICAL CI + P
# ============================================================

def summarize_bootstrap_n11f(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    ci_low, ci_high = np.quantile(

        values,

        [
            0.025,
            0.975
        ]
    )


    p_left = (

        np.sum(
            values <= 0
        )
        +
        1

    ) / (
        len(values)
        +
        1
    )


    p_right = (

        np.sum(
            values >= 0
        )
        +
        1

    ) / (
        len(values)
        +
        1
    )


    p_two = min(

        1.0,

        2.0
        *
        min(
            p_left,
            p_right
        )
    )


    return {

        "CI_Low":
            float(
                ci_low
            ),

        "CI_High":
            float(
                ci_high
            ),

        "P_Value":
            float(
                p_two
            ),

        "Significant_95":
            bool(
                ci_low > 0
                or
                ci_high < 0
            ),
    }


# ============================================================
# N11F.8 FAST LOSS-METRIC BOOTSTRAP
#
# Brier and NLL are mean per-row losses.
#
# Improvement is defined as:
#
#       baseline loss - final loss
#
# Positive delta therefore always means FINAL is better.
#
# We bootstrap the row-level paired differences rather than
# repeatedly calling metric functions.
# ============================================================

def bootstrap_mean_difference_n11f(

    difference,

    n_boot,

    seed,

    batch_size=100,
):

    rng = np.random.default_rng(
        seed
    )


    difference = np.asarray(
        difference,
        dtype=np.float64
    )


    n = len(
        difference
    )


    out = np.empty(
        n_boot,
        dtype=np.float64
    )


    cursor = 0


    while cursor < n_boot:

        current = min(

            batch_size,

            n_boot
            -
            cursor
        )


        # Keep memory bounded.
        #
        # shape approximately:
        # current x N

        idx = rng.integers(

            0,
            n,

            size=(
                current,
                n
            ),

            dtype=np.int32
        )


        out[
            cursor:
            cursor + current
        ] = np.mean(

            difference[
                idx
            ],

            axis=1
        )


        cursor += current


        del idx


    return out


# ============================================================
# N11F.9 FINAL ROW-LEVEL LOSSES
# ============================================================

P_FINAL_CLIP_N11F = np.clip(

    P_FINAL_N11F,

    EPS_N11F,

    1 - EPS_N11F
)


FINAL_BRIER_LOSS_N11F = (

    P_FINAL_N11F
    -
    Y_N11F

) ** 2


FINAL_NLL_LOSS_N11F = -(

    Y_N11F
    *
    np.log(
        P_FINAL_CLIP_N11F
    )

    +

    (
        1
        -
        Y_N11F
    )

    *
    np.log(
        1
        -
        P_FINAL_CLIP_N11F
    )
)


# ============================================================
# N11F.10 RUN ALL BASELINES
# ============================================================

start_n11f = time.perf_counter()


all_rows_n11f = []


for comparison_id, (
    baseline_name,
    p_base

) in enumerate(

    BASELINES_N11F.items(),

    start=1
):

    print(
        "\n"
        + "=" * 110
    )


    print(
        f"[{comparison_id}/"
        f"{len(BASELINES_N11F)}] "
        f"{baseline_name}"
    )


    base_point = point_metrics_n11f(

        Y_N11F,

        p_base
    )


    # ========================================================
    # PR + ROC bootstrap
    # ========================================================

    t0 = time.perf_counter()


    (
        delta_pr_boot,
        delta_roc_boot

    ) = bootstrap_rank_metrics_n11f(

        p_base,

        n_boot=
            N_BOOT_RANK_N11F,

        seed=
            SEED_N11F
            +
            comparison_id
            *
            100
    )


    print(
        f"Ranking bootstrap: "
        f"{time.perf_counter() - t0:.1f}s"
    )


    # PR row

    pr_summary = summarize_bootstrap_n11f(

        delta_pr_boot
    )


    all_rows_n11f.append({

        "Baseline":
            baseline_name,

        "Metric":
            "PR_AUC",

        "Final":
            FINAL_POINT_N11F[
                "PR_AUC"
            ],

        "Baseline_Value":
            base_point[
                "PR_AUC"
            ],

        "Delta":
            (
                FINAL_POINT_N11F[
                    "PR_AUC"
                ]

                -
                base_point[
                    "PR_AUC"
                ]
            ),

        **pr_summary,

        "N_Boot":
            N_BOOT_RANK_N11F,
    })


    # ROC row

    roc_summary = summarize_bootstrap_n11f(

        delta_roc_boot
    )


    all_rows_n11f.append({

        "Baseline":
            baseline_name,

        "Metric":
            "ROC_AUC",

        "Final":
            FINAL_POINT_N11F[
                "ROC_AUC"
            ],

        "Baseline_Value":
            base_point[
                "ROC_AUC"
            ],

        "Delta":
            (
                FINAL_POINT_N11F[
                    "ROC_AUC"
                ]

                -
                base_point[
                    "ROC_AUC"
                ]
            ),

        **roc_summary,

        "N_Boot":
            N_BOOT_RANK_N11F,
    })


    # ========================================================
    # BRIER — FAST ROW-LOSS BOOTSTRAP
    # ========================================================

    baseline_brier_loss = (

        p_base
        -
        Y_N11F

    ) ** 2


    # positive = final better

    brier_difference = (

        baseline_brier_loss

        -
        FINAL_BRIER_LOSS_N11F
    )


    delta_brier_boot = (

        bootstrap_mean_difference_n11f(

            brier_difference,

            n_boot=
                N_BOOT_LOSS_N11F,

            seed=
                SEED_N11F
                +
                comparison_id
                *
                100
                +
                20
        )
    )


    brier_summary = summarize_bootstrap_n11f(

        delta_brier_boot
    )


    all_rows_n11f.append({

        "Baseline":
            baseline_name,

        "Metric":
            "Brier",

        "Final":
            FINAL_POINT_N11F[
                "Brier"
            ],

        "Baseline_Value":
            base_point[
                "Brier"
            ],

        # positive means final has lower Brier
        "Delta":
            (
                base_point[
                    "Brier"
                ]

                -
                FINAL_POINT_N11F[
                    "Brier"
                ]
            ),

        **brier_summary,

        "N_Boot":
            N_BOOT_LOSS_N11F,
    })


    # ========================================================
    # NLL — FAST ROW-LOSS BOOTSTRAP
    # ========================================================

    p_base_clip = np.clip(

        p_base,

        EPS_N11F,

        1 - EPS_N11F
    )


    baseline_nll_loss = -(

        Y_N11F
        *
        np.log(
            p_base_clip
        )

        +

        (
            1
            -
            Y_N11F
        )

        *
        np.log(
            1
            -
            p_base_clip
        )
    )


    # positive = final better

    nll_difference = (

        baseline_nll_loss

        -
        FINAL_NLL_LOSS_N11F
    )


    delta_nll_boot = (

        bootstrap_mean_difference_n11f(

            nll_difference,

            n_boot=
                N_BOOT_LOSS_N11F,

            seed=
                SEED_N11F
                +
                comparison_id
                *
                100
                +
                30
        )
    )


    nll_summary = summarize_bootstrap_n11f(

        delta_nll_boot
    )


    all_rows_n11f.append({

        "Baseline":
            baseline_name,

        "Metric":
            "NLL",

        "Final":
            FINAL_POINT_N11F[
                "NLL"
            ],

        "Baseline_Value":
            base_point[
                "NLL"
            ],

        # positive means final has lower NLL
        "Delta":
            (
                base_point[
                    "NLL"
                ]

                -
                FINAL_POINT_N11F[
                    "NLL"
                ]
            ),

        **nll_summary,

        "N_Boot":
            N_BOOT_LOSS_N11F,
    })


    # clean temporary arrays

    del delta_pr_boot
    del delta_roc_boot

    del baseline_brier_loss
    del brier_difference
    del delta_brier_boot

    del baseline_nll_loss
    del nll_difference
    del delta_nll_boot


# ============================================================
# N11F.11 FULL RESULTS
# ============================================================

SIGNIFICANCE_N11F = pd.DataFrame(

    all_rows_n11f
)


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N11-FAST — ALL PAIRED SIGNIFICANCE RESULTS"
)

print(
    "=" * 165
)


display(
    SIGNIFICANCE_N11F
)


# ============================================================
# N11F.12 PR-AUC SUMMARY
# ============================================================

PR_SIGNIFICANCE_N11F = (

    SIGNIFICANCE_N11F[

        SIGNIFICANCE_N11F[
            "Metric"
        ]
        ==
        "PR_AUC"

    ]

    .sort_values(

        "Delta",

        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N11-FAST — PR-AUC SIGNIFICANCE SUMMARY"
)

print(
    "=" * 165
)


display(
    PR_SIGNIFICANCE_N11F
)


# ============================================================
# N11F.13 COMPACT PAPER TABLE
# ============================================================

PAPER_SIG_N11F = (

    SIGNIFICANCE_N11F

    .pivot(

        index="Baseline",

        columns="Metric",

        values=[
            "Delta",
            "CI_Low",
            "CI_High",
            "P_Value",
        ]
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N11-FAST — COMPACT SIGNIFICANCE TABLE"
)

print(
    "=" * 165
)


display(
    PAPER_SIG_N11F
)


# ============================================================
# N11F.14 SAVE
# ============================================================

SIGNIFICANCE_N11F.to_csv(

    TAB_DIR
    /
    "n11_fast_all_significance.csv",

    index=False
)


PR_SIGNIFICANCE_N11F.to_csv(

    TAB_DIR
    /
    "n11_fast_pr_significance.csv",

    index=False
)


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N11-FAST COMPLETE"
)

print(
    "=" * 165
)


print(
    f"\nTotal runtime: "
    f"{time.perf_counter() - start_n11f:.1f}s"
)

Positive rows: 19114
Negative rows: 183830

[1/6] LSA_HE_5_Anchor
Ranking bootstrap: 82.7s

[2/6] XGB_HARD_DISAGREE_CF
Ranking bootstrap: 82.6s

[3/6] XGB_SMOOTH
Ranking bootstrap: 82.7s

[4/6] CAT_SMOOTH
Ranking bootstrap: 83.1s

[5/6] AUTOINT_REPAIRED
Ranking bootstrap: 82.7s

[6/6] MLP_REPAIRED
Ranking bootstrap: 81.9s

MODULE N11-FAST — ALL PAIRED SIGNIFICANCE RESULTS


,Baseline,Metric,Final,Baseline_Value,Delta,CI_Low,CI_High,P_Value,Significant_95,N_Boot
0,LSA_HE_5_Anchor,PR_AUC,0.379643,0.379645,-2.352783e-06,-0.000055,0.000043,0.834331,False,500
1,LSA_HE_5_Anchor,ROC_AUC,0.850618,0.850617,9.259400e-07,-0.000015,0.000017,0.938124,False,500
2,LSA_HE_5_Anchor,Brier,0.069729,0.069730,7.799871e-07,-0.000001,0.000003,0.487756,False,2000
3,LSA_HE_5_Anchor,NLL,0.235647,0.235667,1.946800e-05,0.000009,0.000030,0.001999,True,2000
4,XGB_HARD_DISAGREE_CF,PR_AUC,0.379643,0.379299,3.438388e-04,-0.000295,0.000963,0.311377,False,500
5,XGB_HARD_DISAGREE_CF,ROC_AUC,0.850618,0.850415,2.023949e-04,0.000018,0.000382,0.031936,True,500
6,XGB_HARD_DISAGREE_CF,Brier,0.069729,0.069907,1.782379e-04,0.000140,0.000217,0.001000,True,2000
7,XGB_HARD_DISAGREE_CF,NLL,0.235647,0.237708,2.060596e-03,0.001839,0.002284,0.001000,True,2000
8,XGB_SMOOTH,PR_AUC,0.379643,0.378606,1.036833e-03,0.000254,0.001754,0.011976,True,500
9,XGB_SMOOTH,ROC_AUC,0.850618,0.850151,4.665393e-04,0.000242,0.000681,0.003992,True,500



N11-FAST — PR-AUC SIGNIFICANCE SUMMARY


,Baseline,Metric,Final,Baseline_Value,Delta,CI_Low,CI_High,P_Value,Significant_95,N_Boot
0,MLP_REPAIRED,PR_AUC,0.379643,0.372623,0.007020,0.005407,0.008696,0.003992,True,500
1,AUTOINT_REPAIRED,PR_AUC,0.379643,0.375803,0.003839,0.002660,0.005096,0.003992,True,500
2,CAT_SMOOTH,PR_AUC,0.379643,0.377730,0.001913,0.000892,0.002934,0.003992,True,500
3,XGB_SMOOTH,PR_AUC,0.379643,0.378606,0.001037,0.000254,0.001754,0.011976,True,500
4,XGB_HARD_DISAGREE_CF,PR_AUC,0.379643,0.379299,0.000344,-0.000295,0.000963,0.311377,False,500
5,LSA_HE_5_Anchor,PR_AUC,0.379643,0.379645,-0.000002,-0.000055,0.000043,0.834331,False,500



N11-FAST — COMPACT SIGNIFICANCE TABLE


Delta                                      CI_Low                                 CI_High                                 P_Value                              
Metric                       Brier       NLL    PR_AUC       ROC_AUC     Brier       NLL    PR_AUC   ROC_AUC     Brier       NLL    PR_AUC   ROC_AUC     Brier       NLL    PR_AUC   ROC_AUC
Baseline                                                                                                                                                                                    
AUTOINT_REPAIRED      2.406999e-04  0.001108  0.003839  1.562499e-03  0.000186  0.000877  0.002660  0.001242  0.000297  0.001337  0.005096  0.001854  0.001000  0.001000  0.003992  0.003992
CAT_SMOOTH            7.934238e-05  0.000177  0.001913  3.630449e-04  0.000038  0.000038  0.000892  0.000151  0.000121  0.000316  0.002934  0.000592  0.001000  0.006997  0.003992  0.007984
LSA_HE_5_Anchor       7.799871e-07  0.000019 -0.000002  9.259400e-07 -0.000001  0.000009 -0.000055 -0.000015  0.000003  0.000030  0.000043  0.000017  0.487756  0.001999  0.834331  0.938124
MLP_REPAIRED          4.426221e-04  0.001788  0.007020  3.001496e-03  0.000371  0.001541  0.005407  0.002526  0.000514  0.002043  0.008696  0.003429  0.001000  0.001000  0.003992  0.003992
XGB_HARD_DISAGREE_CF  1.782379e-04  0.002061  0.000344  2.023949e-04  0.000140  0.001839 -0.000295  0.000018  0.000217  0.002284  0.000963  0.000382  0.001000  0.001000  0.311377  0.031936
XGB_SMOOTH            8.238933e-05  0.000239  0.001037  4.665393e-04  0.000046  0.000123  0.000254  0.000242  0.000118  0.000352  0.001754  0.000681  0.001000  0.001000  0.011976  0.003992


MODULE N11-FAST COMPLETE

Total runtime: 544.7s


In [20]:
# ============================================================
# MODULE N11B-FAST
# OPERATING-POINT STATISTICAL COMPARISON
#
# Adds:
#   Precision
#   Recall / Sensitivity
#   Specificity
#   F1
#   MCC
#   Balanced Accuracy
#   Accuracy
#
# Plus:
#   McNemar test for paired classification errors
#
# Thresholds:
#   Strict cross-fitted F1-optimal threshold for EACH model.
#
# Positive delta always means FINAL MoE is better.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    accuracy_score,
)

from scipy.stats import binomtest


# ============================================================
# N11B.1 CONFIG
# ============================================================

N_BOOT_OPERATING_N11B = 1000

SEED_N11B = SEED + 95000


# ============================================================
# N11B.2 FAST F1 THRESHOLD SEARCH
# ============================================================

def select_f1_threshold_n11b(
    y,
    p,
):

    thresholds = np.linspace(
        0.03,
        0.50,
        300
    )


    best_t = thresholds[0]
    best_f1 = -1.0


    for t in thresholds:

        pred = (
            p >= t
        )


        tp = np.sum(
            pred & (y == 1)
        )

        fp = np.sum(
            pred & (y == 0)
        )

        fn = np.sum(
            (~pred) & (y == 1)
        )


        denom = (
            2 * tp
            +
            fp
            +
            fn
        )


        score = (
            0.0
            if denom == 0
            else
            2 * tp / denom
        )


        if score > best_f1:

            best_f1 = score
            best_t = t


    return float(best_t)


# ============================================================
# N11B.3 CROSS-FITTED OPERATING PREDICTIONS
# ============================================================

def crossfit_class_predictions_n11b(
    p,
):

    p = np.asarray(
        p,
        dtype=float
    )


    pred = np.zeros(
        len(y_DEV),
        dtype=np.int8
    )


    threshold_used = np.zeros(
        len(y_DEV),
        dtype=float
    )


    threshold_rows = []


    for fold in range(
        N_FOLDS
    ):

        train_mask = (
            DEV_FOLD != fold
        )

        valid_mask = (
            DEV_FOLD == fold
        )


        threshold = (
            select_f1_threshold_n11b(

                y_DEV[
                    train_mask
                ],

                p[
                    train_mask
                ]
            )
        )


        pred[
            valid_mask
        ] = (

            p[
                valid_mask
            ]
            >=
            threshold

        ).astype(
            np.int8
        )


        threshold_used[
            valid_mask
        ] = threshold


        threshold_rows.append({

            "Fold":
                fold,

            "Threshold":
                threshold,
        })


    return (
        pred,
        threshold_used,
        pd.DataFrame(
            threshold_rows
        )
    )


# ============================================================
# N11B.4 METRICS FROM CONFUSION COUNTS
# ============================================================

def operating_metrics_n11b(
    y,
    pred,
):

    y = np.asarray(
        y
    )

    pred = np.asarray(
        pred
    )


    tp = np.sum(
        (y == 1)
        &
        (pred == 1)
    )


    tn = np.sum(
        (y == 0)
        &
        (pred == 0)
    )


    fp = np.sum(
        (y == 0)
        &
        (pred == 1)
    )


    fn = np.sum(
        (y == 1)
        &
        (pred == 0)
    )


    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )


    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )


    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )


    f1 = (
        2 * tp
        /
        (
            2 * tp
            +
            fp
            +
            fn
        )

        if (
            2 * tp
            +
            fp
            +
            fn
        ) > 0

        else 0.0
    )


    balanced_accuracy = (
        recall
        +
        specificity
    ) / 2


    accuracy = (
        tp + tn
    ) / (
        tp + tn + fp + fn
    )


    mcc_den = np.sqrt(

        (
            tp + fp
        )

        *
        (
            tp + fn
        )

        *
        (
            tn + fp
        )

        *
        (
            tn + fn
        )
    )


    mcc = (
        (
            tp * tn
            -
            fp * fn
        )
        /
        mcc_den

        if mcc_den > 0

        else 0.0
    )


    return {

        "Precision":
            precision,

        "Recall":
            recall,

        "Specificity":
            specificity,

        "F1":
            f1,

        "MCC":
            mcc,

        "Balanced_Accuracy":
            balanced_accuracy,

        "Accuracy":
            accuracy,

        "TP":
            int(tp),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),
    }


# ============================================================
# N11B.5 BUILD FINAL + BASELINE CLASSIFICATIONS
# ============================================================

P_SYSTEMS_N11B = {

    "Consensus_Regularized_MoE":
        np.asarray(
            P_MOE_N10B,
            dtype=float
        ),

    **BASELINES_N11F
}


CLASS_PRED_N11B = {}

THRESHOLDS_N11B = {}


for system_name, probabilities in (
    P_SYSTEMS_N11B.items()
):

    (
        class_pred,
        threshold_vector,
        threshold_table

    ) = crossfit_class_predictions_n11b(

        probabilities
    )


    CLASS_PRED_N11B[
        system_name
    ] = class_pred


    THRESHOLDS_N11B[
        system_name
    ] = threshold_table


# ============================================================
# N11B.6 POINT METRIC TABLE
# ============================================================

point_rows = []


for system_name, pred in (
    CLASS_PRED_N11B.items()
):

    m = operating_metrics_n11b(

        y_DEV,
        pred
    )


    point_rows.append({

        "System":
            system_name,

        **m,
    })


OPERATING_POINT_N11B = (

    pd.DataFrame(
        point_rows
    )

    .sort_values(
        "F1",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "=" * 165
)

print(
    "N11B — CROSS-FITTED OPERATING-POINT PERFORMANCE"
)

print(
    "=" * 165
)


display(
    OPERATING_POINT_N11B
)


# ============================================================
# N11B.7 PRECOMPUTE STRATA
# ============================================================

pos_idx = np.where(
    y_DEV == 1
)[0]


neg_idx = np.where(
    y_DEV == 0
)[0]


# ============================================================
# N11B.8 PAIRED BOOTSTRAP
#
# Much faster than PR-AUC bootstrap because only confusion
# counts are calculated.
# ============================================================

METRICS_OPERATING_N11B = [

    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "MCC",
    "Balanced_Accuracy",
    "Accuracy",
]


def paired_operating_bootstrap_n11b(

    final_pred,
    baseline_pred,

    n_boot,
    seed,
):

    rng = np.random.default_rng(
        seed
    )


    deltas = {

        metric:
            np.empty(
                n_boot,
                dtype=np.float64
            )

        for metric
        in METRICS_OPERATING_N11B
    }


    for b in range(
        n_boot
    ):

        idx_pos = rng.choice(

            pos_idx,

            size=len(
                pos_idx
            ),

            replace=True
        )


        idx_neg = rng.choice(

            neg_idx,

            size=len(
                neg_idx
            ),

            replace=True
        )


        idx = np.concatenate([

            idx_pos,
            idx_neg

        ])


        m_final = operating_metrics_n11b(

            y_DEV[
                idx
            ],

            final_pred[
                idx
            ]
        )


        m_base = operating_metrics_n11b(

            y_DEV[
                idx
            ],

            baseline_pred[
                idx
            ]
        )


        for metric in (
            METRICS_OPERATING_N11B
        ):

            deltas[
                metric
            ][b] = (

                m_final[
                    metric
                ]

                -

                m_base[
                    metric
                ]
            )


    return deltas


# ============================================================
# N11B.9 RUN COMPARISONS
# ============================================================

final_pred = CLASS_PRED_N11B[

    "Consensus_Regularized_MoE"
]


final_point = operating_metrics_n11b(

    y_DEV,
    final_pred
)


comparison_rows = []


baseline_names = [

    x

    for x in BASELINES_N11F.keys()
]


for comparison_id, baseline_name in enumerate(

    baseline_names,

    start=1
):

    print(
        f"\n[{comparison_id}/"
        f"{len(baseline_names)}] "
        f"{baseline_name}"
    )


    baseline_pred = CLASS_PRED_N11B[
        baseline_name
    ]


    baseline_point = operating_metrics_n11b(

        y_DEV,
        baseline_pred
    )


    bootstrap = (

        paired_operating_bootstrap_n11b(

            final_pred,

            baseline_pred,

            n_boot=
                N_BOOT_OPERATING_N11B,

            seed=
                (
                    SEED_N11B
                    +
                    comparison_id
                )
        )
    )


    for metric in (
        METRICS_OPERATING_N11B
    ):

        values = bootstrap[
            metric
        ]


        ci_low, ci_high = np.quantile(

            values,

            [
                0.025,
                0.975
            ]
        )


        p_left = (

            np.sum(
                values <= 0
            )
            +
            1

        ) / (
            len(values)
            +
            1
        )


        p_right = (

            np.sum(
                values >= 0
            )
            +
            1

        ) / (
            len(values)
            +
            1
        )


        p_value = min(

            1.0,

            2
            *
            min(
                p_left,
                p_right
            )
        )


        comparison_rows.append({

            "Baseline":
                baseline_name,

            "Metric":
                metric,

            "Final":
                final_point[
                    metric
                ],

            "Baseline_Value":
                baseline_point[
                    metric
                ],

            "Delta":
                (
                    final_point[
                        metric
                    ]

                    -
                    baseline_point[
                        metric
                    ]
                ),

            "CI_Low":
                ci_low,

            "CI_High":
                ci_high,

            "P_Value":
                p_value,

            "Significant_95":
                (
                    ci_low > 0
                    or
                    ci_high < 0
                ),
        })


# ============================================================
# N11B.10 RESULTS
# ============================================================

OPERATING_SIGNIFICANCE_N11B = pd.DataFrame(

    comparison_rows
)


print(
    "\n"
    + "=" * 165
)

print(
    "N11B — OPERATING-METRIC SIGNIFICANCE"
)

print(
    "=" * 165
)


display(
    OPERATING_SIGNIFICANCE_N11B
)


# ============================================================
# N11B.11 KEY CLINICAL METRICS ONLY
# ============================================================

KEY_OPERATING_N11B = (

    OPERATING_SIGNIFICANCE_N11B[

        OPERATING_SIGNIFICANCE_N11B[
            "Metric"
        ].isin([

            "Precision",
            "Recall",
            "F1",
            "MCC",
            "Balanced_Accuracy",

        ])
    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N11B — KEY OPERATING METRIC COMPARISONS"
)

print(
    "=" * 165
)


display(
    KEY_OPERATING_N11B
)


# ============================================================
# N11B.12 McNEMAR
#
# Directly asks whether the two models have significantly
# different error patterns on the same subjects.
#
# Exact binomial version.
# ============================================================

mcnemar_rows = []


final_correct = (

    final_pred
    ==
    y_DEV
)


for baseline_name in (
    baseline_names
):

    baseline_pred = CLASS_PRED_N11B[
        baseline_name
    ]


    baseline_correct = (

        baseline_pred
        ==
        y_DEV
    )


    # final correct, baseline wrong

    b = int(

        np.sum(

            final_correct

            &

            (~baseline_correct)
        )
    )


    # final wrong, baseline correct

    c = int(

        np.sum(

            (~final_correct)

            &

            baseline_correct
        )
    )


    discordant = (
        b + c
    )


    if discordant > 0:

        p_value = binomtest(

            min(
                b,
                c
            ),

            n=
                discordant,

            p=0.5,

            alternative="two-sided"
        ).pvalue


    else:

        p_value = 1.0


    mcnemar_rows.append({

        "Baseline":
            baseline_name,

        "Final_Correct_Base_Wrong":
            b,

        "Final_Wrong_Base_Correct":
            c,

        "Net_Correct_Gain":
            b - c,

        "Discordant_N":
            discordant,

        "McNemar_P":
            p_value,

        "Significant_05":
            p_value < 0.05,
    })


MCNEMAR_N11B = pd.DataFrame(

    mcnemar_rows
)


print(
    "\n"
    + "=" * 165
)

print(
    "N11B — McNEMAR PAIRED ERROR COMPARISON"
)

print(
    "=" * 165
)


display(
    MCNEMAR_N11B
)


# ============================================================
# N11B.13 SAVE
# ============================================================

OPERATING_POINT_N11B.to_csv(

    TAB_DIR
    /
    "n11b_operating_point_metrics.csv",

    index=False
)


OPERATING_SIGNIFICANCE_N11B.to_csv(

    TAB_DIR
    /
    "n11b_operating_significance.csv",

    index=False
)


KEY_OPERATING_N11B.to_csv(

    TAB_DIR
    /
    "n11b_key_operating_significance.csv",

    index=False
)


MCNEMAR_N11B.to_csv(

    TAB_DIR
    /
    "n11b_mcnemar.csv",

    index=False
)


print(
    "\nMODULE N11B-FAST COMPLETE"
)

N11B — CROSS-FITTED OPERATING-POINT PERFORMANCE


,System,Precision,Recall,Specificity,F1,MCC,Balanced_Accuracy,Accuracy,TP,TN,FP,FN
0,LSA_HE_5_Anchor,0.348813,0.533640,0.896415,0.421871,0.000000,0.715028,0.862248,10200,164788,19042,8914
1,Consensus_Regularized_MoE,0.347391,0.535262,0.895447,0.421332,0.000000,0.715354,0.861523,10231,164610,19220,8883
2,XGB_HARD_DISAGREE_CF,0.336748,0.562310,0.884845,0.421234,1.834503,0.723578,0.854467,10748,162661,21169,8366
3,XGB_SMOOTH,0.345787,0.537459,0.894272,0.420826,0.000000,0.715866,0.860666,10273,164394,19436,8841
4,CAT_SMOOTH,0.337113,0.555300,0.886466,0.419534,2.149042,0.720883,0.855275,10614,162959,20871,8500
5,AUTOINT_REPAIRED,0.343681,0.537512,0.893271,0.419278,0.000000,0.715391,0.859764,10274,164210,19620,8840
6,MLP_REPAIRED,0.337412,0.551062,0.887483,0.418549,2.468574,0.719273,0.855798,10533,163146,20684,8581



[1/6] LSA_HE_5_Anchor

[2/6] XGB_HARD_DISAGREE_CF

[3/6] XGB_SMOOTH

[4/6] CAT_SMOOTH

[5/6] AUTOINT_REPAIRED

[6/6] MLP_REPAIRED

N11B — OPERATING-METRIC SIGNIFICANCE


,Baseline,Metric,Final,Baseline_Value,Delta,CI_Low,CI_High,P_Value,Significant_95
0,LSA_HE_5_Anchor,Precision,0.347391,0.348813,-0.001423,-0.001924,-9.520866e-04,0.001998,True
1,LSA_HE_5_Anchor,Recall,0.535262,0.533640,0.001622,0.000837,2.355603e-03,0.001998,True
2,LSA_HE_5_Anchor,Specificity,0.895447,0.896415,-0.000968,-0.001137,-8.105315e-04,0.001998,True
3,LSA_HE_5_Anchor,F1,0.421332,0.421871,-0.000539,-0.001122,6.776894e-07,0.053946,False
4,LSA_HE_5_Anchor,MCC,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,False
5,LSA_HE_5_Anchor,Balanced_Accuracy,0.715354,0.715028,0.000327,-0.000073,7.026546e-04,0.109890,False
6,LSA_HE_5_Anchor,Accuracy,0.861523,0.862248,-0.000724,-0.000897,-5.665356e-04,0.001998,True
7,XGB_HARD_DISAGREE_CF,Precision,0.347391,0.336748,0.010642,0.009173,1.218677e-02,0.001998,True
8,XGB_HARD_DISAGREE_CF,Recall,0.535262,0.562310,-0.027048,-0.029716,-2.448467e-02,0.001998,True
9,XGB_HARD_DISAGREE_CF,Specificity,0.895447,0.884845,0.010602,0.010085,1.115174e-02,0.001998,True



N11B — KEY OPERATING METRIC COMPARISONS


,Baseline,Metric,Final,Baseline_Value,Delta,CI_Low,CI_High,P_Value,Significant_95
0,LSA_HE_5_Anchor,Precision,0.347391,0.348813,-0.001423,-0.001924,-9.520866e-04,0.001998,True
1,LSA_HE_5_Anchor,Recall,0.535262,0.533640,0.001622,0.000837,2.355603e-03,0.001998,True
2,LSA_HE_5_Anchor,F1,0.421332,0.421871,-0.000539,-0.001122,6.776894e-07,0.053946,False
3,LSA_HE_5_Anchor,MCC,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,False
4,LSA_HE_5_Anchor,Balanced_Accuracy,0.715354,0.715028,0.000327,-0.000073,7.026546e-04,0.109890,False
5,XGB_HARD_DISAGREE_CF,Precision,0.347391,0.336748,0.010642,0.009173,1.218677e-02,0.001998,True
6,XGB_HARD_DISAGREE_CF,Recall,0.535262,0.562310,-0.027048,-0.029716,-2.448467e-02,0.001998,True
7,XGB_HARD_DISAGREE_CF,F1,0.421332,0.421234,0.000098,-0.001629,1.870313e-03,0.913087,False
8,XGB_HARD_DISAGREE_CF,MCC,0.000000,1.834503,-1.834503,-2.043657,-1.683281e+00,0.001998,True
9,XGB_HARD_DISAGREE_CF,Balanced_Accuracy,0.715354,0.723578,-0.008223,-0.009544,-6.951970e-03,0.001998,True



N11B — McNEMAR PAIRED ERROR COMPARISON


,Baseline,Final_Correct_Base_Wrong,Final_Wrong_Base_Correct,Net_Correct_Gain,Discordant_N,McNemar_P,Significant_05
0,LSA_HE_5_Anchor,78,225,-147,303,1.102663e-17,True
1,XGB_HARD_DISAGREE_CF,2222,790,1432,3012,5.342241e-156,True
2,XGB_SMOOTH,1250,1076,174,2326,3.324461e-04,True
3,CAT_SMOOTH,2133,865,1268,2998,3.385166e-122,True
4,AUTOINT_REPAIRED,1957,1600,357,3557,2.315775e-09,True
5,MLP_REPAIRED,3066,1904,1162,4970,1.750760e-61,True



MODULE N11B-FAST COMPLETE


In [21]:
# ============================================================
# N11B-R — FAST RERUN AFTER MCC FIX
#
# Reuses:
#   CLASS_PRED_N11B
#   BASELINES_N11F
#   y_DEV
#
# No threshold recomputation needed.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import binomtest


N_BOOT_OPERATING_N11BR = 1000

SEED_N11BR = SEED + 97000


METRICS_OPERATING_N11BR = [

    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "MCC",
    "Balanced_Accuracy",
    "Accuracy",
]


# ============================================================
# POINT RESULTS
# ============================================================

point_rows = []


for system_name, pred in CLASS_PRED_N11B.items():

    m = operating_metrics_n11b(
        y_DEV,
        pred
    )

    point_rows.append({
        "System": system_name,
        **m,
    })


OPERATING_POINT_N11BR = (
    pd.DataFrame(point_rows)
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "=" * 160
)

print(
    "N11B-R — CORRECTED OPERATING-POINT PERFORMANCE"
)

print(
    "=" * 160
)

display(
    OPERATING_POINT_N11BR
)


# ============================================================
# STRATA
# ============================================================

pos_idx = np.where(
    y_DEV == 1
)[0]

neg_idx = np.where(
    y_DEV == 0
)[0]


# ============================================================
# BOOTSTRAP
# ============================================================

def paired_operating_bootstrap_n11br(
    final_pred,
    baseline_pred,
    n_boot,
    seed,
):

    rng = np.random.default_rng(seed)

    deltas = {
        metric:
            np.empty(
                n_boot,
                dtype=np.float64
            )
        for metric
        in METRICS_OPERATING_N11BR
    }


    for b in range(n_boot):

        idx_pos = rng.choice(
            pos_idx,
            size=len(pos_idx),
            replace=True
        )

        idx_neg = rng.choice(
            neg_idx,
            size=len(neg_idx),
            replace=True
        )

        idx = np.concatenate([
            idx_pos,
            idx_neg
        ])


        final_m = operating_metrics_n11b(
            y_DEV[idx],
            final_pred[idx]
        )

        base_m = operating_metrics_n11b(
            y_DEV[idx],
            baseline_pred[idx]
        )


        for metric in METRICS_OPERATING_N11BR:

            deltas[metric][b] = (
                final_m[metric]
                -
                base_m[metric]
            )


    return deltas


# ============================================================
# RUN
# ============================================================

final_name = (
    "Consensus_Regularized_MoE"
)

final_pred = CLASS_PRED_N11B[
    final_name
]


final_point = operating_metrics_n11b(
    y_DEV,
    final_pred
)


rows = []


for comparison_id, baseline_name in enumerate(
    BASELINES_N11F.keys(),
    start=1,
):

    print(
        f"[{comparison_id}/"
        f"{len(BASELINES_N11F)}] "
        f"{baseline_name}"
    )


    baseline_pred = CLASS_PRED_N11B[
        baseline_name
    ]


    baseline_point = operating_metrics_n11b(
        y_DEV,
        baseline_pred
    )


    bootstrap = paired_operating_bootstrap_n11br(

        final_pred,
        baseline_pred,

        n_boot=
            N_BOOT_OPERATING_N11BR,

        seed=
            SEED_N11BR
            +
            comparison_id
    )


    for metric in METRICS_OPERATING_N11BR:

        values = bootstrap[
            metric
        ]


        ci_low, ci_high = np.quantile(
            values,
            [0.025, 0.975]
        )


        p_left = (
            np.sum(values <= 0) + 1
        ) / (
            len(values) + 1
        )


        p_right = (
            np.sum(values >= 0) + 1
        ) / (
            len(values) + 1
        )


        p_value = min(
            1.0,
            2.0
            *
            min(
                p_left,
                p_right
            )
        )


        rows.append({

            "Baseline":
                baseline_name,

            "Metric":
                metric,

            "Final":
                final_point[
                    metric
                ],

            "Baseline_Value":
                baseline_point[
                    metric
                ],

            "Delta":
                (
                    final_point[
                        metric
                    ]
                    -
                    baseline_point[
                        metric
                    ]
                ),

            "CI_Low":
                ci_low,

            "CI_High":
                ci_high,

            "P_Value":
                p_value,

            "Significant_95":
                (
                    ci_low > 0
                    or
                    ci_high < 0
                ),
        })


OPERATING_SIGNIFICANCE_N11BR = pd.DataFrame(
    rows
)


print(
    "\n"
    + "=" * 160
)

print(
    "N11B-R — CORRECTED OPERATING-METRIC SIGNIFICANCE"
)

print(
    "=" * 160
)

display(
    OPERATING_SIGNIFICANCE_N11BR
)


# ============================================================
# KEY PAPER METRICS
# ============================================================

KEY_METRICS_N11BR = [

    "Precision",
    "Recall",
    "F1",
    "MCC",
    "Balanced_Accuracy",
]


KEY_OPERATING_N11BR = (

    OPERATING_SIGNIFICANCE_N11BR[

        OPERATING_SIGNIFICANCE_N11BR[
            "Metric"
        ].isin(
            KEY_METRICS_N11BR
        )

    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 160
)

print(
    "N11B-R — KEY CORRECTED OPERATING METRICS"
)

print(
    "=" * 160
)

display(
    KEY_OPERATING_N11BR
)


# ============================================================
# SAVE
# ============================================================

OPERATING_POINT_N11BR.to_csv(

    TAB_DIR
    /
    "n11b_corrected_operating_metrics.csv",

    index=False
)


OPERATING_SIGNIFICANCE_N11BR.to_csv(

    TAB_DIR
    /
    "n11b_corrected_operating_significance.csv",

    index=False
)


KEY_OPERATING_N11BR.to_csv(

    TAB_DIR
    /
    "n11b_corrected_key_metrics.csv",

    index=False
)


print(
    "\nCorrected N11B complete."
)

N11B-R — CORRECTED OPERATING-POINT PERFORMANCE


,System,Precision,Recall,Specificity,F1,MCC,Balanced_Accuracy,Accuracy,TP,TN,FP,FN
0,LSA_HE_5_Anchor,0.348813,0.533640,0.896415,0.421871,0.000000,0.715028,0.862248,10200,164788,19042,8914
1,Consensus_Regularized_MoE,0.347391,0.535262,0.895447,0.421332,0.000000,0.715354,0.861523,10231,164610,19220,8883
2,XGB_HARD_DISAGREE_CF,0.336748,0.562310,0.884845,0.421234,1.834503,0.723578,0.854467,10748,162661,21169,8366
3,XGB_SMOOTH,0.345787,0.537459,0.894272,0.420826,0.000000,0.715866,0.860666,10273,164394,19436,8841
4,CAT_SMOOTH,0.337113,0.555300,0.886466,0.419534,2.149042,0.720883,0.855275,10614,162959,20871,8500
5,AUTOINT_REPAIRED,0.343681,0.537512,0.893271,0.419278,0.000000,0.715391,0.859764,10274,164210,19620,8840
6,MLP_REPAIRED,0.337412,0.551062,0.887483,0.418549,2.468574,0.719273,0.855798,10533,163146,20684,8581


[1/6] LSA_HE_5_Anchor
[2/6] XGB_HARD_DISAGREE_CF
[3/6] XGB_SMOOTH
[4/6] CAT_SMOOTH
[5/6] AUTOINT_REPAIRED
[6/6] MLP_REPAIRED

N11B-R — CORRECTED OPERATING-METRIC SIGNIFICANCE


,Baseline,Metric,Final,Baseline_Value,Delta,CI_Low,CI_High,P_Value,Significant_95
0,LSA_HE_5_Anchor,Precision,0.347391,0.348813,-0.001423,-0.001918,-0.000923,0.001998,True
1,LSA_HE_5_Anchor,Recall,0.535262,0.533640,0.001622,0.000837,0.002511,0.001998,True
2,LSA_HE_5_Anchor,Specificity,0.895447,0.896415,-0.000968,-0.001126,-0.000805,0.001998,True
3,LSA_HE_5_Anchor,F1,0.421332,0.421871,-0.000539,-0.001102,0.000033,0.063936,False
4,LSA_HE_5_Anchor,MCC,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,False
5,LSA_HE_5_Anchor,Balanced_Accuracy,0.715354,0.715028,0.000327,-0.000074,0.000737,0.125874,False
6,LSA_HE_5_Anchor,Accuracy,0.861523,0.862248,-0.000724,-0.000882,-0.000552,0.001998,True
7,XGB_HARD_DISAGREE_CF,Precision,0.347391,0.336748,0.010642,0.009094,0.012111,0.001998,True
8,XGB_HARD_DISAGREE_CF,Recall,0.535262,0.562310,-0.027048,-0.029665,-0.024432,0.001998,True
9,XGB_HARD_DISAGREE_CF,Specificity,0.895447,0.884845,0.010602,0.010075,0.011108,0.001998,True



N11B-R — KEY CORRECTED OPERATING METRICS


,Baseline,Metric,Final,Baseline_Value,Delta,CI_Low,CI_High,P_Value,Significant_95
0,LSA_HE_5_Anchor,Precision,0.347391,0.348813,-0.001423,-0.001918,-0.000923,0.001998,True
1,LSA_HE_5_Anchor,Recall,0.535262,0.533640,0.001622,0.000837,0.002511,0.001998,True
2,LSA_HE_5_Anchor,F1,0.421332,0.421871,-0.000539,-0.001102,0.000033,0.063936,False
3,LSA_HE_5_Anchor,MCC,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,False
4,LSA_HE_5_Anchor,Balanced_Accuracy,0.715354,0.715028,0.000327,-0.000074,0.000737,0.125874,False
5,XGB_HARD_DISAGREE_CF,Precision,0.347391,0.336748,0.010642,0.009094,0.012111,0.001998,True
6,XGB_HARD_DISAGREE_CF,Recall,0.535262,0.562310,-0.027048,-0.029665,-0.024432,0.001998,True
7,XGB_HARD_DISAGREE_CF,F1,0.421332,0.421234,0.000098,-0.001747,0.001797,0.889111,False
8,XGB_HARD_DISAGREE_CF,MCC,0.000000,1.834503,-1.834503,-2.046595,-1.682894,0.001998,True
9,XGB_HARD_DISAGREE_CF,Balanced_Accuracy,0.715354,0.723578,-0.008223,-0.009570,-0.006939,0.001998,True



Corrected N11B complete.


In [22]:
# ============================================================
# N11C — MCC REPAIR + SIGNIFICANCE ONLY
#
# Uses existing:
#   CLASS_PRED_N11B
#   y_DEV
#
# No threshold recalculation.
# No rerunning other metrics.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# SAFE MCC
# ============================================================

def safe_mcc_n11c(y, pred):

    y = np.asarray(y, dtype=np.int8)
    pred = np.asarray(pred, dtype=np.int8)

    tp = float(np.sum((y == 1) & (pred == 1)))
    tn = float(np.sum((y == 0) & (pred == 0)))
    fp = float(np.sum((y == 0) & (pred == 1)))
    fn = float(np.sum((y == 1) & (pred == 0)))

    numerator = (
        tp * tn
        -
        fp * fn
    )

    denominator = np.sqrt(
        (tp + fp)
        *
        (tp + fn)
        *
        (tn + fp)
        *
        (tn + fn)
    )

    if denominator == 0:
        return 0.0

    mcc = numerator / denominator

    # sanity guard
    if not (-1.000001 <= mcc <= 1.000001):

        raise RuntimeError(
            f"Invalid MCC produced: {mcc}"
        )

    return float(mcc)


# ============================================================
# POINT MCC TABLE
# ============================================================

mcc_point_rows = []


for system_name, pred in CLASS_PRED_N11B.items():

    mcc = safe_mcc_n11c(
        y_DEV,
        pred
    )

    mcc_point_rows.append({

        "System":
            system_name,

        "MCC":
            mcc,
    })


MCC_POINT_N11C = pd.DataFrame(
    mcc_point_rows
)


print(
    "=" * 140
)

print(
    "N11C — CORRECTED MCC POINT ESTIMATES"
)

print(
    "=" * 140
)

display(
    MCC_POINT_N11C
)


# ============================================================
# STRATIFIED PAIRED BOOTSTRAP
# ============================================================

N_BOOT_MCC_N11C = 1000

SEED_N11C = SEED + 99000


pos_idx = np.where(
    y_DEV == 1
)[0]


neg_idx = np.where(
    y_DEV == 0
)[0]


final_name = (
    "Consensus_Regularized_MoE"
)


final_pred = CLASS_PRED_N11B[
    final_name
]


final_mcc = safe_mcc_n11c(
    y_DEV,
    final_pred
)


mcc_sig_rows = []


for comparison_id, baseline_name in enumerate(
    BASELINES_N11F.keys(),
    start=1,
):

    baseline_pred = CLASS_PRED_N11B[
        baseline_name
    ]


    baseline_mcc = safe_mcc_n11c(
        y_DEV,
        baseline_pred
    )


    rng = np.random.default_rng(
        SEED_N11C
        +
        comparison_id
    )


    deltas = np.empty(
        N_BOOT_MCC_N11C,
        dtype=float
    )


    for b in range(
        N_BOOT_MCC_N11C
    ):

        sample_pos = rng.choice(
            pos_idx,
            size=len(pos_idx),
            replace=True
        )

        sample_neg = rng.choice(
            neg_idx,
            size=len(neg_idx),
            replace=True
        )

        idx = np.concatenate([
            sample_pos,
            sample_neg
        ])


        final_mcc_b = safe_mcc_n11c(
            y_DEV[idx],
            final_pred[idx]
        )


        baseline_mcc_b = safe_mcc_n11c(
            y_DEV[idx],
            baseline_pred[idx]
        )


        deltas[b] = (
            final_mcc_b
            -
            baseline_mcc_b
        )


    ci_low, ci_high = np.quantile(
        deltas,
        [0.025, 0.975]
    )


    p_left = (
        np.sum(deltas <= 0)
        +
        1
    ) / (
        len(deltas)
        +
        1
    )


    p_right = (
        np.sum(deltas >= 0)
        +
        1
    ) / (
        len(deltas)
        +
        1
    )


    p_value = min(
        1.0,
        2.0
        *
        min(
            p_left,
            p_right
        )
    )


    mcc_sig_rows.append({

        "Baseline":
            baseline_name,

        "Final_MCC":
            final_mcc,

        "Baseline_MCC":
            baseline_mcc,

        "Delta_MCC":
            final_mcc
            -
            baseline_mcc,

        "CI_Low":
            ci_low,

        "CI_High":
            ci_high,

        "P_Value":
            p_value,

        "Significant_95":
            (
                ci_low > 0
                or
                ci_high < 0
            ),
    })


MCC_SIGNIFICANCE_N11C = pd.DataFrame(
    mcc_sig_rows
)


print(
    "\n"
    + "=" * 140
)

print(
    "N11C — CORRECTED MCC SIGNIFICANCE"
)

print(
    "=" * 140
)

display(
    MCC_SIGNIFICANCE_N11C
)


# ============================================================
# SAVE
# ============================================================

MCC_POINT_N11C.to_csv(
    TAB_DIR
    /
    "n11c_corrected_mcc_points.csv",
    index=False
)


MCC_SIGNIFICANCE_N11C.to_csv(
    TAB_DIR
    /
    "n11c_corrected_mcc_significance.csv",
    index=False
)


print(
    "\nN11C complete."
)

N11C — CORRECTED MCC POINT ESTIMATES


,System,MCC
0,Consensus_Regularized_MoE,0.357172
1,LSA_HE_5_Anchor,0.357686
2,XGB_HARD_DISAGREE_CF,0.358756
3,XGB_SMOOTH,0.356727
4,CAT_SMOOTH,0.356405
5,AUTOINT_REPAIRED,0.355030
6,MLP_REPAIRED,0.355045



N11C — CORRECTED MCC SIGNIFICANCE


,Baseline,Final_MCC,Baseline_MCC,Delta_MCC,CI_Low,CI_High,P_Value,Significant_95
0,LSA_HE_5_Anchor,0.357172,0.357686,-0.000515,-0.001174,0.000124,0.119880,False
1,XGB_HARD_DISAGREE_CF,0.357172,0.358756,-0.001584,-0.003595,0.000404,0.131868,False
2,XGB_SMOOTH,0.357172,0.356727,0.000445,-0.001471,0.002280,0.613387,False
3,CAT_SMOOTH,0.357172,0.356405,0.000767,-0.001356,0.002808,0.429570,False
4,AUTOINT_REPAIRED,0.357172,0.355030,0.002142,-0.000188,0.004425,0.071928,False
5,MLP_REPAIRED,0.357172,0.355045,0.002127,-0.000707,0.004861,0.133866,False



N11C complete.


In [23]:
# ============================================================
# MODULE N12 — STRICT MoE RELIABILITY + SELECTIVE REVIEW
#
# Requires completed N10B.
#
# PURPOSE
# ------------------------------------------------------------
# 1. Rebuild reliability analysis strictly for FINAL MoE.
# 2. Remove second-order OOF contamination.
# 3. Compare:
#       CONF
#       DIS
#       LPNS
#       CONF_DIS
#       CONF_LPNS
#       CONF_DIS_LPNS
# 4. Evaluate selective review at:
#       1%, 5%, 10%, 15%, 20%
#
# IMPORTANT
# ------------------------------------------------------------
# For every outer fold:
#
# OUTER TRAIN
#   -> strict inner-cross-fitted expert predictions
#   -> cross-fitted gate predictions
#   -> MoE error labels
#   -> LPNS reference bank
#   -> reliability-head training
#
# OUTER VALID
#   -> strict N10B MoE prediction
#   -> strict expert predictions
#   -> reliability prediction
#
# Thus the current outer-valid fold is excluded from
# reliability-model development.
# ============================================================


import gc
import os
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


# ============================================================
# N12.1 CONFIGURATION
# ============================================================

K_LPNS_N12 = 20

REFERENCE_SIZE_N12 = 80000

GATE_CF_FOLDS_N12 = 3

REVIEW_BUDGETS_N12 = [
    0.01,
    0.05,
    0.10,
    0.15,
    0.20,
]


FINAL_EXPERTS_N12 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


# ============================================================
# N12.2 CHECK REQUIRED N10B OBJECTS
# ============================================================

required_objects_n12 = [

    "P_MOE_N10B",
    "GATE_EPOCHS_N10B",
    "OOF_N7B",
    "build_gate_training_oof_n10b",
    "build_gate_features_n10b",
    "retrain_gate_n10b",
    "predict_gate_n10b",
]


for obj in required_objects_n12:

    if obj not in globals():

        raise RuntimeError(
            f"N12 requires {obj}. "
            "Run N10B first."
        )


# ============================================================
# N12.3 STRICT FINAL OUTER-VALID MoE
# ============================================================

P_FINAL_MOE_N12 = np.asarray(

    P_MOE_N10B,

    dtype=np.float32
)


if np.isnan(
    P_FINAL_MOE_N12
).any():

    raise RuntimeError(
        "Incomplete N10B MoE predictions."
    )


# ============================================================
# N12.4 HELPER — STRICT GATE-CROSSFITTED TRAIN PREDICTIONS
#
# P_train_experts are already strict inner OOF predictions.
#
# But if we train one gate on all of them and predict the same
# rows, its error target would still be optimistic.
#
# Therefore gate itself is cross-fitted again.
# ============================================================

def crossfit_gate_predictions_n12(

    outer_train_idx,
    P_train_experts,
    outer_fold,
    selected_epoch,
):

    y_outer = y_DEV[
        outer_train_idx
    ]


    raw_scaler = StandardScaler()


    raw_scaler.fit(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ]
    )


    X_gate = build_gate_features_n10b(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ],

        P_train_experts,

        raw_scaler
    )


    gate_oof = np.full(

        len(
            outer_train_idx
        ),

        np.nan,

        dtype=np.float32
    )


    cv = StratifiedKFold(

        n_splits=
            GATE_CF_FOLDS_N12,

        shuffle=True,

        random_state=
            SEED
            +
            81000
            +
            outer_fold
    )


    for gate_fold, (
        train_local,
        valid_local

    ) in enumerate(

        cv.split(
            X_gate,
            y_outer
        )
    ):

        gate = retrain_gate_n10b(

            X_gate[
                train_local
            ],

            P_train_experts[
                train_local
            ],

            y_outer[
                train_local
            ],

            n_epochs=
                selected_epoch,

            seed=
                (
                    SEED
                    +
                    82000
                    +
                    outer_fold * 100
                    +
                    gate_fold
                )
        )


        pred, _, _ = predict_gate_n10b(

            gate,

            X_gate[
                valid_local
            ],

            P_train_experts[
                valid_local
            ]
        )


        gate_oof[
            valid_local
        ] = pred


        del gate

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    if np.isnan(
        gate_oof
    ).any():

        raise RuntimeError(
            "Incomplete cross-fitted gate predictions."
        )


    return gate_oof


# ============================================================
# N12.5 LPNS COMPUTATION
#
# query predictions and reference predictions are supplied
# explicitly, so train and valid can use predictions generated
# under different strict protocols.
# ============================================================

def compute_lpns_strict_n12(

    query_global_idx,
    query_pred,

    reference_global_idx,
    reference_pred,

    scaler,
    neighbor_model,

    remove_self=False,
):

    query_global_idx = np.asarray(
        query_global_idx,
        dtype=int
    )


    query_pred = np.asarray(

        query_pred,

        dtype=np.float32
    )


    reference_global_idx = np.asarray(

        reference_global_idx,

        dtype=int
    )


    reference_pred = np.asarray(

        reference_pred,

        dtype=np.float32
    )


    Z_query = scaler.transform(

        X_DEV.iloc[
            query_global_idx
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    extra = (
        1
        if remove_self
        else
        0
    )


    n_neighbors = min(

        K_LPNS_N12
        +
        extra,

        len(
            reference_global_idx
        )
    )


    distances, local_neighbor_idx = (

        neighbor_model.kneighbors(

            Z_query,

            n_neighbors=
                n_neighbors,

            return_distance=True
        )
    )


    selected_local = np.empty(

        (
            len(
                query_global_idx
            ),

            K_LPNS_N12
        ),

        dtype=int
    )


    selected_distance = np.empty(

        (
            len(
                query_global_idx
            ),

            K_LPNS_N12
        ),

        dtype=np.float32
    )


    for i in range(
        len(
            query_global_idx
        )
    ):

        candidate_local = (
            local_neighbor_idx[i]
        )


        candidate_distance = (
            distances[i]
        )


        if remove_self:

            candidate_global = (

                reference_global_idx[
                    candidate_local
                ]
            )


            keep = (

                candidate_global

                !=

                query_global_idx[i]
            )


            candidate_local = (

                candidate_local[
                    keep
                ]
            )


            candidate_distance = (

                candidate_distance[
                    keep
                ]
            )


        if len(
            candidate_local
        ) < K_LPNS_N12:

            raise RuntimeError(

                "Insufficient LPNS neighbors."
            )


        selected_local[
            i
        ] = candidate_local[
            :K_LPNS_N12
        ]


        selected_distance[
            i
        ] = candidate_distance[
            :K_LPNS_N12
        ]


    neighbor_pred = (

        reference_pred[
            selected_local
        ]
    )


    local_scale = np.median(

        selected_distance,

        axis=1,

        keepdims=True
    )


    local_scale = np.maximum(

        local_scale,

        1e-6
    )


    weights = np.exp(

        -selected_distance

        /

        local_scale
    )


    weights /= np.sum(

        weights,

        axis=1,

        keepdims=True
    )


    weighted_mean = np.sum(

        weights[:, :, None]

        *

        neighbor_pred,

        axis=1
    )


    deviation = np.sum(

        weights[:, :, None]

        *

        np.abs(

            neighbor_pred

            -

            query_pred[
                :,
                None,
                :
            ]
        ),

        axis=1
    )


    variance = np.sum(

        weights[:, :, None]

        *

        (

            neighbor_pred

            -

            weighted_mean[
                :,
                None,
                :
            ]

        ) ** 2,

        axis=1
    )


    local_std = np.sqrt(

        np.maximum(
            variance,
            0.0
        )
    )


    local_bias = np.abs(

        query_pred

        -

        weighted_mean
    )


    pieces = []


    for j in range(
        query_pred.shape[1]
    ):

        pieces.extend([

            deviation[:, j],

            local_std[:, j],

            local_bias[:, j],
        ])


    pieces.extend([

        np.mean(
            deviation,
            axis=1
        ),

        np.max(
            deviation,
            axis=1
        ),

        np.mean(
            local_std,
            axis=1
        ),

        np.max(
            local_bias,
            axis=1
        ),
    ])


    result = np.column_stack(

        pieces

    ).astype(
        np.float32
    )


    if result.shape[1] != 19:

        raise RuntimeError(
            "Unexpected LPNS dimension."
        )


    return result


# ============================================================
# N12.6 STORAGE
# ============================================================

BLOCKS_N12 = {

    "CONF":
        ["C"],

    "DIS":
        ["D"],

    "LPNS":
        ["L"],

    "CONF_DIS":
        ["C", "D"],

    "CONF_LPNS":
        ["C", "L"],

    "CONF_DIS_LPNS":
        ["C", "D", "L"],
}


RELIABILITY_OOF_N12 = {

    block:
        np.full(

            len(
                y_DEV
            ),

            np.nan,

            dtype=np.float32
        )

    for block
    in BLOCKS_N12
}


ERROR_TARGET_N12 = np.full(

    len(
        y_DEV
    ),

    -1,

    dtype=np.int8
)


PRED_CLASS_N12 = np.full(

    len(
        y_DEV
    ),

    -1,

    dtype=np.int8
)


THRESHOLD_N12 = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


FOLD_RELIABILITY_ROWS_N12 = []


# ============================================================
# N12.7 HELPER — COMBINE FEATURES
# ============================================================

def combine_blocks_n12(

    codes,

    C,
    D,
    L,
):

    matrices = []


    for code in codes:

        if code == "C":
            matrices.append(C)

        elif code == "D":
            matrices.append(D)

        elif code == "L":
            matrices.append(L)

        else:
            raise ValueError(code)


    return np.concatenate(

        matrices,

        axis=1
    )


# ============================================================
# N12.8 STRICT OUTER LOOP
# ============================================================

total_start_n12 = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 165
    )

    print(
        f"N12 — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 165
    )


    outer_train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    y_train = y_DEV[
        outer_train_idx
    ]


    y_valid = y_DEV[
        outer_valid_idx
    ]


    # ========================================================
    # STEP 1 — strict inner-cross-fitted expert predictions
    # ========================================================

    cache_path = (

        PRED_DIR
        /
        f"N12_outer{outer_fold}_train_experts.npy"
    )


    if os.path.exists(
        cache_path
    ):

        print(
            "\nLoading cached strict expert predictions..."
        )


        P_train_experts = np.load(
            cache_path
        )


    else:

        print(
            "\nGenerating strict inner expert predictions..."
        )


        P_train_experts = (

            build_gate_training_oof_n10b(

                outer_train_idx,

                outer_fold
            )
        )


        np.save(

            cache_path,

            P_train_experts
        )


    # ========================================================
    # STEP 2 — outer-valid strict expert predictions
    # ========================================================

    P_valid_experts = np.column_stack([

        OOF_N7B[
            expert
        ][
            outer_valid_idx
        ]

        for expert
        in FINAL_EXPERTS_N12

    ]).astype(
        np.float32
    )


    # ========================================================
    # STEP 3 — selected N10B gate epoch
    #
    # Selected strictly inside this outer-training partition.
    # ========================================================

    selected_epoch = int(

        GATE_EPOCHS_N10B.loc[

            GATE_EPOCHS_N10B[
                "Fold"
            ]
            ==
            outer_fold,

            "Selected_Epoch"

        ].iloc[0]
    )


    # ========================================================
    # STEP 4 — cross-fitted MoE predictions for outer train
    # ========================================================

    print(
        "Generating cross-fitted train-gate predictions..."
    )


    p_train_moe = (

        crossfit_gate_predictions_n12(

            outer_train_idx,

            P_train_experts,

            outer_fold,

            selected_epoch
        )
    )


    # strict N10B outer-valid prediction

    p_valid_moe = (

        P_FINAL_MOE_N12[
            outer_valid_idx
        ]
    )


    # ========================================================
    # STEP 5 — threshold from outer train only
    # ========================================================

    threshold, train_f1 = (

        select_threshold_n9(

            y_train,

            p_train_moe
        )
    )


    print(
        f"Operating threshold: "
        f"{threshold:.6f}"
    )


    THRESHOLD_N12[
        outer_valid_idx
    ] = threshold


    train_class = (

        p_train_moe
        >=
        threshold

    ).astype(int)


    valid_class = (

        p_valid_moe
        >=
        threshold

    ).astype(int)


    train_error = (

        train_class
        !=
        y_train

    ).astype(int)


    valid_error = (

        valid_class
        !=
        y_valid

    ).astype(int)


    ERROR_TARGET_N12[
        outer_valid_idx
    ] = valid_error


    PRED_CLASS_N12[
        outer_valid_idx
    ] = valid_class


    print(
        f"Train error prevalence: "
        f"{np.mean(train_error):.6f}"
    )


    print(
        f"Valid error prevalence: "
        f"{np.mean(valid_error):.6f}"
    )


    # ========================================================
    # STEP 6 — confidence features
    # ========================================================

    C_train = build_confidence_features_n9(

        p_train_moe,

        threshold
    )


    C_valid = build_confidence_features_n9(

        p_valid_moe,

        threshold
    )


    # ========================================================
    # STEP 7 — disagreement features
    # ========================================================

    D_train = build_disagreement_features_n9(

        P_train_experts
    )


    D_valid = build_disagreement_features_n9(

        P_valid_experts
    )


    # ========================================================
    # STEP 8 — reference bank
    # ========================================================

    rng = np.random.default_rng(

        SEED
        +
        83000
        +
        outer_fold
    )


    reference_n = min(

        REFERENCE_SIZE_N12,

        len(
            outer_train_idx
        )
    )


    reference_local = rng.choice(

        np.arange(
            len(
                outer_train_idx
            )
        ),

        size=
            reference_n,

        replace=False
    )


    reference_global = (

        outer_train_idx[
            reference_local
        ]
    )


    reference_pred = (

        P_train_experts[
            reference_local
        ]
    )


    # ========================================================
    # STEP 9 — feature-space scaler + neighbors
    # ========================================================

    scaler = StandardScaler()


    scaler.fit(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ]
    )


    Z_reference = scaler.transform(

        X_DEV.iloc[
            reference_global
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    nn_model = NearestNeighbors(

        n_neighbors=
            K_LPNS_N12 + 1,

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1,
    )


    nn_model.fit(
        Z_reference
    )


    # ========================================================
    # STEP 10 — strict LPNS
    # ========================================================

    print(
        "Computing strict TRAIN LPNS..."
    )


    L_train = compute_lpns_strict_n12(

        outer_train_idx,

        P_train_experts,

        reference_global,

        reference_pred,

        scaler,

        nn_model,

        remove_self=True,
    )


    print(
        "Computing strict VALID LPNS..."
    )


    L_valid = compute_lpns_strict_n12(

        outer_valid_idx,

        P_valid_experts,

        reference_global,

        reference_pred,

        scaler,

        nn_model,

        remove_self=False,
    )


    # ========================================================
    # STEP 11 — reliability heads
    # ========================================================

    for block_number, (
        block_name,
        codes

    ) in enumerate(
        BLOCKS_N12.items()
    ):

        X_rel_train = combine_blocks_n12(

            codes,

            C_train,

            D_train,

            L_train
        )


        X_rel_valid = combine_blocks_n12(

            codes,

            C_valid,

            D_valid,

            L_valid
        )


        model = make_reliability_head_n9(

            SEED
            +
            84000
            +
            outer_fold * 100
            +
            block_number
        )


        model.fit(

            X_rel_train,

            train_error
        )


        risk = model.predict_proba(

            X_rel_valid

        )[:, 1]


        RELIABILITY_OOF_N12[
            block_name
        ][
            outer_valid_idx
        ] = risk


        m = probability_metrics(

            valid_error,

            risk
        )


        FOLD_RELIABILITY_ROWS_N12.append({

            "Fold":
                outer_fold,

            "Block":
                block_name,

            "N_Features":
                X_rel_train.shape[1],

            "Error_PR_AUC":
                m["PR_AUC"],

            "Error_ROC_AUC":
                m["ROC_AUC"],

            "Error_Brier":
                m["Brier"],

            "Error_NLL":
                m["NLL"],

            "Error_ECE":
                m["ECE"],
        })


        print(

            f"{block_name:16s} | "
            f"ErrorPR={m['PR_AUC']:.6f} | "
            f"ROC={m['ROC_AUC']:.6f}"
        )


        del model
        del X_rel_train
        del X_rel_valid


    del P_train_experts
    del P_valid_experts

    del C_train
    del C_valid

    del D_train
    del D_valid

    del L_train
    del L_valid

    del scaler
    del nn_model
    del Z_reference

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# N12.9 COMPLETENESS
# ============================================================

if np.any(
    ERROR_TARGET_N12 < 0
):

    raise RuntimeError(
        "Incomplete error target."
    )


for block, arr in (
    RELIABILITY_OOF_N12.items()
):

    if np.isnan(
        arr
    ).any():

        raise RuntimeError(

            f"Incomplete reliability predictions: {block}"
        )


# ============================================================
# N12.10 STRICT RELIABILITY RESULTS
# ============================================================

reliability_rows = []


for block, risk in (

    RELIABILITY_OOF_N12.items()
):

    m = probability_metrics(

        ERROR_TARGET_N12,

        risk
    )


    reliability_rows.append({

        "Block":
            block,

        "Error_PR_AUC":
            m["PR_AUC"],

        "Error_ROC_AUC":
            m["ROC_AUC"],

        "Error_Brier":
            m["Brier"],

        "Error_NLL":
            m["NLL"],

        "Error_ECE":
            m["ECE"],
    })


RELIABILITY_RESULTS_N12 = (

    pd.DataFrame(
        reliability_rows
    )

    .sort_values(
        "Error_PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N12 — STRICT FINAL MoE RELIABILITY RESULTS"
)

print(
    "=" * 165
)


print(
    f"\nOverall error prevalence: "
    f"{np.mean(ERROR_TARGET_N12):.6f}"
)


display(
    RELIABILITY_RESULTS_N12
)


# ============================================================
# N12.11 INCREMENTAL LPNS VALUE
# ============================================================

def get_n12(
    block,
    metric
):

    return float(

        RELIABILITY_RESULTS_N12.loc[

            RELIABILITY_RESULTS_N12[
                "Block"
            ]
            ==
            block,

            metric

        ].iloc[0]
    )


INCREMENTAL_N12 = pd.DataFrame([

    {
        "Comparison":
            "CONF_DIS vs CONF",

        "Delta_Error_PR":
            (
                get_n12(
                    "CONF_DIS",
                    "Error_PR_AUC"
                )
                -
                get_n12(
                    "CONF",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_n12(
                    "CONF_DIS",
                    "Error_ROC_AUC"
                )
                -
                get_n12(
                    "CONF",
                    "Error_ROC_AUC"
                )
            ),
    },

    {
        "Comparison":
            "CONF_LPNS vs CONF",

        "Delta_Error_PR":
            (
                get_n12(
                    "CONF_LPNS",
                    "Error_PR_AUC"
                )
                -
                get_n12(
                    "CONF",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_n12(
                    "CONF_LPNS",
                    "Error_ROC_AUC"
                )
                -
                get_n12(
                    "CONF",
                    "Error_ROC_AUC"
                )
            ),
    },

    {
        "Comparison":
            "FULL vs CONF_DIS",

        "Delta_Error_PR":
            (
                get_n12(
                    "CONF_DIS_LPNS",
                    "Error_PR_AUC"
                )
                -
                get_n12(
                    "CONF_DIS",
                    "Error_PR_AUC"
                )
            ),

        "Delta_Error_ROC":
            (
                get_n12(
                    "CONF_DIS_LPNS",
                    "Error_ROC_AUC"
                )
                -
                get_n12(
                    "CONF_DIS",
                    "Error_ROC_AUC"
                )
            ),
    },
])


print(
    "\n"
    + "=" * 165
)

print(
    "N12 — INCREMENTAL LPNS VALUE"
)

print(
    "=" * 165
)


display(
    INCREMENTAL_N12
)


# ============================================================
# N12.12 FOLD STABILITY
# ============================================================

FOLD_RELIABILITY_N12 = pd.DataFrame(

    FOLD_RELIABILITY_ROWS_N12
)


FOLD_STABILITY_N12 = (

    FOLD_RELIABILITY_N12

    .groupby(
        "Block"
    )

    .agg(

        ErrorPR_Mean=(
            "Error_PR_AUC",
            "mean"
        ),

        ErrorPR_SD=(
            "Error_PR_AUC",
            "std"
        ),

        ErrorPR_Min=(
            "Error_PR_AUC",
            "min"
        ),

        ErrorPR_Max=(
            "Error_PR_AUC",
            "max"
        ),

        ErrorROC_Mean=(
            "Error_ROC_AUC",
            "mean"
        ),

        ErrorROC_SD=(
            "Error_ROC_AUC",
            "std"
        ),
    )

    .reset_index()

    .sort_values(
        "ErrorPR_Mean",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N12 — RELIABILITY FOLD STABILITY"
)

print(
    "=" * 165
)


display(
    FOLD_STABILITY_N12
)


# ============================================================
# N12.13 SELECTIVE REVIEW
#
# Review highest-risk cases within each outer fold.
#
# Operational simulation:
#
#   reviewed errors are assumed to be detected/corrected
#   by downstream human review.
#
# Metrics:
#   - errors captured
#   - error capture rate
#   - FN / FP captured
#   - errors per 100 reviewed
#   - residual error rate
#   - post-review accuracy
# ============================================================

SELECTIVE_BLOCKS_N12 = [

    "CONF",

    "CONF_DIS",

    "CONF_LPNS",

    "CONF_DIS_LPNS",
]


selective_rows = []


for block in SELECTIVE_BLOCKS_N12:

    risk_all = (

        RELIABILITY_OOF_N12[
            block
        ]
    )


    for budget in (
        REVIEW_BUDGETS_N12
    ):

        total_reviewed = 0

        total_errors = 0

        captured_errors = 0

        total_fn = 0

        captured_fn = 0

        total_fp = 0

        captured_fp = 0


        for fold in range(
            N_FOLDS
        ):

            fold_idx = np.where(

                DEV_FOLD == fold

            )[0]


            risk = risk_all[
                fold_idx
            ]


            errors = ERROR_TARGET_N12[
                fold_idx
            ]


            pred_class = PRED_CLASS_N12[
                fold_idx
            ]


            truth = y_DEV[
                fold_idx
            ]


            fn = (

                (truth == 1)

                &

                (pred_class == 0)
            )


            fp = (

                (truth == 0)

                &

                (pred_class == 1)
            )


            n_review = max(

                1,

                int(
                    np.ceil(

                        budget

                        *

                        len(
                            fold_idx
                        )
                    )
                )
            )


            ranking = np.argsort(

                -risk

            )


            reviewed = ranking[
                :n_review
            ]


            total_reviewed += (
                n_review
            )


            total_errors += int(

                np.sum(
                    errors
                )
            )


            captured_errors += int(

                np.sum(

                    errors[
                        reviewed
                    ]
                )
            )


            total_fn += int(

                np.sum(
                    fn
                )
            )


            captured_fn += int(

                np.sum(

                    fn[
                        reviewed
                    ]
                )
            )


            total_fp += int(

                np.sum(
                    fp
                )
            )


            captured_fp += int(

                np.sum(

                    fp[
                        reviewed
                    ]
                )
            )


        residual_errors = (

            total_errors

            -
            captured_errors
        )


        selective_rows.append({

            "Block":
                block,

            "Review_Budget":
                budget,

            "Reviewed_N":
                total_reviewed,

            "Total_Errors":
                total_errors,

            "Errors_Captured":
                captured_errors,

            "Error_Capture_Rate":
                (
                    captured_errors
                    /
                    total_errors
                ),

            "Errors_per_100_Reviewed":
                (
                    100
                    *
                    captured_errors
                    /
                    total_reviewed
                ),

            "Total_FN":
                total_fn,

            "FN_Captured":
                captured_fn,

            "FN_Capture_Rate":
                (
                    captured_fn
                    /
                    total_fn
                ),

            "Total_FP":
                total_fp,

            "FP_Captured":
                captured_fp,

            "FP_Capture_Rate":
                (
                    captured_fp
                    /
                    total_fp
                ),

            "Residual_Errors":
                residual_errors,

            "Residual_Error_Rate":
                (
                    residual_errors
                    /
                    len(
                        y_DEV
                    )
                ),

            "PostReview_Accuracy":
                (
                    1.0

                    -

                    residual_errors
                    /
                    len(
                        y_DEV
                    )
                ),
        })


SELECTIVE_REVIEW_N12 = pd.DataFrame(

    selective_rows
)


print(
    "\n"
    + "=" * 165
)

print(
    "N12 — SELECTIVE REVIEW RESULTS"
)

print(
    "=" * 165
)


display(
    SELECTIVE_REVIEW_N12
)


# ============================================================
# N12.14 LPNS GAIN OVER CONFIDENCE AT EACH REVIEW BUDGET
# ============================================================

gain_rows = []


for budget in REVIEW_BUDGETS_N12:

    conf_row = (

        SELECTIVE_REVIEW_N12[

            (
                SELECTIVE_REVIEW_N12[
                    "Block"
                ]
                ==
                "CONF"
            )

            &

            (
                SELECTIVE_REVIEW_N12[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]
    )


    lpns_row = (

        SELECTIVE_REVIEW_N12[

            (
                SELECTIVE_REVIEW_N12[
                    "Block"
                ]
                ==
                "CONF_LPNS"
            )

            &

            (
                SELECTIVE_REVIEW_N12[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]
    )


    dis_row = (

        SELECTIVE_REVIEW_N12[

            (
                SELECTIVE_REVIEW_N12[
                    "Block"
                ]
                ==
                "CONF_DIS"
            )

            &

            (
                SELECTIVE_REVIEW_N12[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]
    )


    gain_rows.append({

        "Review_Budget":
            budget,

        "LPNS_minus_CONF_Errors":
            int(

                lpns_row[
                    "Errors_Captured"
                ]

                -

                conf_row[
                    "Errors_Captured"
                ]
            ),

        "LPNS_minus_CONF_CaptureRate":
            (

                lpns_row[
                    "Error_Capture_Rate"
                ]

                -

                conf_row[
                    "Error_Capture_Rate"
                ]
            ),

        "LPNS_minus_CONF_ErrorsPer100":
            (

                lpns_row[
                    "Errors_per_100_Reviewed"
                ]

                -

                conf_row[
                    "Errors_per_100_Reviewed"
                ]
            ),

        "LPNS_minus_CONF_FN":
            int(

                lpns_row[
                    "FN_Captured"
                ]

                -

                conf_row[
                    "FN_Captured"
                ]
            ),

        "LPNS_minus_CONF_DIS_Errors":
            int(

                lpns_row[
                    "Errors_Captured"
                ]

                -

                dis_row[
                    "Errors_Captured"
                ]
            ),
    })


SELECTIVE_GAIN_N12 = pd.DataFrame(

    gain_rows
)


print(
    "\n"
    + "=" * 165
)

print(
    "N12 — LPNS SELECTIVE-REVIEW GAIN"
)

print(
    "=" * 165
)


display(
    SELECTIVE_GAIN_N12
)


# ============================================================
# N12.15 THRESHOLDS
# ============================================================

THRESHOLD_SUMMARY_N12 = pd.DataFrame({

    "Fold":
        list(
            range(
                N_FOLDS
            )
        ),

    "Threshold":
        [

            float(

                THRESHOLD_N12[
                    DEV_FOLD == fold
                ][0]
            )

            for fold
            in range(
                N_FOLDS
            )
        ]
})


print(
    "\n"
    + "=" * 165
)

print(
    "N12 — FINAL MoE OPERATING THRESHOLDS"
)

print(
    "=" * 165
)


display(
    THRESHOLD_SUMMARY_N12
)


# ============================================================
# N12.16 SAVE
# ============================================================

RELIABILITY_RESULTS_N12.to_csv(

    TAB_DIR
    /
    "n12_strict_moe_reliability.csv",

    index=False
)


INCREMENTAL_N12.to_csv(

    TAB_DIR
    /
    "n12_incremental_lpns.csv",

    index=False
)


FOLD_RELIABILITY_N12.to_csv(

    TAB_DIR
    /
    "n12_reliability_fold_results.csv",

    index=False
)


FOLD_STABILITY_N12.to_csv(

    TAB_DIR
    /
    "n12_reliability_fold_stability.csv",

    index=False
)


SELECTIVE_REVIEW_N12.to_csv(

    TAB_DIR
    /
    "n12_selective_review.csv",

    index=False
)


SELECTIVE_GAIN_N12.to_csv(

    TAB_DIR
    /
    "n12_lpns_selective_gain.csv",

    index=False
)


THRESHOLD_SUMMARY_N12.to_csv(

    TAB_DIR
    /
    "n12_moe_thresholds.csv",

    index=False
)


np.save(

    PRED_DIR
    /
    "N12_FINAL_MOE_ERROR_TARGET.npy",

    ERROR_TARGET_N12
)


np.save(

    PRED_DIR
    /
    "N12_FINAL_MOE_CLASS.npy",

    PRED_CLASS_N12
)


for block, risk in (

    RELIABILITY_OOF_N12.items()
):

    np.save(

        PRED_DIR
        /
        f"N12_{block}_RISK_OOF.npy",

        risk
    )


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N12 COMPLETE"
)

print(
    "=" * 165
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - total_start_n12:.1f}s"
)


print(
    "\nReliability is now aligned to the final MoE."
)

print(
    "Selective-review results use strict cross-fitted risk scores."
)


N12 — OUTER FOLD 0

Generating strict inner expert predictions...

      Gate inner fold 1/3

  Building inner difficulty signals for outer fold 100...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

      Gate inner fold 2/3

  Building inner difficulty signals for outer fold 101...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete

      Gate inner fold 3/3

  Building inner difficulty signals for outer fold 102...
    inner fold 0 complete
    inner fold 1 complete
    inner fold 2 complete
    inner fold 3 complete
Generating cross-fitted train-gate predictions...
Operating threshold: 0.231204
Train error prevalence: 0.134692
Valid error prevalence: 0.130553
Computing strict TRAIN LPNS...
Computing strict VALID LPNS...
CONF             | ErrorPR=0.611691 | ROC=0.903962
DIS              | ErrorPR=0.432057 | ROC=0.855654
LPNS             | ErrorPR=0.487283 | ROC=0.878784
CO

,Block,Error_PR_AUC,Error_ROC_AUC,Error_Brier,Error_NLL,Error_ECE
0,CONF_LPNS,0.653008,0.911809,0.069861,0.236136,0.001794
1,CONF,0.652379,0.911916,0.069826,0.236084,0.001875
2,CONF_DIS_LPNS,0.651332,0.911763,0.069882,0.236173,0.001326
3,CONF_DIS,0.650946,0.911706,0.069890,0.236228,0.001469
4,LPNS,0.518237,0.884499,0.085673,0.273850,0.005871
5,DIS,0.463426,0.860473,0.092907,0.295802,0.013067



N12 — INCREMENTAL LPNS VALUE


,Comparison,Delta_Error_PR,Delta_Error_ROC
0,CONF_DIS vs CONF,-0.001432,-0.000210
1,CONF_LPNS vs CONF,0.000629,-0.000107
2,FULL vs CONF_DIS,0.000386,0.000057



N12 — RELIABILITY FOLD STABILITY


...


N12 — LPNS SELECTIVE-REVIEW GAIN


,Review_Budget,LPNS_minus_CONF_Errors,LPNS_minus_CONF_CaptureRate,LPNS_minus_CONF_ErrorsPer100,LPNS_minus_CONF_FN,LPNS_minus_CONF_DIS_Errors
0,0.01,1,0.000035,0.049261,0,23
1,0.05,-20,-0.000692,-0.197044,0,14
2,0.10,-5,-0.000173,-0.024637,0,17
3,0.15,-2,-0.000069,-0.006569,12,33
4,0.20,-12,-0.000415,-0.029564,-12,-6



N12 — FINAL MoE OPERATING THRESHOLDS


,Fold,Threshold
0,0,0.231204
1,1,0.231204
2,2,0.209197
3,3,0.198194
4,4,0.204482



MODULE N12 COMPLETE

Runtime: 2004.4s

Reliability is now aligned to the final MoE.
Selective-review results use strict cross-fitted risk scores.


In [24]:
# ============================================================
# MODULE N13A
# SOFT-ORACLE DISTILLED BOUNDED MoE
#
# PURPOSE
# ------------------------------------------------------------
# Test whether explicit supervision of expert competence
# creates useful routing.
#
# Strict design:
#
# OUTER TRAIN
#   -> existing strict inner-OOF expert predictions
#   -> construct soft oracle competence targets
#   -> train competence-distilled gate
#
# OUTER VALID
#   -> frozen strict N7B expert predictions
#   -> gate predicts adaptive weights
#
# Final prediction:
#
#   anchor = mean(expert probabilities)
#
#   gated = sum_j w_j * p_j
#
#   p_final =
#       anchor
#       +
#       delta_max *
#       tanh((gated - anchor) / delta_max)
#
# This preserves the strong heterogeneous consensus.
#
# IMPORTANT
# ------------------------------------------------------------
# No 2015 secondary holdout.
# No BRFSS-2020.
# No architecture tuning after seeing outer results.
# ============================================================


import gc
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

from sklearn.model_selection import (
    train_test_split,
)

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# N13A.1 CONFIGURATION
# ============================================================

EXPERTS_N13A = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


N_EXPERTS_N13A = len(
    EXPERTS_N13A
)


# ------------------------------------------------------------
# SOFT ORACLE
# ------------------------------------------------------------

ORACLE_TEMP_N13A = 0.20

ORACLE_SHRINK_N13A = 0.20


# ------------------------------------------------------------
# LOSS
# ------------------------------------------------------------

LAMBDA_ORACLE_N13A = 0.50

LAMBDA_UNIFORM_N13A = 0.005

LAMBDA_RESIDUAL_N13A = 0.01


# ------------------------------------------------------------
# BOUNDED CORRECTION
# ------------------------------------------------------------

DELTA_MAX_N13A = 0.03


# ------------------------------------------------------------
# GATE TRAINING
# ------------------------------------------------------------

MAX_EPOCHS_N13A = 40

PATIENCE_N13A = 6

BATCH_SIZE_N13A = 4096

LR_N13A = 1e-3

WEIGHT_DECAY_N13A = 1e-4


# ============================================================
# N13A.2 SANITY CHECK
# ============================================================

required_n13a = [

    "build_gate_training_oof_n10b",
    "OOF_N7B",
    "X_DEV",
    "y_DEV",
    "DEV_FOLD",
    "FEATURES",
]


for obj in required_n13a:

    if obj not in globals():

        raise RuntimeError(
            f"N13A requires '{obj}'."
        )


# ============================================================
# N13A.3 REPRODUCIBILITY
# ============================================================

def seed_n13a(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


# ============================================================
# N13A.4 METRICS
# ============================================================

def metrics_n13a(
    y,
    p,
):

    p = np.asarray(
        p,
        dtype=float
    )

    p_clip = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )


    # simple ECE

    edges = np.linspace(
        0,
        1,
        16
    )


    ece = 0.0


    for lo, hi in zip(
        edges[:-1],
        edges[1:]
    ):

        if hi == 1.0:

            mask = (
                (p >= lo)
                &
                (p <= hi)
            )

        else:

            mask = (
                (p >= lo)
                &
                (p < hi)
            )


        if np.sum(mask) == 0:

            continue


        ece += (

            np.mean(mask)

            *

            abs(

                np.mean(
                    y[mask]
                )

                -

                np.mean(
                    p[mask]
                )
            )
        )


    return {

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                p_clip
            ),

        "ECE":
            ece,
    }


# ============================================================
# N13A.5 SOFT ORACLE RESPONSIBILITIES
# ============================================================

def soft_oracle_targets_n13a(
    y,
    P,
):

    """
    y : shape [N]
    P : shape [N, E]

    Lower expert BCE loss -> higher responsibility.
    """

    y = np.asarray(
        y,
        dtype=np.float32
    )[:, None]


    P = np.clip(

        np.asarray(
            P,
            dtype=np.float32
        ),

        1e-6,
        1 - 1e-6
    )


    losses = -(

        y
        *
        np.log(P)

        +

        (1 - y)
        *
        np.log(
            1 - P
        )
    )


    logits = (

        -losses

        /
        ORACLE_TEMP_N13A
    )


    logits -= np.max(
        logits,
        axis=1,
        keepdims=True
    )


    q = np.exp(
        logits
    )


    q /= np.sum(
        q,
        axis=1,
        keepdims=True
    )


    # shrink noisy oracle toward uniform

    uniform = np.full_like(

        q,

        1.0
        /
        N_EXPERTS_N13A
    )


    q = (

        (1.0 - ORACLE_SHRINK_N13A)
        *
        q

        +

        ORACLE_SHRINK_N13A
        *
        uniform
    )


    return q.astype(
        np.float32
    )


# ============================================================
# N13A.6 GATE FEATURES
# ============================================================

def build_gate_features_n13a(
    global_idx,
    P_expert,
    scaler,
):

    X_raw = scaler.transform(

        X_DEV.iloc[
            global_idx
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    P_expert = np.asarray(

        P_expert,

        dtype=np.float32
    )


    anchor = np.mean(

        P_expert,

        axis=1,

        keepdims=True
    )


    abs_dev = np.abs(

        P_expert

        -

        anchor
    )


    disagreement = np.column_stack([

        np.std(
            P_expert,
            axis=1
        ),

        np.max(
            P_expert,
            axis=1
        )
        -
        np.min(
            P_expert,
            axis=1
        ),

        np.mean(
            abs_dev,
            axis=1
        ),

        np.max(
            abs_dev,
            axis=1
        ),
    ]).astype(
        np.float32
    )


    return np.concatenate(

        [

            X_raw,
            P_expert,
            anchor,
            abs_dev,
            disagreement,

        ],

        axis=1

    ).astype(
        np.float32
    )


# ============================================================
# N13A.7 GATE MODEL
# ============================================================

class OracleGateN13A(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        n_experts,
    ):

        super().__init__()


        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.GELU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                32
            ),

            nn.GELU(),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                32,
                n_experts
            ),
        )


    def forward(
        self,
        x,
    ):

        logits = self.net(
            x
        )


        return torch.softmax(
            logits,
            dim=1
        )


# ============================================================
# N13A.8 FORWARD MIXTURE
# ============================================================

def bounded_mixture_n13a(
    P,
    W,
):

    anchor = torch.mean(

        P,

        dim=1
    )


    gated = torch.sum(

        P * W,

        dim=1
    )


    correction = (

        DELTA_MAX_N13A

        *

        torch.tanh(

            (
                gated
                -
                anchor
            )

            /
            DELTA_MAX_N13A
        )
    )


    p_final = (

        anchor
        +
        correction
    )


    p_final = torch.clamp(

        p_final,

        1e-6,
        1 - 1e-6
    )


    return (
        p_final,
        anchor,
        correction
    )


# ============================================================
# N13A.9 TRAIN ONE GATE
# ============================================================

def train_oracle_gate_n13a(

    X_train,
    P_train,
    y_train,
    Q_train,

    X_val,
    P_val,
    y_val,
    Q_val,

    seed,
):

    seed_n13a(
        seed
    )


    device = (

        torch.device(
            "cuda"
        )

        if torch.cuda.is_available()

        else torch.device(
            "cpu"
        )
    )


    model = OracleGateN13A(

        X_train.shape[1],

        N_EXPERTS_N13A

    ).to(
        device
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LR_N13A,

        weight_decay=
            WEIGHT_DECAY_N13A
    )


    dataset = TensorDataset(

        torch.tensor(
            X_train,
            dtype=torch.float32
        ),

        torch.tensor(
            P_train,
            dtype=torch.float32
        ),

        torch.tensor(
            y_train,
            dtype=torch.float32
        ),

        torch.tensor(
            Q_train,
            dtype=torch.float32
        ),
    )


    loader = DataLoader(

        dataset,

        batch_size=
            BATCH_SIZE_N13A,

        shuffle=True,

        num_workers=0
    )


    best_state = None

    best_pr = -np.inf

    best_epoch = 1

    no_improve = 0


    uniform_value = (

        1.0
        /
        N_EXPERTS_N13A
    )


    for epoch in range(
        1,
        MAX_EPOCHS_N13A + 1
    ):

        model.train()


        for (
            xb,
            pb,
            yb,
            qb

        ) in loader:

            xb = xb.to(
                device
            )

            pb = pb.to(
                device
            )

            yb = yb.to(
                device
            )

            qb = qb.to(
                device
            )


            optimizer.zero_grad()


            weights = model(
                xb
            )


            (
                p_final,
                anchor,
                correction

            ) = bounded_mixture_n13a(

                pb,
                weights
            )


            # ---------------------------------------------
            # classification loss
            # ---------------------------------------------

            bce = nn.functional.binary_cross_entropy(

                p_final,

                yb
            )


            # ---------------------------------------------
            # soft oracle distillation
            #
            # KL(q || w)
            # ---------------------------------------------

            oracle_kl = torch.mean(

                torch.sum(

                    qb
                    *
                    (

                        torch.log(
                            qb + 1e-8
                        )

                        -

                        torch.log(
                            weights + 1e-8
                        )
                    ),

                    dim=1
                )
            )


            # ---------------------------------------------
            # small regularizer toward consensus
            # ---------------------------------------------

            uniform_kl = torch.mean(

                torch.sum(

                    weights
                    *
                    (

                        torch.log(
                            weights + 1e-8
                        )

                        -

                        np.log(
                            uniform_value
                        )
                    ),

                    dim=1
                )
            )


            # ---------------------------------------------
            # bounded correction penalty
            # ---------------------------------------------

            residual_penalty = torch.mean(

                (
                    correction

                    /
                    DELTA_MAX_N13A
                ) ** 2
            )


            loss = (

                bce

                +

                LAMBDA_ORACLE_N13A
                *
                oracle_kl

                +

                LAMBDA_UNIFORM_N13A
                *
                uniform_kl

                +

                LAMBDA_RESIDUAL_N13A
                *
                residual_penalty
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                max_norm=5.0
            )


            optimizer.step()


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()


        with torch.no_grad():

            xv = torch.tensor(

                X_val,

                dtype=torch.float32,

                device=device
            )


            pv = torch.tensor(

                P_val,

                dtype=torch.float32,

                device=device
            )


            wv = model(
                xv
            )


            p_val_final, _, _ = (

                bounded_mixture_n13a(

                    pv,
                    wv
                )
            )


            p_val_np = (

                p_val_final

                .detach()

                .cpu()

                .numpy()
            )


        val_pr = average_precision_score(

            y_val,

            p_val_np
        )


        if val_pr > best_pr + 1e-7:

            best_pr = val_pr

            best_epoch = epoch

            best_state = {

                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model.state_dict().items()
            }

            no_improve = 0


        else:

            no_improve += 1


        if no_improve >= PATIENCE_N13A:

            break


    model.load_state_dict(
        best_state
    )


    model.to(
        device
    )


    return (
        model,
        best_epoch,
        best_pr
    )


# ============================================================
# N13A.10 PREDICTION
# ============================================================

def predict_oracle_gate_n13a(

    model,
    X_gate,
    P_expert,
):

    device = next(
        model.parameters()
    ).device


    model.eval()


    with torch.no_grad():

        x = torch.tensor(

            X_gate,

            dtype=torch.float32,

            device=device
        )


        p = torch.tensor(

            P_expert,

            dtype=torch.float32,

            device=device
        )


        weights = model(
            x
        )


        (
            p_final,
            anchor,
            correction

        ) = bounded_mixture_n13a(

            p,
            weights
        )


    return (

        p_final.cpu().numpy(),

        weights.cpu().numpy(),

        correction.cpu().numpy(),
    )


# ============================================================
# N13A.11 STORAGE
# ============================================================

P_ORACLE_MOE_N13A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


P_ANCHOR_N13A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


W_ORACLE_N13A = np.full(

    (
        len(
            y_DEV
        ),
        N_EXPERTS_N13A
    ),

    np.nan,

    dtype=np.float32
)


CORRECTION_N13A = np.full(

    len(
        y_DEV
    ),

    np.nan,

    dtype=np.float32
)


fold_rows_n13a = []

epoch_rows_n13a = []


# ============================================================
# N13A.12 OUTER LOOP
# ============================================================

start_n13a = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 160
    )

    print(
        f"N13A — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 160
    )


    outer_train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    outer_valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    y_train = y_DEV[
        outer_train_idx
    ]


    y_valid = y_DEV[
        outer_valid_idx
    ]


    # ========================================================
    # STRICT INNER-OOF EXPERT PREDICTIONS
    #
    # Reuse N12 cache if available.
    # ========================================================

    cache_path = (

        PRED_DIR

        /

        f"N12_outer{outer_fold}_train_experts.npy"
    )


    if cache_path.exists():

        print(
            "Loading cached strict inner-OOF experts..."
        )


        P_train_experts = np.load(
            cache_path
        )


    else:

        print(
            "Generating strict inner-OOF experts..."
        )


        P_train_experts = (

            build_gate_training_oof_n10b(

                outer_train_idx,

                outer_fold
            )
        )


        np.save(

            cache_path,

            P_train_experts
        )


    # ========================================================
    # STRICT OUTER-VALID EXPERT PREDICTIONS
    # ========================================================

    P_valid_experts = np.column_stack([

        OOF_N7B[
            expert
        ][
            outer_valid_idx
        ]

        for expert
        in EXPERTS_N13A

    ]).astype(
        np.float32
    )


    # ========================================================
    # SOFT ORACLE TARGETS
    # ========================================================

    Q_train = soft_oracle_targets_n13a(

        y_train,

        P_train_experts
    )


    # ========================================================
    # GATE FEATURE SCALER
    # ========================================================

    scaler = StandardScaler()


    scaler.fit(

        X_DEV.iloc[
            outer_train_idx
        ][
            FEATURES
        ]
    )


    X_gate_train = build_gate_features_n13a(

        outer_train_idx,

        P_train_experts,

        scaler
    )


    X_gate_valid = build_gate_features_n13a(

        outer_valid_idx,

        P_valid_experts,

        scaler
    )


    # ========================================================
    # INTERNAL GATE VALIDATION SPLIT
    #
    # Only outer-training rows.
    # ========================================================

    train_local, val_local = train_test_split(

        np.arange(
            len(
                outer_train_idx
            )
        ),

        test_size=0.15,

        stratify=
            y_train,

        random_state=
            (
                SEED
                +
                101000
                +
                outer_fold
            )
    )


    # ========================================================
    # TRAIN GATE
    # ========================================================

    model, best_epoch, inner_pr = (

        train_oracle_gate_n13a(

            X_gate_train[
                train_local
            ],

            P_train_experts[
                train_local
            ],

            y_train[
                train_local
            ],

            Q_train[
                train_local
            ],

            X_gate_train[
                val_local
            ],

            P_train_experts[
                val_local
            ],

            y_train[
                val_local
            ],

            Q_train[
                val_local
            ],

            seed=
                (
                    SEED
                    +
                    102000
                    +
                    outer_fold
                )
        )
    )


    # ========================================================
    # OUTER VALID PREDICTION
    # ========================================================

    (
        p_valid,

        w_valid,

        correction_valid

    ) = predict_oracle_gate_n13a(

        model,

        X_gate_valid,

        P_valid_experts
    )


    anchor_valid = np.mean(

        P_valid_experts,

        axis=1
    )


    P_ORACLE_MOE_N13A[
        outer_valid_idx
    ] = p_valid


    P_ANCHOR_N13A[
        outer_valid_idx
    ] = anchor_valid


    W_ORACLE_N13A[
        outer_valid_idx
    ] = w_valid


    CORRECTION_N13A[
        outer_valid_idx
    ] = correction_valid


    # ========================================================
    # FOLD METRICS
    # ========================================================

    anchor_m = metrics_n13a(

        y_valid,

        anchor_valid
    )


    moe_m = metrics_n13a(

        y_valid,

        p_valid
    )


    fold_rows_n13a.append({

        "Fold":
            outer_fold,

        "Anchor_PR":
            anchor_m[
                "PR_AUC"
            ],

        "MoE_PR":
            moe_m[
                "PR_AUC"
            ],

        "Delta_PR":
            (
                moe_m[
                    "PR_AUC"
                ]

                -
                anchor_m[
                    "PR_AUC"
                ]
            ),

        "Anchor_ROC":
            anchor_m[
                "ROC_AUC"
            ],

        "MoE_ROC":
            moe_m[
                "ROC_AUC"
            ],

        "Anchor_Brier":
            anchor_m[
                "Brier"
            ],

        "MoE_Brier":
            moe_m[
                "Brier"
            ],

        "Anchor_NLL":
            anchor_m[
                "NLL"
            ],

        "MoE_NLL":
            moe_m[
                "NLL"
            ],

        "Anchor_ECE":
            anchor_m[
                "ECE"
            ],

        "MoE_ECE":
            moe_m[
                "ECE"
            ],
    })


    epoch_rows_n13a.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            best_epoch,

        "Inner_PR":
            inner_pr,
    })


    print(

        f"Anchor PR = "
        f"{anchor_m['PR_AUC']:.6f}"
    )


    print(

        f"Oracle-MoE PR = "
        f"{moe_m['PR_AUC']:.6f}"
    )


    print(

        f"Delta PR = "
        f"{moe_m['PR_AUC'] - anchor_m['PR_AUC']:+.6f}"
    )


    del model

    del P_train_experts
    del P_valid_experts

    del Q_train

    del X_gate_train
    del X_gate_valid

    del scaler

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# N13A.13 COMPLETENESS
# ============================================================

for name, arr in [

    (
        "P_ORACLE_MOE_N13A",
        P_ORACLE_MOE_N13A
    ),

    (
        "P_ANCHOR_N13A",
        P_ANCHOR_N13A
    ),

]:

    if np.isnan(
        arr
    ).any():

        raise RuntimeError(
            f"Incomplete: {name}"
        )


# ============================================================
# N13A.14 OVERALL RESULTS
# ============================================================

anchor_all = metrics_n13a(

    y_DEV,

    P_ANCHOR_N13A
)


moe_all = metrics_n13a(

    y_DEV,

    P_ORACLE_MOE_N13A
)


RESULTS_N13A = pd.DataFrame([

    {

        "System":
            "Frozen_LSA_HE_5_Anchor",

        **anchor_all,

        "Delta_PR_vs_Anchor":
            0.0,
    },

    {

        "System":
            "Soft_Oracle_Distilled_MoE",

        **moe_all,

        "Delta_PR_vs_Anchor":
            (
                moe_all[
                    "PR_AUC"
                ]

                -
                anchor_all[
                    "PR_AUC"
                ]
            ),
    },
])


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N13A — SOFT-ORACLE DISTILLED MoE"
)

print(
    "=" * 165
)


display(
    RESULTS_N13A
)


# ============================================================
# N13A.15 FOLD RESULTS
# ============================================================

FOLD_RESULTS_N13A = pd.DataFrame(

    fold_rows_n13a
)


print(
    "\n"
    + "=" * 165
)

print(
    "N13A — FOLD RESULTS"
)

print(
    "=" * 165
)


display(
    FOLD_RESULTS_N13A
)


print(

    "\nMean fold delta PR:",

    FOLD_RESULTS_N13A[
        "Delta_PR"
    ].mean()
)


print(

    "Positive-delta folds:",

    int(
        np.sum(

            FOLD_RESULTS_N13A[
                "Delta_PR"
            ]
            >
            0
        )
    ),

    "/",

    N_FOLDS
)


# ============================================================
# N13A.16 ROUTING DISTRIBUTION
# ============================================================

weight_rows = []


for j, expert in enumerate(
    EXPERTS_N13A
):

    w = W_ORACLE_N13A[
        :,
        j
    ]


    weight_rows.append({

        "Expert":
            expert,

        "MeanWeight":
            np.mean(w),

        "SD":
            np.std(w),

        "P10":
            np.quantile(
                w,
                0.10
            ),

        "Median":
            np.median(w),

        "P90":
            np.quantile(
                w,
                0.90
            ),

        "Min":
            np.min(w),

        "Max":
            np.max(w),
    })


WEIGHT_SUMMARY_N13A = pd.DataFrame(

    weight_rows_n13a
    if "weight_rows_n13a" in globals()
    else weight_rows
)


print(
    "\n"
    + "=" * 165
)

print(
    "N13A — EXPERT WEIGHT DISTRIBUTION"
)

print(
    "=" * 165
)


display(
    WEIGHT_SUMMARY_N13A
)


# ============================================================
# N13A.17 ROUTING ENTROPY
# ============================================================

W_clip = np.clip(

    W_ORACLE_N13A,

    1e-12,

    1.0
)


entropy = -(

    W_clip

    *
    np.log(
        W_clip
    )

).sum(
    axis=1
)


normalized_entropy = (

    entropy

    /
    np.log(
        N_EXPERTS_N13A
    )
)


effective_experts = np.exp(
    entropy
)


ENTROPY_N13A = pd.DataFrame([{

    "NormalizedEntropy_Mean":
        np.mean(
            normalized_entropy
        ),

    "NormalizedEntropy_SD":
        np.std(
            normalized_entropy
        ),

    "NormalizedEntropy_P10":
        np.quantile(
            normalized_entropy,
            0.10
        ),

    "NormalizedEntropy_Median":
        np.median(
            normalized_entropy
        ),

    "NormalizedEntropy_P90":
        np.quantile(
            normalized_entropy,
            0.90
        ),

    "EffectiveExperts_Mean":
        np.mean(
            effective_experts
        ),

    "EffectiveExperts_Median":
        np.median(
            effective_experts
        ),
}])


print(
    "\n"
    + "=" * 165
)

print(
    "N13A — ROUTING ENTROPY"
)

print(
    "=" * 165
)


display(
    ENTROPY_N13A
)


# ============================================================
# N13A.18 CORRECTION MAGNITUDE
# ============================================================

abs_corr = np.abs(
    CORRECTION_N13A
)


CORRECTION_SUMMARY_N13A = pd.DataFrame([{

    "MeanSignedCorrection":
        np.mean(
            CORRECTION_N13A
        ),

    "MeanAbsCorrection":
        np.mean(
            abs_corr
        ),

    "MedianAbsCorrection":
        np.median(
            abs_corr
        ),

    "P90AbsCorrection":
        np.quantile(
            abs_corr,
            0.90
        ),

    "P99AbsCorrection":
        np.quantile(
            abs_corr,
            0.99
        ),

    "MaxAbsCorrection":
        np.max(
            abs_corr
        ),
}])


print(
    "\n"
    + "=" * 165
)

print(
    "N13A — BOUNDED CORRECTION MAGNITUDE"
)

print(
    "=" * 165
)


display(
    CORRECTION_SUMMARY_N13A
)


# ============================================================
# N13A.19 DOMINANT EXPERT
# ============================================================

dominant = np.argmax(

    W_ORACLE_N13A,

    axis=1
)


dominant_rows = []


for j, expert in enumerate(
    EXPERTS_N13A
):

    n = int(

        np.sum(
            dominant == j
        )
    )


    dominant_rows.append({

        "Expert":
            expert,

        "Dominant_N":
            n,

        "Dominant_Fraction":
            n
            /
            len(
                dominant
            ),
    })


DOMINANT_N13A = pd.DataFrame(

    dominant_rows
)


print(
    "\n"
    + "=" * 165
)

print(
    "N13A — DOMINANT EXPERT USAGE"
)

print(
    "=" * 165
)


display(
    DOMINANT_N13A
)


# ============================================================
# N13A.20 GATE EPOCHS
# ============================================================

GATE_EPOCHS_N13A = pd.DataFrame(

    epoch_rows_n13a
)


print(
    "\n"
    + "=" * 165
)

print(
    "N13A — GATE EPOCH SELECTION"
)

print(
    "=" * 165
)


display(
    GATE_EPOCHS_N13A
)


# ============================================================
# N13A.21 SAVE
# ============================================================

np.save(

    PRED_DIR
    /
    "N13A_ORACLE_MOE_OOF.npy",

    P_ORACLE_MOE_N13A
)


np.save(

    PRED_DIR
    /
    "N13A_ORACLE_WEIGHTS.npy",

    W_ORACLE_N13A
)


np.save(

    PRED_DIR
    /
    "N13A_CORRECTION.npy",

    CORRECTION_N13A
)


RESULTS_N13A.to_csv(

    TAB_DIR
    /
    "n13a_results.csv",

    index=False
)


FOLD_RESULTS_N13A.to_csv(

    TAB_DIR
    /
    "n13a_fold_results.csv",

    index=False
)


WEIGHT_SUMMARY_N13A.to_csv(

    TAB_DIR
    /
    "n13a_weight_summary.csv",

    index=False
)


ENTROPY_N13A.to_csv(

    TAB_DIR
    /
    "n13a_entropy.csv",

    index=False
)


CORRECTION_SUMMARY_N13A.to_csv(

    TAB_DIR
    /
    "n13a_correction_summary.csv",

    index=False
)


DOMINANT_N13A.to_csv(

    TAB_DIR
    /
    "n13a_dominant_experts.csv",

    index=False
)


GATE_EPOCHS_N13A.to_csv(

    TAB_DIR
    /
    "n13a_gate_epochs.csv",

    index=False
)


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N13A COMPLETE"
)

print(
    "=" * 165
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - start_n13a:.1f}s"
)


print(
    "\nNo secondary 2015 holdout used."
)

print(
    "No BRFSS-2020 used."
)


N13A — OUTER FOLD 0
Loading cached strict inner-OOF experts...
Anchor PR = 0.378993
Oracle-MoE PR = 0.378982
Delta PR = -0.000011

N13A — OUTER FOLD 1
Loading cached strict inner-OOF experts...
Anchor PR = 0.381858
Oracle-MoE PR = 0.381846
Delta PR = -0.000012

N13A — OUTER FOLD 2
Loading cached strict inner-OOF experts...
Anchor PR = 0.379144
Oracle-MoE PR = 0.379130
Delta PR = -0.000014

N13A — OUTER FOLD 3
Loading cached strict inner-OOF experts...
Anchor PR = 0.372725
Oracle-MoE PR = 0.372736
Delta PR = +0.000011

N13A — OUTER FOLD 4
Loading cached strict inner-OOF experts...
Anchor PR = 0.388756
Oracle-MoE PR = 0.388736
Delta PR = -0.000020

MODULE N13A — SOFT-ORACLE DISTILLED MoE


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_Anchor
0,Frozen_LSA_HE_5_Anchor,0.379645,0.850617,0.06973,0.235667,0.002774,0.000000
1,Soft_Oracle_Distilled_MoE,0.379634,0.850617,0.06973,0.235665,0.002787,-0.000011



N13A — FOLD RESULTS


,Fold,Anchor_PR,MoE_PR,Delta_PR,Anchor_ROC,MoE_ROC,Anchor_Brier,MoE_Brier,Anchor_NLL,MoE_NLL,Anchor_ECE,MoE_ECE
0,0,0.378993,0.378982,-0.000011,0.852518,0.852521,0.069673,0.069672,0.234947,0.234944,0.002848,0.002843
1,1,0.381858,0.381846,-0.000012,0.850508,0.850511,0.069755,0.069755,0.235752,0.235748,0.004541,0.004431
2,2,0.379144,0.379130,-0.000014,0.849177,0.849172,0.069828,0.069829,0.236326,0.236328,0.004185,0.004202
3,3,0.372725,0.372736,0.000011,0.849145,0.849155,0.070082,0.070081,0.236670,0.236664,0.003733,0.003955
4,4,0.388756,0.388736,-0.000020,0.851989,0.851985,0.069311,0.069313,0.234637,0.234641,0.005289,0.005380



Mean fold delta PR: -9.112099807362384e-06
Positive-delta folds: 1 / 5

N13A — EXPERT WEIGHT DISTRIBUTION


...


N13A — DOMINANT EXPERT USAGE


,Expert,Dominant_N,Dominant_Fraction
0,XGB_SMOOTH,24165,0.119072
1,CAT_SMOOTH,29301,0.144380
2,MLP_REPAIRED,90687,0.446857
3,AUTOINT_REPAIRED,49389,0.243363
4,XGB_HARD_DISAGREE_CF,9402,0.046328



N13A — GATE EPOCH SELECTION


,Fold,Selected_Epoch,Inner_PR
0,0,36,0.380141
1,1,1,0.384337
2,2,9,0.391617
3,3,1,0.365818
4,4,5,0.374784



MODULE N13A COMPLETE

Runtime: 165.7s

No secondary 2015 holdout used.
No BRFSS-2020 used.


In [25]:
# ============================================================
# MODULE N13B — RESIDUAL-SPECIALIST MoE
#
# Goal:
#   Learn corrections to the strong static anchor rather than
#   train more full disease classifiers.
#
# Strict outer-CV design:
#   outer train -> strict inner-OOF expert predictions
#               -> anchor residuals
#               -> train residual specialists
#
#   outer valid -> frozen strict expert predictions
#               -> anchor
#               -> specialist corrections
#
# No holdout.
# No BRFSS-2020.
# ============================================================

import gc
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# CONFIG
# ============================================================

EXPERTS_N13B = [
    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]

MAX_LOGIT_CORRECTION_N13B = 0.35

GATE_MAX_EPOCHS_N13B = 30
GATE_PATIENCE_N13B = 5
GATE_BATCH_N13B = 4096
GATE_LR_N13B = 1e-3
GATE_WD_N13B = 1e-4

LAMBDA_CORRECTION_N13B = 0.02

N_SPECIALISTS_N13B = 5


# ============================================================
# HELPERS
# ============================================================

def seed_n13b(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_logit_n13b(p):

    p = np.clip(
        np.asarray(p, dtype=float),
        1e-6,
        1 - 1e-6
    )

    return np.log(
        p / (1 - p)
    )


def sigmoid_n13b(z):

    z = np.clip(
        z,
        -30,
        30
    )

    return 1.0 / (
        1.0 + np.exp(-z)
    )


def metrics_n13b(y, p):

    p = np.asarray(
        p,
        dtype=float
    )

    pc = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )

    edges = np.linspace(
        0,
        1,
        16
    )

    ece = 0.0

    for lo, hi in zip(
        edges[:-1],
        edges[1:]
    ):

        if hi == 1:
            mask = (
                (p >= lo)
                &
                (p <= hi)
            )
        else:
            mask = (
                (p >= lo)
                &
                (p < hi)
            )

        if mask.sum() == 0:
            continue

        ece += (
            mask.mean()
            *
            abs(
                y[mask].mean()
                -
                p[mask].mean()
            )
        )

    return {
        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                pc
            ),

        "ECE":
            ece,
    }


# ============================================================
# SPECIALIST FEATURE MATRIX
# ============================================================

def residual_features_n13b(
    global_idx,
    P_expert,
    scaler,
):

    Xraw = scaler.transform(

        X_DEV.iloc[
            global_idx
        ][FEATURES]

    ).astype(
        np.float32
    )

    P = np.asarray(
        P_expert,
        dtype=np.float32
    )

    anchor = P.mean(
        axis=1,
        keepdims=True
    )

    dev = (
        P
        -
        anchor
    )

    abs_dev = np.abs(
        dev
    )

    disagreement = np.column_stack([

        np.std(
            P,
            axis=1
        ),

        np.ptp(
            P,
            axis=1
        ),

        np.mean(
            abs_dev,
            axis=1
        ),

        np.max(
            abs_dev,
            axis=1
        ),

    ]).astype(
        np.float32
    )

    return np.concatenate(
        [
            Xraw,
            P,
            anchor,
            dev,
            abs_dev,
            disagreement,
        ],
        axis=1
    ).astype(
        np.float32
    )


# ============================================================
# TRAIN RESIDUAL SPECIALISTS
# ============================================================

def train_residual_specialists_n13b(
    X,
    y,
    anchor,
    seed,
):

    residual = (
        y
        -
        anchor
    )

    # --------------------------------------------------------
    # Specialist targets
    # --------------------------------------------------------

    target_general = residual

    target_under = np.maximum(
        residual,
        0.0
    )

    target_over = np.minimum(
        residual,
        0.0
    )

    hard_positive_weight = (
        (y == 1)
        *
        (
            1.0
            +
            4.0
            *
            (
                anchor < 0.20
            )
        )
    )

    high_conf_error = np.abs(
        residual
    ) * (
        1.0
        +
        2.0
        *
        (
            np.abs(
                anchor - 0.5
            )
            > 0.30
        )
    )

    configs = [
        (
            target_general,
            None
        ),
        (
            target_under,
            None
        ),
        (
            target_over,
            None
        ),
        (
            residual,
            hard_positive_weight + 1.0
        ),
        (
            residual,
            high_conf_error + 1.0
        ),
    ]

    models = []

    for j, (
        target,
        sample_weight
    ) in enumerate(configs):

        model = HistGradientBoostingRegressor(
            learning_rate=0.04,
            max_iter=140,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=
                seed + j
        )

        model.fit(
            X,
            target,
            sample_weight=
                sample_weight
        )

        models.append(
            model
        )

    return models


# ============================================================
# SPECIALIST CORRECTIONS
# ============================================================

def predict_residual_specialists_n13b(
    models,
    X
):

    preds = np.column_stack([

        m.predict(X)

        for m in models

    ])

    # stabilize raw residual predictions
    preds = np.clip(
        preds,
        -0.20,
        0.20
    )

    return preds.astype(
        np.float32
    )


# ============================================================
# GATE
# ============================================================

class ResidualGateN13B(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        n_specialists
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.GELU(),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                64,
                32
            ),

            nn.GELU(),

            nn.Linear(
                32,
                n_specialists
            ),
        )

    def forward(
        self,
        x
    ):

        return torch.softmax(
            self.net(x),
            dim=1
        )


# ============================================================
# FINAL FORWARD
#
# Residual specialists predict probability-space correction.
# We convert the weighted correction to a bounded logit shift.
# ============================================================

def forward_residual_moe_n13b(
    anchor,
    residual_matrix,
    weights
):

    weighted_residual = torch.sum(
        residual_matrix
        *
        weights,
        dim=1
    )

    correction_logit = (
        MAX_LOGIT_CORRECTION_N13B
        *
        torch.tanh(
            weighted_residual
            /
            0.05
        )
    )

    anchor = torch.clamp(
        anchor,
        1e-6,
        1 - 1e-6
    )

    anchor_logit = torch.log(
        anchor
        /
        (
            1 - anchor
        )
    )

    final_logit = (
        anchor_logit
        +
        correction_logit
    )

    p_final = torch.sigmoid(
        final_logit
    )

    return (
        p_final,
        correction_logit,
        weighted_residual
    )


# ============================================================
# GATE TRAINING
# ============================================================

def train_residual_gate_n13b(
    X_train,
    anchor_train,
    R_train,
    y_train,

    X_val,
    anchor_val,
    R_val,
    y_val,

    seed
):

    seed_n13b(
        seed
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = ResidualGateN13B(
        X_train.shape[1],
        N_SPECIALISTS_N13B
    ).to(
        device
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR_N13B,
        weight_decay=GATE_WD_N13B
    )

    ds = TensorDataset(

        torch.tensor(
            X_train,
            dtype=torch.float32
        ),

        torch.tensor(
            anchor_train,
            dtype=torch.float32
        ),

        torch.tensor(
            R_train,
            dtype=torch.float32
        ),

        torch.tensor(
            y_train,
            dtype=torch.float32
        ),
    )

    loader = DataLoader(
        ds,
        batch_size=
            GATE_BATCH_N13B,
        shuffle=True,
        num_workers=0
    )

    best_state = None
    best_pr = -np.inf
    best_epoch = 1
    no_improve = 0

    for epoch in range(
        1,
        GATE_MAX_EPOCHS_N13B + 1
    ):

        model.train()

        for (
            xb,
            ab,
            rb,
            yb
        ) in loader:

            xb = xb.to(device)
            ab = ab.to(device)
            rb = rb.to(device)
            yb = yb.to(device)

            opt.zero_grad()

            w = model(
                xb
            )

            (
                pf,
                correction,
                weighted_residual
            ) = forward_residual_moe_n13b(
                ab,
                rb,
                w
            )

            bce = nn.functional.binary_cross_entropy(
                pf,
                yb
            )

            penalty = torch.mean(
                correction ** 2
            )

            loss = (
                bce
                +
                LAMBDA_CORRECTION_N13B
                *
                penalty
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            opt.step()

        # ---------------------------------------------
        # validation
        # ---------------------------------------------

        model.eval()

        with torch.no_grad():

            xv = torch.tensor(
                X_val,
                dtype=torch.float32,
                device=device
            )

            av = torch.tensor(
                anchor_val,
                dtype=torch.float32,
                device=device
            )

            rv = torch.tensor(
                R_val,
                dtype=torch.float32,
                device=device
            )

            wv = model(
                xv
            )

            pv, _, _ = (
                forward_residual_moe_n13b(
                    av,
                    rv,
                    wv
                )
            )

            pv = (
                pv
                .cpu()
                .numpy()
            )

        pr = average_precision_score(
            y_val,
            pv
        )

        if pr > best_pr + 1e-7:

            best_pr = pr
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model.state_dict().items()
            }

            no_improve = 0

        else:

            no_improve += 1

        if (
            no_improve
            >=
            GATE_PATIENCE_N13B
        ):
            break

    model.load_state_dict(
        best_state
    )

    model.to(
        device
    )

    return (
        model,
        best_epoch,
        best_pr
    )


# ============================================================
# STORAGE
# ============================================================

P_RESIDUAL_MOE_N13B = np.full(
    len(y_DEV),
    np.nan,
    dtype=np.float32
)

P_ANCHOR_N13B = np.full(
    len(y_DEV),
    np.nan,
    dtype=np.float32
)

W_RESIDUAL_N13B = np.full(
    (
        len(y_DEV),
        N_SPECIALISTS_N13B
    ),
    np.nan,
    dtype=np.float32
)

CORRECTION_LOGIT_N13B = np.full(
    len(y_DEV),
    np.nan,
    dtype=np.float32
)

fold_rows_n13b = []
epoch_rows_n13b = []


# ============================================================
# OUTER LOOP
# ============================================================

start_n13b = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        +
        "=" * 165
    )

    print(
        f"N13B — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 165
    )

    train_idx = np.where(
        DEV_FOLD != outer_fold
    )[0]

    valid_idx = np.where(
        DEV_FOLD == outer_fold
    )[0]

    y_train = y_DEV[
        train_idx
    ]

    y_valid = y_DEV[
        valid_idx
    ]

    # --------------------------------------------------------
    # strict inner OOF experts
    # --------------------------------------------------------

    cache_path = (
        PRED_DIR
        /
        f"N12_outer{outer_fold}_train_experts.npy"
    )

    if cache_path.exists():

        P_train_experts = np.load(
            cache_path
        )

    else:

        P_train_experts = (
            build_gate_training_oof_n10b(
                train_idx,
                outer_fold
            )
        )

        np.save(
            cache_path,
            P_train_experts
        )

    # --------------------------------------------------------
    # strict outer valid experts
    # --------------------------------------------------------

    P_valid_experts = np.column_stack([

        OOF_N7B[
            e
        ][
            valid_idx
        ]

        for e in EXPERTS_N13B

    ]).astype(
        np.float32
    )

    anchor_train = (
        P_train_experts.mean(
            axis=1
        )
    )

    anchor_valid = (
        P_valid_experts.mean(
            axis=1
        )
    )

    # --------------------------------------------------------
    # scaler
    # --------------------------------------------------------

    scaler = StandardScaler()

    scaler.fit(
        X_DEV.iloc[
            train_idx
        ][FEATURES]
    )

    X_train_res = (
        residual_features_n13b(
            train_idx,
            P_train_experts,
            scaler
        )
    )

    X_valid_res = (
        residual_features_n13b(
            valid_idx,
            P_valid_experts,
            scaler
        )
    )

    # --------------------------------------------------------
    # split outer train for residual/gate construction
    # --------------------------------------------------------

    resid_local, gate_local = (
        train_test_split(

            np.arange(
                len(train_idx)
            ),

            test_size=0.35,

            stratify=
                y_train,

            random_state=
                SEED
                +
                120000
                +
                outer_fold
        )
    )

    gate_train_local, gate_val_local = (
        train_test_split(

            gate_local,

            test_size=0.25,

            stratify=
                y_train[
                    gate_local
                ],

            random_state=
                SEED
                +
                121000
                +
                outer_fold
        )
    )

    # --------------------------------------------------------
    # train residual specialists
    # --------------------------------------------------------

    specialists = (
        train_residual_specialists_n13b(

            X_train_res[
                resid_local
            ],

            y_train[
                resid_local
            ],

            anchor_train[
                resid_local
            ],

            seed=
                SEED
                +
                122000
                +
                outer_fold
        )
    )

    # --------------------------------------------------------
    # specialist predictions
    # --------------------------------------------------------

    R_gate_train = (
        predict_residual_specialists_n13b(

            specialists,

            X_train_res[
                gate_train_local
            ]
        )
    )

    R_gate_val = (
        predict_residual_specialists_n13b(

            specialists,

            X_train_res[
                gate_val_local
            ]
        )
    )

    R_valid = (
        predict_residual_specialists_n13b(

            specialists,

            X_valid_res
        )
    )

    # --------------------------------------------------------
    # train gate
    # --------------------------------------------------------

    gate, best_epoch, inner_pr = (
        train_residual_gate_n13b(

            X_train_res[
                gate_train_local
            ],

            anchor_train[
                gate_train_local
            ],

            R_gate_train,

            y_train[
                gate_train_local
            ],

            X_train_res[
                gate_val_local
            ],

            anchor_train[
                gate_val_local
            ],

            R_gate_val,

            y_train[
                gate_val_local
            ],

            seed=
                SEED
                +
                123000
                +
                outer_fold
        )
    )

    # --------------------------------------------------------
    # outer valid
    # --------------------------------------------------------

    device = next(
        gate.parameters()
    ).device

    gate.eval()

    with torch.no_grad():

        xv = torch.tensor(
            X_valid_res,
            dtype=torch.float32,
            device=device
        )

        av = torch.tensor(
            anchor_valid,
            dtype=torch.float32,
            device=device
        )

        rv = torch.tensor(
            R_valid,
            dtype=torch.float32,
            device=device
        )

        wv = gate(
            xv
        )

        (
            pf,
            correction_logit,
            weighted_residual
        ) = forward_residual_moe_n13b(
            av,
            rv,
            wv
        )

        p_valid = (
            pf
            .cpu()
            .numpy()
        )

        weights_valid = (
            wv
            .cpu()
            .numpy()
        )

        correction_valid = (
            correction_logit
            .cpu()
            .numpy()
        )

    P_RESIDUAL_MOE_N13B[
        valid_idx
    ] = p_valid

    P_ANCHOR_N13B[
        valid_idx
    ] = anchor_valid

    W_RESIDUAL_N13B[
        valid_idx
    ] = weights_valid

    CORRECTION_LOGIT_N13B[
        valid_idx
    ] = correction_valid

    # --------------------------------------------------------
    # metrics
    # --------------------------------------------------------

    ma = metrics_n13b(
        y_valid,
        anchor_valid
    )

    mm = metrics_n13b(
        y_valid,
        p_valid
    )

    fold_rows_n13b.append({

        "Fold":
            outer_fold,

        "Anchor_PR":
            ma["PR_AUC"],

        "MoE_PR":
            mm["PR_AUC"],

        "Delta_PR":
            mm["PR_AUC"]
            -
            ma["PR_AUC"],

        "Anchor_ROC":
            ma["ROC_AUC"],

        "MoE_ROC":
            mm["ROC_AUC"],

        "Anchor_Brier":
            ma["Brier"],

        "MoE_Brier":
            mm["Brier"],

        "Anchor_NLL":
            ma["NLL"],

        "MoE_NLL":
            mm["NLL"],

        "Anchor_ECE":
            ma["ECE"],

        "MoE_ECE":
            mm["ECE"],
    })

    epoch_rows_n13b.append({

        "Fold":
            outer_fold,

        "Selected_Epoch":
            best_epoch,

        "Inner_PR":
            inner_pr,
    })

    print(
        f"Anchor PR: "
        f"{ma['PR_AUC']:.6f}"
    )

    print(
        f"Residual-MoE PR: "
        f"{mm['PR_AUC']:.6f}"
    )

    print(
        f"Delta PR: "
        f"{mm['PR_AUC'] - ma['PR_AUC']:+.6f}"
    )

    del specialists
    del gate

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# RESULTS
# ============================================================

anchor_all = metrics_n13b(
    y_DEV,
    P_ANCHOR_N13B
)

moe_all = metrics_n13b(
    y_DEV,
    P_RESIDUAL_MOE_N13B
)


RESULTS_N13B = pd.DataFrame([

    {
        "System":
            "Frozen_LSA_HE_5_Anchor",

        **anchor_all,

        "Delta_PR_vs_Anchor":
            0.0,
    },

    {
        "System":
            "Residual_Specialist_MoE",

        **moe_all,

        "Delta_PR_vs_Anchor":
            (
                moe_all["PR_AUC"]
                -
                anchor_all["PR_AUC"]
            ),
    },
])


FOLD_RESULTS_N13B = pd.DataFrame(
    fold_rows_n13b
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "MODULE N13B — RESIDUAL-SPECIALIST MoE RESULTS"
)

print(
    "=" * 165
)

display(
    RESULTS_N13B
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13B — FOLD RESULTS"
)

print(
    "=" * 165
)

display(
    FOLD_RESULTS_N13B
)


print(
    "\nMean fold delta PR:",
    FOLD_RESULTS_N13B[
        "Delta_PR"
    ].mean()
)


print(
    "Positive-delta folds:",
    int(
        (
            FOLD_RESULTS_N13B[
                "Delta_PR"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_FOLDS
)


# ============================================================
# ROUTING ENTROPY
# ============================================================

W = np.clip(
    W_RESIDUAL_N13B,
    1e-12,
    1.0
)

entropy = -np.sum(
    W
    *
    np.log(W),
    axis=1
)

norm_entropy = (
    entropy
    /
    np.log(
        N_SPECIALISTS_N13B
    )
)

effective = np.exp(
    entropy
)


ENTROPY_N13B = pd.DataFrame([{

    "NormalizedEntropy_Mean":
        norm_entropy.mean(),

    "NormalizedEntropy_SD":
        norm_entropy.std(),

    "NormalizedEntropy_P10":
        np.quantile(
            norm_entropy,
            0.10
        ),

    "NormalizedEntropy_Median":
        np.median(
            norm_entropy
        ),

    "NormalizedEntropy_P90":
        np.quantile(
            norm_entropy,
            0.90
        ),

    "EffectiveExperts_Mean":
        effective.mean(),

    "EffectiveExperts_Median":
        np.median(
            effective
        ),
}])


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13B — ROUTING ENTROPY"
)

print(
    "=" * 165
)

display(
    ENTROPY_N13B
)


# ============================================================
# CORRECTION DISTRIBUTION
# ============================================================

abs_corr = np.abs(
    CORRECTION_LOGIT_N13B
)


CORRECTION_N13B = pd.DataFrame([{

    "MeanSignedLogitCorrection":
        CORRECTION_LOGIT_N13B.mean(),

    "MeanAbsLogitCorrection":
        abs_corr.mean(),

    "MedianAbsLogitCorrection":
        np.median(
            abs_corr
        ),

    "P90AbsLogitCorrection":
        np.quantile(
            abs_corr,
            0.90
        ),

    "P99AbsLogitCorrection":
        np.quantile(
            abs_corr,
            0.99
        ),

    "MaxAbsLogitCorrection":
        abs_corr.max(),
}])


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13B — CORRECTION MAGNITUDE"
)

print(
    "=" * 165
)

display(
    CORRECTION_N13B
)


# ============================================================
# DOMINANT SPECIALIST
# ============================================================

specialist_names_n13b = [

    "GeneralResidual",
    "Underestimate",
    "Overestimate",
    "HardPositive",
    "HighConfidenceError",
]


dominant = np.argmax(
    W_RESIDUAL_N13B,
    axis=1
)


DOMINANT_N13B = pd.DataFrame([

    {
        "Specialist":
            name,

        "Dominant_N":
            int(
                np.sum(
                    dominant == j
                )
            ),

        "Dominant_Fraction":
            np.mean(
                dominant == j
            ),
    }

    for j, name
    in enumerate(
        specialist_names_n13b
    )
])


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13B — DOMINANT SPECIALIST USAGE"
)

print(
    "=" * 165
)

display(
    DOMINANT_N13B
)


GATE_EPOCHS_N13B = pd.DataFrame(
    epoch_rows_n13b
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13B — GATE EPOCHS"
)

print(
    "=" * 165
)

display(
    GATE_EPOCHS_N13B
)


# ============================================================
# SAVE
# ============================================================

np.save(
    PRED_DIR
    /
    "N13B_RESIDUAL_MOE_OOF.npy",

    P_RESIDUAL_MOE_N13B
)


np.save(
    PRED_DIR
    /
    "N13B_RESIDUAL_WEIGHTS.npy",

    W_RESIDUAL_N13B
)


np.save(
    PRED_DIR
    /
    "N13B_LOGIT_CORRECTIONS.npy",

    CORRECTION_LOGIT_N13B
)


RESULTS_N13B.to_csv(
    TAB_DIR
    /
    "n13b_results.csv",
    index=False
)


FOLD_RESULTS_N13B.to_csv(
    TAB_DIR
    /
    "n13b_fold_results.csv",
    index=False
)


ENTROPY_N13B.to_csv(
    TAB_DIR
    /
    "n13b_entropy.csv",
    index=False
)


CORRECTION_N13B.to_csv(
    TAB_DIR
    /
    "n13b_corrections.csv",
    index=False
)


DOMINANT_N13B.to_csv(
    TAB_DIR
    /
    "n13b_dominant_specialists.csv",
    index=False
)


GATE_EPOCHS_N13B.to_csv(
    TAB_DIR
    /
    "n13b_gate_epochs.csv",
    index=False
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "MODULE N13B COMPLETE"
)

print(
    "=" * 165
)

print(
    f"\nRuntime: "
    f"{time.perf_counter() - start_n13b:.1f}s"
)


N13B — OUTER FOLD 0
Anchor PR: 0.378993
Residual-MoE PR: 0.377311
Delta PR: -0.001681

N13B — OUTER FOLD 1
Anchor PR: 0.381858
Residual-MoE PR: 0.382242
Delta PR: +0.000384

N13B — OUTER FOLD 2
Anchor PR: 0.379144
Residual-MoE PR: 0.379443
Delta PR: +0.000299

N13B — OUTER FOLD 3
Anchor PR: 0.372725
Residual-MoE PR: 0.372470
Delta PR: -0.000255

N13B — OUTER FOLD 4
Anchor PR: 0.388756
Residual-MoE PR: 0.388771
Delta PR: +0.000016

MODULE N13B — RESIDUAL-SPECIALIST MoE RESULTS


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_Anchor
0,Frozen_LSA_HE_5_Anchor,0.379645,0.850617,0.069730,0.235667,0.002774,0.00000
1,Residual_Specialist_MoE,0.378725,0.850178,0.069819,0.236053,0.004903,-0.00092



N13B — FOLD RESULTS


,Fold,Anchor_PR,MoE_PR,Delta_PR,Anchor_ROC,MoE_ROC,Anchor_Brier,MoE_Brier,Anchor_NLL,MoE_NLL,Anchor_ECE,MoE_ECE
0,0,0.378993,0.377311,-0.001681,0.852518,0.852163,0.069673,0.069762,0.234947,0.235215,0.002848,0.005402
1,1,0.381858,0.382242,0.000384,0.850508,0.850520,0.069755,0.069798,0.235752,0.235993,0.004541,0.006509
2,2,0.379144,0.379443,0.000299,0.849177,0.849138,0.069828,0.069800,0.236326,0.236264,0.004185,0.004418
3,3,0.372725,0.372470,-0.000255,0.849145,0.849116,0.070082,0.070187,0.236670,0.236972,0.003733,0.006571
4,4,0.388756,0.388771,0.000016,0.851989,0.851979,0.069311,0.069549,0.234637,0.235820,0.005289,0.013894



Mean fold delta PR: -0.00024749532861291225
Positive-delta folds: 3 / 5

N13B — ROUTING ENTROPY


...


N13B — DOMINANT SPECIALIST USAGE


,Specialist,Dominant_N,Dominant_Fraction
0,GeneralResidual,69754,0.343711
1,Underestimate,0,0.000000
2,Overestimate,132529,0.653032
3,HardPositive,661,0.003257
4,HighConfidenceError,0,0.000000



N13B — GATE EPOCHS


,Fold,Selected_Epoch,Inner_PR
0,0,5,0.373175
1,1,4,0.388283
2,2,10,0.383959
3,3,3,0.350558
4,4,1,0.376603



MODULE N13B COMPLETE

Runtime: 67.2s


In [26]:
# ============================================================
# MODULE N13C — DIFFICULTY-ADAPTIVE SPARSE RESIDUAL MoE
#
# IDEA
# ------------------------------------------------------------
# Easy patients:
#       final = frozen static anchor
#
# Difficult patients:
#       final = anchor + bounded residual specialist correction
#
# Difficulty is estimated ONLY from inference-available
# quantities:
#       expert disagreement
#       probability spread
#       anchor uncertainty
#       expert-anchor deviations
#
# Threshold is selected inside outer training only.
#
# No 2015 secondary holdout.
# No BRFSS-2020.
# ============================================================

import gc
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# CONFIG
# ============================================================

EXPERTS_N13C = [
    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]

N_SPECIALISTS_N13C = 5

MAX_LOGIT_CORRECTION_N13C = 0.20

GATE_MAX_EPOCHS_N13C = 25
GATE_PATIENCE_N13C = 5
GATE_BATCH_N13C = 4096
GATE_LR_N13C = 1e-3
GATE_WD_N13C = 1e-4

# fixed candidate routing coverages
# selected ONLY inside outer-train
DIFFICULTY_COVERAGES_N13C = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.30,
]

LAMBDA_CORRECTION_N13C = 0.03


# ============================================================
# REPRODUCIBILITY
# ============================================================

def seed_n13c(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# METRICS
# ============================================================

def metrics_n13c(y, p):

    p = np.asarray(
        p,
        dtype=float
    )

    pc = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )

    # ECE
    bins = np.linspace(
        0,
        1,
        16
    )

    ece = 0.0

    for lo, hi in zip(
        bins[:-1],
        bins[1:]
    ):

        if hi == 1:

            m = (
                (p >= lo)
                &
                (p <= hi)
            )

        else:

            m = (
                (p >= lo)
                &
                (p < hi)
            )

        if m.sum() == 0:
            continue

        ece += (
            m.mean()
            *
            abs(
                y[m].mean()
                -
                p[m].mean()
            )
        )

    return {

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                pc
            ),

        "ECE":
            ece,
    }


# ============================================================
# DIFFICULTY SCORE
#
# Purely inference-time observable.
#
# Higher = more difficult / unstable.
# ============================================================

def difficulty_score_n13c(
    P
):

    P = np.asarray(
        P,
        dtype=np.float32
    )

    anchor = P.mean(
        axis=1
    )

    std = P.std(
        axis=1
    )

    spread = np.ptp(
        P,
        axis=1
    )

    abs_dev = np.abs(
        P
        -
        anchor[:, None]
    )

    mean_dev = abs_dev.mean(
        axis=1
    )

    max_dev = abs_dev.max(
        axis=1
    )

    # uncertainty around 0.5
    uncertainty = (
        1.0
        -
        2.0
        *
        np.abs(
            anchor
            -
            0.5
        )
    )

    # normalize each component robustly

    components = [
        std,
        spread,
        mean_dev,
        max_dev,
        uncertainty,
    ]

    normalized = []

    for x in components:

        lo = np.quantile(
            x,
            0.05
        )

        hi = np.quantile(
            x,
            0.95
        )

        z = (
            x - lo
        ) / (
            hi - lo + 1e-8
        )

        z = np.clip(
            z,
            0,
            1
        )

        normalized.append(
            z
        )

    score = np.mean(
        np.column_stack(
            normalized
        ),
        axis=1
    )

    return score.astype(
        np.float32
    )


# ============================================================
# FEATURE MATRIX
# ============================================================

def residual_features_n13c(
    global_idx,
    P,
    scaler
):

    Xraw = scaler.transform(

        X_DEV.iloc[
            global_idx
        ][FEATURES]

    ).astype(
        np.float32
    )

    P = np.asarray(
        P,
        dtype=np.float32
    )

    anchor = P.mean(
        axis=1,
        keepdims=True
    )

    dev = (
        P
        -
        anchor
    )

    abs_dev = np.abs(
        dev
    )

    difficulty = difficulty_score_n13c(
        P
    )[:, None]

    return np.concatenate(
        [
            Xraw,
            P,
            anchor,
            dev,
            abs_dev,
            difficulty,
        ],
        axis=1
    ).astype(
        np.float32
    )


# ============================================================
# SPECIALISTS
# ============================================================

def train_specialists_n13c(
    X,
    y,
    anchor,
    seed
):

    residual = (
        y - anchor
    )

    # General residual
    target_general = residual

    # Underestimate correction
    target_under = np.maximum(
        residual,
        0.0
    )

    # Overestimate correction
    target_over = np.minimum(
        residual,
        0.0
    )

    # Hard-positive weighted residual
    weight_positive = (
        1.0
        +
        5.0
        *
        (
            (y == 1)
            &
            (anchor < 0.20)
        )
    )

    # Large-error specialist
    weight_large_error = (
        1.0
        +
        4.0
        *
        (
            np.abs(
                residual
            )
            >
            np.quantile(
                np.abs(residual),
                0.80
            )
        )
    )

    configs = [

        (
            target_general,
            None
        ),

        (
            target_under,
            None
        ),

        (
            target_over,
            None
        ),

        (
            residual,
            weight_positive
        ),

        (
            residual,
            weight_large_error
        ),
    ]

    models = []

    for j, (
        target,
        sample_weight
    ) in enumerate(
        configs
    ):

        model = HistGradientBoostingRegressor(

            learning_rate=0.04,

            max_iter=120,

            max_leaf_nodes=15,

            l2_regularization=1.5,

            random_state=
                seed + j
        )

        model.fit(
            X,
            target,
            sample_weight=
                sample_weight
        )

        models.append(
            model
        )

    return models


def predict_specialists_n13c(
    models,
    X
):

    R = np.column_stack([

        m.predict(
            X
        )

        for m in models
    ])

    return np.clip(
        R,
        -0.15,
        0.15
    ).astype(
        np.float32
    )


# ============================================================
# SPARSE ROUTER
# ============================================================

class SparseResidualGateN13C(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        n_specialists
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.GELU(),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                64,
                32
            ),

            nn.GELU(),

            nn.Linear(
                32,
                n_specialists
            ),
        )


    def forward(
        self,
        x
    ):

        return torch.softmax(
            self.net(x),
            dim=1
        )


# ============================================================
# FORWARD
# ============================================================

def forward_sparse_residual_n13c(
    anchor,
    R,
    W
):

    weighted_residual = torch.sum(
        R * W,
        dim=1
    )

    logit_shift = (
        MAX_LOGIT_CORRECTION_N13C
        *
        torch.tanh(
            weighted_residual
            /
            0.05
        )
    )

    anchor = torch.clamp(
        anchor,
        1e-6,
        1 - 1e-6
    )

    anchor_logit = torch.log(
        anchor
        /
        (
            1 - anchor
        )
    )

    p = torch.sigmoid(
        anchor_logit
        +
        logit_shift
    )

    return (
        p,
        logit_shift
    )


# ============================================================
# GATE TRAINING
#
# Train ONLY on difficult cases.
# ============================================================

def train_sparse_gate_n13c(

    X_train,
    anchor_train,
    R_train,
    y_train,

    X_val,
    anchor_val,
    R_val,
    y_val,

    seed
):

    seed_n13c(
        seed
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = SparseResidualGateN13C(

        X_train.shape[1],

        N_SPECIALISTS_N13C

    ).to(
        device
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            GATE_LR_N13C,

        weight_decay=
            GATE_WD_N13C
    )

    dataset = TensorDataset(

        torch.tensor(
            X_train,
            dtype=torch.float32
        ),

        torch.tensor(
            anchor_train,
            dtype=torch.float32
        ),

        torch.tensor(
            R_train,
            dtype=torch.float32
        ),

        torch.tensor(
            y_train,
            dtype=torch.float32
        ),
    )

    loader = DataLoader(

        dataset,

        batch_size=
            min(
                GATE_BATCH_N13C,
                len(dataset)
            ),

        shuffle=True,

        num_workers=0
    )

    best_state = None
    best_pr = -np.inf
    best_epoch = 1
    no_improve = 0


    for epoch in range(
        1,
        GATE_MAX_EPOCHS_N13C + 1
    ):

        model.train()

        for (
            xb,
            ab,
            rb,
            yb
        ) in loader:

            xb = xb.to(device)
            ab = ab.to(device)
            rb = rb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            w = model(
                xb
            )

            (
                pf,
                shift
            ) = forward_sparse_residual_n13c(
                ab,
                rb,
                w
            )

            bce = nn.functional.binary_cross_entropy(
                pf,
                yb
            )

            penalty = torch.mean(
                shift ** 2
            )

            loss = (
                bce
                +
                LAMBDA_CORRECTION_N13C
                *
                penalty
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            optimizer.step()


        # validation
        model.eval()

        with torch.no_grad():

            xv = torch.tensor(
                X_val,
                dtype=torch.float32,
                device=device
            )

            av = torch.tensor(
                anchor_val,
                dtype=torch.float32,
                device=device
            )

            rv = torch.tensor(
                R_val,
                dtype=torch.float32,
                device=device
            )

            wv = model(
                xv
            )

            pv, _ = forward_sparse_residual_n13c(
                av,
                rv,
                wv
            )

            pv = pv.cpu().numpy()


        pr = average_precision_score(
            y_val,
            pv
        )

        if pr > best_pr + 1e-7:

            best_pr = pr
            best_epoch = epoch

            best_state = {

                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model.state_dict().items()
            }

            no_improve = 0

        else:

            no_improve += 1


        if (
            no_improve
            >=
            GATE_PATIENCE_N13C
        ):

            break


    model.load_state_dict(
        best_state
    )

    model.to(
        device
    )

    return (
        model,
        best_epoch,
        best_pr
    )


# ============================================================
# STORAGE
# ============================================================

P_N13C = np.full(
    len(y_DEV),
    np.nan,
    dtype=np.float32
)

P_ANCHOR_N13C = np.full(
    len(y_DEV),
    np.nan,
    dtype=np.float32
)

DIFFICULT_N13C = np.zeros(
    len(y_DEV),
    dtype=np.int8
)

CORRECTION_N13C = np.zeros(
    len(y_DEV),
    dtype=np.float32
)

W_N13C = np.full(
    (
        len(y_DEV),
        N_SPECIALISTS_N13C
    ),
    np.nan,
    dtype=np.float32
)


fold_rows_n13c = []
coverage_rows_n13c = []
epoch_rows_n13c = []


# ============================================================
# OUTER LOOP
# ============================================================

start_n13c = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        +
        "=" * 165
    )

    print(
        f"N13C — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 165
    )


    train_idx = np.where(
        DEV_FOLD != outer_fold
    )[0]

    valid_idx = np.where(
        DEV_FOLD == outer_fold
    )[0]

    y_train = y_DEV[
        train_idx
    ]

    y_valid = y_DEV[
        valid_idx
    ]


    # --------------------------------------------------------
    # strict inner-OOF expert predictions
    # --------------------------------------------------------

    cache_path = (
        PRED_DIR
        /
        f"N12_outer{outer_fold}_train_experts.npy"
    )

    if cache_path.exists():

        P_train = np.load(
            cache_path
        )

    else:

        P_train = (
            build_gate_training_oof_n10b(
                train_idx,
                outer_fold
            )
        )

        np.save(
            cache_path,
            P_train
        )


    P_valid = np.column_stack([

        OOF_N7B[
            e
        ][
            valid_idx
        ]

        for e
        in EXPERTS_N13C

    ]).astype(
        np.float32
    )


    anchor_train = P_train.mean(
        axis=1
    )

    anchor_valid = P_valid.mean(
        axis=1
    )


    # --------------------------------------------------------
    # features
    # --------------------------------------------------------

    scaler = StandardScaler()

    scaler.fit(
        X_DEV.iloc[
            train_idx
        ][FEATURES]
    )

    X_train = residual_features_n13c(
        train_idx,
        P_train,
        scaler
    )

    X_valid = residual_features_n13c(
        valid_idx,
        P_valid,
        scaler
    )


    difficulty_train = difficulty_score_n13c(
        P_train
    )

    difficulty_valid = difficulty_score_n13c(
        P_valid
    )


    # --------------------------------------------------------
    # split outer train
    # --------------------------------------------------------

    specialist_local, routing_local = (
        train_test_split(

            np.arange(
                len(train_idx)
            ),

            test_size=0.40,

            stratify=
                y_train,

            random_state=
                SEED
                +
                130000
                +
                outer_fold
        )
    )


    router_train_local, router_val_local = (
        train_test_split(

            routing_local,

            test_size=0.30,

            stratify=
                y_train[
                    routing_local
                ],

            random_state=
                SEED
                +
                131000
                +
                outer_fold
        )
    )


    # --------------------------------------------------------
    # train specialists
    # --------------------------------------------------------

    specialists = train_specialists_n13c(

        X_train[
            specialist_local
        ],

        y_train[
            specialist_local
        ],

        anchor_train[
            specialist_local
        ],

        seed=
            SEED
            +
            132000
            +
            outer_fold
    )


    R_router_train = predict_specialists_n13c(

        specialists,

        X_train[
            router_train_local
        ]
    )


    R_router_val = predict_specialists_n13c(

        specialists,

        X_train[
            router_val_local
        ]
    )


    R_valid = predict_specialists_n13c(

        specialists,

        X_valid
    )


    # ========================================================
    # SELECT DIFFICULTY COVERAGE INSIDE OUTER TRAIN
    # ========================================================

    candidate_rows = []

    best_pr = -np.inf
    best_coverage = None
    best_threshold = None
    best_gate_state = None
    best_epoch = None


    for coverage in DIFFICULTY_COVERAGES_N13C:

        threshold = np.quantile(

            difficulty_train[
                router_train_local
            ],

            1.0
            -
            coverage
        )


        train_hard = (
            difficulty_train[
                router_train_local
            ]
            >=
            threshold
        )


        val_hard = (
            difficulty_train[
                router_val_local
            ]
            >=
            threshold
        )


        # safety
        if (
            train_hard.sum() < 500
            or
            val_hard.sum() < 100
        ):

            continue


        gate, epoch, _ = train_sparse_gate_n13c(

            X_train[
                router_train_local
            ][
                train_hard
            ],

            anchor_train[
                router_train_local
            ][
                train_hard
            ],

            R_router_train[
                train_hard
            ],

            y_train[
                router_train_local
            ][
                train_hard
            ],

            X_train[
                router_val_local
            ][
                val_hard
            ],

            anchor_train[
                router_val_local
            ][
                val_hard
            ],

            R_router_val[
                val_hard
            ],

            y_train[
                router_val_local
            ][
                val_hard
            ],

            seed=
                SEED
                +
                133000
                +
                outer_fold * 100
                +
                int(
                    coverage * 100
                )
        )


        # ---------------------------------------------
        # evaluate FULL router-validation partition
        #
        # easy cases remain anchor
        # ---------------------------------------------

        p_val_full = anchor_train[
            router_val_local
        ].copy()


        device = next(
            gate.parameters()
        ).device


        with torch.no_grad():

            xv = torch.tensor(

                X_train[
                    router_val_local
                ][
                    val_hard
                ],

                dtype=torch.float32,

                device=device
            )


            av = torch.tensor(

                anchor_train[
                    router_val_local
                ][
                    val_hard
                ],

                dtype=torch.float32,

                device=device
            )


            rv = torch.tensor(

                R_router_val[
                    val_hard
                ],

                dtype=torch.float32,

                device=device
            )


            wv = gate(
                xv
            )


            pv_hard, _ = forward_sparse_residual_n13c(

                av,
                rv,
                wv
            )


        p_val_full[
            val_hard
        ] = (
            pv_hard
            .cpu()
            .numpy()
        )


        full_pr = average_precision_score(

            y_train[
                router_val_local
            ],

            p_val_full
        )


        candidate_rows.append({

            "Fold":
                outer_fold,

            "Coverage":
                coverage,

            "DifficultyThreshold":
                threshold,

            "Validation_PR":
                full_pr,

            "Routed_Validation_N":
                int(
                    val_hard.sum()
                ),
        })


        if full_pr > best_pr:

            best_pr = full_pr

            best_coverage = coverage

            best_threshold = threshold

            best_epoch = epoch

            best_gate_state = {

                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in gate.state_dict().items()
            }


        del gate

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    if best_gate_state is None:

        raise RuntimeError(
            "No valid N13C routing candidate."
        )


    # ========================================================
    # REBUILD FINAL GATE FOR CHOSEN COVERAGE
    #
    # train on all routing rows inside difficult region
    # ========================================================

    routing_hard = (
        difficulty_train[
            routing_local
        ]
        >=
        best_threshold
    )


    R_routing_all = predict_specialists_n13c(

        specialists,

        X_train[
            routing_local
        ]
    )


    final_gate = SparseResidualGateN13C(

        X_train.shape[1],

        N_SPECIALISTS_N13C

    )


    # use selected gate state as initialization
    final_gate.load_state_dict(
        best_gate_state
    )


    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    final_gate = final_gate.to(
        device
    )


    optimizer = torch.optim.AdamW(

        final_gate.parameters(),

        lr=
            GATE_LR_N13C,

        weight_decay=
            GATE_WD_N13C
    )


    ds = TensorDataset(

        torch.tensor(
            X_train[
                routing_local
            ][
                routing_hard
            ],
            dtype=torch.float32
        ),

        torch.tensor(
            anchor_train[
                routing_local
            ][
                routing_hard
            ],
            dtype=torch.float32
        ),

        torch.tensor(
            R_routing_all[
                routing_hard
            ],
            dtype=torch.float32
        ),

        torch.tensor(
            y_train[
                routing_local
            ][
                routing_hard
            ],
            dtype=torch.float32
        ),
    )


    loader = DataLoader(

        ds,

        batch_size=
            min(
                GATE_BATCH_N13C,
                len(ds)
            ),

        shuffle=True,

        num_workers=0
    )


    final_gate.train()


    for epoch in range(
        best_epoch
    ):

        for (
            xb,
            ab,
            rb,
            yb
        ) in loader:

            xb = xb.to(device)
            ab = ab.to(device)
            rb = rb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            w = final_gate(
                xb
            )

            pf, shift = forward_sparse_residual_n13c(

                ab,
                rb,
                w
            )

            loss = (

                nn.functional.binary_cross_entropy(
                    pf,
                    yb
                )

                +

                LAMBDA_CORRECTION_N13C
                *
                torch.mean(
                    shift ** 2
                )
            )

            loss.backward()

            optimizer.step()


    # ========================================================
    # OUTER VALID
    # ========================================================

    valid_hard = (
        difficulty_valid
        >=
        best_threshold
    )


    p_final = anchor_valid.copy()


    # uniform weights for easy cases purely for storage
    W_fold = np.full(

        (
            len(valid_idx),
            N_SPECIALISTS_N13C
        ),

        1.0
        /
        N_SPECIALISTS_N13C,

        dtype=np.float32
    )


    correction_fold = np.zeros(

        len(
            valid_idx
        ),

        dtype=np.float32
    )


    if valid_hard.sum() > 0:

        final_gate.eval()

        with torch.no_grad():

            xv = torch.tensor(

                X_valid[
                    valid_hard
                ],

                dtype=torch.float32,

                device=device
            )


            av = torch.tensor(

                anchor_valid[
                    valid_hard
                ],

                dtype=torch.float32,

                device=device
            )


            rv = torch.tensor(

                R_valid[
                    valid_hard
                ],

                dtype=torch.float32,

                device=device
            )


            wv = final_gate(
                xv
            )


            pv, shift = forward_sparse_residual_n13c(

                av,
                rv,
                wv
            )


        p_final[
            valid_hard
        ] = (
            pv
            .cpu()
            .numpy()
        )


        W_fold[
            valid_hard
        ] = (
            wv
            .cpu()
            .numpy()
        )


        correction_fold[
            valid_hard
        ] = (
            shift
            .cpu()
            .numpy()
        )


    # ========================================================
    # STORE
    # ========================================================

    P_N13C[
        valid_idx
    ] = p_final


    P_ANCHOR_N13C[
        valid_idx
    ] = anchor_valid


    DIFFICULT_N13C[
        valid_idx
    ] = valid_hard.astype(
        np.int8
    )


    W_N13C[
        valid_idx
    ] = W_fold


    CORRECTION_N13C[
        valid_idx
    ] = correction_fold


    # ========================================================
    # METRICS
    # ========================================================

    ma = metrics_n13c(

        y_valid,
        anchor_valid
    )


    mm = metrics_n13c(

        y_valid,
        p_final
    )


    fold_rows_n13c.append({

        "Fold":
            outer_fold,

        "SelectedCoverage":
            best_coverage,

        "RoutedFraction":
            valid_hard.mean(),

        "Anchor_PR":
            ma["PR_AUC"],

        "MoE_PR":
            mm["PR_AUC"],

        "Delta_PR":
            mm["PR_AUC"]
            -
            ma["PR_AUC"],

        "Anchor_ROC":
            ma["ROC_AUC"],

        "MoE_ROC":
            mm["ROC_AUC"],

        "Anchor_Brier":
            ma["Brier"],

        "MoE_Brier":
            mm["Brier"],

        "Anchor_NLL":
            ma["NLL"],

        "MoE_NLL":
            mm["NLL"],

        "Anchor_ECE":
            ma["ECE"],

        "MoE_ECE":
            mm["ECE"],
    })


    coverage_rows_n13c.extend(
        candidate_rows
    )


    epoch_rows_n13c.append({

        "Fold":
            outer_fold,

        "Coverage":
            best_coverage,

        "DifficultyThreshold":
            best_threshold,

        "SelectedEpoch":
            best_epoch,

        "InnerValidationPR":
            best_pr,
    })


    print(
        f"Selected coverage: "
        f"{best_coverage:.2f}"
    )

    print(
        f"Outer routed fraction: "
        f"{valid_hard.mean():.4f}"
    )

    print(
        f"Anchor PR: "
        f"{ma['PR_AUC']:.6f}"
    )

    print(
        f"Sparse MoE PR: "
        f"{mm['PR_AUC']:.6f}"
    )

    print(
        f"Delta PR: "
        f"{mm['PR_AUC'] - ma['PR_AUC']:+.6f}"
    )


    del specialists
    del final_gate

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# FINAL RESULTS
# ============================================================

anchor_all = metrics_n13c(
    y_DEV,
    P_ANCHOR_N13C
)

moe_all = metrics_n13c(
    y_DEV,
    P_N13C
)


RESULTS_N13C = pd.DataFrame([

    {
        "System":
            "Frozen_LSA_HE_5_Anchor",

        **anchor_all,

        "Delta_PR_vs_Anchor":
            0.0,
    },

    {
        "System":
            "Difficulty_Adaptive_Residual_MoE",

        **moe_all,

        "Delta_PR_vs_Anchor":
            (
                moe_all["PR_AUC"]
                -
                anchor_all["PR_AUC"]
            ),
    },
])


FOLD_RESULTS_N13C = pd.DataFrame(
    fold_rows_n13c
)


COVERAGE_SEARCH_N13C = pd.DataFrame(
    coverage_rows_n13c
)


EPOCHS_N13C = pd.DataFrame(
    epoch_rows_n13c
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "MODULE N13C — DIFFICULTY-ADAPTIVE RESIDUAL MoE"
)

print(
    "=" * 165
)

display(
    RESULTS_N13C
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13C — FOLD RESULTS"
)

print(
    "=" * 165
)

display(
    FOLD_RESULTS_N13C
)


print(
    "\nMean fold delta PR:",
    FOLD_RESULTS_N13C[
        "Delta_PR"
    ].mean()
)


print(
    "Positive folds:",
    int(
        (
            FOLD_RESULTS_N13C[
                "Delta_PR"
            ]
            >
            0
        ).sum()
    ),
    "/",
    N_FOLDS
)


print(
    "\nOverall routed fraction:",
    DIFFICULT_N13C.mean()
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13C — COVERAGE SEARCH"
)

print(
    "=" * 165
)

display(
    COVERAGE_SEARCH_N13C
)


# ============================================================
# HARD-REGION PERFORMANCE
# ============================================================

hard = (
    DIFFICULT_N13C
    ==
    1
)


if hard.sum() > 0:

    HARD_REGION_N13C = pd.DataFrame([

        {
            "System":
                "Anchor",

            **metrics_n13c(
                y_DEV[hard],
                P_ANCHOR_N13C[hard]
            ),
        },

        {
            "System":
                "SparseResidualMoE",

            **metrics_n13c(
                y_DEV[hard],
                P_N13C[hard]
            ),
        },

    ])


    print(
        "\n"
        +
        "=" * 165
    )

    print(
        "N13C — ROUTED HARD-REGION PERFORMANCE"
    )

    print(
        "=" * 165
    )

    display(
        HARD_REGION_N13C
    )


# ============================================================
# CORRECTION MAGNITUDE
# ============================================================

routed_corr = np.abs(
    CORRECTION_N13C[
        hard
    ]
)


CORRECTION_SUMMARY_N13C = pd.DataFrame([{

    "Routed_N":
        int(
            hard.sum()
        ),

    "Routed_Fraction":
        hard.mean(),

    "MeanAbsLogitCorrection":
        (
            routed_corr.mean()
            if len(routed_corr)
            else 0
        ),

    "MedianAbsLogitCorrection":
        (
            np.median(
                routed_corr
            )
            if len(routed_corr)
            else 0
        ),

    "P90AbsLogitCorrection":
        (
            np.quantile(
                routed_corr,
                0.90
            )
            if len(routed_corr)
            else 0
        ),

    "MaxAbsLogitCorrection":
        (
            routed_corr.max()
            if len(routed_corr)
            else 0
        ),
}])


print(
    "\n"
    +
    "=" * 165
)

print(
    "N13C — ROUTED CORRECTION MAGNITUDE"
)

print(
    "=" * 165
)

display(
    CORRECTION_SUMMARY_N13C
)


# ============================================================
# SAVE
# ============================================================

np.save(
    PRED_DIR
    /
    "N13C_DIFFICULTY_ADAPTIVE_MOE.npy",

    P_N13C
)


np.save(
    PRED_DIR
    /
    "N13C_DIFFICULT_MASK.npy",

    DIFFICULT_N13C
)


RESULTS_N13C.to_csv(
    TAB_DIR
    /
    "n13c_results.csv",
    index=False
)


FOLD_RESULTS_N13C.to_csv(
    TAB_DIR
    /
    "n13c_fold_results.csv",
    index=False
)


COVERAGE_SEARCH_N13C.to_csv(
    TAB_DIR
    /
    "n13c_coverage_search.csv",
    index=False
)


EPOCHS_N13C.to_csv(
    TAB_DIR
    /
    "n13c_selected_parameters.csv",
    index=False
)


CORRECTION_SUMMARY_N13C.to_csv(
    TAB_DIR
    /
    "n13c_correction_summary.csv",
    index=False
)


print(
    "\n"
    +
    "=" * 165
)

print(
    "MODULE N13C COMPLETE"
)

print(
    "=" * 165
)

print(
    f"\nRuntime: "
    f"{time.perf_counter() - start_n13c:.1f}s"
)


N13C — OUTER FOLD 0
Selected coverage: 0.15
Outer routed fraction: 0.1467
Anchor PR: 0.378993
Sparse MoE PR: 0.377784
Delta PR: -0.001208

N13C — OUTER FOLD 1
Selected coverage: 0.05
Outer routed fraction: 0.0477
Anchor PR: 0.381858
Sparse MoE PR: 0.381609
Delta PR: -0.000249

N13C — OUTER FOLD 2
Selected coverage: 0.15
Outer routed fraction: 0.1571
Anchor PR: 0.379144
Sparse MoE PR: 0.378770
Delta PR: -0.000373

N13C — OUTER FOLD 3
Selected coverage: 0.05
Outer routed fraction: 0.0483
Anchor PR: 0.372725
Sparse MoE PR: 0.372999
Delta PR: +0.000274

N13C — OUTER FOLD 4
Selected coverage: 0.20
Outer routed fraction: 0.2073
Anchor PR: 0.388756
Sparse MoE PR: 0.388393
Delta PR: -0.000363

MODULE N13C — DIFFICULTY-ADAPTIVE RESIDUAL MoE


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE,Delta_PR_vs_Anchor
0,Frozen_LSA_HE_5_Anchor,0.379645,0.850617,0.069730,0.235667,0.002774,0.000000
1,Difficulty_Adaptive_Residual_MoE,0.379359,0.850530,0.069741,0.235701,0.002725,-0.000286



N13C — FOLD RESULTS


,Fold,SelectedCoverage,RoutedFraction,Anchor_PR,MoE_PR,Delta_PR,Anchor_ROC,MoE_ROC,Anchor_Brier,MoE_Brier,Anchor_NLL,MoE_NLL,Anchor_ECE,MoE_ECE
0,0,0.15,0.146739,0.378993,0.377784,-0.001208,0.852518,0.852342,0.069673,0.069721,0.234947,0.235077,0.002848,0.004148
1,1,0.05,0.047698,0.381858,0.381609,-0.000249,0.850508,0.850491,0.069755,0.069767,0.235752,0.235778,0.004541,0.004111
2,2,0.15,0.157136,0.379144,0.378770,-0.000373,0.849177,0.849117,0.069828,0.069836,0.236326,0.236373,0.004185,0.004586
3,3,0.05,0.048264,0.372725,0.372999,0.000274,0.849145,0.849167,0.070082,0.070067,0.236670,0.236639,0.003733,0.003779
4,4,0.20,0.207303,0.388756,0.388393,-0.000363,0.851989,0.851908,0.069311,0.069312,0.234637,0.234640,0.005289,0.004776



Mean fold delta PR: -0.0003840058319558914
Positive folds: 1 / 5

Overall routed fraction: 0.12142758593503626

N13C — COVERAGE SEARCH


...


N13C — ROUTED HARD-REGION PERFORMANCE


,System,PR_AUC,ROC_AUC,Brier,NLL,ECE
0,Anchor,0.487936,0.687371,0.190271,0.563208,0.006504
1,SparseResidualMoE,0.487486,0.686777,0.190360,0.563494,0.006226



N13C — ROUTED CORRECTION MAGNITUDE


,Routed_N,Routed_Fraction,MeanAbsLogitCorrection,MedianAbsLogitCorrection,P90AbsLogitCorrection,MaxAbsLogitCorrection
0,24643,0.121428,0.066025,0.058588,0.133081,0.191893



MODULE N13C COMPLETE

Runtime: 71.8s


In [27]:
# ============================================================
# MODULE N14
# STRICT CLASS-CONDITIONAL RELIABILITY
# + DUAL SELECTIVE REVIEW
#
# FINAL PREDICTOR:
#   N10B Consensus-Regularized MoE
#
# TWO RELIABILITY TASKS
# ------------------------------------------------------------
# FN task:
#   Only predicted-negative cases
#   target = 1 if true class is positive
#
#   P(FN | predicted negative, features)
#
# FP task:
#   Only predicted-positive cases
#   target = 1 if true class is negative
#
#   P(FP | predicted positive, features)
#
# FEATURE BLOCKS
# ------------------------------------------------------------
#   CONF
#   CONF_DIS
#   CONF_LPNS
#   CONF_DIS_LPNS
#
# STRICTNESS
# ------------------------------------------------------------
# Outer training:
#   cached strict inner-OOF experts
#   -> cross-fitted final MoE
#   -> operating threshold
#   -> FN/FP reliability heads
#
# Outer validation:
#   strict N10B final MoE
#   -> reliability prediction
#
# LPNS reference bank contains outer-training cases only.
#
# No 2015 secondary holdout.
# No BRFSS-2020.
# ============================================================


import gc
import os
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# N14.1 CONFIG
# ============================================================

EXPERTS_N14 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


K_LPNS_N14 = 20

REFERENCE_SIZE_N14 = 80000


REVIEW_BUDGETS_N14 = [

    0.01,
    0.025,
    0.05,
    0.10,
    0.15,
    0.20,
]


BLOCKS_N14 = {

    "CONF":
        ["C"],

    "CONF_DIS":
        ["C", "D"],

    "CONF_LPNS":
        ["C", "L"],

    "CONF_DIS_LPNS":
        ["C", "D", "L"],
}


# ============================================================
# N14.2 REQUIRED OBJECTS
# ============================================================

required_n14 = [

    "P_MOE_N10B",
    "GATE_EPOCHS_N10B",

    "crossfit_gate_predictions_n12",

    "build_confidence_features_n9",
    "build_disagreement_features_n9",

    "compute_lpns_strict_n12",

    "make_reliability_head_n9",

    "select_threshold_n9",

    "OOF_N7B",
    "X_DEV",
    "y_DEV",
    "DEV_FOLD",
    "FEATURES",
]


for obj in required_n14:

    if obj not in globals():

        raise RuntimeError(
            f"N14 requires '{obj}'."
        )


# ============================================================
# N14.3 FINAL MoE
# ============================================================

P_FINAL_N14 = np.asarray(

    P_MOE_N10B,

    dtype=np.float32
)


if np.isnan(
    P_FINAL_N14
).any():

    raise RuntimeError(
        "Incomplete final N10B MoE probabilities."
    )


# ============================================================
# N14.4 FEATURE COMBINATION
# ============================================================

def combine_blocks_n14(
    codes,
    C,
    D,
    L,
):

    matrices = []

    for code in codes:

        if code == "C":

            matrices.append(C)

        elif code == "D":

            matrices.append(D)

        elif code == "L":

            matrices.append(L)

        else:

            raise ValueError(
                code
            )

    return np.concatenate(
        matrices,
        axis=1
    )


# ============================================================
# N14.5 PROBABILITY METRICS
#
# Safe for class-conditional reliability tasks.
# ============================================================

def reliability_metrics_n14(
    y,
    p,
):

    y = np.asarray(
        y,
        dtype=np.int8
    )

    p = np.asarray(
        p,
        dtype=float
    )

    pc = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )


    result = {

        "N":
            len(y),

        "Positive_N":
            int(
                np.sum(
                    y == 1
                )
            ),

        "Prevalence":
            float(
                np.mean(y)
            ),

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                pc,
                labels=[0, 1]
            ),
    }


    # ROC requires both classes

    if len(
        np.unique(y)
    ) == 2:

        result[
            "ROC_AUC"
        ] = roc_auc_score(
            y,
            p
        )

    else:

        result[
            "ROC_AUC"
        ] = np.nan


    return result


# ============================================================
# N14.6 STORAGE
#
# Risk arrays exist only for their relevant predicted class.
# Other rows remain NaN.
# ============================================================

FN_RISK_N14 = {

    block:
        np.full(
            len(y_DEV),
            np.nan,
            dtype=np.float32
        )

    for block
    in BLOCKS_N14
}


FP_RISK_N14 = {

    block:
        np.full(
            len(y_DEV),
            np.nan,
            dtype=np.float32
        )

    for block
    in BLOCKS_N14
}


PRED_CLASS_N14 = np.full(

    len(y_DEV),

    -1,

    dtype=np.int8
)


THRESHOLD_N14 = np.full(

    len(y_DEV),

    np.nan,

    dtype=np.float32
)


fold_rows_n14 = []


# ============================================================
# N14.7 OUTER LOOP
# ============================================================

start_n14 = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 165
    )

    print(
        f"N14 — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 165
    )


    train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    y_train = y_DEV[
        train_idx
    ]


    y_valid = y_DEV[
        valid_idx
    ]


    # ========================================================
    # STEP 1
    # STRICT INNER-OOF EXPERT PREDICTIONS
    # ========================================================

    cache_path = (

        PRED_DIR
        /
        f"N12_outer{outer_fold}_train_experts.npy"
    )


    if not cache_path.exists():

        raise RuntimeError(

            f"Missing N12 expert cache: "
            f"{cache_path}"
        )


    print(
        "Loading cached strict train experts..."
    )


    P_train_experts = np.load(

        cache_path

    ).astype(
        np.float32
    )


    # ========================================================
    # STEP 2
    # STRICT OUTER-VALID EXPERTS
    # ========================================================

    P_valid_experts = np.column_stack([

        OOF_N7B[
            expert
        ][
            valid_idx
        ]

        for expert
        in EXPERTS_N14

    ]).astype(
        np.float32
    )


    # ========================================================
    # STEP 3
    # CROSS-FITTED FINAL MoE ON OUTER TRAIN
    # ========================================================

    selected_epoch = int(

        GATE_EPOCHS_N10B.loc[

            GATE_EPOCHS_N10B[
                "Fold"
            ]
            ==
            outer_fold,

            "Selected_Epoch"

        ].iloc[0]
    )


    print(
        "Generating cross-fitted train MoE..."
    )


    p_train_moe = (

        crossfit_gate_predictions_n12(

            train_idx,

            P_train_experts,

            outer_fold,

            selected_epoch
        )
    )


    p_valid_moe = (

        P_FINAL_N14[
            valid_idx
        ]
    )


    # ========================================================
    # STEP 4
    # OPERATING THRESHOLD
    #
    # Chosen using outer-training rows only.
    # ========================================================

    threshold, train_f1 = (

        select_threshold_n9(

            y_train,

            p_train_moe
        )
    )


    train_pred = (

        p_train_moe
        >=
        threshold

    ).astype(
        np.int8
    )


    valid_pred = (

        p_valid_moe
        >=
        threshold

    ).astype(
        np.int8
    )


    PRED_CLASS_N14[
        valid_idx
    ] = valid_pred


    THRESHOLD_N14[
        valid_idx
    ] = threshold


    print(
        f"Threshold: "
        f"{threshold:.6f}"
    )


    # ========================================================
    # STEP 5
    # CLASS-CONDITIONAL TARGETS
    # ========================================================

    # --------------------------------------------------------
    # Predicted negatives
    #
    # target=1 means FALSE NEGATIVE
    # --------------------------------------------------------

    train_neg = (
        train_pred == 0
    )


    valid_neg = (
        valid_pred == 0
    )


    y_fn_train = (

        y_train[
            train_neg
        ]
        ==
        1

    ).astype(
        np.int8
    )


    y_fn_valid = (

        y_valid[
            valid_neg
        ]
        ==
        1

    ).astype(
        np.int8
    )


    # --------------------------------------------------------
    # Predicted positives
    #
    # target=1 means FALSE POSITIVE
    # --------------------------------------------------------

    train_pos = (
        train_pred == 1
    )


    valid_pos = (
        valid_pred == 1
    )


    y_fp_train = (

        y_train[
            train_pos
        ]
        ==
        0

    ).astype(
        np.int8
    )


    y_fp_valid = (

        y_valid[
            valid_pos
        ]
        ==
        0

    ).astype(
        np.int8
    )


    print(
        f"Train predicted-negative N: "
        f"{train_neg.sum():,} | "
        f"FN prevalence: "
        f"{y_fn_train.mean():.5f}"
    )


    print(
        f"Train predicted-positive N: "
        f"{train_pos.sum():,} | "
        f"FP prevalence: "
        f"{y_fp_train.mean():.5f}"
    )


    print(
        f"Valid predicted-negative N: "
        f"{valid_neg.sum():,} | "
        f"FN prevalence: "
        f"{y_fn_valid.mean():.5f}"
    )


    print(
        f"Valid predicted-positive N: "
        f"{valid_pos.sum():,} | "
        f"FP prevalence: "
        f"{y_fp_valid.mean():.5f}"
    )


    # ========================================================
    # STEP 6
    # CONFIDENCE FEATURES
    # ========================================================

    C_train_all = (

        build_confidence_features_n9(

            p_train_moe,

            threshold
        )
    )


    C_valid_all = (

        build_confidence_features_n9(

            p_valid_moe,

            threshold
        )
    )


    # ========================================================
    # STEP 7
    # DISAGREEMENT FEATURES
    # ========================================================

    D_train_all = (

        build_disagreement_features_n9(

            P_train_experts
        )
    )


    D_valid_all = (

        build_disagreement_features_n9(

            P_valid_experts
        )
    )


    # ========================================================
    # STEP 8
    # STRICT LPNS REFERENCE BANK
    # ========================================================

    rng = np.random.default_rng(

        SEED
        +
        141000
        +
        outer_fold
    )


    reference_n = min(

        REFERENCE_SIZE_N14,

        len(
            train_idx
        )
    )


    reference_local = rng.choice(

        np.arange(
            len(
                train_idx
            )
        ),

        size=
            reference_n,

        replace=False
    )


    reference_global = (

        train_idx[
            reference_local
        ]
    )


    reference_pred = (

        P_train_experts[
            reference_local
        ]
    )


    scaler = StandardScaler()


    scaler.fit(

        X_DEV.iloc[
            train_idx
        ][
            FEATURES
        ]
    )


    Z_reference = scaler.transform(

        X_DEV.iloc[
            reference_global
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    nn_model = NearestNeighbors(

        n_neighbors=
            K_LPNS_N14 + 1,

        metric="euclidean",

        algorithm="auto",

        n_jobs=-1
    )


    nn_model.fit(
        Z_reference
    )


    # ========================================================
    # STEP 9
    # LPNS
    # ========================================================

    print(
        "Computing strict train LPNS..."
    )


    L_train_all = (

        compute_lpns_strict_n12(

            train_idx,

            P_train_experts,

            reference_global,

            reference_pred,

            scaler,

            nn_model,

            remove_self=True
        )
    )


    print(
        "Computing strict valid LPNS..."
    )


    L_valid_all = (

        compute_lpns_strict_n12(

            valid_idx,

            P_valid_experts,

            reference_global,

            reference_pred,

            scaler,

            nn_model,

            remove_self=False
        )
    )


    # ========================================================
    # STEP 10
    # TRAIN SEPARATE FN / FP RELIABILITY HEADS
    # ========================================================

    valid_neg_global = (

        valid_idx[
            valid_neg
        ]
    )


    valid_pos_global = (

        valid_idx[
            valid_pos
        ]
    )


    for block_number, (
        block_name,
        codes

    ) in enumerate(
        BLOCKS_N14.items()
    ):


        # ----------------------------------------------------
        # FULL FEATURE MATRICES
        # ----------------------------------------------------

        X_rel_train_all = (

            combine_blocks_n14(

                codes,

                C_train_all,

                D_train_all,

                L_train_all
            )
        )


        X_rel_valid_all = (

            combine_blocks_n14(

                codes,

                C_valid_all,

                D_valid_all,

                L_valid_all
            )
        )


        # ====================================================
        # FN HEAD
        # ====================================================

        X_fn_train = (

            X_rel_train_all[
                train_neg
            ]
        )


        X_fn_valid = (

            X_rel_valid_all[
                valid_neg
            ]
        )


        fn_model = make_reliability_head_n9(

            SEED
            +
            142000
            +
            outer_fold * 100
            +
            block_number
        )


        fn_model.fit(

            X_fn_train,

            y_fn_train
        )


        fn_risk = (

            fn_model

            .predict_proba(
                X_fn_valid
            )[:, 1]
        )


        FN_RISK_N14[
            block_name
        ][
            valid_neg_global
        ] = fn_risk


        fn_metrics = (

            reliability_metrics_n14(

                y_fn_valid,

                fn_risk
            )
        )


        # ====================================================
        # FP HEAD
        # ====================================================

        X_fp_train = (

            X_rel_train_all[
                train_pos
            ]
        )


        X_fp_valid = (

            X_rel_valid_all[
                valid_pos
            ]
        )


        fp_model = make_reliability_head_n9(

            SEED
            +
            143000
            +
            outer_fold * 100
            +
            block_number
        )


        fp_model.fit(

            X_fp_train,

            y_fp_train
        )


        fp_risk = (

            fp_model

            .predict_proba(
                X_fp_valid
            )[:, 1]
        )


        FP_RISK_N14[
            block_name
        ][
            valid_pos_global
        ] = fp_risk


        fp_metrics = (

            reliability_metrics_n14(

                y_fp_valid,

                fp_risk
            )
        )


        # ====================================================
        # FOLD STORAGE
        # ====================================================

        fold_rows_n14.append({

            "Fold":
                outer_fold,

            "Block":
                block_name,

            "Task":
                "FN",

            **fn_metrics,
        })


        fold_rows_n14.append({

            "Fold":
                outer_fold,

            "Block":
                block_name,

            "Task":
                "FP",

            **fp_metrics,
        })


        print(

            f"{block_name:16s} | "

            f"FN_PR="
            f"{fn_metrics['PR_AUC']:.6f} | "

            f"FN_ROC="
            f"{fn_metrics['ROC_AUC']:.6f} | "

            f"FP_PR="
            f"{fp_metrics['PR_AUC']:.6f} | "

            f"FP_ROC="
            f"{fp_metrics['ROC_AUC']:.6f}"
        )


        del fn_model
        del fp_model

        del X_rel_train_all
        del X_rel_valid_all

        del X_fn_train
        del X_fn_valid

        del X_fp_train
        del X_fp_valid


    # ========================================================
    # CLEAN FOLD
    # ========================================================

    del P_train_experts
    del P_valid_experts

    del C_train_all
    del C_valid_all

    del D_train_all
    del D_valid_all

    del L_train_all
    del L_valid_all

    del scaler
    del nn_model
    del Z_reference

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# N14.8 COMPLETENESS
# ============================================================

if np.any(
    PRED_CLASS_N14 < 0
):

    raise RuntimeError(
        "Incomplete N14 predictions."
    )


NEG_MASK_N14 = (

    PRED_CLASS_N14 == 0
)


POS_MASK_N14 = (

    PRED_CLASS_N14 == 1
)


FN_TARGET_N14 = (

    y_DEV[
        NEG_MASK_N14
    ]
    ==
    1

).astype(
    np.int8
)


FP_TARGET_N14 = (

    y_DEV[
        POS_MASK_N14
    ]
    ==
    0

).astype(
    np.int8
)


for block in BLOCKS_N14:

    if np.isnan(

        FN_RISK_N14[
            block
        ][
            NEG_MASK_N14
        ]

    ).any():

        raise RuntimeError(

            f"Incomplete FN risk: {block}"
        )


    if np.isnan(

        FP_RISK_N14[
            block
        ][
            POS_MASK_N14
        ]

    ).any():

        raise RuntimeError(

            f"Incomplete FP risk: {block}"
        )


# ============================================================
# N14.9 POOLED CLASS-CONDITIONAL RELIABILITY
# ============================================================

pooled_rows_n14 = []


for block in BLOCKS_N14:


    # FN
    fn_metrics = reliability_metrics_n14(

        FN_TARGET_N14,

        FN_RISK_N14[
            block
        ][
            NEG_MASK_N14
        ]
    )


    pooled_rows_n14.append({

        "Task":
            "FN",

        "Block":
            block,

        **fn_metrics,
    })


    # FP
    fp_metrics = reliability_metrics_n14(

        FP_TARGET_N14,

        FP_RISK_N14[
            block
        ][
            POS_MASK_N14
        ]
    )


    pooled_rows_n14.append({

        "Task":
            "FP",

        "Block":
            block,

        **fp_metrics,
    })


CLASS_CONDITIONAL_RESULTS_N14 = (

    pd.DataFrame(
        pooled_rows_n14
    )

    .sort_values(
        [
            "Task",
            "PR_AUC"
        ],

        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — STRICT CLASS-CONDITIONAL RELIABILITY"
)

print(
    "=" * 165
)


display(
    CLASS_CONDITIONAL_RESULTS_N14
)


# ============================================================
# N14.10 FN RESULTS ONLY
# ============================================================

FN_RESULTS_N14 = (

    CLASS_CONDITIONAL_RESULTS_N14[

        CLASS_CONDITIONAL_RESULTS_N14[
            "Task"
        ]
        ==
        "FN"

    ]

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — FALSE-NEGATIVE RISK RESULTS"
)

print(
    "=" * 165
)


display(
    FN_RESULTS_N14
)


# ============================================================
# N14.11 FP RESULTS ONLY
# ============================================================

FP_RESULTS_N14 = (

    CLASS_CONDITIONAL_RESULTS_N14[

        CLASS_CONDITIONAL_RESULTS_N14[
            "Task"
        ]
        ==
        "FP"

    ]

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — FALSE-POSITIVE RISK RESULTS"
)

print(
    "=" * 165
)


display(
    FP_RESULTS_N14
)


# ============================================================
# N14.12 INCREMENTAL LPNS
# ============================================================

def get_metric_n14(
    task,
    block,
    metric,
):

    return float(

        CLASS_CONDITIONAL_RESULTS_N14.loc[

            (
                CLASS_CONDITIONAL_RESULTS_N14[
                    "Task"
                ]
                ==
                task
            )

            &

            (
                CLASS_CONDITIONAL_RESULTS_N14[
                    "Block"
                ]
                ==
                block
            ),

            metric

        ].iloc[0]
    )


incremental_rows_n14 = []


for task in [
    "FN",
    "FP",
]:

    incremental_rows_n14.extend([

        {

            "Task":
                task,

            "Comparison":
                "CONF_LPNS vs CONF",

            "Delta_PR":
                (
                    get_metric_n14(
                        task,
                        "CONF_LPNS",
                        "PR_AUC"
                    )

                    -

                    get_metric_n14(
                        task,
                        "CONF",
                        "PR_AUC"
                    )
                ),

            "Delta_ROC":
                (
                    get_metric_n14(
                        task,
                        "CONF_LPNS",
                        "ROC_AUC"
                    )

                    -

                    get_metric_n14(
                        task,
                        "CONF",
                        "ROC_AUC"
                    )
                ),
        },

        {

            "Task":
                task,

            "Comparison":
                "CONF_DIS vs CONF",

            "Delta_PR":
                (
                    get_metric_n14(
                        task,
                        "CONF_DIS",
                        "PR_AUC"
                    )

                    -

                    get_metric_n14(
                        task,
                        "CONF",
                        "PR_AUC"
                    )
                ),

            "Delta_ROC":
                (
                    get_metric_n14(
                        task,
                        "CONF_DIS",
                        "ROC_AUC"
                    )

                    -

                    get_metric_n14(
                        task,
                        "CONF",
                        "ROC_AUC"
                    )
                ),
        },

        {

            "Task":
                task,

            "Comparison":
                "FULL vs CONF",

            "Delta_PR":
                (
                    get_metric_n14(
                        task,
                        "CONF_DIS_LPNS",
                        "PR_AUC"
                    )

                    -

                    get_metric_n14(
                        task,
                        "CONF",
                        "PR_AUC"
                    )
                ),

            "Delta_ROC":
                (
                    get_metric_n14(
                        task,
                        "CONF_DIS_LPNS",
                        "ROC_AUC"
                    )

                    -

                    get_metric_n14(
                        task,
                        "CONF",
                        "ROC_AUC"
                    )
                ),
        },
    ])


INCREMENTAL_N14 = pd.DataFrame(
    incremental_rows_n14
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — INCREMENTAL FEATURE VALUE"
)

print(
    "=" * 165
)


display(
    INCREMENTAL_N14
)


# ============================================================
# N14.13 FOLD STABILITY
# ============================================================

FOLD_RESULTS_N14 = pd.DataFrame(
    fold_rows_n14
)


FOLD_STABILITY_N14 = (

    FOLD_RESULTS_N14

    .groupby(
        [
            "Task",
            "Block"
        ]
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_SD=(
            "ROC_AUC",
            "std"
        ),

        Prevalence_Mean=(
            "Prevalence",
            "mean"
        ),
    )

    .reset_index()

    .sort_values(
        [
            "Task",
            "PR_Mean"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — FOLD STABILITY"
)

print(
    "=" * 165
)


display(
    FOLD_STABILITY_N14
)


# ============================================================
# N14.14 DUAL SELECTIVE REVIEW
#
# Review budgets are calculated WITHIN predicted class.
#
# FN:
#   review highest FN-risk among predicted negatives
#
# FP:
#   review highest FP-risk among predicted positives
# ============================================================

review_rows_n14 = []


for block in BLOCKS_N14:

    # ========================================================
    # FN SELECTIVE REVIEW
    # ========================================================

    for budget in REVIEW_BUDGETS_N14:

        total_candidates = 0
        total_targets = 0
        reviewed_n = 0
        captured = 0


        for fold in range(
            N_FOLDS
        ):

            fold_idx = np.where(

                DEV_FOLD == fold

            )[0]


            candidate_idx = fold_idx[

                PRED_CLASS_N14[
                    fold_idx
                ]
                ==
                0
            ]


            risk = FN_RISK_N14[
                block
            ][
                candidate_idx
            ]


            target = (

                y_DEV[
                    candidate_idx
                ]
                ==
                1

            ).astype(
                np.int8
            )


            n_review = max(

                1,

                int(
                    np.ceil(

                        budget

                        *
                        len(
                            candidate_idx
                        )
                    )
                )
            )


            ranked = np.argsort(
                -risk
            )


            selected = ranked[
                :n_review
            ]


            total_candidates += len(
                candidate_idx
            )


            total_targets += int(
                target.sum()
            )


            reviewed_n += n_review


            captured += int(

                target[
                    selected
                ].sum()
            )


        review_rows_n14.append({

            "Task":
                "FN",

            "Block":
                block,

            "Review_Budget":
                budget,

            "Candidate_N":
                total_candidates,

            "Target_Errors":
                total_targets,

            "Reviewed_N":
                reviewed_n,

            "Errors_Captured":
                captured,

            "Capture_Rate":
                (
                    captured
                    /
                    total_targets
                ),

            "Errors_per_100_Reviewed":
                (
                    100.0
                    *
                    captured
                    /
                    reviewed_n
                ),

            "Residual_Errors":
                (
                    total_targets
                    -
                    captured
                ),

            "Residual_Error_Rate_within_Class":
                (
                    (
                        total_targets
                        -
                        captured
                    )

                    /
                    total_candidates
                ),
        })


    # ========================================================
    # FP SELECTIVE REVIEW
    # ========================================================

    for budget in REVIEW_BUDGETS_N14:

        total_candidates = 0
        total_targets = 0
        reviewed_n = 0
        captured = 0


        for fold in range(
            N_FOLDS
        ):

            fold_idx = np.where(

                DEV_FOLD == fold

            )[0]


            candidate_idx = fold_idx[

                PRED_CLASS_N14[
                    fold_idx
                ]
                ==
                1
            ]


            risk = FP_RISK_N14[
                block
            ][
                candidate_idx
            ]


            target = (

                y_DEV[
                    candidate_idx
                ]
                ==
                0

            ).astype(
                np.int8
            )


            n_review = max(

                1,

                int(
                    np.ceil(

                        budget

                        *
                        len(
                            candidate_idx
                        )
                    )
                )
            )


            ranked = np.argsort(
                -risk
            )


            selected = ranked[
                :n_review
            ]


            total_candidates += len(
                candidate_idx
            )


            total_targets += int(
                target.sum()
            )


            reviewed_n += n_review


            captured += int(

                target[
                    selected
                ].sum()
            )


        review_rows_n14.append({

            "Task":
                "FP",

            "Block":
                block,

            "Review_Budget":
                budget,

            "Candidate_N":
                total_candidates,

            "Target_Errors":
                total_targets,

            "Reviewed_N":
                reviewed_n,

            "Errors_Captured":
                captured,

            "Capture_Rate":
                (
                    captured
                    /
                    total_targets
                ),

            "Errors_per_100_Reviewed":
                (
                    100.0
                    *
                    captured
                    /
                    reviewed_n
                ),

            "Residual_Errors":
                (
                    total_targets
                    -
                    captured
                ),

            "Residual_Error_Rate_within_Class":
                (
                    (
                        total_targets
                        -
                        captured
                    )

                    /
                    total_candidates
                ),
        })


DUAL_REVIEW_N14 = pd.DataFrame(
    review_rows_n14
)


# ============================================================
# N14.15 FALSE-NEGATIVE REVIEW
# ============================================================

FN_REVIEW_N14 = (

    DUAL_REVIEW_N14[

        DUAL_REVIEW_N14[
            "Task"
        ]
        ==
        "FN"

    ]

    .sort_values(
        [
            "Review_Budget",
            "Capture_Rate"
        ],

        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — PREDICTED-NEGATIVE REVIEW / FN RECOVERY"
)

print(
    "=" * 165
)


display(
    FN_REVIEW_N14
)


# ============================================================
# N14.16 FALSE-POSITIVE REVIEW
# ============================================================

FP_REVIEW_N14 = (

    DUAL_REVIEW_N14[

        DUAL_REVIEW_N14[
            "Task"
        ]
        ==
        "FP"

    ]

    .sort_values(
        [
            "Review_Budget",
            "Capture_Rate"
        ],

        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — PREDICTED-POSITIVE REVIEW / FP RECOVERY"
)

print(
    "=" * 165
)


display(
    FP_REVIEW_N14
)


# ============================================================
# N14.17 LPNS SELECTIVE-REVIEW GAIN
#
# Main comparison:
#     CONF_LPNS vs CONF
# ============================================================

gain_rows_n14 = []


for task in [
    "FN",
    "FP",
]:

    for budget in REVIEW_BUDGETS_N14:

        conf = DUAL_REVIEW_N14[

            (
                DUAL_REVIEW_N14[
                    "Task"
                ]
                ==
                task
            )

            &

            (
                DUAL_REVIEW_N14[
                    "Block"
                ]
                ==
                "CONF"
            )

            &

            (
                DUAL_REVIEW_N14[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]


        lpns = DUAL_REVIEW_N14[

            (
                DUAL_REVIEW_N14[
                    "Task"
                ]
                ==
                task
            )

            &

            (
                DUAL_REVIEW_N14[
                    "Block"
                ]
                ==
                "CONF_LPNS"
            )

            &

            (
                DUAL_REVIEW_N14[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]


        full = DUAL_REVIEW_N14[

            (
                DUAL_REVIEW_N14[
                    "Task"
                ]
                ==
                task
            )

            &

            (
                DUAL_REVIEW_N14[
                    "Block"
                ]
                ==
                "CONF_DIS_LPNS"
            )

            &

            (
                DUAL_REVIEW_N14[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]


        gain_rows_n14.append({

            "Task":
                task,

            "Review_Budget":
                budget,

            "LPNS_minus_CONF_Errors":
                int(

                    lpns[
                        "Errors_Captured"
                    ]

                    -

                    conf[
                        "Errors_Captured"
                    ]
                ),

            "LPNS_minus_CONF_CaptureRate":
                (

                    lpns[
                        "Capture_Rate"
                    ]

                    -

                    conf[
                        "Capture_Rate"
                    ]
                ),

            "LPNS_minus_CONF_ErrorsPer100":
                (

                    lpns[
                        "Errors_per_100_Reviewed"
                    ]

                    -

                    conf[
                        "Errors_per_100_Reviewed"
                    ]
                ),

            "FULL_minus_CONF_Errors":
                int(

                    full[
                        "Errors_Captured"
                    ]

                    -

                    conf[
                        "Errors_Captured"
                    ]
                ),

            "FULL_minus_CONF_CaptureRate":
                (

                    full[
                        "Capture_Rate"
                    ]

                    -

                    conf[
                        "Capture_Rate"
                    ]
                ),
        })


REVIEW_GAIN_N14 = pd.DataFrame(
    gain_rows_n14
)


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — CLASS-CONDITIONAL SELECTIVE-REVIEW GAIN"
)

print(
    "=" * 165
)


display(
    REVIEW_GAIN_N14
)


# ============================================================
# N14.18 THRESHOLD SUMMARY
# ============================================================

THRESHOLD_SUMMARY_N14 = pd.DataFrame({

    "Fold":
        list(
            range(
                N_FOLDS
            )
        ),

    "Threshold":
        [

            float(

                THRESHOLD_N14[
                    DEV_FOLD == fold
                ][0]
            )

            for fold
            in range(
                N_FOLDS
            )
        ]
})


print(
    "\n"
    + "=" * 165
)

print(
    "N14 — OPERATING THRESHOLDS"
)

print(
    "=" * 165
)


display(
    THRESHOLD_SUMMARY_N14
)


# ============================================================
# N14.19 SAVE
# ============================================================

CLASS_CONDITIONAL_RESULTS_N14.to_csv(

    TAB_DIR
    /
    "n14_class_conditional_reliability.csv",

    index=False
)


FN_RESULTS_N14.to_csv(

    TAB_DIR
    /
    "n14_fn_reliability.csv",

    index=False
)


FP_RESULTS_N14.to_csv(

    TAB_DIR
    /
    "n14_fp_reliability.csv",

    index=False
)


INCREMENTAL_N14.to_csv(

    TAB_DIR
    /
    "n14_incremental_features.csv",

    index=False
)


FOLD_RESULTS_N14.to_csv(

    TAB_DIR
    /
    "n14_fold_results.csv",

    index=False
)


FOLD_STABILITY_N14.to_csv(

    TAB_DIR
    /
    "n14_fold_stability.csv",

    index=False
)


FN_REVIEW_N14.to_csv(

    TAB_DIR
    /
    "n14_fn_selective_review.csv",

    index=False
)


FP_REVIEW_N14.to_csv(

    TAB_DIR
    /
    "n14_fp_selective_review.csv",

    index=False
)


REVIEW_GAIN_N14.to_csv(

    TAB_DIR
    /
    "n14_class_conditional_review_gain.csv",

    index=False
)


THRESHOLD_SUMMARY_N14.to_csv(

    TAB_DIR
    /
    "n14_thresholds.csv",

    index=False
)


for block in BLOCKS_N14:

    np.save(

        PRED_DIR
        /
        f"N14_FN_{block}_RISK.npy",

        FN_RISK_N14[
            block
        ]
    )


    np.save(

        PRED_DIR
        /
        f"N14_FP_{block}_RISK.npy",

        FP_RISK_N14[
            block
        ]
    )


np.save(

    PRED_DIR
    /
    "N14_PRED_CLASS.npy",

    PRED_CLASS_N14
)


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N14 COMPLETE"
)

print(
    "=" * 165
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - start_n14:.1f}s"
)


print(
    "\nFinal predictor remained frozen."
)

print(
    "No secondary 2015 holdout used."
)

print(
    "No BRFSS-2020 used."
)


N14 — OUTER FOLD 0
Loading cached strict train experts...
Generating cross-fitted train MoE...
Threshold: 0.231204
Train predicted-negative N: 139,986 | FN prevalence: 0.05283
Train predicted-positive N: 22,369 | FP prevalence: 0.64701
Valid predicted-negative N: 35,323 | FN prevalence: 0.05458
Valid predicted-positive N: 5,266 | FP prevalence: 0.64014
Computing strict train LPNS...
Computing strict valid LPNS...
CONF             | FN_PR=0.173680 | FN_ROC=0.814152 | FP_PR=0.727713 | FP_ROC=0.628140
CONF_DIS         | FN_PR=0.171100 | FN_ROC=0.812895 | FP_PR=0.722153 | FP_ROC=0.618971
CONF_LPNS        | FN_PR=0.171855 | FN_ROC=0.813112 | FP_PR=0.718992 | FP_ROC=0.620825
CONF_DIS_LPNS    | FN_PR=0.170922 | FN_ROC=0.812302 | FP_PR=0.718707 | FP_ROC=0.616751

N14 — OUTER FOLD 1
Loading cached strict train experts...
Generating cross-fitted train MoE...
Threshold: 0.231204
Train predicted-negative N: 139,791 | FN prevalence: 0.05229
Train predicted-positive N: 22,564 | FP prevalence: 0.646

,Task,Block,N,Positive_N,Prevalence,PR_AUC,Brier,NLL,ROC_AUC
0,FN,CONF,172328,8692,0.050439,0.155078,0.044676,0.169301,0.805539
1,FN,CONF_DIS,172328,8692,0.050439,0.154903,0.044727,0.169459,0.804499
2,FN,CONF_LPNS,172328,8692,0.050439,0.154230,0.044740,0.169454,0.804655
3,FN,CONF_DIS_LPNS,172328,8692,0.050439,0.153865,0.044743,0.169514,0.804372
4,FP,CONF,30616,20194,0.659590,0.746658,0.212308,0.614603,0.635290
5,FP,CONF_DIS,30616,20194,0.659590,0.746636,0.212581,0.615228,0.632406
6,FP,CONF_DIS_LPNS,30616,20194,0.659590,0.744810,0.212884,0.615916,0.631162
7,FP,CONF_LPNS,30616,20194,0.659590,0.742518,0.212927,0.616124,0.631218



N14 — FALSE-NEGATIVE RISK RESULTS


,Task,Block,N,Positive_N,Prevalence,PR_AUC,Brier,NLL,ROC_AUC
0,FN,CONF,172328,8692,0.050439,0.155078,0.044676,0.169301,0.805539
1,FN,CONF_DIS,172328,8692,0.050439,0.154903,0.044727,0.169459,0.804499
2,FN,CONF_LPNS,172328,8692,0.050439,0.154230,0.044740,0.169454,0.804655
3,FN,CONF_DIS_LPNS,172328,8692,0.050439,0.153865,0.044743,0.169514,0.804372



N14 — FALSE-POSITIVE RISK RESULTS


...


N14 — CLASS-CONDITIONAL SELECTIVE-REVIEW GAIN


,Task,Review_Budget,LPNS_minus_CONF_Errors,LPNS_minus_CONF_CaptureRate,LPNS_minus_CONF_ErrorsPer100,FULL_minus_CONF_Errors,FULL_minus_CONF_CaptureRate
0,FN,0.010,-2,-0.000230,-0.115875,-8,-0.000920
1,FN,0.025,5,0.000575,0.115982,-28,-0.003221
2,FN,0.050,-20,-0.002301,-0.232045,-24,-0.002761
3,FN,0.100,-99,-0.011390,-0.574413,-73,-0.008399
4,FN,0.150,-16,-0.001841,-0.061891,-22,-0.002531
5,FN,0.200,-21,-0.002416,-0.060926,5,0.000575
6,FP,0.010,-9,-0.000446,-2.912621,0,0.000000
7,FP,0.025,-23,-0.001139,-2.994792,-6,-0.000297
8,FP,0.050,8,0.000396,0.521512,20,0.000990
9,FP,0.100,16,0.000792,0.522193,40,0.001981



N14 — OPERATING THRESHOLDS


,Fold,Threshold
0,0,0.231204
1,1,0.231204
2,2,0.209197
3,3,0.198194
4,4,0.204482



MODULE N14 COMPLETE

Runtime: 279.3s

Final predictor remained frozen.
No secondary 2015 holdout used.
No BRFSS-2020 used.


In [28]:
# ============================================================
# MODULE N15
# FEATURE-AWARE CLASS-CONDITIONAL META-RELIABILITY
#
# Question:
#   Does original patient context provide information about
#   MoE errors beyond prediction confidence?
#
# FINAL PREDICTOR IS FROZEN:
#   N10B Consensus-Regularized MoE
#
# RELIABILITY REPRESENTATIONS:
#
#   CONF
#       C
#
#   X
#       original 21 patient variables
#
#   X_CONF
#       patient variables + confidence
#
#   X_CONF_EXPERTS
#       patient variables + confidence
#       + five expert probabilities
#
# TWO TASKS:
#
#   FN task:
#       among predicted-negative cases,
#       predict whether the case is actually positive
#
#   FP task:
#       among predicted-positive cases,
#       predict whether the case is actually negative
#
# STRICTNESS:
#   - outer-training reliability model only
#   - cross-fitted final MoE probabilities on outer train
#   - strict outer-valid evaluation
#   - no 2015 secondary holdout
#   - no BRFSS-2020
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# N15.1 CONFIG
# ============================================================

EXPERTS_N15 = [

    "XGB_SMOOTH",
    "CAT_SMOOTH",
    "MLP_REPAIRED",
    "AUTOINT_REPAIRED",
    "XGB_HARD_DISAGREE_CF",
]


BLOCKS_N15 = [

    "CONF",
    "X",
    "X_CONF",
    "X_CONF_EXPERTS",
]


REVIEW_BUDGETS_N15 = [

    0.01,
    0.025,
    0.05,
    0.10,
    0.15,
    0.20,
]


# ============================================================
# N15.2 REQUIRED OBJECTS
# ============================================================

required_n15 = [

    "P_MOE_N10B",
    "GATE_EPOCHS_N10B",

    "crossfit_gate_predictions_n12",

    "build_confidence_features_n9",

    "make_reliability_head_n9",

    "select_threshold_n9",

    "OOF_N7B",
    "X_DEV",
    "y_DEV",
    "DEV_FOLD",
    "FEATURES",
    "N_FOLDS",

    "PRED_DIR",
    "TAB_DIR",
    "SEED",
]


for obj in required_n15:

    if obj not in globals():

        raise RuntimeError(
            f"N15 requires '{obj}'."
        )


# ============================================================
# N15.3 FINAL MoE
# ============================================================

P_FINAL_N15 = np.asarray(

    P_MOE_N10B,

    dtype=np.float32
)


if np.isnan(
    P_FINAL_N15
).any():

    raise RuntimeError(
        "Incomplete N10B final MoE probabilities."
    )


# ============================================================
# N15.4 METRICS
# ============================================================

def reliability_metrics_n15(
    y,
    p,
):

    y = np.asarray(
        y,
        dtype=np.int8
    )

    p = np.asarray(
        p,
        dtype=float
    )

    pc = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )


    out = {

        "N":
            len(y),

        "Positive_N":
            int(
                y.sum()
            ),

        "Prevalence":
            float(
                y.mean()
            ),

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                pc,
                labels=[0, 1]
            ),
    }


    if len(
        np.unique(y)
    ) == 2:

        out[
            "ROC_AUC"
        ] = roc_auc_score(
            y,
            p
        )

    else:

        out[
            "ROC_AUC"
        ] = np.nan


    return out


# ============================================================
# N15.5 FEATURE CONSTRUCTION
# ============================================================

def construct_features_n15(

    block,
    X_profile,
    C,
    P_experts,

):

    if block == "CONF":

        return C.astype(
            np.float32
        )


    if block == "X":

        return X_profile.astype(
            np.float32
        )


    if block == "X_CONF":

        return np.concatenate(

            [
                X_profile,
                C,
            ],

            axis=1

        ).astype(
            np.float32
        )


    if block == "X_CONF_EXPERTS":

        return np.concatenate(

            [
                X_profile,
                C,
                P_experts,
            ],

            axis=1

        ).astype(
            np.float32
        )


    raise ValueError(
        f"Unknown block: {block}"
    )


# ============================================================
# N15.6 STORAGE
# ============================================================

FN_RISK_N15 = {

    block:
        np.full(
            len(y_DEV),
            np.nan,
            dtype=np.float32
        )

    for block
    in BLOCKS_N15
}


FP_RISK_N15 = {

    block:
        np.full(
            len(y_DEV),
            np.nan,
            dtype=np.float32
        )

    for block
    in BLOCKS_N15
}


PRED_CLASS_N15 = np.full(

    len(y_DEV),

    -1,

    dtype=np.int8
)


THRESHOLD_N15 = np.full(

    len(y_DEV),

    np.nan,

    dtype=np.float32
)


fold_rows_n15 = []


# ============================================================
# N15.7 OUTER LOOP
# ============================================================

start_n15 = time.perf_counter()


for outer_fold in range(
    N_FOLDS
):

    print(
        "\n"
        + "=" * 165
    )

    print(
        f"N15 — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 165
    )


    train_idx = np.where(

        DEV_FOLD != outer_fold

    )[0]


    valid_idx = np.where(

        DEV_FOLD == outer_fold

    )[0]


    y_train = y_DEV[
        train_idx
    ]


    y_valid = y_DEV[
        valid_idx
    ]


    # ========================================================
    # STEP 1
    # STRICT INNER-OOF EXPERTS
    # ========================================================

    expert_cache = (

        PRED_DIR
        /
        f"N12_outer{outer_fold}_train_experts.npy"
    )


    if not expert_cache.exists():

        raise RuntimeError(
            f"Missing expert cache: {expert_cache}"
        )


    P_train_experts = np.load(

        expert_cache

    ).astype(
        np.float32
    )


    P_valid_experts = np.column_stack([

        OOF_N7B[
            expert
        ][
            valid_idx
        ]

        for expert
        in EXPERTS_N15

    ]).astype(
        np.float32
    )


    # ========================================================
    # STEP 2
    # STRICT CROSS-FITTED FINAL MoE ON OUTER TRAIN
    #
    # Cache this now because later modules may reuse it.
    # ========================================================

    moe_cache = (

        PRED_DIR
        /
        f"N15_outer{outer_fold}_train_moe.npy"
    )


    if moe_cache.exists():

        print(
            "Loading cached train MoE probabilities..."
        )

        p_train_moe = np.load(
            moe_cache
        )


    else:

        selected_epoch = int(

            GATE_EPOCHS_N10B.loc[

                GATE_EPOCHS_N10B[
                    "Fold"
                ]
                ==
                outer_fold,

                "Selected_Epoch"

            ].iloc[0]
        )


        print(
            "Generating strict cross-fitted train MoE..."
        )


        p_train_moe = (

            crossfit_gate_predictions_n12(

                train_idx,

                P_train_experts,

                outer_fold,

                selected_epoch
            )
        )


        np.save(

            moe_cache,

            p_train_moe
        )


    p_valid_moe = (

        P_FINAL_N15[
            valid_idx
        ]
    )


    # ========================================================
    # STEP 3
    # STRICT OPERATING THRESHOLD
    # ========================================================

    threshold, _ = (

        select_threshold_n9(

            y_train,

            p_train_moe
        )
    )


    train_pred = (

        p_train_moe
        >=
        threshold

    ).astype(
        np.int8
    )


    valid_pred = (

        p_valid_moe
        >=
        threshold

    ).astype(
        np.int8
    )


    PRED_CLASS_N15[
        valid_idx
    ] = valid_pred


    THRESHOLD_N15[
        valid_idx
    ] = threshold


    print(
        f"Threshold = "
        f"{threshold:.6f}"
    )


    # ========================================================
    # STEP 4
    # CLASS-CONDITIONAL TARGETS
    # ========================================================

    train_neg = (
        train_pred == 0
    )

    valid_neg = (
        valid_pred == 0
    )


    train_pos = (
        train_pred == 1
    )

    valid_pos = (
        valid_pred == 1
    )


    y_fn_train = (

        y_train[
            train_neg
        ]
        ==
        1

    ).astype(
        np.int8
    )


    y_fn_valid = (

        y_valid[
            valid_neg
        ]
        ==
        1

    ).astype(
        np.int8
    )


    y_fp_train = (

        y_train[
            train_pos
        ]
        ==
        0

    ).astype(
        np.int8
    )


    y_fp_valid = (

        y_valid[
            valid_pos
        ]
        ==
        0

    ).astype(
        np.int8
    )


    print(
        f"Pred-neg valid N={valid_neg.sum():,} | "
        f"FN prevalence={y_fn_valid.mean():.5f}"
    )


    print(
        f"Pred-pos valid N={valid_pos.sum():,} | "
        f"FP prevalence={y_fp_valid.mean():.5f}"
    )


    # ========================================================
    # STEP 5
    # PATIENT PROFILE FEATURES
    #
    # Standardization fit on outer train only.
    # ========================================================

    scaler = StandardScaler()


    scaler.fit(

        X_DEV.iloc[
            train_idx
        ][
            FEATURES
        ]
    )


    X_train_profile = scaler.transform(

        X_DEV.iloc[
            train_idx
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    X_valid_profile = scaler.transform(

        X_DEV.iloc[
            valid_idx
        ][
            FEATURES
        ]

    ).astype(
        np.float32
    )


    # ========================================================
    # STEP 6
    # CONFIDENCE FEATURES
    # ========================================================

    C_train = build_confidence_features_n9(

        p_train_moe,

        threshold
    ).astype(
        np.float32
    )


    C_valid = build_confidence_features_n9(

        p_valid_moe,

        threshold
    ).astype(
        np.float32
    )


    # ========================================================
    # STEP 7
    # TRAIN CLASS-CONDITIONAL HEADS
    # ========================================================

    valid_neg_global = valid_idx[
        valid_neg
    ]


    valid_pos_global = valid_idx[
        valid_pos
    ]


    for block_no, block in enumerate(
        BLOCKS_N15
    ):

        # ----------------------------------------------------
        # construct full feature matrices
        # ----------------------------------------------------

        F_train = construct_features_n15(

            block,

            X_train_profile,

            C_train,

            P_train_experts
        )


        F_valid = construct_features_n15(

            block,

            X_valid_profile,

            C_valid,

            P_valid_experts
        )


        # ====================================================
        # FN HEAD
        # ====================================================

        fn_model = make_reliability_head_n9(

            SEED
            +
            151000
            +
            outer_fold * 100
            +
            block_no
        )


        fn_model.fit(

            F_train[
                train_neg
            ],

            y_fn_train
        )


        fn_risk = (

            fn_model

            .predict_proba(

                F_valid[
                    valid_neg
                ]

            )[:, 1]
        )


        FN_RISK_N15[
            block
        ][
            valid_neg_global
        ] = fn_risk


        fn_metrics = reliability_metrics_n15(

            y_fn_valid,

            fn_risk
        )


        # ====================================================
        # FP HEAD
        # ====================================================

        fp_model = make_reliability_head_n9(

            SEED
            +
            152000
            +
            outer_fold * 100
            +
            block_no
        )


        fp_model.fit(

            F_train[
                train_pos
            ],

            y_fp_train
        )


        fp_risk = (

            fp_model

            .predict_proba(

                F_valid[
                    valid_pos
                ]

            )[:, 1]
        )


        FP_RISK_N15[
            block
        ][
            valid_pos_global
        ] = fp_risk


        fp_metrics = reliability_metrics_n15(

            y_fp_valid,

            fp_risk
        )


        # ====================================================
        # STORE
        # ====================================================

        fold_rows_n15.append({

            "Fold":
                outer_fold,

            "Task":
                "FN",

            "Block":
                block,

            **fn_metrics,
        })


        fold_rows_n15.append({

            "Fold":
                outer_fold,

            "Task":
                "FP",

            "Block":
                block,

            **fp_metrics,
        })


        print(

            f"{block:16s} | "
            f"FN_PR={fn_metrics['PR_AUC']:.6f} | "
            f"FN_ROC={fn_metrics['ROC_AUC']:.6f} | "
            f"FP_PR={fp_metrics['PR_AUC']:.6f} | "
            f"FP_ROC={fp_metrics['ROC_AUC']:.6f}"
        )


        del fn_model
        del fp_model

        del F_train
        del F_valid


    del scaler

    del X_train_profile
    del X_valid_profile

    del C_train
    del C_valid

    del P_train_experts
    del P_valid_experts

    gc.collect()


# ============================================================
# N15.8 GLOBAL CLASS-CONDITIONAL MASKS
# ============================================================

NEG_MASK_N15 = (

    PRED_CLASS_N15 == 0
)


POS_MASK_N15 = (

    PRED_CLASS_N15 == 1
)


if np.any(
    PRED_CLASS_N15 < 0
):

    raise RuntimeError(
        "Incomplete N15 predicted classes."
    )


FN_TARGET_N15 = (

    y_DEV[
        NEG_MASK_N15
    ]
    ==
    1

).astype(
    np.int8
)


FP_TARGET_N15 = (

    y_DEV[
        POS_MASK_N15
    ]
    ==
    0

).astype(
    np.int8
)


# ============================================================
# N15.9 COMPLETENESS CHECK
# ============================================================

for block in BLOCKS_N15:

    if np.isnan(

        FN_RISK_N15[
            block
        ][
            NEG_MASK_N15
        ]

    ).any():

        raise RuntimeError(
            f"Incomplete FN risk: {block}"
        )


    if np.isnan(

        FP_RISK_N15[
            block
        ][
            POS_MASK_N15
        ]

    ).any():

        raise RuntimeError(
            f"Incomplete FP risk: {block}"
        )


# ============================================================
# N15.10 POOLED RESULTS
# ============================================================

pooled_rows_n15 = []


for block in BLOCKS_N15:

    fn_metrics = reliability_metrics_n15(

        FN_TARGET_N15,

        FN_RISK_N15[
            block
        ][
            NEG_MASK_N15
        ]
    )


    pooled_rows_n15.append({

        "Task":
            "FN",

        "Block":
            block,

        **fn_metrics,
    })


    fp_metrics = reliability_metrics_n15(

        FP_TARGET_N15,

        FP_RISK_N15[
            block
        ][
            POS_MASK_N15
        ]
    )


    pooled_rows_n15.append({

        "Task":
            "FP",

        "Block":
            block,

        **fp_metrics,
    })


RESULTS_N15 = pd.DataFrame(
    pooled_rows_n15
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — FEATURE-AWARE CLASS-CONDITIONAL RELIABILITY"
)

print(
    "=" * 165
)


display(

    RESULTS_N15.sort_values(

        [
            "Task",
            "PR_AUC"
        ],

        ascending=[
            True,
            False
        ]
    )
)


# ============================================================
# N15.11 FN RESULTS
# ============================================================

FN_RESULTS_N15 = (

    RESULTS_N15[

        RESULTS_N15[
            "Task"
        ]
        ==
        "FN"

    ]

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — FALSE-NEGATIVE META-RISK"
)

print(
    "=" * 165
)


display(
    FN_RESULTS_N15
)


# ============================================================
# N15.12 FP RESULTS
# ============================================================

FP_RESULTS_N15 = (

    RESULTS_N15[

        RESULTS_N15[
            "Task"
        ]
        ==
        "FP"

    ]

    .sort_values(
        "PR_AUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — FALSE-POSITIVE META-RISK"
)

print(
    "=" * 165
)


display(
    FP_RESULTS_N15
)


# ============================================================
# N15.13 INCREMENTAL VALUE VS CONFIDENCE
# ============================================================

incremental_rows_n15 = []


for task in [
    "FN",
    "FP",
]:

    base = RESULTS_N15[

        (
            RESULTS_N15[
                "Task"
            ]
            ==
            task
        )

        &

        (
            RESULTS_N15[
                "Block"
            ]
            ==
            "CONF"
        )

    ].iloc[0]


    for block in [

        "X",
        "X_CONF",
        "X_CONF_EXPERTS",

    ]:

        row = RESULTS_N15[

            (
                RESULTS_N15[
                    "Task"
                ]
                ==
                task
            )

            &

            (
                RESULTS_N15[
                    "Block"
                ]
                ==
                block
            )

        ].iloc[0]


        incremental_rows_n15.append({

            "Task":
                task,

            "Block":
                block,

            "Delta_PR_vs_CONF":
                (
                    row[
                        "PR_AUC"
                    ]
                    -
                    base[
                        "PR_AUC"
                    ]
                ),

            "Delta_ROC_vs_CONF":
                (
                    row[
                        "ROC_AUC"
                    ]
                    -
                    base[
                        "ROC_AUC"
                    ]
                ),

            "Delta_Brier_vs_CONF":
                (
                    base[
                        "Brier"
                    ]
                    -
                    row[
                        "Brier"
                    ]
                ),

            "Delta_NLL_vs_CONF":
                (
                    base[
                        "NLL"
                    ]
                    -
                    row[
                        "NLL"
                    ]
                ),
        })


INCREMENTAL_N15 = pd.DataFrame(
    incremental_rows_n15
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — INCREMENTAL VALUE VS CONFIDENCE"
)

print(
    "=" * 165
)


display(
    INCREMENTAL_N15
)


# ============================================================
# N15.14 FOLD STABILITY
# ============================================================

FOLD_RESULTS_N15 = pd.DataFrame(
    fold_rows_n15
)


FOLD_STABILITY_N15 = (

    FOLD_RESULTS_N15

    .groupby(
        [
            "Task",
            "Block"
        ]
    )

    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_SD=(
            "PR_AUC",
            "std"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_SD=(
            "ROC_AUC",
            "std"
        ),
    )

    .reset_index()

    .sort_values(
        [
            "Task",
            "PR_Mean"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — FOLD STABILITY"
)

print(
    "=" * 165
)


display(
    FOLD_STABILITY_N15
)


# ============================================================
# N15.15 CLASS-CONDITIONAL SELECTIVE REVIEW
#
# IMPORTANT:
# review budget is WITHIN predicted class.
# ============================================================

review_rows_n15 = []


for task in [
    "FN",
    "FP",
]:

    for block in BLOCKS_N15:

        for budget in REVIEW_BUDGETS_N15:

            total_candidates = 0
            total_errors = 0
            reviewed_n = 0
            captured = 0


            for fold in range(
                N_FOLDS
            ):

                fold_idx = np.where(

                    DEV_FOLD == fold

                )[0]


                if task == "FN":

                    candidate_idx = fold_idx[

                        PRED_CLASS_N15[
                            fold_idx
                        ]
                        ==
                        0
                    ]


                    target = (

                        y_DEV[
                            candidate_idx
                        ]
                        ==
                        1

                    ).astype(
                        np.int8
                    )


                    risk = FN_RISK_N15[
                        block
                    ][
                        candidate_idx
                    ]


                else:

                    candidate_idx = fold_idx[

                        PRED_CLASS_N15[
                            fold_idx
                        ]
                        ==
                        1
                    ]


                    target = (

                        y_DEV[
                            candidate_idx
                        ]
                        ==
                        0

                    ).astype(
                        np.int8
                    )


                    risk = FP_RISK_N15[
                        block
                    ][
                        candidate_idx
                    ]


                n_review = max(

                    1,

                    int(
                        np.ceil(

                            budget
                            *
                            len(
                                candidate_idx
                            )
                        )
                    )
                )


                order = np.argsort(
                    -risk
                )


                chosen = order[
                    :n_review
                ]


                total_candidates += len(
                    candidate_idx
                )


                total_errors += int(
                    target.sum()
                )


                reviewed_n += n_review


                captured += int(

                    target[
                        chosen
                    ].sum()
                )


            review_rows_n15.append({

                "Task":
                    task,

                "Block":
                    block,

                "Review_Budget":
                    budget,

                "Candidate_N":
                    total_candidates,

                "Target_Errors":
                    total_errors,

                "Reviewed_N":
                    reviewed_n,

                "Errors_Captured":
                    captured,

                "Capture_Rate":
                    (
                        captured
                        /
                        total_errors
                    ),

                "Errors_per_100_Reviewed":
                    (
                        100.0
                        *
                        captured
                        /
                        reviewed_n
                    ),

                "Residual_Errors":
                    (
                        total_errors
                        -
                        captured
                    ),

                "Residual_Error_Rate":
                    (
                        total_errors
                        -
                        captured
                    )
                    /
                    total_candidates,
            })


REVIEW_N15 = pd.DataFrame(
    review_rows_n15
)


# ============================================================
# N15.16 FN REVIEW RESULTS
# ============================================================

FN_REVIEW_N15 = (

    REVIEW_N15[

        REVIEW_N15[
            "Task"
        ]
        ==
        "FN"

    ]

    .sort_values(
        [
            "Review_Budget",
            "Capture_Rate"
        ],

        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — PREDICTED-NEGATIVE REVIEW / FN RECOVERY"
)

print(
    "=" * 165
)


display(
    FN_REVIEW_N15
)


# ============================================================
# N15.17 FP REVIEW RESULTS
# ============================================================

FP_REVIEW_N15 = (

    REVIEW_N15[

        REVIEW_N15[
            "Task"
        ]
        ==
        "FP"

    ]

    .sort_values(
        [
            "Review_Budget",
            "Capture_Rate"
        ],

        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — PREDICTED-POSITIVE REVIEW / FP RECOVERY"
)

print(
    "=" * 165
)


display(
    FP_REVIEW_N15
)


# ============================================================
# N15.18 REVIEW GAIN VS CONFIDENCE
# ============================================================

gain_rows_n15 = []


for task in [
    "FN",
    "FP",
]:

    for budget in REVIEW_BUDGETS_N15:

        conf = REVIEW_N15[

            (
                REVIEW_N15[
                    "Task"
                ]
                ==
                task
            )

            &

            (
                REVIEW_N15[
                    "Block"
                ]
                ==
                "CONF"
            )

            &

            (
                REVIEW_N15[
                    "Review_Budget"
                ]
                ==
                budget
            )

        ].iloc[0]


        for block in [

            "X",
            "X_CONF",
            "X_CONF_EXPERTS",

        ]:

            row = REVIEW_N15[

                (
                    REVIEW_N15[
                        "Task"
                    ]
                    ==
                    task
                )

                &

                (
                    REVIEW_N15[
                        "Block"
                    ]
                    ==
                    block
                )

                &

                (
                    REVIEW_N15[
                        "Review_Budget"
                    ]
                    ==
                    budget
                )

            ].iloc[0]


            gain_rows_n15.append({

                "Task":
                    task,

                "Review_Budget":
                    budget,

                "Block":
                    block,

                "Errors_Gained_vs_CONF":
                    int(
                        row[
                            "Errors_Captured"
                        ]
                        -
                        conf[
                            "Errors_Captured"
                        ]
                    ),

                "CaptureRate_Gain_vs_CONF":
                    (
                        row[
                            "Capture_Rate"
                        ]
                        -
                        conf[
                            "Capture_Rate"
                        ]
                    ),

                "ErrorsPer100_Gain_vs_CONF":
                    (
                        row[
                            "Errors_per_100_Reviewed"
                        ]
                        -
                        conf[
                            "Errors_per_100_Reviewed"
                        ]
                    ),
            })


REVIEW_GAIN_N15 = pd.DataFrame(
    gain_rows_n15
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — SELECTIVE-REVIEW GAIN VS CONFIDENCE"
)

print(
    "=" * 165
)


display(
    REVIEW_GAIN_N15
)


# ============================================================
# N15.19 KEY FN SUMMARY
# ============================================================

key_budgets = [
    0.05,
    0.10,
    0.20,
]


KEY_FN_N15 = (

    FN_REVIEW_N15[

        FN_REVIEW_N15[
            "Review_Budget"
        ].isin(
            key_budgets
        )

    ]

    .sort_values(
        [
            "Review_Budget",
            "Capture_Rate"
        ],

        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — KEY FN RECOVERY SUMMARY"
)

print(
    "=" * 165
)


display(
    KEY_FN_N15
)


# ============================================================
# N15.20 THRESHOLDS
# ============================================================

THRESHOLD_SUMMARY_N15 = pd.DataFrame({

    "Fold":
        range(
            N_FOLDS
        ),

    "Threshold":
        [

            float(

                THRESHOLD_N15[

                    DEV_FOLD == fold

                ][0]
            )

            for fold
            in range(
                N_FOLDS
            )
        ]
})


print(
    "\n"
    + "=" * 165
)

print(
    "N15 — OPERATING THRESHOLDS"
)

print(
    "=" * 165
)


display(
    THRESHOLD_SUMMARY_N15
)


# ============================================================
# N15.21 SAVE
# ============================================================

RESULTS_N15.to_csv(

    TAB_DIR
    /
    "n15_feature_aware_reliability.csv",

    index=False
)


FN_RESULTS_N15.to_csv(

    TAB_DIR
    /
    "n15_fn_meta_risk.csv",

    index=False
)


FP_RESULTS_N15.to_csv(

    TAB_DIR
    /
    "n15_fp_meta_risk.csv",

    index=False
)


INCREMENTAL_N15.to_csv(

    TAB_DIR
    /
    "n15_incremental_vs_confidence.csv",

    index=False
)


FOLD_RESULTS_N15.to_csv(

    TAB_DIR
    /
    "n15_fold_results.csv",

    index=False
)


FOLD_STABILITY_N15.to_csv(

    TAB_DIR
    /
    "n15_fold_stability.csv",

    index=False
)


FN_REVIEW_N15.to_csv(

    TAB_DIR
    /
    "n15_fn_selective_review.csv",

    index=False
)


FP_REVIEW_N15.to_csv(

    TAB_DIR
    /
    "n15_fp_selective_review.csv",

    index=False
)


REVIEW_GAIN_N15.to_csv(

    TAB_DIR
    /
    "n15_review_gain_vs_confidence.csv",

    index=False
)


KEY_FN_N15.to_csv(

    TAB_DIR
    /
    "n15_key_fn_recovery.csv",

    index=False
)


THRESHOLD_SUMMARY_N15.to_csv(

    TAB_DIR
    /
    "n15_thresholds.csv",

    index=False
)


for block in BLOCKS_N15:

    np.save(

        PRED_DIR
        /
        f"N15_FN_{block}_RISK.npy",

        FN_RISK_N15[
            block
        ]
    )


    np.save(

        PRED_DIR
        /
        f"N15_FP_{block}_RISK.npy",

        FP_RISK_N15[
            block
        ]
    )


np.save(

    PRED_DIR
    /
    "N15_PRED_CLASS.npy",

    PRED_CLASS_N15
)


print(
    "\n"
    + "=" * 165
)

print(
    "MODULE N15 COMPLETE"
)

print(
    "=" * 165
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - start_n15:.1f}s"
)


print(
    "\nFinal N10B predictor remained frozen."
)

print(
    "No secondary 2015 holdout used."
)

print(
    "No BRFSS-2020 used."
)


N15 — OUTER FOLD 0
Generating strict cross-fitted train MoE...
Threshold = 0.231204
Pred-neg valid N=35,323 | FN prevalence=0.05458
Pred-pos valid N=5,266 | FP prevalence=0.64014
CONF             | FN_PR=0.173680 | FN_ROC=0.814152 | FP_PR=0.727713 | FP_ROC=0.628140
X                | FN_PR=0.170162 | FN_ROC=0.812210 | FP_PR=0.717113 | FP_ROC=0.616320
X_CONF           | FN_PR=0.174331 | FN_ROC=0.815028 | FP_PR=0.720956 | FP_ROC=0.624118
X_CONF_EXPERTS   | FN_PR=0.178726 | FN_ROC=0.814548 | FP_PR=0.722054 | FP_ROC=0.624089

N15 — OUTER FOLD 1
Generating strict cross-fitted train MoE...
Threshold = 0.231204
Pred-neg valid N=34,982 | FN prevalence=0.05271
Pred-pos valid N=5,607 | FP prevalence=0.64705
CONF             | FN_PR=0.164243 | FN_ROC=0.807310 | FP_PR=0.719926 | FP_ROC=0.622534
X                | FN_PR=0.163061 | FN_ROC=0.808091 | FP_PR=0.726478 | FP_ROC=0.622892
X_CONF           | FN_PR=0.161218 | FN_ROC=0.808192 | FP_PR=0.723545 | FP_ROC=0.624425
X_CONF_EXPERTS   | FN_PR=0.1608

,Task,Block,N,Positive_N,Prevalence,PR_AUC,Brier,NLL,ROC_AUC
6,FN,X_CONF_EXPERTS,172328,8692,0.050439,0.155526,0.044704,0.169278,0.805558
0,FN,CONF,172328,8692,0.050439,0.155078,0.044676,0.169301,0.805539
4,FN,X_CONF,172328,8692,0.050439,0.153781,0.044722,0.169353,0.805253
2,FN,X,172328,8692,0.050439,0.152382,0.044765,0.169419,0.804786
7,FP,X_CONF_EXPERTS,30616,20194,0.659590,0.747076,0.212196,0.614411,0.634828
5,FP,X_CONF,30616,20194,0.659590,0.746882,0.212127,0.614239,0.635471
1,FP,CONF,30616,20194,0.659590,0.746658,0.212308,0.614603,0.635290
3,FP,X,30616,20194,0.659590,0.745119,0.212297,0.614621,0.632175



N15 — FALSE-NEGATIVE META-RISK


,Task,Block,N,Positive_N,Prevalence,PR_AUC,Brier,NLL,ROC_AUC
0,FN,X_CONF_EXPERTS,172328,8692,0.050439,0.155526,0.044704,0.169278,0.805558
1,FN,CONF,172328,8692,0.050439,0.155078,0.044676,0.169301,0.805539
2,FN,X_CONF,172328,8692,0.050439,0.153781,0.044722,0.169353,0.805253
3,FN,X,172328,8692,0.050439,0.152382,0.044765,0.169419,0.804786



N15 — FALSE-POSITIVE META-RISK


...


N15 — KEY FN RECOVERY SUMMARY


,Task,Block,Review_Budget,Candidate_N,Target_Errors,Reviewed_N,Errors_Captured,Capture_Rate,Errors_per_100_Reviewed,Residual_Errors,Residual_Error_Rate
0,FN,CONF,0.05,172328,8692,8619,1690,0.194432,19.607843,7002,0.040632
1,FN,X_CONF_EXPERTS,0.05,172328,8692,8619,1666,0.191671,19.329389,7026,0.040771
2,FN,X_CONF,0.05,172328,8692,8619,1642,0.188909,19.050934,7050,0.040910
3,FN,X,0.05,172328,8692,8619,1599,0.183962,18.552036,7093,0.041160
4,FN,CONF,0.10,172328,8692,17235,3103,0.356995,18.004062,5589,0.032432
5,FN,X_CONF,0.10,172328,8692,17235,3055,0.351473,17.725558,5637,0.032711
6,FN,X_CONF_EXPERTS,0.10,172328,8692,17235,3053,0.351243,17.713954,5639,0.032722
7,FN,X,0.10,172328,8692,17235,3014,0.346756,17.487670,5678,0.032949
8,FN,X_CONF,0.20,172328,8692,34468,5190,0.597101,15.057445,3502,0.020322
9,FN,X_CONF_EXPERTS,0.20,172328,8692,34468,5179,0.595835,15.025531,3513,0.020386



N15 — OPERATING THRESHOLDS


,Fold,Threshold
0,0,0.231204
1,1,0.231204
2,2,0.209197
3,3,0.198194
4,4,0.204482



MODULE N15 COMPLETE

Runtime: 101.2s

Final N10B predictor remained frozen.
No secondary 2015 holdout used.
No BRFSS-2020 used.


In [29]:
# ============================================================
# MODULE N16A
# EXACT-PROFILE DEPENDENCE ROBUSTNESS
#
# Final predictor remains frozen:
#   N10B Consensus-Regularized MoE
#
# PURPOSE
# ------------------------------------------------------------
# 1. Audit exact duplicate feature profiles.
# 2. Check conflicting labels among identical profiles.
# 3. Compare:
#       - ordinary row-level evaluation
#       - equal-weight profile evaluation
#       - singleton-only evaluation
#       - duplicated-profile evaluation
# 4. Profile-cluster bootstrap CIs.
#
# No model development.
# No holdout.
# No BRFSS-2020.
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================
# N16A.1 REQUIREMENTS
# ============================================================

required_n16a = [
    "X_DEV",
    "y_DEV",
    "FEATURES",
    "P_MOE_N10B",
    "TAB_DIR",
    "SEED",
]

for obj in required_n16a:
    if obj not in globals():
        raise RuntimeError(
            f"N16A requires '{obj}'."
        )


start_n16a = time.perf_counter()


# ============================================================
# N16A.2 DATA
# ============================================================

X_profile_n16a = (
    X_DEV[FEATURES]
    .reset_index(drop=True)
    .copy()
)

y_n16a = np.asarray(
    y_DEV,
    dtype=np.int8
)

p_n16a = np.asarray(
    P_MOE_N10B,
    dtype=float
)

if len(X_profile_n16a) != len(y_n16a):
    raise RuntimeError(
        "Length mismatch."
    )

if np.isnan(p_n16a).any():
    raise RuntimeError(
        "Incomplete final MoE predictions."
    )


# ============================================================
# N16A.3 EXACT PROFILE ID
#
# pandas hashing avoids constructing huge strings.
# ============================================================

PROFILE_HASH_N16A = pd.util.hash_pandas_object(
    X_profile_n16a,
    index=False
).to_numpy(
    dtype=np.uint64
)


profile_frame_n16a = pd.DataFrame({
    "PROFILE_ID":
        PROFILE_HASH_N16A,
    "y":
        y_n16a,
    "p":
        p_n16a,
})


# ============================================================
# N16A.4 PROFILE SUMMARY
# ============================================================

PROFILE_SUMMARY_N16A = (
    profile_frame_n16a
    .groupby(
        "PROFILE_ID",
        sort=False
    )
    .agg(
        N=("y", "size"),
        Positive_N=("y", "sum"),
        Mean_Target=("y", "mean"),
        Mean_Probability=("p", "mean"),
        Min_Probability=("p", "min"),
        Max_Probability=("p", "max"),
    )
    .reset_index()
)


PROFILE_SUMMARY_N16A[
    "Negative_N"
] = (
    PROFILE_SUMMARY_N16A["N"]
    -
    PROFILE_SUMMARY_N16A["Positive_N"]
)


PROFILE_SUMMARY_N16A[
    "Conflicting_Label"
] = (
    (PROFILE_SUMMARY_N16A["Positive_N"] > 0)
    &
    (PROFILE_SUMMARY_N16A["Negative_N"] > 0)
)


PROFILE_SUMMARY_N16A[
    "Duplicate_Profile"
] = (
    PROFILE_SUMMARY_N16A["N"] > 1
)


profile_size_map_n16a = (
    PROFILE_SUMMARY_N16A
    .set_index("PROFILE_ID")["N"]
)


PROFILE_SIZE_PER_ROW_N16A = pd.Series(
    PROFILE_HASH_N16A
).map(
    profile_size_map_n16a
).to_numpy()


DUP_ROW_MASK_N16A = (
    PROFILE_SIZE_PER_ROW_N16A > 1
)

SINGLETON_MASK_N16A = (
    PROFILE_SIZE_PER_ROW_N16A == 1
)


# ============================================================
# N16A.5 AUDIT
# ============================================================

n_rows = len(y_n16a)

n_profiles = len(
    PROFILE_SUMMARY_N16A
)

n_duplicate_profiles = int(
    PROFILE_SUMMARY_N16A[
        "Duplicate_Profile"
    ].sum()
)

n_conflicting_profiles = int(
    PROFILE_SUMMARY_N16A[
        "Conflicting_Label"
    ].sum()
)

n_rows_in_duplicate_profiles = int(
    DUP_ROW_MASK_N16A.sum()
)

n_rows_in_conflicting_profiles = int(
    PROFILE_SUMMARY_N16A.loc[
        PROFILE_SUMMARY_N16A[
            "Conflicting_Label"
        ],
        "N"
    ].sum()
)


AUDIT_N16A = pd.DataFrame([{
    "Rows":
        n_rows,

    "Unique_Profiles":
        n_profiles,

    "Duplicate_Profiles":
        n_duplicate_profiles,

    "Rows_in_Duplicate_Profiles":
        n_rows_in_duplicate_profiles,

    "Duplicate_Row_Fraction":
        n_rows_in_duplicate_profiles
        /
        n_rows,

    "Conflicting_Label_Profiles":
        n_conflicting_profiles,

    "Conflicting_Profile_Fraction":
        n_conflicting_profiles
        /
        n_profiles,

    "Rows_in_Conflicting_Profiles":
        n_rows_in_conflicting_profiles,

    "Max_Profile_Size":
        int(
            PROFILE_SUMMARY_N16A[
                "N"
            ].max()
        ),

    "Median_Profile_Size":
        float(
            PROFILE_SUMMARY_N16A[
                "N"
            ].median()
        ),
}])


print(
    "=" * 160
)

print(
    "N16A — EXACT PROFILE AUDIT"
)

print(
    "=" * 160
)

display(
    AUDIT_N16A
)


# ============================================================
# N16A.6 METRICS
# ============================================================

def metrics_n16a(
    y,
    p,
):

    y = np.asarray(y)
    p = np.asarray(p)

    pc = np.clip(
        p,
        1e-7,
        1 - 1e-7
    )

    result = {
        "N":
            len(y),

        "Prevalence":
            float(
                np.mean(y)
            ),

        "PR_AUC":
            average_precision_score(
                y,
                p
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                p
            ),

        "Brier":
            brier_score_loss(
                y,
                p
            ),

        "NLL":
            log_loss(
                y,
                pc
            ),
    }

    return result


# ============================================================
# N16A.7 ROW-LEVEL METRICS
# ============================================================

m_all = metrics_n16a(
    y_n16a,
    p_n16a
)

m_singletons = metrics_n16a(
    y_n16a[
        SINGLETON_MASK_N16A
    ],
    p_n16a[
        SINGLETON_MASK_N16A
    ]
)

m_duplicates = metrics_n16a(
    y_n16a[
        DUP_ROW_MASK_N16A
    ],
    p_n16a[
        DUP_ROW_MASK_N16A
    ]
)


# ============================================================
# N16A.8 ONE PROFILE = ONE OBSERVATION
#
# Mean target can be fractional when an identical feature
# profile contains conflicting labels.
#
# AP/ROC require binary labels, so for ranking metrics we use
# only label-consistent profiles.
#
# Brier/NLL can additionally be calculated against empirical
# profile event rate.
# ============================================================

consistent_profiles_n16a = (
    ~PROFILE_SUMMARY_N16A[
        "Conflicting_Label"
    ]
)


profile_y_binary_n16a = (
    PROFILE_SUMMARY_N16A.loc[
        consistent_profiles_n16a,
        "Positive_N"
    ]
    > 0
).astype(
    np.int8
).to_numpy()


profile_p_binary_n16a = (
    PROFILE_SUMMARY_N16A.loc[
        consistent_profiles_n16a,
        "Mean_Probability"
    ]
    .to_numpy()
)


m_unique_consistent = metrics_n16a(
    profile_y_binary_n16a,
    profile_p_binary_n16a
)


# ------------------------------------------------------------
# Profile-level probability quality against empirical event rate
# ------------------------------------------------------------

profile_rate_n16a = (
    PROFILE_SUMMARY_N16A[
        "Mean_Target"
    ].to_numpy(
        dtype=float
    )
)

profile_prob_n16a = (
    PROFILE_SUMMARY_N16A[
        "Mean_Probability"
    ].to_numpy(
        dtype=float
    )
)


profile_brier_empirical = np.mean(
    (
        profile_prob_n16a
        -
        profile_rate_n16a
    ) ** 2
)


eps_n16a = 1e-7

profile_prob_clip_n16a = np.clip(
    profile_prob_n16a,
    eps_n16a,
    1 - eps_n16a
)


profile_nll_empirical = -np.mean(
    profile_rate_n16a
    *
    np.log(
        profile_prob_clip_n16a
    )
    +
    (
        1
        -
        profile_rate_n16a
    )
    *
    np.log(
        1
        -
        profile_prob_clip_n16a
    )
)


# ============================================================
# N16A.9 COMPARISON TABLE
# ============================================================

COMPARISON_N16A = pd.DataFrame([

    {
        "Evaluation":
            "All rows",

        **m_all,
    },

    {
        "Evaluation":
            "Singleton rows only",

        **m_singletons,
    },

    {
        "Evaluation":
            "Rows belonging to duplicate profiles",

        **m_duplicates,
    },

    {
        "Evaluation":
            "Unique label-consistent profiles",

        **m_unique_consistent,
    },

])


print(
    "\n"
    + "=" * 160
)

print(
    "N16A — DUPLICATE-PROFILE PERFORMANCE"
)

print(
    "=" * 160
)

display(
    COMPARISON_N16A
)


PROFILE_PROBABILITY_QUALITY_N16A = (
    pd.DataFrame([{
        "Evaluation":
            "Equal-weight all unique profiles",

        "N_Profiles":
            len(
                PROFILE_SUMMARY_N16A
            ),

        "Empirical_Profile_Brier":
            profile_brier_empirical,

        "Empirical_Profile_NLL":
            profile_nll_empirical,
    }])
)


print(
    "\n"
    + "=" * 160
)

print(
    "N16A — EQUAL-WEIGHT PROFILE PROBABILITY QUALITY"
)

print(
    "=" * 160
)

display(
    PROFILE_PROBABILITY_QUALITY_N16A
)


# ============================================================
# N16A.10 LABEL-CONFLICT CHARACTERIZATION
# ============================================================

CONFLICTING_PROFILES_N16A = (
    PROFILE_SUMMARY_N16A[
        PROFILE_SUMMARY_N16A[
            "Conflicting_Label"
        ]
    ]
    .copy()
)


if len(
    CONFLICTING_PROFILES_N16A
) > 0:

    CONFLICT_SUMMARY_N16A = pd.DataFrame([{

        "Conflicting_Profiles":
            len(
                CONFLICTING_PROFILES_N16A
            ),

        "Rows":
            int(
                CONFLICTING_PROFILES_N16A[
                    "N"
                ].sum()
            ),

        "Mean_Profile_Size":
            float(
                CONFLICTING_PROFILES_N16A[
                    "N"
                ].mean()
            ),

        "Mean_Event_Rate":
            float(
                CONFLICTING_PROFILES_N16A[
                    "Mean_Target"
                ].mean()
            ),

        "Mean_Predicted_Probability":
            float(
                CONFLICTING_PROFILES_N16A[
                    "Mean_Probability"
                ].mean()
            ),
    }])

else:

    CONFLICT_SUMMARY_N16A = pd.DataFrame([{

        "Conflicting_Profiles":
            0,

        "Rows":
            0,

        "Mean_Profile_Size":
            np.nan,

        "Mean_Event_Rate":
            np.nan,

        "Mean_Predicted_Probability":
            np.nan,
    }])


print(
    "\n"
    + "=" * 160
)

print(
    "N16A — CONFLICTING IDENTICAL PROFILES"
)

print(
    "=" * 160
)

display(
    CONFLICT_SUMMARY_N16A
)


# ============================================================
# N16A.11 PROFILE-CLUSTER BOOTSTRAP
#
# Sampling unit = unique feature profile.
#
# This addresses dependence among repeated rows.
# ============================================================

N_BOOT_N16A = 1000

rng_n16a = np.random.default_rng(
    SEED + 160000
)


profile_groups_n16a = (
    profile_frame_n16a
    .groupby(
        "PROFILE_ID",
        sort=False
    )
    .indices
)


profile_ids_n16a = np.array(
    list(
        profile_groups_n16a.keys()
    ),
    dtype=np.uint64
)


group_indices_n16a = [
    np.asarray(
        profile_groups_n16a[pid],
        dtype=np.int64
    )
    for pid
    in profile_ids_n16a
]


boot_rows_n16a = []


for b in range(
    N_BOOT_N16A
):

    sampled_profile_positions = rng_n16a.integers(

        0,

        len(
            profile_ids_n16a
        ),

        size=len(
            profile_ids_n16a
        )
    )


    sampled_rows = np.concatenate([

        group_indices_n16a[
            pos
        ]

        for pos
        in sampled_profile_positions

    ])


    yb = y_n16a[
        sampled_rows
    ]

    pb = p_n16a[
        sampled_rows
    ]


    # extraordinarily unlikely, but guard
    if len(
        np.unique(yb)
    ) < 2:
        continue


    boot_rows_n16a.append({

        "PR_AUC":
            average_precision_score(
                yb,
                pb
            ),

        "ROC_AUC":
            roc_auc_score(
                yb,
                pb
            ),

        "Brier":
            brier_score_loss(
                yb,
                pb
            ),

        "NLL":
            log_loss(
                yb,
                np.clip(
                    pb,
                    1e-7,
                    1 - 1e-7
                )
            ),
    })


BOOT_N16A = pd.DataFrame(
    boot_rows_n16a
)


ci_rows_n16a = []


for metric in [
    "PR_AUC",
    "ROC_AUC",
    "Brier",
    "NLL",
]:

    values = BOOT_N16A[
        metric
    ].to_numpy()


    ci_rows_n16a.append({

        "Metric":
            metric,

        "Point_Estimate":
            m_all[
                metric
            ],

        "ClusterBoot_Mean":
            values.mean(),

        "CI_Low":
            np.quantile(
                values,
                0.025
            ),

        "CI_High":
            np.quantile(
                values,
                0.975
            ),

        "Bootstrap_N":
            len(values),
    })


CLUSTER_BOOTSTRAP_N16A = pd.DataFrame(
    ci_rows_n16a
)


print(
    "\n"
    + "=" * 160
)

print(
    "N16A — PROFILE-CLUSTER BOOTSTRAP"
)

print(
    "=" * 160
)

display(
    CLUSTER_BOOTSTRAP_N16A
)


# ============================================================
# N16A.12 METRIC DELTAS
# ============================================================

DELTA_N16A = pd.DataFrame([

    {
        "Comparison":
            "Singleton minus All",

        "Delta_PR":
            m_singletons["PR_AUC"]
            -
            m_all["PR_AUC"],

        "Delta_ROC":
            m_singletons["ROC_AUC"]
            -
            m_all["ROC_AUC"],

        "Delta_Brier":
            m_singletons["Brier"]
            -
            m_all["Brier"],

        "Delta_NLL":
            m_singletons["NLL"]
            -
            m_all["NLL"],
    },

    {
        "Comparison":
            "Unique-consistent profiles minus All",

        "Delta_PR":
            m_unique_consistent["PR_AUC"]
            -
            m_all["PR_AUC"],

        "Delta_ROC":
            m_unique_consistent["ROC_AUC"]
            -
            m_all["ROC_AUC"],

        "Delta_Brier":
            m_unique_consistent["Brier"]
            -
            m_all["Brier"],

        "Delta_NLL":
            m_unique_consistent["NLL"]
            -
            m_all["NLL"],
    },
])


print(
    "\n"
    + "=" * 160
)

print(
    "N16A — SENSITIVITY DELTAS"
)

print(
    "=" * 160
)

display(
    DELTA_N16A
)


# ============================================================
# N16A.13 PROFILE-SIZE DISTRIBUTION
# ============================================================

bins_n16a = [
    1,
    2,
    3,
    4,
    5,
    10,
    20,
    50,
    100,
    np.inf,
]

labels_n16a = [
    "1",
    "2",
    "3",
    "4",
    "5-9",
    "10-19",
    "20-49",
    "50-99",
    "100+",
]


PROFILE_SUMMARY_N16A[
    "Size_Bin"
] = pd.cut(
    PROFILE_SUMMARY_N16A[
        "N"
    ],
    bins=bins_n16a,
    labels=labels_n16a,
    right=False
)


PROFILE_SIZE_DISTRIBUTION_N16A = (
    PROFILE_SUMMARY_N16A
    .groupby(
        "Size_Bin",
        observed=False
    )
    .agg(
        Profiles=(
            "PROFILE_ID",
            "size"
        ),

        Rows=(
            "N",
            "sum"
        ),

        Conflicting=(
            "Conflicting_Label",
            "sum"
        ),
    )
    .reset_index()
)


print(
    "\n"
    + "=" * 160
)

print(
    "N16A — PROFILE SIZE DISTRIBUTION"
)

print(
    "=" * 160
)

display(
    PROFILE_SIZE_DISTRIBUTION_N16A
)


# ============================================================
# N16A.14 SAVE
# ============================================================

AUDIT_N16A.to_csv(
    TAB_DIR
    /
    "n16a_profile_audit.csv",
    index=False
)


COMPARISON_N16A.to_csv(
    TAB_DIR
    /
    "n16a_profile_performance.csv",
    index=False
)


PROFILE_PROBABILITY_QUALITY_N16A.to_csv(
    TAB_DIR
    /
    "n16a_profile_probability_quality.csv",
    index=False
)


CONFLICT_SUMMARY_N16A.to_csv(
    TAB_DIR
    /
    "n16a_conflicting_profiles.csv",
    index=False
)


CLUSTER_BOOTSTRAP_N16A.to_csv(
    TAB_DIR
    /
    "n16a_profile_cluster_bootstrap.csv",
    index=False
)


DELTA_N16A.to_csv(
    TAB_DIR
    /
    "n16a_sensitivity_deltas.csv",
    index=False
)


PROFILE_SIZE_DISTRIBUTION_N16A.to_csv(
    TAB_DIR
    /
    "n16a_profile_size_distribution.csv",
    index=False
)


print(
    "\n"
    + "=" * 160
)

print(
    "MODULE N16A COMPLETE"
)

print(
    "=" * 160
)


print(
    f"\nRuntime: "
    f"{time.perf_counter() - start_n16a:.1f}s"
)


print(
    "\nFrozen final predictor only."
)

print(
    "No model selection performed."
)

print(
    "No secondary 2015 holdout used."
)

print(
    "No BRFSS-2020 used."
)

N16A — EXACT PROFILE AUDIT


,Rows,Unique_Profiles,Duplicate_Profiles,Rows_in_Duplicate_Profiles,Duplicate_Row_Fraction,Conflicting_Label_Profiles,Conflicting_Profile_Fraction,Rows_in_Conflicting_Profiles,Max_Profile_Size,Median_Profile_Size
0,202944,185039,8753,26658,0.131356,732,0.003956,2403,50,1.0



N16A — DUPLICATE-PROFILE PERFORMANCE


,Evaluation,N,Prevalence,PR_AUC,ROC_AUC,Brier,NLL
0,All rows,202944,0.094184,0.379643,0.850618,0.069729,0.235647
1,Singleton rows only,176286,0.103383,0.385989,0.841765,0.075980,0.254574
2,Rows belonging to duplicate profiles,26658,0.033348,0.228111,0.873996,0.028394,0.110488
3,Unique label-consistent profiles,184307,0.099117,0.384788,0.846878,0.072972,0.245361



N16A — EQUAL-WEIGHT PROFILE PROBABILITY QUALITY


...


N16A — SENSITIVITY DELTAS


,Comparison,Delta_PR,Delta_ROC,Delta_Brier,Delta_NLL
0,Singleton minus All,0.006346,-0.008853,0.006251,0.018927
1,Unique-consistent profiles minus All,0.005146,-0.003740,0.003243,0.009714



N16A — PROFILE SIZE DISTRIBUTION


,Size_Bin,Profiles,Rows,Conflicting
0,1,176286,176286,0
1,2,5691,11382,403
2,3,1506,4518,149
3,4,599,2396,74
4,5-9,716,4428,85
5,10-19,187,2435,17
6,20-49,53,1449,4
7,50-99,1,50,0
8,100+,0,0,0



MODULE N16A COMPLETE

Runtime: 241.6s

Frozen final predictor only.
No model selection performed.
No secondary 2015 holdout used.
No BRFSS-2020 used.


In [30]:
# ============================================================
# MODULE N16B-1
# EXACT-PROFILE STRATIFIED GROUPED FOLDS
#
# Purpose:
#   Ensure identical 21-feature profiles never appear
#   in both train and validation folds.
#
# Architecture/hyperparameters remain frozen.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold


# ============================================================
# REQUIREMENTS
# ============================================================

required_n16b = [
    "X_DEV",
    "y_DEV",
    "FEATURES",
    "SEED",
    "TAB_DIR",
]

for obj in required_n16b:
    if obj not in globals():
        raise RuntimeError(
            f"N16B requires '{obj}'."
        )


# ============================================================
# PROFILE GROUPS
# ============================================================

X_group_n16b = (
    X_DEV[FEATURES]
    .reset_index(drop=True)
)

y_group_n16b = np.asarray(
    y_DEV,
    dtype=np.int8
)


GROUP_ID_N16B = pd.util.hash_pandas_object(
    X_group_n16b,
    index=False
).to_numpy(
    dtype=np.uint64
)


# ============================================================
# STRATIFIED GROUP 5-FOLD
# ============================================================

sgkf_n16b = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED + 160100
)


GROUP_FOLD_N16B = np.full(
    len(y_group_n16b),
    -1,
    dtype=np.int8
)


fold_rows_n16b = []


for fold, (train_idx, valid_idx) in enumerate(
    sgkf_n16b.split(
        X_group_n16b,
        y_group_n16b,
        groups=GROUP_ID_N16B
    )
):

    GROUP_FOLD_N16B[
        valid_idx
    ] = fold


    train_groups = set(
        GROUP_ID_N16B[
            train_idx
        ].tolist()
    )


    valid_groups = set(
        GROUP_ID_N16B[
            valid_idx
        ].tolist()
    )


    overlap = (
        train_groups
        &
        valid_groups
    )


    fold_rows_n16b.append({

        "Fold":
            fold,

        "Train_N":
            len(train_idx),

        "Valid_N":
            len(valid_idx),

        "Train_Pos":
            int(
                y_group_n16b[
                    train_idx
                ].sum()
            ),

        "Valid_Pos":
            int(
                y_group_n16b[
                    valid_idx
                ].sum()
            ),

        "Train_Prevalence":
            float(
                y_group_n16b[
                    train_idx
                ].mean()
            ),

        "Valid_Prevalence":
            float(
                y_group_n16b[
                    valid_idx
                ].mean()
            ),

        "Train_Profiles":
            len(
                train_groups
            ),

        "Valid_Profiles":
            len(
                valid_groups
            ),

        "Profile_Overlap":
            len(
                overlap
            ),
    })


GROUP_FOLD_SUMMARY_N16B = pd.DataFrame(
    fold_rows_n16b
)


print(
    "=" * 150
)

print(
    "N16B — EXACT-PROFILE GROUPED FOLDS"
)

print(
    "=" * 150
)

display(
    GROUP_FOLD_SUMMARY_N16B
)


# ============================================================
# SANITY CHECKS
# ============================================================

if np.any(
    GROUP_FOLD_N16B < 0
):
    raise RuntimeError(
        "Some rows did not receive a grouped fold."
    )


if (
    GROUP_FOLD_SUMMARY_N16B[
        "Profile_Overlap"
    ]
    >
    0
).any():

    raise RuntimeError(
        "Profile leakage detected."
    )


# same profile must have exactly one fold
check_n16b = pd.DataFrame({
    "group":
        GROUP_ID_N16B,
    "fold":
        GROUP_FOLD_N16B,
})


max_fold_count_n16b = (
    check_n16b
    .groupby("group")["fold"]
    .nunique()
    .max()
)


print(
    "\nMaximum number of folds occupied "
    "by one exact profile:",
    max_fold_count_n16b
)


print(
    "\nFold prevalence range:",
    GROUP_FOLD_SUMMARY_N16B[
        "Valid_Prevalence"
    ].min(),
    "to",
    GROUP_FOLD_SUMMARY_N16B[
        "Valid_Prevalence"
    ].max()
)


# ============================================================
# SAVE
# ============================================================

np.save(
    PRED_DIR
    /
    "N16B_GROUP_FOLD.npy",
    GROUP_FOLD_N16B
)


GROUP_FOLD_SUMMARY_N16B.to_csv(
    TAB_DIR
    /
    "n16b_group_fold_summary.csv",
    index=False
)


print(
    "\nN16B grouped-fold construction complete."
)

N16B — EXACT-PROFILE GROUPED FOLDS


,Fold,Train_N,Valid_N,Train_Pos,Valid_Pos,Train_Prevalence,Valid_Prevalence,Train_Profiles,Valid_Profiles,Profile_Overlap
0,0,162325,40619,15328,3786,0.094428,0.093208,148020,37019,0
1,1,162262,40682,15293,3821,0.094249,0.093924,148020,37019,0
2,2,162377,40567,15261,3853,0.093985,0.094979,148049,36990,0
3,3,162496,40448,15253,3861,0.093867,0.095456,148034,37005,0
4,4,162316,40628,15321,3793,0.094390,0.093359,148033,37006,0



Maximum number of folds occupied by one exact profile: 1

Fold prevalence range: 0.09320761220118663 to 0.09545589398734178

N16B grouped-fold construction complete.
